In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

x = tf.keras.Input(shape=(1200, 1))

layers = [
    (256, 36, 134),
    (128, 25, 17),
    (128, 25, 4),
    (64, 16, 1),
]

for i, (filters, kernel, expected) in enumerate(layers, 1):
    matches = []

    for padding in ["valid", "same"]:
        for stride in range(1, 21):
            try:
                layer = tf.keras.layers.Conv1D(
                    filters=filters,
                    kernel_size=kernel,
                    strides=stride,
                    padding=padding
                )
                output = layer(x)
                if output.shape[1] == expected:
                    matches.append((padding, stride))
            except ValueError:
                pass

    print(f"Layer {i}: expected length {expected}")
    print("Possible settings:", matches)

    x = tf.keras.Input(shape=(expected, filters))

In [ ]:
import tensorflow as tf

# Remaining Conv1D layers reported in Table 2
kernel_sizes = [9, 4, 2, 4, 4, 2, 1]

x = tf.keras.Input(shape=(1, 64))

for i, kernel in enumerate(kernel_sizes, start=5):
    layer = tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=kernel,
        strides=1,
        padding="same",
        activation=tf.nn.leaky_relu
    )

    x = layer(x)

    print(
        f"Conv1D {i}:",
        f"kernel={kernel},",
        f"output={x.shape},",
        f"parameters={layer.count_params():,}"
    )

In [ ]:
import tensorflow as tf

# Decoder specifications from manuscript Table 2
specs = [
    (64, 4, 3),
    (64, 4, 6),
    (64, 4, 19),
    (64, 4, 22),
    (64, 5, 47),
    (64, 4, 47),
    (64, 4, 96),
    (64, 4, 194),
    (64, 3, 197),
    (64, 2, 199),
    (64, 1, 200),
    (1, 1, 200),
]

current_length = 1
current_channels = 64

for i, (filters, kernel, expected) in enumerate(specs, 1):
    matches = []

    for padding in ["valid", "same"]:
        for stride in range(1, 21):
            try:
                x = tf.keras.Input(
                    shape=(current_length, current_channels)
                )

                layer = tf.keras.layers.Conv1DTranspose(
                    filters=filters,
                    kernel_size=kernel,
                    strides=stride,
                    padding=padding
                )

                output = layer(x)

                if output.shape[1] == expected:
                    matches.append((padding, stride))

            except (ValueError, TypeError):
                pass

    print(
        f"Decoder layer {i}: "
        f"{current_length} -> {expected}, "
        f"kernel={kernel}"
    )
    print("Possible settings:", matches)

    current_length = expected
    current_channels = filters

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

transitions = [
    (1, 3, 4),
    (3, 6, 4),
    (6, 19, 4),
    (19, 22, 4),
    (22, 47, 5),
    (47, 47, 4),
    (47, 96, 4),
    (96, 194, 4),
    (194, 197, 3),
    (197, 199, 2),
    (199, 200, 1),
]

for input_length, expected_length, kernel in transitions:
    matches = []

    for padding in ["same", "valid"]:
        for stride in range(1, 21):
            try:
                model = tf.keras.Sequential([
                    layers.Input(shape=(input_length, 64)),
                    layers.Conv1DTranspose(
                        filters=64,
                        kernel_size=kernel,
                        strides=stride,
                        padding=padding
                    )
                ])

                actual_length = model.output_shape[1]

                if actual_length == expected_length:
                    matches.append((stride, padding))

            except Exception:
                pass

    print(
        f"{input_length:>3} -> {expected_length:<3} "
        f"| kernel={kernel} | matches={matches}"
    )

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

transitions = [
    (194, 197, 3),
    (197, 199, 2),
    (199, 200, 1),
]

for input_length, target_length, kernel in transitions:
    print(f"\nTransition {input_length} -> {target_length}, kernel={kernel}")

    solutions = []

    for padding in ["same", "valid"]:
        for stride in range(1, 6):
            for output_padding in [None] + list(range(stride)):
                try:
                    model = tf.keras.Sequential([
                        layers.Input(shape=(input_length, 64)),
                        layers.Conv1DTranspose(
                            64,
                            kernel_size=kernel,
                            strides=stride,
                            padding=padding,
                            output_padding=output_padding
                        )
                    ])

                    actual = model.output_shape[1]

                    if actual == target_length:
                        solutions.append({
                            "stride": stride,
                            "padding": padding,
                            "output_padding": output_padding
                        })

                except (ValueError, TypeError):
                    pass

    print("Solutions:", solutions if solutions else "None")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

transitions = [
    (194, 197, 3),
    (197, 199, 2),
    (199, 200, 1)
]

for input_length, target_length, kernel in transitions:
    model = tf.keras.Sequential([
        layers.Input(shape=(input_length, 64)),
        layers.Conv1DTranspose(
            filters=64,
            kernel_size=kernel,
            strides=1,
            padding="same"
        ),
        layers.ZeroPadding1D(
            padding=(0, target_length - input_length)
        )
    ])

    print(
        f"{input_length} -> {target_length}: "
        f"Actual output = {model.output_shape}, "
        f"Parameters = {model.count_params():,}"
    )

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

latent_dim = 20

inputs = layers.Input(shape=(64,), name="flattened_features")

# Mean of the latent distribution
z_mean = layers.Dense(
    latent_dim, name="z_mean"
)(inputs)

# Log variance of the latent distribution
z_log_var = layers.Dense(
    latent_dim, name="z_log_var"
)(inputs)

encoder_heads = Model(
    inputs,
    [z_mean, z_log_var],
    name="vae_latent_heads"
)

encoder_heads.summary()

print("\nReported final Dense parameters:", 1300)
print("Two Gaussian latent heads:", encoder_heads.count_params())
print("Additional parameters:",
      encoder_heads.count_params() - 1300)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

latent_dim = 20

inputs = layers.Input(shape=(64,), name="flattened_features")

# Single trainable latent mean layer
z_mean = layers.Dense(
    latent_dim,
    name="z_mean"
)(inputs)

# Fixed unit variance (no additional trainable parameters)
z_log_var = layers.Lambda(
    lambda x: tf.zeros_like(x),
    name="fixed_log_variance"
)(z_mean)

encoder = Model(
    inputs,
    [z_mean, z_log_var],
    name="single_head_encoder"
)

encoder.summary()

print("\nTrainable parameters:", encoder.count_params())
print("Expected parameters:", 1300)

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Illustrative numerical sequence, not original experimental data
sequence = np.array([
    [0.2],
    [0.5],
    [1.3],
    [2.1],
    [3.4]
])

scaler = MinMaxScaler()

scaled = scaler.fit_transform(sequence)
recovered = scaler.inverse_transform(scaled)

print("Original shape:", sequence.shape)
print("Scaled shape:", scaled.shape)
print("Recovered shape:", recovered.shape)

print("\nOriginal values:")
print(sequence.flatten())

print("\nRecovered values:")
print(recovered.flatten())

print("\nNumerical recovery:",
      np.allclose(sequence, recovered))

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Illustrative vocabulary, not the original study vocabulary
index_to_word = {
    1: "aircraft",
    2: "engine",
    3: "failure",
    4: "landing",
    5: "pilot"
}

token_ids = np.array([[2], [3], [4], [1]], dtype=float)

scaler = MinMaxScaler()
scaled_ids = scaler.fit_transform(token_ids)

# Simulated decoder predictions with small numerical errors
predicted_scaled = scaled_ids + np.array([
    [0.01], [-0.02], [0.01], [-0.01]
])

recovered_values = scaler.inverse_transform(predicted_scaled)
recovered_ids = np.rint(recovered_values).astype(int).flatten()

recovered_words = [
    index_to_word.get(int(idx), "<UNK>")
    for idx in recovered_ids
]

print("Original token IDs:", token_ids.flatten().astype(int))
print("Recovered token IDs:", recovered_ids)
print("Recovered words:", recovered_words)
print("Exact ID recovery:", np.array_equal(
    token_ids.flatten().astype(int), recovered_ids
))

In [ ]:
import numpy as np

true_ids = np.array([2, 3, 4, 1])

errors = [0.1, 0.3, 0.49, 0.51, 0.8, 1.2]

print("Error | Recovered IDs | Exact recovery")
print("-" * 50)

for error in errors:
    predicted = true_ids.astype(float) + error
    recovered = np.rint(predicted).astype(int)

    exact = np.array_equal(true_ids, recovered)

    print(
        f"{error:>5} | "
        f"{recovered.tolist()} | "
        f"{exact}"
    )

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

# Automatically detect the delimiter
df = pd.read_csv(filename, sep=None, engine="python")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

if {"analysisNarrative", "probableCause"}.issubset(df.columns):
    paired = df.dropna(
        subset=["analysisNarrative", "probableCause"]
    ).copy()

    print("\nComplete narrative–cause pairs:", len(paired))

    print(
        "Duplicate narrative–cause pairs:",
        paired.duplicated(
            subset=["analysisNarrative", "probableCause"]
        ).sum()
    )

In [ ]:
# Retain complete narrative–cause pairs
paired = df.dropna(
    subset=["analysisNarrative", "probableCause"]
).copy()

# Remove empty or whitespace-only text
for col in ["analysisNarrative", "probableCause"]:
    paired[col] = paired[col].astype(str).str.strip()

paired = paired[
    (paired["analysisNarrative"] != "") &
    (paired["probableCause"] != "")
].copy()

print("Original records:", len(df))
print("Complete, nonempty pairs:", len(paired))

print("\nExact duplicate pairs:",
      paired.duplicated(
          subset=["analysisNarrative", "probableCause"]
      ).sum())

print("Duplicate narratives:",
      paired.duplicated(
          subset=["analysisNarrative"]
      ).sum())

print("Duplicate probable-cause statements:",
      paired.duplicated(
          subset=["probableCause"]
      ).sum())

print("\nNarrative word lengths:")
print(
    paired["analysisNarrative"]
    .str.split()
    .str.len()
    .describe()
)

print("\nProbable-cause word lengths:")
print(
    paired["probableCause"]
    .str.split()
    .str.len()
    .describe()
)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42

train_df, test_df = train_test_split(
    paired,
    test_size=0.15,
    random_state=SEED,
    shuffle=True
)

print("Total usable pairs:", len(paired))
print("Development records:", len(train_df))
print("Test records:", len(test_df))

print("\nOverlap checks:")
print(
    "Narratives shared between development and test:",
    len(
        set(train_df["analysisNarrative"])
        & set(test_df["analysisNarrative"])
    )
)

print(
    "Probable-cause statements shared between development and test:",
    len(
        set(train_df["probableCause"])
        & set(test_df["probableCause"])
    )
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)

# Fit vocabulary and IDF weights on development targets only.
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2
)

Y_train = vectorizer.fit_transform(
    train_df["probableCause"].astype(str)
)

Y_test = vectorizer.transform(
    test_df["probableCause"].astype(str)
)

print("Training target matrix:", Y_train.shape)
print("Test target matrix:", Y_test.shape)

# Baseline 1: Predict the same mean target for every test case.
mean_target = np.asarray(
    Y_train.mean(axis=0)
).ravel()

mean_norm = np.linalg.norm(mean_target)

if mean_norm == 0:
    raise ValueError("Mean target has zero norm.")

mean_target = mean_target / mean_norm

# TF-IDF vectors are L2-normalized by default.
# Account explicitly for any zero vectors.
test_norms = np.sqrt(
    Y_test.multiply(Y_test).sum(axis=1)
).A1

mean_scores = np.asarray(
    Y_test @ mean_target
).ravel()

mean_scores = np.divide(
    mean_scores,
    test_norms,
    out=np.zeros_like(mean_scores),
    where=test_norms > 0
)

# Baseline 2: Randomly select one development target
# for each test case, with replacement.
random_indices = rng.integers(
    low=0,
    high=Y_train.shape[0],
    size=Y_test.shape[0]
)

Y_random = Y_train[random_indices]

random_scores = np.asarray(
    Y_test.multiply(Y_random).sum(axis=1)
).ravel()

random_norms = np.sqrt(
    Y_random.multiply(Y_random).sum(axis=1)
).A1

random_scores = np.divide(
    random_scores,
    test_norms * random_norms,
    out=np.zeros_like(random_scores),
    where=(test_norms * random_norms) > 0
)

results = pd.DataFrame({
    "Baseline": [
        "Mean target (TF-IDF)",
        "Random target (TF-IDF)"
    ],
    "Mean cosine similarity": [
        mean_scores.mean(),
        random_scores.mean()
    ],
    "Standard deviation": [
        mean_scores.std(ddof=1),
        random_scores.std(ddof=1)
    ]
})

print("\nBASELINE RESULTS")
print(results.to_string(index=False))

print("\nZero-vector test targets:",
      int((test_norms == 0).sum()))

print("Zero-vector development targets:",
      int((np.sqrt(
          Y_train.multiply(Y_train).sum(axis=1)
      ).A1 == 0).sum()))

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Narrative representations
narrative_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2
)

X_train = narrative_vectorizer.fit_transform(
    train_df["analysisNarrative"].astype(str)
)

X_test = narrative_vectorizer.transform(
    test_df["analysisNarrative"].astype(str)
)

print("Narrative training shape:", X_train.shape)
print("Narrative test shape:", X_test.shape)

# Reduce target dimensionality to make regression efficient
svd = TruncatedSVD(
    n_components=128,
    random_state=42
)

Y_train_reduced = svd.fit_transform(Y_train)

# Regularized linear regression
model = Ridge(
    alpha=10.0,
    solver="lsqr"
)

model.fit(X_train, Y_train_reduced)

Y_pred_reduced = model.predict(X_test)

# Return predictions to the original target TF-IDF space
Y_pred = svd.inverse_transform(Y_pred_reduced)

# Normalize predicted representations
Y_pred = normalize(Y_pred, norm="l2")

# Cosine similarity against actual test targets
ridge_scores = np.asarray(
    Y_test.multiply(Y_pred).sum(axis=1)
).ravel()

print("\nTF-IDF + Ridge regression")
print("Mean cosine similarity:", round(ridge_scores.mean(), 6))
print("Standard deviation:", round(ridge_scores.std(ddof=1), 6))

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# X_train and X_test are narrative TF-IDF matrices
# Y_train and Y_test are probable-cause TF-IDF matrices

# Find the most similar development narrative
narrative_similarities = cosine_similarity(
    X_test,
    X_train
)

nearest_indices = narrative_similarities.argmax(axis=1)

# Retrieve the corresponding development probable causes
Y_nearest = Y_train[nearest_indices]

# Compare retrieved causes against actual test causes
nearest_scores = np.asarray(
    Y_test.multiply(Y_nearest).sum(axis=1)
).ravel()

test_norms = np.sqrt(
    Y_test.multiply(Y_test).sum(axis=1)
).A1

nearest_norms = np.sqrt(
    Y_nearest.multiply(Y_nearest).sum(axis=1)
).A1

denominator = test_norms * nearest_norms

nearest_scores = np.divide(
    nearest_scores,
    denominator,
    out=np.zeros_like(nearest_scores),
    where=denominator > 0
)

print("NEAREST-NEIGHBOR RETRIEVAL BASELINE")
print("Mean cosine similarity:",
      round(nearest_scores.mean(), 6))
print("Standard deviation:",
      round(nearest_scores.std(ddof=1), 6))

print("\nAverage narrative retrieval similarity:",
      round(
          narrative_similarities.max(axis=1).mean(),
          6
      ))

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def content_tokens(text):
    tokens = re.findall(r"[a-z]+", str(text).lower())
    return set(
        t for t in tokens
        if len(t) > 2 and t not in ENGLISH_STOP_WORDS
    )

def overlap_metrics(prediction, reference):
    p = content_tokens(prediction)
    r = content_tokens(reference)

    if not p or not r:
        return 0.0, 0.0, 0.0

    shared = len(p & r)

    precision = shared / len(p)
    recall = shared / len(r)

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    return precision, recall, f1

# 1. Overlap between actual test narratives and actual causes
narrative_cause = np.array([
    overlap_metrics(narrative, cause)
    for narrative, cause in zip(
        test_df["analysisNarrative"],
        test_df["probableCause"]
    )
])

# 2. Overlap between nearest-neighbor predicted causes
# and actual test causes
retrieved_causes = train_df.iloc[
    nearest_indices
]["probableCause"].to_numpy()

retrieval_cause = np.array([
    overlap_metrics(predicted, actual)
    for predicted, actual in zip(
        retrieved_causes,
        test_df["probableCause"]
    )
])

print("NARRATIVE–CAUSE CONTENT-WORD OVERLAP")
print("Precision:", round(narrative_cause[:, 0].mean(), 4))
print("Recall:", round(narrative_cause[:, 1].mean(), 4))
print("F1:", round(narrative_cause[:, 2].mean(), 4))

print("\nRETRIEVAL PREDICTION–REFERENCE OVERLAP")
print("Precision:", round(retrieval_cause[:, 0].mean(), 4))
print("Recall:", round(retrieval_cause[:, 1].mean(), 4))
print("F1:", round(retrieval_cause[:, 2].mean(), 4))

In [ ]:
import numpy as np

def prediction_source_overlap(prediction, narrative):
    predicted_words = content_tokens(prediction)
    narrative_words = content_tokens(narrative)

    if not predicted_words:
        return 0.0

    return len(predicted_words & narrative_words) / len(predicted_words)

source_overlap = np.array([
    prediction_source_overlap(predicted, narrative)
    for predicted, narrative in zip(
        retrieved_causes,
        test_df["analysisNarrative"]
    )
])

print("PREDICTED CAUSE–SOURCE NARRATIVE OVERLAP")
print("Mean:", round(source_overlap.mean(), 4))
print("Median:", round(np.median(source_overlap), 4))
print("Standard deviation:", round(source_overlap.std(ddof=1), 4))

print("\nPercentage of predictions with:")
print("No lexical overlap:",
      round(100 * np.mean(source_overlap == 0), 2), "%")
print("At least 50% overlap:",
      round(100 * np.mean(source_overlap >= 0.5), 2), "%")
print("Complete overlap:",
      round(100 * np.mean(source_overlap == 1), 2), "%")

In [ ]:
import numpy as np

# Y_pred: Ridge-predicted TF-IDF vectors
# Y_test: reference probable-cause TF-IDF vectors
# vectorizer: target TF-IDF vectorizer

feature_names = vectorizer.get_feature_names_out()

# Use unigrams only to avoid counting bigrams as individual words
unigram_indices = np.array([
    i for i, term in enumerate(feature_names)
    if " " not in term
])

unigram_features = feature_names[unigram_indices]

reference_texts = test_df["probableCause"].astype(str).tolist()

precision_scores = []
recall_scores = []
f1_scores = []

for i, reference in enumerate(reference_texts):

    reference_words = content_tokens(reference)
    k = len(reference_words)

    if k == 0:
        continue

    # Rank predicted unigram features by their Ridge scores
    weights = Y_pred[i, unigram_indices]

    top_indices = np.argsort(weights)[::-1][:k]

    predicted_words = set(
        unigram_features[top_indices]
    )

    shared = len(predicted_words & reference_words)

    precision = shared / len(predicted_words)
    recall = shared / len(reference_words)

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

print("RIDGE REGRESSION: KEYWORD RECOVERY")
print("Mean precision:", round(np.mean(precision_scores), 4))
print("Mean recall:", round(np.mean(recall_scores), 4))
print("Mean F1:", round(np.mean(f1_scores), 4))
print("Evaluated test cases:", len(f1_scores))

In [ ]:
import numpy as np

K = 10

precision_scores = []
recall_scores = []
f1_scores = []
source_overlap_scores = []

for i, (reference, narrative) in enumerate(zip(
    test_df["probableCause"].astype(str),
    test_df["analysisNarrative"].astype(str)
)):
    reference_words = content_tokens(reference)
    narrative_words = content_tokens(narrative)

    weights = Y_pred[i, unigram_indices]

    top_indices = np.argsort(weights)[::-1][:K]

    predicted_words = set(
        unigram_features[top_indices]
    )

    shared = len(predicted_words & reference_words)

    precision = shared / len(predicted_words)

    recall = (
        shared / len(reference_words)
        if reference_words else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    source_overlap = (
        len(predicted_words & narrative_words)
        / len(predicted_words)
    )

    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    source_overlap_scores.append(source_overlap)

print("RIDGE: FIXED TOP-10 KEYWORD EVALUATION")
print("Precision:", round(np.mean(precision_scores), 4))
print("Recall:", round(np.mean(recall_scores), 4))
print("F1:", round(np.mean(f1_scores), 4))
print("Predicted keyword–narrative overlap:",
      round(np.mean(source_overlap_scores), 4))
print("Test records:", len(f1_scores))

In [ ]:
import json
from google.colab import files

uploaded = files.upload()

for filename in uploaded:
    if filename.endswith(".ipynb"):
        with open(filename, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        keywords = [
            "glove", "embedding", "word2vec",
            "reshape", "pad_sequences",
            "tokenizer", "word_index",
            "inverse", "nearest", "cosine",
            "wordcloud", "word_cloud",
            "decode", "latent", "z_mean",
            "z_log_var", "sampling"
        ]

        print("\nNOTEBOOK:", filename)

        for i, cell in enumerate(notebook.get("cells", [])):
            source = "".join(cell.get("source", []))

            if any(k in source.lower() for k in keywords):
                print("\n" + "=" * 60)
                print("CELL:", i)
                print(source[:3500])

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Small demonstration using aviation-related sentences
causes = [
    "The pilot failed to maintain adequate airspeed",
    "Engine failure resulted in a forced landing",
    "Improper maintenance caused loss of engine power"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(causes)

max_len = 12

sequences = tokenizer.texts_to_sequences(causes)

padded = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post"
)

# Fixed global scaling, preserving integer token IDs
max_id = max(tokenizer.word_index.values())
normalized = padded.astype(float) / max_id

# Simulate numerical model output using known values
predicted = normalized[0].copy()

# Recover integer token IDs
recovered_ids = np.rint(
    np.clip(predicted, 0, 1) * max_id
).astype(int)

# Convert IDs back to words, preserving sequence order
recovered_words = [
    tokenizer.index_word.get(int(i), "<UNK>")
    for i in recovered_ids
    if i != 0
]

print("Original:", causes[0])
print("Token IDs:", padded[0].tolist())
print("Recovered:", " ".join(recovered_words))
print("Exact ID recovery:",
      np.array_equal(recovered_ids, padded[0]))

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

original_ids = padded[0]
original_values = normalized[0]

noise_levels = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]

print(f"{'Noise':<10} {'Token accuracy':<18} Recovered words")
print("-" * 90)

for noise in noise_levels:
    noisy_prediction = original_values + rng.normal(
        0, noise, size=original_values.shape
    )

    recovered_ids = np.rint(
        np.clip(noisy_prediction, 0, 1) * max_id
    ).astype(int)

    # Accuracy on actual tokens, excluding padding
    valid = original_ids != 0
    accuracy = np.mean(
        recovered_ids[valid] == original_ids[valid]
    )

    recovered_words = [
        tokenizer.index_word.get(int(i), "<UNK>")
        for i in recovered_ids
        if i != 0
    ]

    print(
        f"{noise:<10.3f} "
        f"{accuracy:<18.3f} "
        f"{' '.join(recovered_words)}"
    )

In [ ]:
import glob
import os
import csv

files = glob.glob("/content/ntsb_dataset_2000_2020*.csv")

for path in files:
    print("\nFILE:", os.path.basename(path))
    print("SIZE:", os.path.getsize(path))

    with open(path, "rb") as f:
        sample = f.read(20000)

    print("First 300 bytes:")
    print(repr(sample[:300]))

    print("\nDetected delimiters:")
    print("Tabs:", sample.count(b"\t"))
    print("Commas:", sample.count(b","))
    print("Semicolons:", sample.count(b";"))

    try:
        dialect = csv.Sniffer().sniff(
            sample.decode("utf-8-sig", errors="replace"),
            delimiters=",\t;"
        )
        print("Sniffer delimiter:", repr(dialect.delimiter))
    except csv.Error:
        print("Sniffer could not identify delimiter")

In [ ]:
import pandas as pd
import csv

file_path = "/content/ntsb_dataset_2000_2020.csv"

df = pd.read_csv(
    file_path,
    sep="\t",
    engine="python",
    quoting=csv.QUOTE_NONE,
    encoding="utf-8",
    on_bad_lines="error"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

required = ["analysisNarrative", "probableCause"]
assert all(c in df.columns for c in required)

valid = df[required].dropna().copy()

for col in required:
    valid[col] = valid[col].astype(str).str.strip()

valid = valid[
    valid["analysisNarrative"].ne("") &
    valid["probableCause"].ne("")
]

print("Complete narrative-cause pairs:", len(valid))
print("First probable cause:")
print(valid["probableCause"].iloc[0][:300])

In [ ]:
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Use the correctly parsed dataset
causes = valid["probableCause"].head(1000).tolist()

# Earlier notebook's tokenizer configuration
tokenizer = Tokenizer(num_words=7000)
tokenizer.fit_on_texts(causes)

sequences = tokenizer.texts_to_sequences(causes)

padded = pad_sequences(
    sequences,
    maxlen=121,
    padding="post",
    truncating="post"
)

# Simplified numerical encoding for sensitivity testing
max_id = max(1, int(padded.max()))
normalized = padded.astype(np.float64) / max_id

noise_levels = [0, 0.001, 0.005, 0.01, 0.02, 0.05]
repetitions = 30
rng = np.random.default_rng(42)

mask = padded != 0
results = []

for noise in noise_levels:
    accuracies = []

    for _ in range(repetitions):
        predictions = normalized + rng.normal(
            0, noise, normalized.shape
        )

        recovered = np.rint(
            np.clip(predictions, 0, 1) * max_id
        ).astype(int)

        accuracy = np.mean(
            recovered[mask] == padded[mask]
        )

        accuracies.append(accuracy)

    results.append({
        "Noise": noise,
        "Mean accuracy": np.mean(accuracies),
        "SD": np.std(accuracies)
    })

results_df = pd.DataFrame(results)

print("Statements:", len(causes))
print("Non-padding tokens:", int(mask.sum()))
print("Maximum token ID:", max_id)
print()
print(results_df.to_string(index=False))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Use the existing, correctly parsed dataset
data = valid[["analysisNarrative", "probableCause"]].copy()
data = data.reset_index(drop=True)

# Normalize text for exact-duplicate detection
def normalize_text(series):
    return (
        series.astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

data["narrative_norm"] = normalize_text(data["analysisNarrative"])
data["cause_norm"] = normalize_text(data["probableCause"])

# Reproduce the supplementary 85/15 split
train, test = train_test_split(
    data,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print("Development records:", len(train))
print("Test records:", len(test))

# Duplicates within the entire supplementary dataset
print("\nFULL DATASET")
print("Repeated narratives:",
      data.duplicated("narrative_norm").sum())
print("Repeated probable causes:",
      data.duplicated("cause_norm").sum())
print("Repeated narrative-cause pairs:",
      data.duplicated(
          ["narrative_norm", "cause_norm"]
      ).sum())

# Exact matches crossing development and test partitions
train_narratives = set(train["narrative_norm"])
train_causes = set(train["cause_norm"])
train_pairs = set(zip(
    train["narrative_norm"],
    train["cause_norm"]
))

test_narrative_matches = test["narrative_norm"].isin(
    train_narratives
)

test_cause_matches = test["cause_norm"].isin(
    train_causes
)

test_pair_matches = pd.Series(
    [
        (n, c) in train_pairs
        for n, c in zip(
            test["narrative_norm"],
            test["cause_norm"]
        )
    ],
    index=test.index
)

print("\nCROSS-SPLIT EXACT MATCHES")
print("Test narratives also in development:",
      int(test_narrative_matches.sum()))
print("Test probable causes also in development:",
      int(test_cause_matches.sum()))
print("Test narrative-cause pairs also in development:",
      int(test_pair_matches.sum()))

print("\nCROSS-SPLIT PERCENTAGES")
print("Narrative overlap: {:.2f}%".format(
    100 * test_narrative_matches.mean()
))
print("Probable-cause overlap: {:.2f}%".format(
    100 * test_cause_matches.mean()
))
print("Complete-pair overlap: {:.2f}%".format(
    100 * test_pair_matches.mean()
))

In [ ]:

import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize

# Reuse the verified complete-pair dataset
data = valid[["analysisNarrative", "probableCause"]].copy()
data = data.reset_index(drop=True)

dev, test = train_test_split(
    data, test_size=0.15, random_state=42, shuffle=True
)

# Target representation
target_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    max_features=10000,
    min_df=2
)

Y_dev = target_vectorizer.fit_transform(dev["probableCause"])
Y_test = target_vectorizer.transform(test["probableCause"])

# Narrative representation
narrative_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    max_features=10000,
    min_df=2
)

X_dev = narrative_vectorizer.fit_transform(dev["analysisNarrative"])
X_test = narrative_vectorizer.transform(test["analysisNarrative"])

# Reduced target space
svd = TruncatedSVD(n_components=128, random_state=42)
Y_dev_reduced = svd.fit_transform(Y_dev)

# Supervised regression baseline
ridge = Ridge(alpha=10.0, solver="lsqr")
ridge.fit(X_dev, Y_dev_reduced)

# Reconstruct predictions in target TF-IDF space
Y_pred_reduced = ridge.predict(X_test)
Y_pred = Y_pred_reduced @ svd.components_

# Cosine similarity in the common target space
Y_pred_norm = normalize(Y_pred)
Y_test_norm = normalize(Y_test)

scores = np.asarray(
    Y_test_norm.multiply(Y_pred_norm).sum(axis=1)
).ravel()

# Lexical overlap
def content_words(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    return set(w for w in words if w not in ENGLISH_STOP_WORDS)

def cause_recall(narrative, cause):
    a = content_words(narrative)
    b = content_words(cause)
    return len(a & b) / len(b) if b else np.nan

results = test.copy()
results["cosine"] = scores
results["narrative_words"] = (
    results["analysisNarrative"].str.split().str.len()
)
results["cause_words"] = (
    results["probableCause"].str.split().str.len()
)
results["lexical_overlap"] = [
    cause_recall(n, c)
    for n, c in zip(
        results["analysisNarrative"],
        results["probableCause"]
    )
]

# Descriptive performance groups
results["performance_group"] = pd.qcut(
    results["cosine"].rank(method="first"),
    q=3,
    labels=["Low", "Medium", "High"]
)

print("Mean baseline cosine:", round(scores.mean(), 4))
print("SD:", round(scores.std(), 4))

print("\nPERFORMANCE GROUP SUMMARY")
print(
    results.groupby(
        "performance_group", observed=True
    )[[
        "cosine",
        "narrative_words",
        "cause_words",
        "lexical_overlap"
    ]].agg(["mean", "median"]).round(4)
)

print("\nCORRELATIONS WITH COSINE")
print(
    results[[
        "cosine",
        "narrative_words",
        "cause_words",
        "lexical_overlap"
    ]].corr()["cosine"].round(4)
)

print("\nTHREE REPRESENTATIVE RECORDS")
for group in ["Low", "Medium", "High"]:
    subset = results[
        results["performance_group"] == group
    ].sort_values("cosine")

    example = subset.iloc[len(subset) // 2]

    print("\nGROUP:", group)
    print("Cosine:", round(example["cosine"], 4))
    print("Narrative words:", example["narrative_words"])
    print("Lexical overlap:", round(example["lexical_overlap"], 4))
    print("Expert probable cause:")
    print(example["probableCause"][:600])


In [ ]:

import numpy as np
import pandas as pd

# Confirm that predictions and test records align
assert len(results) == Y_pred.shape[0]
assert Y_pred.shape[1] == len(
    target_vectorizer.get_feature_names_out()
)

terms = target_vectorizer.get_feature_names_out()

def predicted_terms(row_index, top_k=10):
    """Retrieve highest-scoring target vocabulary terms."""
    weights = np.asarray(Y_pred[row_index]).ravel()
    indices = np.argsort(weights)[::-1]

    # Keep only positive-weight terms
    indices = [i for i in indices if weights[i] > 0]

    return [terms[i] for i in indices[:top_k]]

# Reset indices to align rows with prediction matrix
case_results = results.reset_index(drop=True)

case_table = []

for group in ["Low", "Medium", "High"]:
    indices = np.where(
        case_results["performance_group"].astype(str).values
        == group
    )[0]

    ordered = indices[
        np.argsort(case_results.loc[indices, "cosine"].values)
    ]

    # Representative case near group median
    idx = int(ordered[len(ordered) // 2])
    record = case_results.iloc[idx]

    case_table.append({
        "Group": group,
        "Cosine": round(float(record["cosine"]), 4),
        "Predicted terms": ", ".join(predicted_terms(idx)),
        "Expert probable cause": record["probableCause"]
    })

case_table = pd.DataFrame(case_table)

pd.set_option("display.max_colwidth", 500)
display(case_table)

print("\nCOPYABLE OUTPUT\n")
for _, row in case_table.iterrows():
    print("GROUP:", row["Group"])
    print("COSINE:", row["Cosine"])
    print("PREDICTED TERMS:", row["Predicted terms"])
    print("EXPERT PROBABLE CAUSE:", row["Expert probable cause"])
    print("-" * 70)


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

summary = (
    results.groupby("performance_group", observed=True)
    .agg(
        cosine=("cosine", "mean"),
        narrative_words=("narrative_words", "mean"),
        lexical_overlap=("lexical_overlap", "mean")
    )
    .reindex(["Low", "Medium", "High"])
)

groups = ["Low", "Medium", "High"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

measures = [
    ("cosine", "Mean cosine similarity", "(a)"),
    ("narrative_words", "Mean narrative length (words)", "(b)"),
    ("lexical_overlap", "Mean lexical overlap", "(c)")
]

for ax, (column, ylabel, panel) in zip(axes, measures):
    values = summary[column].values
    bars = ax.bar(groups, values, color="#356B9B", width=0.62)

    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(panel, fontsize=11, loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_ylim(0, max(values) * 1.25)

    for bar, value in zip(bars, values):
        label = f"{value:.2f}" if column == "narrative_words" \
                else f"{value:.3f}"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.035,
            label,
            ha="center",
            fontsize=9
        )

plt.tight_layout()

plt.savefig(
    "supplementary_error_analysis.pdf",
    bbox_inches="tight"
)
plt.savefig(
    "supplementary_error_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import numpy as np
import pandas as pd

# Use the complete-pair dataset from our earlier experiments
df = valid.copy()

narrative_lengths = (
    df["analysisNarrative"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

cause_lengths = (
    df["probableCause"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print("Complete records:", len(df))

print("\nNARRATIVES")
print("Mean tokens:", round(narrative_lengths.mean(), 2))
print("Maximum tokens:", narrative_lengths.max())
print("Records exceeding 1200:", (narrative_lengths > 1200).sum())
print(
    "Percentage exceeding 1200:",
    round((narrative_lengths > 1200).mean() * 100, 2),
    "%"
)

print("\nPROBABLE CAUSES")
print("Mean tokens:", round(cause_lengths.mean(), 2))
print("Maximum tokens:", cause_lengths.max())
print("Records exceeding 200:", (cause_lengths > 200).sum())
print(
    "Percentage exceeding 200:",
    round((cause_lengths > 200).mean() * 100, 2),
    "%"
)

print("\nLENGTH DISTRIBUTIONS")
print(
    pd.DataFrame({
        "Narrative": narrative_lengths.describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        ),
        "Probable cause": cause_lengths.describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        )
    }).round(2)
)


In [ ]:

import re
from collections import Counter

# Use the same supplementary dataset
df = valid.copy()

def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", str(text).lower())

narrative_tokens = []
cause_tokens = []

for text in df["analysisNarrative"].fillna(""):
    narrative_tokens.extend(tokenize(text))

for text in df["probableCause"].fillna(""):
    cause_tokens.extend(tokenize(text))

narrative_vocab = set(narrative_tokens)
cause_vocab = set(cause_tokens)
combined_vocab = narrative_vocab | cause_vocab

print("SUPPLEMENTARY DATASET VOCABULARY AUDIT")
print("Complete records:", len(df))
print("Narrative vocabulary:", len(narrative_vocab))
print("Probable-cause vocabulary:", len(cause_vocab))
print("Combined vocabulary:", len(combined_vocab))

print("\nTOKEN COUNTS")
print("Narrative tokens:", len(narrative_tokens))
print("Probable-cause tokens:", len(cause_tokens))
print("Combined tokens:", len(narrative_tokens) + len(cause_tokens))

print("\nVOCABULARY OVERLAP")
print("Shared vocabulary:", len(narrative_vocab & cause_vocab))

print("\nMOST FREQUENT WORDS")
for word, count in Counter(
    narrative_tokens + cause_tokens
).most_common(15):
    print(word, count)


In [ ]:
def loss_func(encoder_mu, encoder_log_var):

    def vae_kl_loss(encoder_mu, encoder_log_var):
        kl_loss = -0.5 * K.sum(
            1.0 + encoder_log_var
            - K.square(encoder_mu)
            - K.exp(encoder_log_var),
            axis=1
        )
        return kl_loss

In [ ]:
for name in [
    "narrative",
    "prob_cause",
    "sequences_matrix_analysis",
    "sequences_matrix_cause",
    "encoder_mu_log_var_model",
    "encoder",
    "decoder",
    "vae"
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT LOADED'}")

In [ ]:
print("CURRENT NOTEBOOK STATE")
print("=" * 55)

for name in ["narrative", "encoder", "prob_cause", "vae"]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}")
        print("Type:", type(obj).__name__)
        print("Shape:", getattr(obj, "shape", "Not available"))

        if name == "encoder":
            if hasattr(obj, "summary"):
                obj.summary()
            else:
                print("Not a Keras model with a summary method.")
    else:
        print(f"\n{name}: NOT LOADED")

print("\nAVAILABLE MODEL-RELATED VARIABLES")
print("=" * 55)

for name in sorted(globals()):
    if any(term in name.lower() for term in
           ["model", "weight", "checkpoint", "encoder", "decoder", "train", "test"]):
        if not name.startswith("_"):
            print(name)

In [ ]:
print("EXPERIMENT IDENTIFICATION")
print("=" * 55)

for name in [
    "X_train", "X_test",
    "Y_train", "Y_test",
    "train_df", "test_df",
    "model", "Model",
    "encoder_heads"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("Type:", type(obj).__name__)
        print("Shape:", getattr(obj, "shape", "N/A"))

        if name in ["model", "encoder_heads"]:
            print("Model name:", getattr(obj, "name", "N/A"))
            print("Input shape:", getattr(obj, "input_shape", "N/A"))
            print("Output shape:", getattr(obj, "output_shape", "N/A"))

In [ ]:
import numpy as np

print("SUPPLEMENTARY EXPERIMENT AUDIT")
print("=" * 60)

for name in [
    "Y_train_reduced",
    "Y_test_norm",
    "model",
    "encoder",
    "encoder_heads",
    "weights"
]:
    if name not in globals():
        print(f"{name}: NOT AVAILABLE")
        continue

    obj = globals()[name]

    print(f"\n{name}")
    print("Type:", type(obj).__name__)
    print("Shape:", getattr(obj, "shape", "N/A"))

    if name == "model":
        print("Parameters:", obj.get_params())

    if name == "weights":
        print("Type:", type(obj).__name__)
        if isinstance(obj, (list, tuple)):
            print("Number of arrays:", len(obj))
            print("Array shapes:", [
                np.shape(w) for w in obj[:10]
            ])

In [ ]:
import numpy as np

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# --------------------------------------------------
# 1. Reduce narrative feature dimensionality
# --------------------------------------------------

svd_x = TruncatedSVD(
    n_components=128,
    random_state=42
)

X_train_128 = svd_x.fit_transform(X_train)
X_test_128 = svd_x.transform(X_test)

print("Reduced narrative shapes:")
print(X_train_128.shape, X_test_128.shape)

# --------------------------------------------------
# 2. Train Ridge on reduced input features
# --------------------------------------------------

ridge_128 = Ridge(alpha=10.0, solver="lsqr")

ridge_128.fit(
    X_train_128,
    Y_train_reduced
)

ridge_predictions_128 = ridge_128.predict(X_test_128)

# --------------------------------------------------
# 3. Train nonlinear Random Forest baseline
# --------------------------------------------------

rf_128 = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_128.fit(
    X_train_128,
    Y_train_reduced
)

rf_predictions_128 = rf_128.predict(X_test_128)

# --------------------------------------------------
# 4. Recover target-space predictions
# --------------------------------------------------

# Identify the fitted target SVD from the current session.
# Do not refit it on test data.

target_svd_candidates = [
    (name, obj)
    for name, obj in globals().items()
    if isinstance(obj, TruncatedSVD)
    and hasattr(obj, "components_")
    and obj.components_.shape == (128, Y_train.shape[1])
]

if len(target_svd_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one fitted 128-dimensional target SVD. "
        f"Found: {[name for name, _ in target_svd_candidates]}. "
        "Please share this output so we can identify the correct transformer."
    )

target_svd_name, target_svd = target_svd_candidates[0]

print("Using target SVD:", target_svd_name)

ridge_pred_original = target_svd.inverse_transform(
    ridge_predictions_128
)

rf_pred_original = target_svd.inverse_transform(
    rf_predictions_128
)

# --------------------------------------------------
# 5. Evaluate against actual test targets
# --------------------------------------------------

Y_test_dense = Y_test.toarray()

def evaluate_predictions(y_true, y_pred):
    true_norm = normalize(y_true)
    pred_norm = normalize(y_pred)

    similarities = np.sum(
        true_norm * pred_norm,
        axis=1
    )

    return (
        float(np.mean(similarities)),
        float(np.std(similarities))
    )

ridge_mean, ridge_sd = evaluate_predictions(
    Y_test_dense,
    ridge_pred_original
)

rf_mean, rf_sd = evaluate_predictions(
    Y_test_dense,
    rf_pred_original
)

print("\nMATCHED SUPPLEMENTARY BASELINE COMPARISON")
print("=" * 55)

print(
    f"Ridge + input SVD: "
    f"{ridge_mean:.4f} ± {ridge_sd:.4f}"
)

print(
    f"Random Forest + input SVD: "
    f"{rf_mean:.4f} ± {rf_sd:.4f}"
)

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

def per_case_cosine(y_true, y_pred):
    a = normalize(y_true)
    b = normalize(y_pred)
    return np.sum(a * b, axis=1)

ridge_scores = per_case_cosine(
    Y_test_dense, ridge_pred_original
)

rf_scores = per_case_cosine(
    Y_test_dense, rf_pred_original
)

differences = ridge_scores - rf_scores

rng = np.random.default_rng(42)
n_bootstrap = 5000

bootstrap_differences = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    indices = rng.integers(
        0, len(differences), len(differences)
    )
    bootstrap_differences[i] = differences[indices].mean()

ci_lower, ci_upper = np.percentile(
    bootstrap_differences, [2.5, 97.5]
)

print("PAIRED BASELINE COMPARISON")
print("=" * 50)
print(f"Test cases: {len(differences)}")
print(f"Mean difference (Ridge - RF): {differences.mean():.4f}")
print(f"95% bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Ridge better on: {(differences > 0).sum()} cases")
print(f"Random Forest better on: {(differences < 0).sum()} cases")

In [ ]:
import re

terms = [
    "latent_space_dim",
    "reconstruction_loss_factor",
    "train_test_split",
    "Conv1D",
    "Conv2D",
    "GloVe",
    "glove",
    "vae_kl_loss",
    "vae_loss",
    "model.compile",
    "vae.compile",
]

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

for term in terms:
    matches = [
        (i, line.strip()[:300])
        for i, line in enumerate(history)
        if term.lower() in line.lower()
    ]
    print(f"\n{term}:")
    for i, line in matches[-5:]:
        print(f"  Cell {i}: {line}")

In [ ]:
import json
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    if not filename.endswith(".ipynb"):
        continue

    notebook = json.loads(content.decode("utf-8"))

    keywords = [
        "latent_space_dim",
        "reconstruction_loss_factor",
        "train_test_split(",
        "Conv1D(",
        "Conv2D(",
        "glove",
        "vae_kl_loss",
        "compile(",
        ".fit("
    ]

    print("\nORIGINAL NOTEBOOK CONFIGURATION AUDIT")
    print("=" * 60)

    for i, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        matched = [
            term for term in keywords
            if term.lower() in source.lower()
        ]

        if matched:
            print(f"\nNOTEBOOK CELL {i}")
            print("Matched:", ", ".join(matched))
            print(source[:2500])
            print("-" * 60)

In [ ]:
def vae_loss(y_true, y_predict):
    reconstruction_loss = cosine_similarity(y_true, y_predict)
    kl_loss = vae_kl_loss(y_true, y_predict)

    loss = reconstruction_loss + kl_loss
    return loss

In [ ]:
import json
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    if not filename.endswith(".ipynb"):
        continue

    notebook = json.loads(content.decode("utf-8"))

    print("SAVED TRAINING AND EVALUATION OUTPUTS")
    print("=" * 65)

    for i, cell in enumerate(notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        keywords = [
            "vae.fit",
            "vae.evaluate",
            "predict(",
            "cosine_similarity",
            "model.summary",
            "vae.summary",
        ]

        if not any(k.lower() in source.lower() for k in keywords):
            continue

        print(f"\nNOTEBOOK CELL {i}")
        print("SOURCE:")
        print(source[:1200])

        outputs = cell.get("outputs", [])

        if not outputs:
            print("SAVED OUTPUT: None")
            continue

        for output in outputs:
            text_output = "".join(output.get("text", []))

            data = output.get("data", {})
            text_output += "".join(
                data.get("text/plain", [])
            )

            if text_output.strip():
                print("SAVED OUTPUT:")
                print(text_output[-3000:])
            else:
                print(
                    "Saved output exists, but is not plain text."
                )

In [ ]:
import pandas as pd
import numpy as np
import re

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("/content/ntsb_two_columns_clean.csv")

# Basic word tokenization, without stemming or lemmatization yet
def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower())

input_tokens = df["analysisNarrative"].map(tokenize)
target_tokens = df["probableCause"].map(tokenize)

input_lengths = input_tokens.map(len)
target_lengths = target_tokens.map(len)

print("Number of paired records:", len(df))
print("Unique narrative tokens:",
      len(set(w for row in input_tokens for w in row)))
print("Unique probable-cause tokens:",
      len(set(w for row in target_tokens for w in row)))

print("\nNarratives exceeding 1200 tokens:",
      (input_lengths > 1200).sum())
print("Probable causes exceeding 200 tokens:",
      (target_lengths > 200).sum())

print("\nExample narrative tokens:")
print(input_tokens.iloc[0][:20])

print("\nExample probable-cause tokens:")
print(target_tokens.iloc[0][:20])

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Total records:", len(train_df) + len(test_df))

# Save the split for reproducibility
train_df.to_csv("ntsb_train.csv", index=False)
test_df.to_csv("ntsb_test.csv", index=False)

In [ ]:
import gensim
import gensim.downloader as api

print("Gensim version:", gensim.__version__)

models = [
    name for name in api.info()["models"]
    if "glove" in name.lower()
]

print("Available GloVe models:")
print(models)

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

print("GloVe vocabulary size:", len(glove))
print("Embedding dimensions:", glove.vector_size)

word = "aircraft"

if word in glove:
    print("Example word:", word)
    print("First 10 embedding values:", glove[word][:10])

In [ ]:
from collections import Counter
import re

def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower())

# Use training data only
narrative_tokens = train_df["analysisNarrative"].map(tokenize)
cause_tokens = train_df["probableCause"].map(tokenize)

all_tokens = [
    token
    for document in list(narrative_tokens) + list(cause_tokens)
    for token in document
]

counts = Counter(all_tokens)

total_tokens = sum(counts.values())
covered_tokens = sum(
    frequency
    for word, frequency in counts.items()
    if word in glove
)

covered_words = sum(word in glove for word in counts)

print("Total training tokens:", total_tokens)
print("Unique training words:", len(counts))

print(
    "Token coverage:",
    f"{100 * covered_tokens / total_tokens:.2f}%"
)

print(
    "Vocabulary coverage:",
    f"{100 * covered_words / len(counts):.2f}%"
)

print("\n20 most frequent words missing from GloVe:")

missing = [
    (word, frequency)
    for word, frequency in counts.most_common()
    if word not in glove
]

print(missing[:20])

In [ ]:
def normalize_for_glove(word):
    if word.endswith("'s") and len(word) > 2:
        return word[:-2]
    return word

normalized_counts = Counter()

for word, frequency in counts.items():
    normalized_counts[normalize_for_glove(word)] += frequency

normalized_covered = sum(
    frequency
    for word, frequency in normalized_counts.items()
    if word in glove
)

normalized_vocab_covered = sum(
    word in glove for word in normalized_counts
)

print(
    "Original token coverage:",
    f"{100 * covered_tokens / total_tokens:.2f}%"
)

print(
    "Normalized token coverage:",
    f"{100 * normalized_covered / total_tokens:.2f}%"
)

print(
    "Normalized vocabulary coverage:",
    f"{100 * normalized_vocab_covered / len(normalized_counts):.2f}%"
)

print("\nExamples:")
for word in ["pilot's", "airplane's", "aircraft's", "preimpact"]:
    normalized = normalize_for_glove(word)
    print(word, "->", normalized, "| In GloVe:", normalized in glove)

In [ ]:
import numpy as np

def text_to_glove(text, max_length):
    tokens = [
        normalize_for_glove(w)
        for w in tokenize(text)
    ]

    matrix = np.zeros(
        (max_length, glove.vector_size),
        dtype=np.float32
    )

    for i, word in enumerate(tokens[:max_length]):
        if word in glove:
            matrix[i] = glove[word]

    return matrix

example_x = text_to_glove(
    train_df["analysisNarrative"].iloc[0], 1200
)

example_y = text_to_glove(
    train_df["probableCause"].iloc[0], 200
)

print("Narrative tensor:", example_x.shape)
print("Probable-cause tensor:", example_y.shape)

print("First narrative word vector:")
print(example_x[0][:10])

print("Nonzero narrative positions:",
      np.count_nonzero(np.any(example_x != 0, axis=1)))

In [ ]:
import numpy as np

batch_size = 32

def glove_batch(texts, max_length):
    return np.stack([
        text_to_glove(text, max_length)
        for text in texts
    ])

batch_x = glove_batch(
    train_df["analysisNarrative"].iloc[:batch_size],
    1200
)

batch_y = glove_batch(
    train_df["probableCause"].iloc[:batch_size],
    200
)

print("Input batch:", batch_x.shape)
print("Target batch:", batch_y.shape)

print("Input contains NaN:", np.isnan(batch_x).any())
print("Target contains NaN:", np.isnan(batch_y).any())

print("Input memory (MB):", round(batch_x.nbytes / 1e6, 2))
print("Target memory (MB):", round(batch_y.nbytes / 1e6, 2))

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.backend.clear_session()

LATENT_DIM = 20

# Encoder
encoder_input = layers.Input(shape=(1200, 100))

x = layers.Conv1D(64, 5, strides=2, padding="same",
                  activation="relu")(encoder_input)
x = layers.Conv1D(128, 5, strides=2, padding="same",
                  activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)

z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)

encoder = Model(
    encoder_input,
    [z_mean, z_log_var],
    name="encoder"
)

# Decoder
decoder_input = layers.Input(shape=(LATENT_DIM,))

y = layers.Dense(50 * 128, activation="relu")(decoder_input)
y = layers.Reshape((50, 128))(y)

y = layers.UpSampling1D(2)(y)
y = layers.Conv1D(128, 5, padding="same",
                  activation="relu")(y)

y = layers.UpSampling1D(2)(y)
y = layers.Conv1D(64, 5, padding="same",
                  activation="relu")(y)

decoder_output = layers.Conv1D(
    100, 1, padding="same", activation="linear"
)(y)

decoder = Model(
    decoder_input,
    decoder_output,
    name="decoder"
)

print("Encoder:")
encoder.summary()

print("\nDecoder:")
decoder.summary()

print("\nDecoder output shape:", decoder.output_shape)

In [ ]:
import tensorflow as tf

class NarrativeCauseVAE(tf.keras.Model):
    def __init__(self, encoder, decoder, beta=0.01):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.recon_tracker,
            self.kl_tracker
        ]

    def sample(self, z_mean, z_log_var):
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    def compute_losses(self, x, y, training):
        z_mean, z_log_var = self.encoder(x, training=training)

        if training:
            z = self.sample(z_mean, z_log_var)
        else:
            z = z_mean

        prediction = self.decoder(z, training=training)

        # Identify real target tokens, excluding zero padding
        mask = tf.reduce_any(tf.not_equal(y, 0.0), axis=-1)
        mask = tf.cast(mask, tf.float32)

        # Cosine similarity for each target position
        cosine = tf.keras.losses.cosine_similarity(
            y, prediction, axis=-1
        )

        # Keras cosine_similarity returns negative cosine
        cosine_loss = 1.0 + cosine

        # Average over real target positions for each record
        per_record_loss = (
            tf.reduce_sum(cosine_loss * mask, axis=1)
            / tf.maximum(tf.reduce_sum(mask, axis=1), 1.0)
        )

        reconstruction_loss = tf.reduce_mean(per_record_loss)

        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1.0 + z_log_var
                - tf.square(z_mean)
                - tf.exp(z_log_var),
                axis=1
            )
        )

        total_loss = reconstruction_loss + self.beta * kl_loss

        return total_loss, reconstruction_loss, kl_loss

    def train_step(self, data):
        x, y = data

        with tf.GradientTape() as tape:
            total_loss, recon_loss, kl_loss = self.compute_losses(
                x, y, training=True
            )

        gradients = tape.gradient(
            total_loss, self.trainable_weights
        )

        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_weights)
        )

        self.loss_tracker.update_state(total_loss)
        self.recon_tracker.update_state(recon_loss)
        self.kl_tracker.update_state(kl_loss)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data

        total_loss, recon_loss, kl_loss = self.compute_losses(
            x, y, training=False
        )

        self.loss_tracker.update_state(total_loss)
        self.recon_tracker.update_state(recon_loss)
        self.kl_tracker.update_state(kl_loss)

        return {m.name: m.result() for m in self.metrics}


vae = NarrativeCauseVAE(encoder, decoder, beta=0.01)

vae.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    )
)

print("VAE initialized successfully.")
print("Latent dimension:", LATENT_DIM)
print("KL weight:", vae.beta)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("ntsb_two_columns_clean.csv")

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Total records:", len(df))
print("Training records:", len(train_df))
print("Testing records:", len(test_df))

In [ ]:
for name in ["glove", "glove_batch", "encoder", "decoder", "vae"]:
    print(name, "Available" if name in globals() else "MISSING")

In [ ]:
%pip install -q gensim

import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

print("GloVe loaded successfully")
print("Vocabulary size:", len(glove))
print("Embedding dimensions:", glove.vector_size)

In [ ]:
import numpy as np
import re

def tokenize(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

def normalize_for_glove(word):
    if word.endswith("'s") and len(word) > 2:
        return word[:-2]
    return word

def text_to_glove(text, max_length):
    tokens = [
        normalize_for_glove(w)
        for w in tokenize(text)
    ]

    matrix = np.zeros(
        (max_length, glove.vector_size),
        dtype=np.float32
    )

    for i, word in enumerate(tokens[:max_length]):
        if word in glove:
            matrix[i] = glove[word]

    return matrix

def glove_batch(texts, max_length):
    return np.stack([
        text_to_glove(text, max_length)
        for text in texts
    ])

print("Preprocessing functions restored.")

In [ ]:
variables = [
    "train_df",
    "test_df",
    "glove",
    "glove_batch",
    "encoder",
    "decoder",
    "vae"
]

for name in variables:
    print(
        name,
        "READY" if name in globals() else "MISSING"
    )

print("\nGPU:", __import__("tensorflow").config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split

# Reproduce the same validation split
model_train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

model_train_df = model_train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Model training:", len(model_train_df))
print("Validation:", len(val_df))
print("Held-out testing:", len(test_df))

BATCH_SIZE = 32

def make_dataset(dataframe, batch_size=32, shuffle=False):

    def generator():
        # Shuffle record indices at the start of each training epoch
        indices = np.arange(len(dataframe))

        if shuffle:
            np.random.shuffle(indices)

        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start:start + batch_size]
            batch = dataframe.iloc[batch_indices]

            x = glove_batch(
                batch["analysisNarrative"],
                1200
            )

            y = glove_batch(
                batch["probableCause"],
                200
            )

            yield (
                x.astype(np.float32),
                y.astype(np.float32)
            )

    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(
                shape=(None, 1200, 100),
                dtype=tf.float32
            ),
            tf.TensorSpec(
                shape=(None, 200, 100),
                dtype=tf.float32
            )
        )
    ).prefetch(tf.data.AUTOTUNE)


train_dataset = make_dataset(
    model_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = make_dataset(
    val_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Verify one batch
for x_batch, y_batch in train_dataset.take(1):
    print("\nInput batch:", x_batch.shape)
    print("Target batch:", y_batch.shape)
    print("Input contains NaN:", bool(tf.reduce_any(tf.math.is_nan(x_batch))))
    print("Target contains NaN:", bool(tf.reduce_any(tf.math.is_nan(y_batch))))

In [ ]:
import tensorflow as tf
import numpy as np

# Get one batch from the training dataset
x_batch, y_batch = next(iter(train_dataset))

# Check encoder outputs
z_mean, z_log_var = vae.encoder(
    x_batch, training=False
)

print("Latent mean shape:", z_mean.shape)
print("Latent log variance shape:", z_log_var.shape)

# Check decoder output
predictions = vae.decoder(
    z_mean, training=False
)

print("Predicted target shape:", predictions.shape)
print("Expected target shape:", y_batch.shape)

# Calculate losses without updating model weights
total_loss, recon_loss, kl_loss = vae.compute_losses(
    x_batch,
    y_batch,
    training=False
)

print("\nInitial losses:")
print("Total loss:", float(total_loss.numpy()))
print("Reconstruction loss:", float(recon_loss.numpy()))
print("KL divergence:", float(kl_loss.numpy()))

# Verify all losses are finite
losses = [total_loss, recon_loss, kl_loss]

print(
    "\nAll losses finite:",
    all(bool(tf.reduce_all(tf.math.is_finite(v))) for v in losses)
)

# Verify output shape
assert predictions.shape == y_batch.shape, "Output shape mismatch"

print("Forward-pass test completed successfully.")

In [ ]:
import tensorflow as tf

# Save initial model weights so we can restore them later
initial_encoder_weights = encoder.get_weights()
initial_decoder_weights = decoder.get_weights()

# Short training test
history_test = vae.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3,
    verbose=1
)

print("\nTraining test completed.")

for i in range(len(history_test.history["loss"])):
    print(
        f"Epoch {i+1}: "
        f"Train loss = {history_test.history['loss'][i]:.4f}, "
        f"Validation loss = {history_test.history['val_loss'][i]:.4f}"
    )

In [ ]:
import numpy as np
import tensorflow as tf

# Use validation records, not test records
texts = val_df["analysisNarrative"].iloc[:16]

x = glove_batch(texts, 1200)

# Deterministic predictions using latent means
z_mean, z_log_var = encoder.predict(x, verbose=0)
predictions = decoder.predict(z_mean, verbose=0)

# Compare latent vectors across different narratives
latent_variation = np.mean(np.std(z_mean, axis=0))

# Compare predictions across different narratives
prediction_variation = np.mean(
    np.std(predictions, axis=0)
)

# Compare two predicted sequences using cosine similarity
a = predictions[0].flatten()
b = predictions[1].flatten()

prediction_similarity = np.dot(a, b) / (
    np.linalg.norm(a) * np.linalg.norm(b) + 1e-8
)

print("Mean latent variation:", latent_variation)
print("Mean prediction variation:", prediction_variation)
print("Similarity between two predictions:", prediction_similarity)

print("KL divergence at epoch 3:",
      history_test.history["kl_loss"][-1])

In [ ]:
import numpy as np

# Predictions from real narratives
real_predictions = decoder.predict(z_mean, verbose=0)

# Predictions from zero latent vectors
zero_latent = np.zeros_like(z_mean)
zero_predictions = decoder.predict(zero_latent, verbose=0)

# Compare the two sets of predictions
differences = np.abs(real_predictions - zero_predictions)

print("Mean absolute difference:", differences.mean())
print("Maximum absolute difference:", differences.max())

# Check the magnitude of the actual latent vectors
print("Mean absolute latent value:", np.abs(z_mean).mean())

In [ ]:
import os

os.makedirs("/content/vae_reproduction", exist_ok=True)

encoder.save_weights(
    "/content/vae_reproduction/experimental_encoder.weights.h5"
)

decoder.save_weights(
    "/content/vae_reproduction/experimental_decoder.weights.h5"
)

print("Experimental model weights saved.")
print("Encoder parameters:", encoder.count_params())
print("Decoder parameters:", decoder.count_params())
print(
    "Total parameters:",
    encoder.count_params() + decoder.count_params()
)

In [ ]:
encoder_params = [
    9472, 819328, 409728, 131136,
    36928, 16448, 8256, 16448,
    16448, 8256, 4160, 1300
]

decoder_params = [
    1344, 16448, 16448, 16448,
    16448, 20544, 16448, 16448,
    16448, 12352, 8256, 4160, 65
]

print("Encoder parameters:", sum(encoder_params))
print("Decoder parameters:", sum(decoder_params))
print("Total parameters:", sum(encoder_params) + sum(decoder_params))

assert sum(encoder_params) == 1477908
assert sum(decoder_params) == 161857

print("Architecture parameter arithmetic verified.")

In [ ]:
import math

# Encoder layers from the submitted manuscript
lengths = [1200, 134, 17, 4, 1, 1, 1, 1, 1, 1, 1, 1]
kernels = [36, 25, 25, 16, 9, 4, 2, 4, 4, 2, 1]

print("Layer | Input | Kernel | Output | Possible VALID strides")

for i, kernel in enumerate(kernels):
    input_length = lengths[i]
    target_length = lengths[i + 1]

    possible = [
        s for s in range(1, 100)
        if input_length >= kernel
        and math.floor((input_length - kernel) / s) + 1 == target_length
    ]

    print(
        f"{i+1:5} | "
        f"{input_length:5} | "
        f"{kernel:6} | "
        f"{target_length:6} | "
        f"{possible}"
    )

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

inputs = layers.Input(shape=(1200, 1))

filters = [256, 128, 128, 64, 64, 64, 64, 64, 64, 64, 64]
kernels = [36, 25, 25, 16, 9, 4, 2, 4, 4, 2, 1]
strides = [9, 8, 5, 4, 3, 3, 2, 4, 3, 2, 1]

x = inputs

for i, (f, k, s) in enumerate(zip(filters, kernels, strides)):
    x = layers.Conv1D(
        filters=f,
        kernel_size=k,
        strides=s,
        padding="same",
        activation=layers.LeakyReLU(negative_slope=0.2),
        name=f"original_conv_{i+1}"
    )(x)

    print(f"Layer {i+1}: {x.shape}")

encoder_test = Model(inputs, x)

print("\nConvolutional parameters:", encoder_test.count_params())

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

decoder_input = layers.Input(shape=(20,))

x = layers.Dense(64)(decoder_input)
x = layers.LeakyReLU(negative_slope=0.2)(x)
x = layers.Reshape((1, 64))(x)

filters = [64] * 11 + [1]
kernels = [4, 4, 4, 4, 5, 4, 4, 4, 3, 2, 1, 1]

# Candidate strides based on the reported output lengths
strides = [3, 2, 3, 1, 2, 1, 2, 2, 1, 1, 1, 1]

for i, (f, k, s) in enumerate(zip(filters, kernels, strides)):
    x = layers.Conv1DTranspose(
        filters=f,
        kernel_size=k,
        strides=s,
        padding="valid",
        activation=layers.LeakyReLU(negative_slope=0.2),
        name=f"decoder_conv_{i+1}"
    )(x)

    print(f"Layer {i+1}: {x.shape}")

decoder_test = Model(decoder_input, x)

print("\nDecoder parameters:", decoder_test.count_params())

In [ ]:
# Reported decoder sequence lengths
target_lengths = [
    1, 3, 6, 19, 22, 47, 47,
    96, 194, 197, 199, 200, 200
]

kernels = [4, 4, 4, 4, 5, 4, 4, 4, 3, 2, 1, 1]

print("Layer | Input | Kernel | Target | Possible settings")

for i, kernel in enumerate(kernels):
    length_in = target_lengths[i]
    target = target_lengths[i + 1]

    matches = []

    for stride in range(1, 20):
        for padding in ["valid", "same"]:
            try:
                inp = tf.keras.Input(shape=(length_in, 64))

                out = tf.keras.layers.Conv1DTranspose(
                    filters=64,
                    kernel_size=kernel,
                    strides=stride,
                    padding=padding
                )(inp)

                if out.shape[1] == target:
                    matches.append((stride, padding))

            except (ValueError, TypeError):
                pass

    print(
        f"{i+1:5} | "
        f"{length_in:5} | "
        f"{kernel:6} | "
        f"{target:6} | "
        f"{matches}"
    )

In [ ]:
import tensorflow as tf

remaining = [
    (194, 3, 197),
    (197, 2, 199),
    (199, 1, 200)
]

for length_in, kernel, target in remaining:
    matches = []

    for stride in range(1, 10):
        for padding in ["valid", "same"]:
            for output_padding in range(stride):
                try:
                    inp = tf.keras.Input(shape=(length_in, 64))

                    out = tf.keras.layers.Conv1DTranspose(
                        filters=64,
                        kernel_size=kernel,
                        strides=stride,
                        padding=padding,
                        output_padding=output_padding
                    )(inp)

                    if out.shape[1] == target:
                        matches.append(
                            (stride, padding, output_padding)
                        )

                except (ValueError, TypeError):
                    pass

    print(
        f"{length_in} -> {target}, "
        f"kernel={kernel}: {matches}"
    )

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

inp = layers.Input(shape=(20,))

x = layers.Dense(64)(inp)
x = layers.LeakyReLU(negative_slope=0.2)(x)
x = layers.Reshape((1, 64))(x)

# Candidate configurations for the first eight layers
settings = [
    (4, 3, "same"),
    (4, 2, "same"),
    (4, 3, "valid"),
    (4, 1, "valid"),
    (5, 2, "valid"),
    (4, 1, "same"),
    (4, 2, "valid"),
    (4, 2, "valid")
]

for i, (kernel, stride, padding) in enumerate(settings):
    x = layers.Conv1DTranspose(
        64,
        kernel_size=kernel,
        strides=stride,
        padding=padding,
        activation="linear",
        name=f"decoder_layer_{i+1}"
    )(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)
    print(f"Layer {i+1}:", x.shape)

print("Expected shape after layer 8: (None, 194, 64)")

In [ ]:
# Continue from x after decoder layer 8: (None, 194, 64)

remaining_kernels = [3, 2, 1]

for i, kernel in enumerate(remaining_kernels, start=9):

    # Add positions explicitly
    x = layers.ZeroPadding1D(
        padding=(0, kernel)
    )(x)

    x = layers.Conv1DTranspose(
        filters=64,
        kernel_size=kernel,
        strides=1,
        padding="same",
        activation="linear",
        name=f"decoder_layer_{i}"
    )(x)

    x = layers.LeakyReLU(negative_slope=0.2)(x)

    print(f"Layer {i}:", x.shape)

# Final output layer
x = layers.Conv1DTranspose(
    filters=1,
    kernel_size=1,
    strides=1,
    padding="same",
    name="decoder_output"
)(x)

decoder_reconstructed = Model(inp, x)

print("\nFinal decoder output:", decoder_reconstructed.output_shape)
print("Decoder parameters:", decoder_reconstructed.count_params())

In [ ]:
import os

search_dirs = ["/content", "/content/drive/MyDrive"]

extensions = (
    ".py", ".ipynb", ".npy", ".npz",
    ".h5", ".keras", ".pkl", ".pickle"
)

for folder in search_dirs:
    if not os.path.exists(folder):
        continue

    print(f"\nSearching: {folder}")

    for root, dirs, files in os.walk(folder):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__", "node_modules"}
        ]

        for filename in files:
            if filename.lower().endswith(extensions):
                print(os.path.join(root, filename))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# Use the previously verified convolutional encoder
inputs = encoder_test.input

x = encoder_test.output
x = layers.Flatten()(x)

# One trainable Dense(20) layer: 1,300 parameters
z_mean = layers.Dense(
    20,
    activation=layers.LeakyReLU(negative_slope=0.2),
    name="latent_mean"
)(x)

# Candidate fixed-variance posterior: sigma = 1
# This is an assumption for testing, NOT a recovered original setting.
class FixedVarianceSampling(layers.Layer):
    def call(self, mean):
        epsilon = tf.random.normal(tf.shape(mean))
        return mean + epsilon

z = FixedVarianceSampling(name="latent_sampling")(z_mean)

encoder_candidate = Model(
    inputs,
    [z_mean, z],
    name="candidate_probabilistic_encoder"
)

# Check output dimensions and parameters
sample = tf.random.normal((4, 1200, 1))

mean_values, sampled_values = encoder_candidate(sample)

# With sigma = 1, KL(q || N(0,I)) = 0.5 * sum(mu^2)
kl_per_sample = 0.5 * tf.reduce_sum(
    tf.square(mean_values),
    axis=1
)

print("Latent mean shape:", mean_values.shape)
print("Sampled latent shape:", sampled_values.shape)
print("Encoder parameters:", encoder_candidate.count_params())
print("KL values:", kl_per_sample.numpy())
print("All KL values nonnegative:",
      bool(tf.reduce_all(kl_per_sample >= 0).numpy()))
print("All KL values finite:",
      bool(tf.reduce_all(tf.math.is_finite(kl_per_sample)).numpy()))

In [ ]:
import tensorflow as tf
import numpy as np

# Use one fixed input
x = tf.random.normal((1, 1200, 1))

# Encode the same input twice
mean_1, sample_1 = encoder_candidate(x, training=False)
mean_2, sample_2 = encoder_candidate(x, training=False)

print("Mean difference:",
      np.max(np.abs(mean_1.numpy() - mean_2.numpy())))

print("Sample difference:",
      np.mean(np.abs(sample_1.numpy() - sample_2.numpy())))

print("Sample 1 first five values:",
      sample_1.numpy()[0, :5])

print("Sample 2 first five values:",
      sample_2.numpy()[0, :5])

In [ ]:
import tensorflow as tf

alpha = 1000.0

# Synthetic target and prediction vectors
y_true = tf.random.uniform((4, 200, 1))
y_pred = tf.Variable(tf.random.uniform((4, 200, 1)))

with tf.GradientTape() as tape:

    true_flat = tf.reshape(y_true, (4, 200))
    pred_flat = tf.reshape(y_pred, (4, 200))

    cosine = tf.keras.losses.cosine_similarity(
        true_flat, pred_flat, axis=-1
    ) * -1.0

    # Numerical safeguard for the denominator
    denominator = tf.maximum(1.0 + cosine, 1e-7)

    semantic_loss = tf.reduce_mean(
        alpha / denominator
    )

    # Fixed-variance Gaussian KL term
    kl_loss = tf.reduce_mean(
        0.5 * tf.reduce_sum(
            tf.square(mean_values), axis=1
        )
    )

    total_loss = semantic_loss + kl_loss

gradients = tape.gradient(total_loss, y_pred)

print("Mean cosine similarity:", cosine.numpy().mean())
print("Semantic loss:", semantic_loss.numpy())
print("KL loss:", kl_loss.numpy())
print("Total loss:", total_loss.numpy())

print(
    "Gradients finite:",
    bool(tf.reduce_all(tf.math.is_finite(gradients)).numpy())
)

print(
    "Gradient magnitude:",
    tf.reduce_mean(tf.abs(gradients)).numpy()
)

In [ ]:
import tensorflow as tf
import numpy as np

# Models constructed in our previous steps:
# encoder_candidate: (1200,1) -> [z_mean, z]
# decoder_reconstructed: (20,) -> (200,1)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Synthetic inputs and targets for a computational test only
x_batch = tf.random.uniform((4, 1200, 1))
y_batch = tf.random.uniform((4, 200, 1))

with tf.GradientTape() as tape:

    z_mean, z = encoder_candidate(x_batch, training=True)

    predictions = decoder_reconstructed(z, training=True)

    y_flat = tf.reshape(y_batch, (4, 200))
    p_flat = tf.reshape(predictions, (4, 200))

    cosine = -tf.keras.losses.cosine_similarity(
        y_flat, p_flat, axis=-1
    )

    semantic_loss = tf.reduce_mean(
        1000.0 / tf.maximum(1.0 + cosine, 1e-7)
    )

    kl_loss = tf.reduce_mean(
        0.5 * tf.reduce_sum(tf.square(z_mean), axis=1)
    )

    total_loss = semantic_loss + kl_loss

variables = (
    encoder_candidate.trainable_variables
    + decoder_reconstructed.trainable_variables
)

gradients = tape.gradient(total_loss, variables)

encoder_count = len(encoder_candidate.trainable_variables)

encoder_grads = gradients[:encoder_count]
decoder_grads = gradients[encoder_count:]

def gradient_report(grads):
    available = [g for g in grads if g is not None]

    if not available:
        return "No gradients"

    finite = all(
        bool(tf.reduce_all(tf.math.is_finite(g)).numpy())
        for g in available
    )

    nonzero = sum(
        bool(tf.reduce_any(tf.not_equal(g, 0)).numpy())
        for g in available
    )

    return {
        "total_variables": len(grads),
        "variables_with_gradients": len(available),
        "variables_with_nonzero_gradients": nonzero,
        "all_available_gradients_finite": finite
    }

print("Prediction shape:", predictions.shape)
print("Encoder parameters:", encoder_candidate.count_params())
print("Decoder parameters:", decoder_reconstructed.count_params())
print("Combined parameters:",
      encoder_candidate.count_params()
      + decoder_reconstructed.count_params())

print("Mean cosine similarity:", float(tf.reduce_mean(cosine)))
print("Semantic loss:", float(semantic_loss))
print("KL loss:", float(kl_loss))
print("Total loss:", float(total_loss))

print("\nEncoder gradients:", gradient_report(encoder_grads))
print("Decoder gradients:", gradient_report(decoder_grads))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# Previously loaded GloVe model
# glove_model should contain 100-dimensional embeddings.
glove = globals().get("glove_model", globals().get("glove"))

print("GloVe available:", glove is not None)

if glove is None:
    print("Please tell me the variable name of your loaded GloVe model.")

else:
    print("Embedding dimension:", glove.vector_size)

    # Sample common aviation terms
    words = [
        "aircraft", "pilot", "engine",
        "weather", "runway", "failure",
        "maintenance", "collision"
    ]

    vectors = np.array([
        glove[w] for w in words
    ], dtype=np.float32)

    mean_values = vectors.mean(axis=1)
    norm_values = np.linalg.norm(vectors, axis=1)

    # Exploratory PCA fitted only to these sample words.
    # This is NOT a production PCA model.
    pca = PCA(n_components=1)
    pca_values = pca.fit_transform(vectors).ravel()

    results = pd.DataFrame({
        "word": words,
        "mean_projection": mean_values,
        "vector_norm": norm_values,
        "sample_PCA_projection": pca_values
    })

    print(results.to_string(index=False))

In [ ]:
import tensorflow as tf
import numpy as np

# Use the encoder reconstructed earlier
x_batch = tf.random.normal((4, 1200, 1))

# Obtain the existing 20-dimensional latent representation
z_mean, _ = encoder_candidate(x_batch, training=False)

# Candidate A: fixed log-variance
logvar_fixed = tf.zeros_like(z_mean)

# Candidate B: constrained, input-dependent log-variance
# This is an illustrative assumption, not the original implementation.
logvar_dependent = tf.tanh(z_mean)

def test_posterior(mean, logvar):
    std = tf.exp(0.5 * logvar)
    epsilon = tf.random.normal(tf.shape(mean))
    z = mean + std * epsilon

    kl = -0.5 * tf.reduce_sum(
        1.0 + logvar
        - tf.square(mean)
        - tf.exp(logvar),
        axis=1
    )

    return z, kl

z_fixed, kl_fixed = test_posterior(
    z_mean, logvar_fixed
)

z_dependent, kl_dependent = test_posterior(
    z_mean, logvar_dependent
)

print("Latent mean shape:", z_mean.shape)
print("Fixed-variance sample shape:", z_fixed.shape)
print("Dependent-variance sample shape:", z_dependent.shape)

print("\nFixed-variance KL:", kl_fixed.numpy())
print("Dependent-variance KL:", kl_dependent.numpy())

print("\nVariance values for first sample:")
print(
    "Fixed:",
    tf.exp(logvar_fixed[0, :5]).numpy()
)
print(
    "Dependent:",
    tf.exp(logvar_dependent[0, :5]).numpy()
)

print("\nEncoder parameters:",
      encoder_candidate.count_params())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# Previously loaded GloVe embeddings
glove = globals().get("glove_model", globals().get("glove"))

# Previously prepared dataset
df = pd.read_csv("/content/ntsb_two_columns_clean.csv")

print("Dataset records:", len(df))
print("Columns:", df.columns.tolist())

# Recreate the previously established split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# Fit PCA using vocabulary from training records only
import re

def tokenize(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

train_words = set()

for column in ["analysisNarrative", "probableCause"]:
    for text in train_df[column].fillna(""):
        train_words.update(tokenize(text))

known_words = sorted(
    word for word in train_words if word in glove
)

embedding_matrix = np.stack(
    [glove[word] for word in known_words]
).astype(np.float32)

pca = PCA(n_components=1, random_state=42)
pca.fit(embedding_matrix)

print("\nTraining vocabulary:", len(train_words))
print("GloVe-covered vocabulary:", len(known_words))
print("PCA explained variance ratio:",
      pca.explained_variance_ratio_[0])

# Build a deterministic scalar lookup
projected_values = pca.transform(
    embedding_matrix
).reshape(-1)

scalar_lookup = dict(
    zip(known_words, projected_values)
)

# Inspect selected words
for word in [
    "aircraft", "pilot", "engine",
    "weather", "runway", "failure"
]:
    print(
        word,
        scalar_lookup.get(word, "not in training vocabulary")
    )

In [ ]:
import numpy as np
from scipy.stats import spearmanr

words = [
    "aircraft", "airplane", "pilot", "aviator",
    "engine", "motor", "runway", "airport",
    "weather", "storm", "failure", "malfunction",
    "maintenance", "repair", "collision", "impact"
]

words = [
    w for w in words
    if w in glove and w in scalar_lookup
]

vectors = np.stack([glove[w] for w in words])
scalars = np.array([scalar_lookup[w] for w in words])

# Original GloVe cosine similarity
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
normalized = vectors / np.maximum(norms, 1e-12)
original_similarity = normalized @ normalized.T

# One-dimensional projection:
# compare pairwise distances, not 1D cosine similarity
original_distance = 1.0 - original_similarity
projected_distance = np.abs(
    scalars[:, None] - scalars[None, :]
)

upper = np.triu_indices(len(words), k=1)

rho, p_value = spearmanr(
    original_distance[upper],
    projected_distance[upper]
)

print("Words evaluated:", len(words))
print("Spearman correlation:", rho)
print("p-value:", p_value)

print("\nSelected word pairs:")
pairs = [
    ("aircraft", "airplane"),
    ("pilot", "aviator"),
    ("engine", "motor"),
    ("weather", "storm"),
    ("maintenance", "repair"),
    ("collision", "impact")
]

for a, b in pairs:
    if a in words and b in words:
        i, j = words.index(a), words.index(b)

        print(
            f"{a:12} - {b:12} "
            f"GloVe cosine={original_similarity[i,j]:.3f}, "
            f"PCA distance={projected_distance[i,j]:.3f}"
        )

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

# Fit tokenizer on training data only
texts = (
    train_df["analysisNarrative"].fillna("").astype(str).tolist()
    + train_df["probableCause"].fillna("").astype(str).tolist()
)

tokenizer = Tokenizer(
    num_words=150000,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(texts)

words = [
    "aircraft", "pilot", "engine",
    "weather", "runway", "failure"
]

print("Word | Token ID | Normalized ID | PCA scalar")

vocab_size = len(tokenizer.word_index) + 1

for word in words:
    token_id = tokenizer.word_index.get(word)

    normalized_id = (
        token_id / max(vocab_size - 1, 1)
        if token_id is not None else None
    )

    pca_scalar = scalar_lookup.get(word)

    print(
        word,
        token_id,
        normalized_id,
        pca_scalar
    )

print("\nTokenizer vocabulary size:", vocab_size)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Held-out probable-cause statements
test_causes = (
    test_df["probableCause"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Convert to integer token IDs
sequences = tokenizer.texts_to_sequences(test_causes)

# Pad to the reported output length
Y = pad_sequences(
    sequences,
    maxlen=200,
    padding="post",
    truncating="post"
).astype(np.float32)

# Normalize token IDs to [0, 1]
Y = Y / max(vocab_size - 1, 1)

# Randomly pair different test records
rng = np.random.default_rng(42)
permutation = rng.permutation(len(Y))

# Avoid self-pairs
while np.any(permutation == np.arange(len(Y))):
    permutation = rng.permutation(len(Y))

A = Y
B = Y[permutation]

# Cosine similarity
dot = np.sum(A * B, axis=1)
norm_a = np.linalg.norm(A, axis=1)
norm_b = np.linalg.norm(B, axis=1)

similarity = dot / np.maximum(
    norm_a * norm_b, 1e-12
)

print("Random-pair mean cosine:", similarity.mean())
print("Random-pair SD:", similarity.std())
print("Minimum:", similarity.min())
print("Maximum:", similarity.max())
print("Pairs above 0.5:", np.mean(similarity > 0.5))
print("Pairs above 0.786:", np.mean(similarity > 0.786))

In [ ]:
# Find the five highest-scoring random pairs
top_indices = np.argsort(similarity)[-5:][::-1]

for rank, i in enumerate(top_indices, start=1):
    j = permutation[i]

    print("=" * 70)
    print("PAIR:", rank)
    print("COSINE:", round(float(similarity[i]), 4))

    print("\nStatement A:")
    print(test_causes[i][:700])

    print("\nStatement B:")
    print(test_causes[j][:700])

    print("\nExact text match:",
          test_causes[i].strip().lower()
          == test_causes[j].strip().lower())

    print()

In [ ]:
import numpy as np
import re

def glove_sentence_vector(text):
    words = re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

    vectors = [
        glove[w]
        for w in words
        if w in glove
    ]

    if not vectors:
        return np.zeros(100, dtype=np.float32)

    return np.mean(vectors, axis=0)

def cosine(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)

print("Pair | Token-ID cosine | Mean-GloVe cosine")
print("-" * 55)

for rank, i in enumerate(top_indices, start=1):
    j = permutation[i]

    a = glove_sentence_vector(test_causes[i])
    b = glove_sentence_vector(test_causes[j])

    semantic_similarity = cosine(a, b)

    print(
        f"{rank:4d} | "
        f"{similarity[i]:15.4f} | "
        f"{semantic_similarity:17.4f}"
    )

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def mean_glove(text, remove_stopwords=False):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

    if remove_stopwords:
        words = [
            w for w in words
            if w not in ENGLISH_STOP_WORDS
        ]

    vectors = [glove[w] for w in words if w in glove]

    if not vectors:
        return np.zeros(glove.vector_size, dtype=np.float32)

    return np.mean(vectors, axis=0)

print("Pair | Original GloVe | Without stopwords")
print("-" * 48)

for rank, i in enumerate(top_indices, start=1):
    j = permutation[i]

    original = cosine(
        mean_glove(test_causes[i]),
        mean_glove(test_causes[j])
    )

    filtered = cosine(
        mean_glove(test_causes[i], True),
        mean_glove(test_causes[j], True)
    )

    print(f"{rank:4d} | {original:14.4f} | {filtered:17.4f}")

In [ ]:

import os

root = "/content/drive/MyDrive"

keywords = [
    "checkpoint", "roberta", "transformer",
    "vae", "epoch", "model", "weights"
]

extensions = (".pt", ".pth", ".bin", ".safetensors")

matches = []

for folder, dirs, files in os.walk(root):
    for filename in files:
        name = filename.lower()

        if name.endswith(extensions) and any(
            word in name for word in keywords
        ):
            path = os.path.join(folder, filename)
            size_mb = os.path.getsize(path) / (1024 ** 2)
            matches.append((path, size_mb))

print("Model files found:", len(matches))

for path, size in matches:
    print(f"{size:.2f} MB | {path}")


In [ ]:

import pandas as pd
import os

print("=== COMPUTING ENVIRONMENT ===")
print("GPU required: NO")

print("\n=== AVAILABLE DATAFRAMES ===")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\nDataFrame: {name}")
        print(f"Shape: {obj.shape}")
        print(f"Columns: {list(obj.columns)}")

        for col in obj.columns:
            if "narrative" in col.lower() or "cause" in col.lower():
                print(
                    f"{col}: "
                    f"{obj[col].notna().sum()} non-null values"
                )

print("\n=== WORKING DIRECTORY ===")
print(os.getcwd())


In [ ]:

import os

print("=== GOOGLE DRIVE STATUS ===")
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

print("\n=== FILES IN /content ===")
for item in os.listdir("/content"):
    print(item)

print("\n=== POSSIBLE DATASET FILES ===")

search_dirs = [
    "/content",
    "/content/drive/MyDrive"
]

extensions = (".csv", ".xlsx", ".parquet", ".pkl", ".json")

for folder in search_dirs:
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
            ]

            for filename in files:
                if filename.lower().endswith(extensions):
                    print(os.path.join(root, filename))


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

import os

print("=== SEARCHING FOR NTSB DATASET ===")

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    dirs[:] = [d for d in dirs if not d.startswith('.')]

    for filename in files:
        if 'ntsb_dataset_2000_2020' in filename.lower():
            full_path = os.path.join(root, filename)
            print("FOUND:", full_path)


In [ ]:

import pandas as pd

# Load the original dataset
dataset_path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

df = pd.read_csv(dataset_path, low_memory=False)

print("=== DATASET OVERVIEW ===")
print("Total records:", len(df))
print("\nColumns:")
print(df.columns.tolist())

# Find the relevant columns, ignoring capitalization
columns = {c.lower(): c for c in df.columns}

narrative_col = columns.get("analysisnarrative")
cause_col = columns.get("probablecause")

print("\n=== TARGET FIELD AVAILABILITY ===")

if narrative_col and cause_col:
    narrative = df[narrative_col].fillna("").astype(str).str.strip()
    cause = df[cause_col].fillna("").astype(str).str.strip()

    valid_narrative = narrative.ne("")
    valid_cause = cause.ne("")

    print("Records with analysisNarrative:", valid_narrative.sum())
    print("Records with probableCause:", valid_cause.sum())
    print("Records with BOTH:", (valid_narrative & valid_cause).sum())
    print("Narrative present, cause missing:",
          (valid_narrative & ~valid_cause).sum())
    print("Both fields missing:",
          (~valid_narrative & ~valid_cause).sum())
else:
    print("Exact column names not found.")
    print("We will inspect the available column names.")


In [ ]:

import pandas as pd

dataset_path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

try:
    df = pd.read_csv(
        dataset_path,
        sep="\t",
        encoding="utf-8-sig",
        low_memory=False
    )

    print("=== DATASET LOADED SUCCESSFULLY ===")
    print("Total records:", len(df))
    print("Number of columns:", len(df.columns))
    print("Column names:", df.columns.tolist())

    print("\n=== DATA AVAILABILITY ===")

    narrative = df["analysisNarrative"].fillna("").astype(str).str.strip()
    cause = df["probableCause"].fillna("").astype(str).str.strip()

    has_narrative = narrative.ne("")
    has_cause = cause.ne("")

    print("Records with analysisNarrative:", has_narrative.sum())
    print("Records with probableCause:", has_cause.sum())
    print("Records with BOTH:", (has_narrative & has_cause).sum())
    print("Narrative present, cause missing:",
          (has_narrative & ~has_cause).sum())

except Exception as e:
    print("Loading error:", type(e).__name__, str(e)[:500])


In [ ]:

import os

path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

print("=== FILE INFORMATION ===")
print("Size (MB):", round(os.path.getsize(path) / (1024**2), 2))

print("\n=== FIRST 5 PHYSICAL LINES ===")

with open(path, "rb") as f:
    for i in range(5):
        line = f.readline()
        print(f"\nLine {i+1}:")
        print("Length in bytes:", len(line))
        print("Tab count:", line.count(b"\t"))
        print("Comma count:", line.count(b","))
        print("First 250 characters:", repr(line[:250]))

print("\n=== HEADER ===")
with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
    print(repr(f.readline()[:500]))


In [ ]:

import pandas as pd
import csv

path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

try:
    df = pd.read_csv(
        path,
        sep="\t",
        engine="python",
        quoting=csv.QUOTE_NONE,
        encoding="utf-8-sig",
        on_bad_lines="error"
    )

    print("=== DATASET LOADED ===")
    print("Total records:", len(df))
    print("Columns:", df.columns.tolist())

    narrative = df["analysisNarrative"].fillna("").astype(str).str.strip()
    cause = df["probableCause"].fillna("").astype(str).str.strip()

    has_narrative = narrative.ne("")
    has_cause = cause.ne("")

    print("\n=== DATA AVAILABILITY ===")
    print("Narrative present:", has_narrative.sum())
    print("Probable cause present:", has_cause.sum())
    print("Both present:", (has_narrative & has_cause).sum())
    print("Narrative present, cause missing:",
          (has_narrative & ~has_cause).sum())

except Exception as e:
    print("Loading error:", type(e).__name__, str(e)[:500])


In [ ]:

from google.colab import files

uploaded = files.upload()

print("\n=== UPLOADED FILES ===")
for filename in uploaded:
    print(filename)


In [ ]:

import os
import pandas as pd
import csv

filenames = list(uploaded.keys())

print("=== FILE INSPECTION ===")

for filename in filenames:
    path = os.path.join("/content", filename)

    print("\nFile:", filename)
    print("Size (MB):", round(os.path.getsize(path) / 1024**2, 2))

    try:
        if filename.lower().endswith((".xlsx", ".xls")):
            sample = pd.read_excel(path, nrows=3)
        else:
            sample = pd.read_csv(
                path,
                sep=None,
                engine="python",
                nrows=3,
                encoding="utf-8-sig"
            )

        print("Columns:", sample.columns.tolist())
        print("Sample rows:", len(sample))

    except Exception as e:
        print("Inspection error:", str(e)[:300])


In [ ]:

import pandas as pd
import os

filenames = list(uploaded.keys())
dataframes = []

print("=== LOADING FILES ===")

for filename in filenames:
    path = os.path.join("/content", filename)

    temp = pd.read_csv(
        path,
        encoding="utf-8-sig",
        low_memory=False
    )

    temp["source_file"] = filename
    dataframes.append(temp)

    print(f"{filename}: {len(temp):,} rows")

# Combine without removing any records
combined_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\n=== COMBINED DATASET ===")
print("Total rows:", f"{len(combined_df):,}")
print("Total columns:", len(combined_df.columns))

# Check investigation identifiers
ids = combined_df["NtsbNo"].astype("string").str.strip()
valid_ids = ids.notna() & ids.ne("")

print("\n=== INVESTIGATION IDENTIFIERS ===")
print("Rows with valid NtsbNo:", int(valid_ids.sum()))
print("Unique NtsbNo:", ids[valid_ids].nunique())
print("Repeated identifier rows:",
      int(ids[valid_ids].duplicated().sum()))

# Check probable-cause availability
cause = combined_df["ProbableCause"].fillna("").astype(str).str.strip()
has_cause = cause.ne("")

print("\n=== PROBABLE CAUSE ===")
print("Records with probable cause:", int(has_cause.sum()))
print("Records without probable cause:", int((~has_cause).sum()))

print("\n=== EVENT TYPES ===")
print(combined_df["EventType"].value_counts(dropna=False).head(10))

print("\n=== REPORT STATUS ===")
print(combined_df["ReportStatus"].value_counts(dropna=False).head(10))


In [ ]:

import csv
import os

filename = "carol_investigations_summary (3).csv"
path = os.path.join("/content", filename)

print("=== FILE DIAGNOSTIC ===")

with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
    lines = f.readlines()

print("Physical lines:", len(lines))

for line_number in [1, 189, 190, 191]:
    if line_number <= len(lines):
        line = lines[line_number - 1]

        print(f"\nLine {line_number}")
        print("Length:", len(line))
        print("Commas:", line.count(","))
        print("Tabs:", line.count("\t"))
        print("Beginning:", repr(line[:180]))
        print("End:", repr(line[-180:]))

print("\n=== CSV PARSER TEST ===")

with open(path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader, start=1):
        if i == 1:
            print("Header fields:", len(row))
        if 185 <= i <= 195:
            print(f"CSV record {i}: {len(row)} fields")
        if i > 195:
            break


In [ ]:

import csv

path = "/content/carol_investigations_summary (3).csv"

with open(path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, strict=False)
    header = next(reader)

    for record_number, row in enumerate(reader, start=2):
        if len(row) != len(header):
            print("=== FIRST MALFORMED RECORD ===")
            print("CSV record:", record_number)
            print("Physical line at end:", reader.line_num)
            print("Expected fields:", len(header))
            print("Actual fields:", len(row))

            print("\n=== FIRST 15 FIELDS ===")
            for i, value in enumerate(row[:15]):
                print(i, header[i] if i < len(header) else "EXTRA",
                      ":", repr(value[:100]))

            print("\n=== LAST 8 FIELDS ===")
            for i in range(max(0, len(row)-8), len(row)):
                print(i, ":", repr(row[i][:200]))

            break


In [ ]:

import csv
import os
from collections import Counter

filenames = list(uploaded.keys())

print("=== CSV STRUCTURE AUDIT ===")

for filename in filenames:
    path = os.path.join("/content", filename)

    counts = Counter()
    safe_trailing = 0
    other_malformed = 0
    examples = []

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        expected = len(header)

        for record_no, row in enumerate(reader, start=2):
            counts[len(row)] += 1

            if len(row) > expected:
                extra = row[expected:]

                if all(not x.strip() for x in extra):
                    safe_trailing += 1
                else:
                    other_malformed += 1
                    if len(examples) < 3:
                        examples.append(
                            (record_no, len(row), repr(extra)[:150])
                        )

            elif len(row) < expected:
                other_malformed += 1
                if len(examples) < 3:
                    examples.append(
                        (record_no, len(row), "Missing fields")
                    )

    print(f"\n{filename}")
    print("Expected columns:", expected)
    print("Record count:", sum(counts.values()))
    print("Field counts:", dict(counts))
    print("Trailing-empty-field cases:", safe_trailing)
    print("Other malformed records:", other_malformed)

    if examples:
        print("Examples:", examples)


In [ ]:

import csv
import os

problem_files = [
    "carol_investigations_summary (3).csv",
    "carol_investigations_summary (2).csv"
]

for filename in problem_files:
    path = os.path.join("/content", filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        expected = len(header)

        for record_no, row in enumerate(reader, start=2):
            if len(row) > expected and any(
                str(x).strip() for x in row[expected:]
            ):
                print("\n" + "=" * 60)
                print("FILE:", filename)
                print("RECORD:", record_no)
                print("NTSB ID:", row[0])
                print("FIELDS:", len(row), "EXPECTED:", expected)

                print("\nLAST SIX FIELDS:")
                for i in range(max(0, len(row) - 6), len(row)):
                    print(f"Field {i}: {repr(row[i][:250])}")


In [ ]:

import csv
import os

targets = {
    "ERA18FA167",
    "WPR17FA066",
    "ERA14FA120",
    "WPR13FA073"
}

files_to_check = [
    "carol_investigations_summary (3).csv",
    "carol_investigations_summary (2).csv"
]

for filename in files_to_check:
    path = os.path.join("/content", filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if row and row[0] in targets:
                print("\n" + "=" * 55)
                print("NTSB ID:", row[0])
                print("File:", filename)

                print("\nFIELDS 95–119:")
                for i in range(95, len(row)):
                    expected_name = (
                        header[i] if i < len(header)
                        else "EXTRA FIELD"
                    )
                    value = row[i].replace("\n", " ").strip()
                    print(f"{i:3} | {expected_name:30} | {value[:65]}")


In [ ]:

import csv
import os

targets = {
    "ERA18FA167",
    "WPR17FA066",
    "ERA14FA120",
    "WPR13FA073"
}

for filename in [
    "carol_investigations_summary (3).csv",
    "carol_investigations_summary (2).csv"
]:
    path = os.path.join("/content", filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if row and row[0] in targets:
                print("\n" + "=" * 55)
                print("NTSB ID:", row[0])

                for i in range(0, 55):
                    value = row[i].replace("\n", " ").strip()
                    print(
                        f"{i:3} | {header[i]:28} | {value[:65]}"
                    )


In [ ]:

import csv
import os

targets = {
    "ERA18FA167",
    "WPR17FA066",
    "ERA14FA120",
    "WPR13FA073"
}

for filename in [
    "carol_investigations_summary (3).csv",
    "carol_investigations_summary (2).csv"
]:
    path = os.path.join("/content", filename)

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if row and row[0] in targets:
                print("\n" + "=" * 50)
                print("NTSB ID:", row[0])

                for i in range(30, 53):
                    print(
                        f"{i:2} | {header[i]:25} | "
                        f"{row[i][:100]}"
                    )


In [ ]:

import csv
import os
import pandas as pd
from collections import Counter

filenames = list(uploaded.keys())

# Verified manufacturer repairs for four investigations
manufacturer_repairs = {
    "ERA18FA167": "Continental Motors, Inc.; Continental Motors, Inc.",
    "WPR17FA066": "Continental Motors, Inc.; Continental Motors, Inc.",
    "ERA14FA120": "Continental Motors, Inc.",
    "WPR13FA073": "National Flight Service, Inc.; National Flight Service, Inc."
}

all_records = []
repair_log = []
file_counts = {}

for filename in filenames:
    path = os.path.join("/content", filename)
    count = 0

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        expected = len(header)

        assert expected == 118, f"Unexpected columns in {filename}"

        for record_no, row in enumerate(reader, start=2):
            original_length = len(row)
            ntsb_id = row[0].strip() if row else ""

            # Four verified manufacturer-name repairs
            if ntsb_id in manufacturer_repairs and len(row) == 120:
                row = (
                    row[:40]
                    + [manufacturer_repairs[ntsb_id]]
                    + row[43:]
                )
                repair_type = "manufacturer_field_reconstructed"

            # Remove only extra empty fields at the end
            elif len(row) > expected and all(
                not str(x).strip() for x in row[expected:]
            ):
                row = row[:expected]
                repair_type = "trailing_empty_fields_removed"

            elif len(row) == expected:
                repair_type = None

            else:
                raise ValueError(
                    f"Unresolved record: {filename}, "
                    f"record {record_no}, ID {ntsb_id}, "
                    f"fields {len(row)}"
                )

            if len(row) != expected:
                raise ValueError(
                    f"Repair failed: {filename}, ID {ntsb_id}"
                )

            record = dict(zip(header, row))
            record["source_file"] = filename
            all_records.append(record)
            count += 1

            if repair_type:
                repair_log.append({
                    "source_file": filename,
                    "record_number": record_no,
                    "NtsbNo": ntsb_id,
                    "original_fields": original_length,
                    "repair": repair_type
                })

    file_counts[filename] = count

combined_df = pd.DataFrame(all_records)

print("=== FILE COUNTS ===")
for name, count in file_counts.items():
    print(f"{name}: {count:,}")

print("\n=== COMBINED DATA ===")
print("Rows:", f"{len(combined_df):,}")
print("Columns:", len(combined_df.columns))
print("Repairs applied:", len(repair_log))

print("\n=== IDENTIFIER AUDIT ===")
ids = combined_df["NtsbNo"].astype("string").str.strip()
valid_ids = ids.notna() & ids.ne("")

print("Valid IDs:", int(valid_ids.sum()))
print("Unique IDs:", ids[valid_ids].nunique())
print("Repeated-ID rows:", int(ids[valid_ids].duplicated().sum()))

print("\n=== PROBABLE CAUSE ===")
cause = combined_df["ProbableCause"].fillna("").astype(str).str.strip()
print("Present:", int(cause.ne("").sum()))
print("Missing:", int(cause.eq("").sum()))

print("\n=== REPAIR SUMMARY ===")
print(pd.DataFrame(repair_log)["repair"].value_counts())


In [ ]:

import pandas as pd

print("=== FINAL DATA INTEGRITY CHECK ===")

# Verify identifiers
assert len(combined_df) == 37843
assert combined_df["NtsbNo"].nunique() == 37843

# Check categorical fields for possible shifts
for col in ["Mode", "EventType", "ReportStatus"]:
    print(f"\n{col}:")
    print(combined_df[col].value_counts(dropna=False).head(10))

# Inspect probable-cause text lengths
cause = combined_df["ProbableCause"].fillna("").astype(str).str.strip()
nonempty = cause[cause.ne("")]

print("\n=== PROBABLE CAUSE LENGTH ===")
print("Nonempty records:", len(nonempty))
print("Median words:", nonempty.str.split().str.len().median())
print("Maximum words:", nonempty.str.split().str.len().max())

# Check suspicious probable-cause values
suspicious = nonempty[
    nonempty.str.fullmatch(r"\d+|True|False", case=False)
]

print("\nSuspicious numeric/boolean causes:", len(suspicious))
print("Sample suspicious IDs:",
      combined_df.loc[suspicious.index, "NtsbNo"].head(10).tolist())

# Check source files
print("\n=== SOURCE FILE DISTRIBUTION ===")
print(combined_df["source_file"].value_counts())


In [ ]:

import pandas as pd

dates = pd.to_datetime(combined_df["EventDate"], errors="coerce")
period = dates.dt.year.between(2005, 2020)

cause = combined_df["ProbableCause"].fillna("").astype(str).str.strip()
has_cause = cause.ne("")

print("=== NTSB DATASET AUDIT ===")
print("Total records (2000–2020 export):", len(combined_df))
print("Records dated 2005–2020:", int(period.sum()))
print("With probable cause:", int((period & has_cause).sum()))
print("Without probable cause:", int((period & ~has_cause).sum()))
print("Invalid or missing event dates:", int(dates.isna().sum()))

print("\n=== YEARLY DISTRIBUTION ===")
summary = pd.DataFrame({
    "Year": dates.dt.year,
    "HasCause": has_cause
})

print(
    summary.groupby("Year")["HasCause"]
    .agg(Total="size", WithCause="sum")
    .loc[2005:2020]
    .to_string()
)


In [ ]:

import pandas as pd

df_period = combined_df.loc[
    pd.to_datetime(combined_df["EventDate"], errors="coerce")
    .dt.year.between(2005, 2020)
].copy()

print("=== FILTERING DIAGNOSTIC ===")
print("Initial 2005–2020 records:", len(df_period))

for col in [
    "Mode",
    "EventType",
    "ReportStatus",
    "MostRecentReportType",
    "ReportType",
    "AirCraftCategory"
]:
    print(f"\n=== {col} ===")
    print(df_period[col].value_counts(dropna=False).head(15).to_string())

print("\n=== PROBABLE CAUSE BY REPORT STATUS ===")
df_period["has_cause"] = (
    df_period["ProbableCause"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print(
    df_period.groupby("ReportStatus", dropna=False)["has_cause"]
    .agg(total="size", with_cause="sum")
    .to_string()
)


In [ ]:

valid_statuses = {"Completed", "N/A", ""}

suspicious = df_period.loc[
    ~df_period["ReportStatus"]
    .fillna("")
    .astype(str)
    .str.strip()
    .isin(valid_statuses)
]

print("Suspicious rows:", len(suspicious))

columns = [
    "NtsbNo",
    "EventDate",
    "EventType",
    "ReportStatus",
    "ReportType",
    "ProbableCause",
    "source_file"
]

print(
    suspicious[columns]
    .head(15)
    .to_string(index=False, max_colwidth=45)
)


In [ ]:

import csv
import os

case_id = "ERA19LA228"
filename = "carol_investigations_summary (3).csv"

# Locate original CSV
paths = [
    filename,
    f"/content/{filename}"
]

file_path = next(
    (p for p in paths if os.path.exists(p)),
    None
)

if file_path is None:
    print("Original CSV not found. Please check its location.")
else:
    with open(file_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if row and row[0].strip() == case_id:
                print("Original header fields:", len(header))
                print("Original row fields:", len(row))

                for col in [
                    "EventType",
                    "ReportStatus",
                    "ReportType",
                    "ProbableCause"
                ]:
                    idx = header.index(col)
                    print(f"\n{col}:")
                    print("Original:", repr(row[idx][:180]))
                    print(
                        "Combined:",
                        repr(
                            str(
                                combined_df.loc[
                                    combined_df["NtsbNo"] == case_id,
                                    col
                                ].iloc[0]
                            )[:180]
                        )
                    )
                break
        else:
            print("Case not found in original CSV.")


In [ ]:

import csv

filename = "/content/carol_investigations_summary (3).csv"

with open(filename, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

    for row in reader:
        if row and row[0].strip() == "ERA19LA228":
            print("Header fields:", len(header))
            print("Row fields:", len(row))
            print("Extra fields:", repr(row[len(header):]))
            print("Engine manufacturer:", repr(row[40:44]))
            break


In [ ]:

import pandas as pd

df = combined_df.copy()

df["EventDate"] = pd.to_datetime(df["EventDate"], errors="coerce")
df = df[df["EventDate"].dt.year.between(2005, 2020)].copy()

status = df["ReportStatus"].fillna("").astype(str).str.strip()
mode = df["Mode"].fillna("").astype(str).str.strip()
event = df["EventType"].fillna("").astype(str).str.strip()

completed = status.eq("Completed")
aviation = mode.str.contains("Aviation", case=False, na=False)
accident_incident = event.isin(["Accident", "Incident"])

print("=== SELECTION CRITERIA COMPARISON ===")
print("All 2005–2020 records:", len(df))
print("Completed only:", completed.sum())
print("Aviation only:", aviation.sum())
print("Accidents and incidents only:", accident_incident.sum())
print("Completed + aviation:", (completed & aviation).sum())
print("Completed + accidents/incidents:",
      (completed & accident_incident).sum())
print("Aviation + accidents/incidents:",
      (aviation & accident_incident).sum())
print("All three criteria:",
      (completed & aviation & accident_incident).sum())

print("\n=== MODE VALUES ===")
print(mode.value_counts(dropna=False).to_string())

print("\n=== EVENT TYPE VALUES ===")
print(event.value_counts(dropna=False).head(15).to_string())


In [ ]:

from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)


In [ ]:

import os

file_path = "/content/avall.mdb"

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"Database found: {file_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("Database not found. Check the upload location.")


In [ ]:

import subprocess

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "mdbtools"],
    check=True
)

print("MDB tools installed successfully.")


In [ ]:

import subprocess

result = subprocess.run(
    ["mdb-tables", "-1", "/content/avall.mdb"],
    capture_output=True,
    text=True,
    check=True
)

tables = result.stdout.splitlines()

print("Number of tables:", len(tables))
print("\nRelevant tables:")

for table in tables:
    if any(word in table.lower() for word in [
        "event", "narrative", "finding", "dictionary"
    ]):
        print(table)


In [ ]:

import subprocess
import os

db_path = "/content/avall.mdb"

tables_to_export = ["events", "narratives"]

for table in tables_to_export:
    output_path = f"/content/ntsb_{table}_raw.csv"

    with open(output_path, "wb") as output_file:
        subprocess.run(
            ["mdb-export", db_path, table],
            stdout=output_file,
            check=True
        )

    size_mb = os.path.getsize(output_path) / (1024 ** 2)

    print(f"{table}: exported successfully")
    print(f"File: {output_path}")
    print(f"Size: {size_mb:.2f} MB\n")


In [ ]:
import os

path = "/content/ntsb_narratives_raw.csv"

if os.path.exists(path):
    print("Narratives file exists")
    print("Size:", round(os.path.getsize(path) / 1024**2, 2), "MB")
else:
    print("Narratives file not yet created")

In [ ]:

import pandas as pd

events = pd.read_csv(
    "/content/ntsb_events_raw.csv",
    low_memory=False
)

narratives = pd.read_csv(
    "/content/ntsb_narratives_raw.csv",
    low_memory=False
)

print("EVENTS")
print("Shape:", events.shape)
print("Columns:", events.columns.tolist())

print("\nNARRATIVES")
print("Shape:", narratives.shape)
print("Columns:", narratives.columns.tolist())

print("\nNARRATIVE FIELD AVAILABILITY")

for col in ["narr_accp", "narr_accf", "narr_cause", "narr_inc"]:
    if col in narratives.columns:
        valid = (
            narratives[col].notna()
            & narratives[col].astype(str).str.strip().ne("")
        )
        print(f"{col}: {valid.sum():,} nonempty records")


In [ ]:

import pandas as pd

# Filter events to the original study's available years
events["ev_date"] = pd.to_datetime(
    events["ev_date"], errors="coerce"
)

events_period = events[
    events["ev_date"].dt.year.between(2008, 2020)
].copy()

# Join investigation metadata with aircraft-level narratives
joined = events_period[
    ["ev_id", "ntsb_no", "ev_date", "ev_type"]
].merge(
    narratives[
        ["ev_id", "Aircraft_Key",
         "narr_accp", "narr_accf", "narr_cause"]
    ],
    on="ev_id",
    how="left",
    validate="one_to_many"
)

def has_text(series):
    return series.notna() & series.astype(str).str.strip().ne("")

for field in ["narr_accp", "narr_accf", "narr_cause"]:
    joined[field + "_present"] = has_text(joined[field])

print("Events in period:", len(events_period))
print("Joined rows:", len(joined))
print("Unique investigations:", joined["ev_id"].nunique())

print("\nNarrative availability (joined rows):")
for field in ["narr_accp", "narr_accf", "narr_cause"]:
    print(field, int(joined[field + "_present"].sum()))

print("\nUnique investigations with factual narrative:",
      joined.loc[joined["narr_accp_present"], "ev_id"].nunique())

print("Unique investigations with final narrative:",
      joined.loc[joined["narr_accf_present"], "ev_id"].nunique())

print("Unique investigations with probable cause:",
      joined.loc[joined["narr_cause_present"], "ev_id"].nunique())

print("\nFirst example with all three fields:")
examples = joined[
    joined["narr_accp_present"]
    & joined["narr_accf_present"]
    & joined["narr_cause_present"]
]

if not examples.empty:
    example = examples.iloc[0]
    print("NTSB number:", example["ntsb_no"])

    for field in ["narr_accp", "narr_accf", "narr_cause"]:
        print(f"\n{field}:")
        print(str(example[field])[:1000])
else:
    print("No complete example found.")


In [ ]:

# Investigation-level availability:
# An investigation may have more than one aircraft narrative.

availability = joined.groupby("ev_id").agg(
    has_factual=("narr_accp_present", "any"),
    has_final=("narr_accf_present", "any"),
    has_cause=("narr_cause_present", "any")
)

for field, label in [
    ("has_factual", "Factual narrative"),
    ("has_final", "Final narrative")
]:
    has_input = availability[field]
    has_target = availability["has_cause"]

    print(f"\n{label}")
    print("-" * 45)
    print("Narrative + probable cause:",
          int((has_input & has_target).sum()))
    print("Narrative without probable cause:",
          int((has_input & ~has_target).sum()))
    print("Probable cause without narrative:",
          int((~has_input & has_target).sum()))
    print("Neither:",
          int((~has_input & ~has_target).sum()))


In [ ]:
import os

old_path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

print("Original dataset found:", os.path.exists(old_path))

In [ ]:

import pandas as pd
import csv
import re
import os

old_path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

if not os.path.exists(old_path):
    print("Original dataset not found. Check Google Drive mounting.")
else:
    old_df = pd.read_csv(
        old_path,
        sep="\t",
        engine="python",
        quoting=csv.QUOTE_NONE,
        encoding="utf-8-sig",
        on_bad_lines="error"
    )

    def normalize_text(value):
        if pd.isna(value):
            return ""
        return re.sub(r"\s+", " ", str(value)).strip().lower()

    original_texts = {
        normalize_text(x)
        for x in old_df["analysisNarrative"]
        if normalize_text(x)
    }

    print("Original dataset rows:", len(old_df))
    print("Unique nonempty original narratives:", len(original_texts))

    for field in ["narr_accp", "narr_accf"]:
        current_texts = {
            normalize_text(x)
            for x in joined[field]
            if normalize_text(x)
        }

        matches = original_texts & current_texts

        print(f"\n{field}")
        print("Unique current narratives:", len(current_texts))
        print("Exact normalized matches:", len(matches))
        print(
            "Original narrative match rate:",
            f"{len(matches) / len(original_texts):.2%}"
            if original_texts else "N/A"
        )


In [ ]:

import re

# Distinctive passage from the original narrative
search_phrase = "The instrument rated pilot received three weather briefings"

def normalize_for_search(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()

matches = joined[
    joined["narr_accp"].fillna("").str.contains(
        "three weather briefings", case=False, regex=False
    )
    |
    joined["narr_accf"].fillna("").str.contains(
        "three weather briefings", case=False, regex=False
    )
    |
    joined["narr_cause"].fillna("").str.contains(
        "inadequate in-flight planning", case=False, regex=False
    )
]

print("Matching rows:", len(matches))

if not matches.empty:
    for _, record in matches.head(3).iterrows():
        print("\nNTSB number:", record["ntsb_no"])
        print("Event date:", record["ev_date"])

        for field in ["narr_accp", "narr_accf", "narr_cause"]:
            text = record[field]
            print(f"\n{field}:")
            print(str(text)[:1500] if pd.notna(text) else "[Missing]")
else:
    print("No matching investigation found in 2008–2020.")
    print("The original example may be from an earlier year.")


In [ ]:

print("ORIGINAL ANALYSIS NARRATIVE")
print("-" * 60)

sample = old_df["analysisNarrative"].dropna()
sample = sample[sample.astype(str).str.strip().ne("")]

print(str(sample.iloc[0])[:2500])

print("\nORIGINAL PROBABLE CAUSE")
print("-" * 60)

print(str(old_df.loc[sample.index[0], "probableCause"])[:1000])


In [ ]:

# Search the full narratives table, not just the 2008–2020 subset.

phrase = "inadequate in-flight planning"

matches = narratives[
    narratives["narr_cause"].fillna("").astype(str).str.contains(
        phrase,
        case=False,
        regex=False
    )
].copy()

print("Matching narrative records:", len(matches))

if not matches.empty:
    results = matches.merge(
        events[["ev_id", "ntsb_no", "ev_date"]],
        on="ev_id",
        how="left"
    )

    for _, record in results.head(10).iterrows():
        print("\nNTSB number:", record["ntsb_no"])
        print("Event date:", record["ev_date"])
        print("Probable cause:", str(record["narr_cause"])[:700])
        print("Factual narrative:", str(record["narr_accp"])[:400])
        print("Final narrative:", str(record["narr_accf"])[:400])
else:
    print("No matching probable cause found in this database.")


In [ ]:

# Search the full NTSB narratives table using distinctive
# phrases from the original analysis narrative.

phrases = [
    "Miami Automated Flight Service Station",
    "video integrator and processor",
    "instrument recency of experience",
    "descended from 7,600 to 6,100"
]

mask = pd.Series(False, index=narratives.index)

for field in ["narr_accp", "narr_accf"]:
    text = narratives[field].fillna("").astype(str)

    for phrase in phrases:
        mask |= text.str.contains(
            phrase,
            case=False,
            regex=False
        )

found = narratives.loc[
    mask,
    ["ev_id", "Aircraft_Key", "narr_accp",
     "narr_accf", "narr_cause"]
].merge(
    events[["ev_id", "ntsb_no", "ev_date"]],
    on="ev_id",
    how="left"
)

print("Matching records:", len(found))

for _, record in found.head(10).iterrows():
    print("\n" + "=" * 60)
    print("NTSB number:", record["ntsb_no"])
    print("Event date:", record["ev_date"])

    for field in ["narr_accp", "narr_accf", "narr_cause"]:
        value = record[field]
        print(f"\n{field}:")
        print(str(value)[:1200] if pd.notna(value) else "[Missing]")


In [ ]:

import re

original_cause = str(old_df.loc[sample.index[0], "probableCause"])

def normalize(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()

target = normalize(original_cause)

causes = narratives["narr_cause"].map(normalize)

exact_matches = narratives.loc[causes == target].copy()

partial_matches = narratives.loc[
    causes.map(lambda x: target in x if x else False)
].copy()

print("Original probable cause:")
print(original_cause)

print("\nExact matches:", len(exact_matches))
print("Contained within NTSB cause:", len(partial_matches))

if not partial_matches.empty:
    results = partial_matches.merge(
        events[["ev_id", "ntsb_no", "ev_date"]],
        on="ev_id",
        how="left"
    )

    print(
        results[
            ["ntsb_no", "ev_date", "narr_cause"]
        ].to_string(index=False)
    )


In [ ]:

phrases = [
    "Miami Automated Flight Service Station",
    "crash was located 138 degrees",
    "Between 1631:04",
    "video integrator and processor",
    "instrument recency of experience"
]

for phrase in phrases:
    print(f"\nPhrase: {phrase}")

    for field in ["narr_accp", "narr_accf"]:
        mask = narratives[field].fillna("").astype(str).str.contains(
            phrase, case=False, regex=False
        )

        count = int(mask.sum())
        print(f"  {field}: {count} matches")

        if count:
            matched_ids = narratives.loc[mask, "ev_id"].unique()
            matching_events = events[
                events["ev_id"].isin(matched_ids)
            ]

            print(
                matching_events[
                    ["ntsb_no", "ev_date"]
                ].head(5).to_string(index=False)
            )


In [ ]:

import subprocess
import re

db_path = "/content/avall.mdb"

result = subprocess.run(
    ["mdb-schema", db_path, "postgres"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

if result.returncode != 0:
    print("ERROR:", result.stderr[:2000])
else:
    schema = result.stdout

    print("Schema characters:", len(schema))

    keywords = [
        "analysis", "analys", "narr",
        "cause", "probable", "abstract", "description"
    ]

    current_table = None
    matches = 0

    for line in schema.splitlines():
        table_match = re.search(
            r'CREATE TABLE\s+"?([^"\s(]+)',
            line,
            flags=re.IGNORECASE
        )

        if table_match:
            current_table = table_match.group(1)

        if any(word in line.lower() for word in keywords):
            print(f"{current_table}: {line.strip()}")
            matches += 1

    print("\nMatching schema lines:", matches)


In [ ]:

import glob
import os
import pandas as pd

files = glob.glob("/content/*carol*.csv")

print("CAROL files found:", len(files))

for path in files:
    print("\nFILE:", os.path.basename(path))

    df = pd.read_csv(
        path,
        nrows=3,
        low_memory=False,
        on_bad_lines="skip"
    )

    print("Columns:", len(df.columns))

    relevant = [
        col for col in df.columns
        if any(word in col.lower() for word in [
            "analys", "narr", "cause",
            "abstract", "description", "report"
        ])
    ]

    print("Relevant fields:", relevant)


In [ ]:

import pandas as pd

df = pd.read_csv(
    "/content/carol_investigations_summary.csv",
    low_memory=False,
    on_bad_lines="skip"
)

print("Total records:", len(df))
print("Reports with URLs:", df["ReportUrl"].notna().sum())

print("\nSample report URLs:")
for url in df["ReportUrl"].dropna().head(5):
    print(url)


In [ ]:

import pandas as pd

# Use the events table already extracted from avall.mdb
df = events.copy()

df["ev_date"] = pd.to_datetime(df["ev_date"], errors="coerce")
df["Year"] = df["ev_date"].dt.year

print("Total event records:", len(df))
print("Unique investigations:", df["ev_id"].nunique())
print("Earliest year:", df["Year"].min())
print("Latest year:", df["Year"].max())

print("\nInvestigations by decade:")
df["Decade"] = (df["Year"] // 10) * 10
print(df.groupby("Decade")["ev_id"].nunique().to_string())

print("\nInvestigations before 2000:",
      df.loc[df["Year"] < 2000, "ev_id"].nunique())


In [ ]:

import pandas as pd

# Use the previously extracted Access database tables
e = events.copy()
n = narratives.copy()

e["ev_date"] = pd.to_datetime(e["ev_date"], errors="coerce")

# One record per investigation
e = e.drop_duplicates(subset=["ev_id"])

# Narrative availability per investigation
for col in ["narr_accp", "narr_accf", "narr_cause"]:
    n[col] = n[col].fillna("").astype(str).str.strip()

coverage = n.groupby("ev_id").agg(
    factual=("narr_accp", lambda x: x.ne("").any()),
    final=("narr_accf", lambda x: x.ne("").any()),
    cause=("narr_cause", lambda x: x.ne("").any())
).reset_index()

merged = e.merge(coverage, on="ev_id", how="left")

for col in ["factual", "final", "cause"]:
    merged[col] = merged[col].fillna(False)

merged["Year"] = merged["ev_date"].dt.year
merged["Decade"] = (merged["Year"] // 10) * 10

print("ACCESS DATABASE INVESTIGATION SUMMARY")
print("=" * 55)
print("Total investigations:", len(merged))
print("Earliest event date:", merged["ev_date"].min())
print("Latest event date:", merged["ev_date"].max())
print("With factual narratives:", int(merged["factual"].sum()))
print("With final narratives:", int(merged["final"].sum()))
print("With probable causes:", int(merged["cause"].sum()))

print("\nINVESTIGATIONS BY DECADE")
print(merged.groupby("Decade").agg(
    investigations=("ev_id", "count"),
    factual_narratives=("factual", "sum"),
    final_narratives=("final", "sum"),
    probable_causes=("cause", "sum")
).to_string())


In [ ]:

from google.colab import files
import os

print("Select your FOUR CAROL JSON files.")
print("Please wait until all uploads finish.")

uploaded = files.upload()

print("\nUPLOAD SUMMARY")
print("=" * 55)

for filename in uploaded:
    path = os.path.join("/content", filename)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{filename}: {size_mb:.2f} MB")

print("\nTotal files uploaded:", len(uploaded))


In [ ]:

import json
import glob
import os
import pandas as pd

json_files = sorted(
    glob.glob("/content/carol_investigations_data*.json")
)

all_records = []

print("CAROL JSON DATASET INSPECTION")
print("=" * 65)

for path in json_files:
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        # Handle a JSON object containing a list of records
        records = next(
            (v for v in data.values()
             if isinstance(v, list)),
            []
        )
    else:
        records = []

    print(f"{os.path.basename(path)}: {len(records):,} records")

    for record in records:
        if isinstance(record, dict):
            all_records.append({
                "NTSBNumber": record.get("NTSBNumber"),
                "EventDateTimeUTC": record.get("EventDateTimeUTC"),
                "AnalysisNarrative": record.get("AnalysisNarrative"),
                "ProbableCause": record.get("ProbableCause"),
                "ConcatenatedFactualNarrative":
                    record.get("ConcatenatedFactualNarrative")
            })

df_json = pd.DataFrame(all_records)

print("\nCOMBINED DATASET")
print("=" * 65)
print("Total records:", len(df_json))
print("Unique investigations:", df_json["NTSBNumber"].nunique())
print("Duplicate investigation IDs:",
      df_json["NTSBNumber"].duplicated().sum())

def has_text(series):
    return series.fillna("").astype(str).str.strip().ne("")

analysis = has_text(df_json["AnalysisNarrative"])
cause = has_text(df_json["ProbableCause"])
factual = has_text(df_json["ConcatenatedFactualNarrative"])

print("\nNARRATIVE AVAILABILITY")
print("With analysis narratives:", analysis.sum())
print("With probable causes:", cause.sum())
print("With factual narratives:", factual.sum())
print("With analysis AND probable cause:", (analysis & cause).sum())
print("With analysis but NO probable cause:", (analysis & ~cause).sum())

dates = pd.to_datetime(
    df_json["EventDateTimeUTC"], errors="coerce"
)

print("\nDATE COVERAGE")
print("Earliest event:", dates.min())
print("Latest event:", dates.max())
print("Invalid/missing dates:", dates.isna().sum())

print("\nINVESTIGATIONS BY DECADE")
decades = (dates.dt.year // 10) * 10
print(decades.value_counts().sort_index().to_string())

print("\nInspection complete.")


In [ ]:

import pandas as pd

df = df_json.copy()

df["EventDate"] = pd.to_datetime(
    df["EventDateTimeUTC"], errors="coerce"
)

df["Year"] = df["EventDate"].dt.year

analysis = df["AnalysisNarrative"].fillna("").astype(str).str.strip().ne("")
cause = df["ProbableCause"].fillna("").astype(str).str.strip().ne("")

period = df["Year"].between(2005, 2020)

print("CAROL JSON: 2005–2020 STUDY PERIOD")
print("=" * 60)
print("Total investigations:", int(period.sum()))
print("With analysis:", int((period & analysis).sum()))
print("With probable cause:", int((period & cause).sum()))
print("With BOTH:", int((period & analysis & cause).sum()))
print("Analysis WITHOUT cause:", int((period & analysis & ~cause).sum()))
print("Cause WITHOUT analysis:", int((period & ~analysis & cause).sum()))
print("Neither:", int((period & ~analysis & ~cause).sum()))

print("\nINVESTIGATIONS BY YEAR")
summary = pd.DataFrame({
    "Year": df["Year"],
    "Analysis": analysis,
    "Cause": cause
})

summary = summary[summary["Year"].between(2005, 2020)]

print(summary.groupby("Year").agg(
    Total=("Year", "size"),
    With_Analysis=("Analysis", "sum"),
    With_Cause=("Cause", "sum")
).to_string())

print("\nRECORDS WITH MISSING OR INVALID DATES")
print(df.loc[df["EventDate"].isna(), "NTSBNumber"].head(10).to_string(index=False))


In [ ]:

import pandas as pd
import re
import csv

old_path = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

old = pd.read_csv(
    old_path,
    sep="\t",
    engine="python",
    quoting=csv.QUOTE_NONE,
    encoding="utf-8-sig",
    on_bad_lines="error"
)

def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().lower()

old_analysis = {
    normalize_text(x)
    for x in old["analysisNarrative"]
    if normalize_text(x)
}

new_analysis = {
    normalize_text(x)
    for x in df_json["AnalysisNarrative"]
    if normalize_text(x)
}

matches = old_analysis & new_analysis

print("ORIGINAL DATASET VS CAROL JSON")
print("=" * 55)
print("Original unique analysis narratives:", len(old_analysis))
print("CAROL unique analysis narratives:", len(new_analysis))
print("Exact normalized text matches:", len(matches))
print("Original narratives matched (%):",
      round(100 * len(matches) / len(old_analysis), 2)
      if old_analysis else 0)

print("\nComparison complete.")


In [ ]:

import re

def clean_display(text):
    return re.sub(r"\s+", " ", str(text)).strip()

print("ORIGINAL DATASET — FIRST ANALYSIS")
print("=" * 65)
print(clean_display(old["analysisNarrative"].dropna().iloc[0])[:1200])

print("\n\nCAROL JSON — FIRST ANALYSIS")
print("=" * 65)
print(clean_display(
    df_json["AnalysisNarrative"].dropna().iloc[0]
)[:1200])


In [ ]:

print("ORIGINAL DATASET INSPECTION")
print("=" * 60)

print("Shape:", old.shape)
print("\nColumns:")
print(old.columns.tolist())

print("\nNonempty values:")
for col in old.columns:
    nonempty = old[col].fillna("").astype(str).str.strip().ne("")
    print(f"{col}: {nonempty.sum():,}")

print("\nDuplicate rows:", old.duplicated().sum())

print("\nFirst 3 records (text lengths only):")
for i, row in old.head(3).iterrows():
    print({
        "row": i,
        "analysis_length": len(str(row.get("analysisNarrative", ""))),
        "cause_length": len(str(row.get("probableCause", "")))
    })


In [ ]:

import json
import glob
import pandas as pd

records = []

for path in sorted(glob.glob("/content/carol_investigations_data*.json")):
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    records.extend(data)

df_full = pd.DataFrame(records)

dates = pd.to_datetime(
    df_full["EventDateTimeUTC"], errors="coerce"
)
period = dates.dt.year.between(2005, 2020)

status = (
    df_full["CompletionStatus"]
    .fillna("")
    .astype(str)
    .str.strip()
)

completed = status.str.casefold().eq("completed")

analysis = (
    df_full["AnalysisNarrative"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

cause = (
    df_full["ProbableCause"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print("COMPLETION STATUS VALUES")
print("=" * 60)
print(status.value_counts(dropna=False).to_string())

print("\n2005–2020 DATA CLEANING")
print("=" * 60)
print("All investigations:", int(period.sum()))
print("Completed investigations:", int((period & completed).sum()))
print("Completed with analysis:", int((period & completed & analysis).sum()))
print("Completed with cause:", int((period & completed & cause).sum()))
print("Completed with BOTH:", int((period & completed & analysis & cause).sum()))
print("Completed with analysis but NO cause:",
      int((period & completed & analysis & ~cause).sum()))

print("\nORIGINAL MANUSCRIPT TARGET: 20,919")
print("Difference from completed + analysis:",
      int((period & completed & analysis).sum()) - 20919)
print("Difference from completed + analysis + cause:",
      int((period & completed & analysis & cause).sum()) - 20919)


In [ ]:

import pandas as pd

clean = df_full.loc[
    period & completed & analysis
].copy()

clean["analysis_words"] = (
    clean["AnalysisNarrative"]
    .astype(str)
    .str.split()
    .str.len()
)

clean["cause_words"] = (
    clean["ProbableCause"]
    .astype(str)
    .str.split()
    .str.len()
)

print("COMPLETED INVESTIGATIONS WITH ANALYSIS")
print("=" * 60)
print("Records:", len(clean))
print("Unique NTSB numbers:", clean["NTSBNumber"].nunique())
print("Duplicate analysis narratives:",
      clean["AnalysisNarrative"].duplicated().sum())

print("\nANALYSIS NARRATIVE LENGTH")
print(clean["analysis_words"].describe().to_string())

print("\nCOUNTS BY MINIMUM ANALYSIS LENGTH")
for minimum in [10, 20, 30, 50, 75, 100]:
    count = (clean["analysis_words"] >= minimum).sum()
    print(f"At least {minimum:3d} words: {count:,}")

print("\nPROBABLE CAUSE LENGTH")
print(clean["cause_words"].describe().to_string())

print("\nINVESTIGATIONS BY EVENT TYPE")
print(clean["EventType"].value_counts(dropna=False).to_string())


In [ ]:

print("AVAILABLE DATE-RELATED FIELDS")
print("=" * 60)

date_columns = [
    col for col in df_full.columns
    if any(term in col.lower() for term in [
        "date", "published", "completed"
    ])
]

print(date_columns)

print("\nCOUNTS USING DIFFERENT DATE FIELDS")
print("=" * 60)

for col in date_columns:
    parsed = pd.to_datetime(df_full[col], errors="coerce")
    in_period = parsed.dt.year.between(2005, 2020)

    count = (completed & analysis & in_period).sum()

    print(f"{col}: {count:,} completed records with analysis")


In [ ]:

import pandas as pd

event_date = pd.to_datetime(
    df_full["EventDateTimeUTC"], errors="coerce"
)

published_date = pd.to_datetime(
    df_full["OriginalPublishedDate"], errors="coerce"
)

report_date = pd.to_datetime(
    df_full["ReportDate"], errors="coerce"
)

base = completed & analysis

event_period = event_date.dt.year.between(2005, 2020)
published_period = published_date.dt.year.between(2005, 2020)
report_period = report_date.dt.year.between(2005, 2020)

print("DATE COMBINATION INSPECTION")
print("=" * 60)

conditions = {
    "Event date only": event_period,
    "Publication date only": published_period,
    "Report date only": report_period,
    "Event AND publication": event_period & published_period,
    "Event AND report": event_period & report_period,
    "Publication AND report": published_period & report_period,
    "All three dates": event_period & published_period & report_period
}

for label, condition in conditions.items():
    count = int((base & condition).sum())
    print(f"{label:25s}: {count:,}")

print("\nOriginal manuscript count: 20,919")


In [ ]:

import pandas as pd

# Use completed records with analysis and event dates in 2005–2020
study = df_full.loc[
    completed & analysis & event_period
].copy()

study["analysis_words"] = (
    study["AnalysisNarrative"].fillna("")
    .astype(str).str.split().str.len()
)

study["cause_words"] = (
    study["ProbableCause"].fillna("")
    .astype(str).str.split().str.len()
)

print("DATA QUALITY INSPECTION")
print("=" * 60)
print("Starting records:", len(study))

print("\nMINIMUM CAUSE LENGTH")
for minimum in [2, 5, 10, 15, 20]:
    count = (study["cause_words"] >= minimum).sum()
    print(f"At least {minimum:2d} words: {count:,}")

print("\nCOMMON SHORT PROBABLE CAUSES")
short = study[study["cause_words"] <= 5]
print(short["ProbableCause"].value_counts().head(15).to_string())

print("\nCOMMON SHORT ANALYSIS NARRATIVES")
short_analysis = study[study["analysis_words"] <= 20]
print(short_analysis["AnalysisNarrative"].value_counts().head(10).to_string())


In [ ]:

fields = [
    "HighestInjuryLevel",
    "EventType",
    "AnalysisNarrative",
    "ProbableCause"
]

print("STRUCTURED FIELD AVAILABILITY")
print("=" * 60)
print("Starting records:", len(study))

for field in fields:
    if field in study.columns:
        valid = (
            study[field].fillna("")
            .astype(str).str.strip().ne("")
        )
        print(f"{field}: {valid.sum():,} populated")
    else:
        print(f"{field}: NOT FOUND")

print("\nADDITIONAL RELEVANT FIELDS")
print([
    col for col in study.columns
    if any(term in col.lower() for term in [
        "injury", "damage", "aircraft",
        "operation", "vehicle"
    ])
])


In [ ]:

import pandas as pd

check = study.copy()

check["Published"] = pd.to_datetime(
    check["OriginalPublishedDate"], errors="coerce"
)

print("COMPLETED INVESTIGATIONS BY PUBLICATION YEAR")
print("=" * 60)

print(
    check["Published"].dt.year
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nPUBLICATION DATE AVAILABILITY")
print("Missing publication dates:", check["Published"].isna().sum())

print("\nCUMULATIVE INVESTIGATIONS BY PUBLICATION CUTOFF")
for year in range(2018, 2027):
    count = (check["Published"].dt.year <= year).sum()
    print(f"Published by {year}: {count:,}")

print("\nOriginal reported sample: 20,919")


In [ ]:

import pandas as pd

check = study.copy()

check["Published"] = pd.to_datetime(
    check["OriginalPublishedDate"], errors="coerce"
)

check["ReportDateParsed"] = pd.to_datetime(
    check["ReportDate"], errors="coerce"
)

cutoff = pd.Timestamp("2022-11-30 23:59:59")

print("HISTORICAL CUTOFF: NOVEMBER 2022")
print("=" * 60)

published_by_cutoff = check["Published"].le(cutoff)
reported_by_cutoff = check["ReportDateParsed"].le(cutoff)

print("Original published by cutoff:",
      int(published_by_cutoff.sum()))

print("Report dated by cutoff:",
      int(reported_by_cutoff.sum()))

print("Both dates by cutoff:",
      int((published_by_cutoff & reported_by_cutoff).sum()))

print("\nOriginal manuscript:", 20919)

print("\nDIFFERENCES")
print("Published cutoff difference:",
      int(published_by_cutoff.sum()) - 20919)

print("Report cutoff difference:",
      int(reported_by_cutoff.sum()) - 20919)

print("\nPUBLICATIONS DURING NOVEMBER 2022")
november = check["Published"].between(
    "2022-11-01", "2022-11-30 23:59:59"
)
print(int(november.sum()))


In [ ]:

import pandas as pd

check = df_full.copy()

event = pd.to_datetime(
    check["EventDateTimeUTC"], errors="coerce"
)
report = pd.to_datetime(
    check["ReportDate"], errors="coerce"
)

status_ok = (
    check["CompletionStatus"].fillna("")
    .astype(str).str.strip().str.casefold().eq("completed")
)

analysis_ok = (
    check["AnalysisNarrative"].fillna("")
    .astype(str).str.strip().ne("")
)

cause_ok = (
    check["ProbableCause"].fillna("")
    .astype(str).str.strip().ne("")
)

base = status_ok & analysis_ok & cause_ok

event_period = event.dt.year.between(2005, 2020)
report_period = report.dt.year.between(2005, 2020)
cutoff = report.le(pd.Timestamp("2022-11-30 23:59:59"))

print("EVENT DATE VS REPORT DATE")
print("=" * 60)

print("Event 2005–2020:",
      int((base & event_period).sum()))

print("Report 2005–2020:",
      int((base & report_period).sum()))

print("Both event and report 2005–2020:",
      int((base & event_period & report_period).sum()))

print("Event 2005–2020, report by Nov 2022:",
      int((base & event_period & cutoff).sum()))

print("\nOriginal reported count: 20,919")


In [ ]:

import pandas as pd

dataset = study[
    ["NTSBNumber", "AnalysisNarrative", "ProbableCause"]
].copy()

dataset = dataset.rename(columns={
    "NTSBNumber": "investigation_id",
    "AnalysisNarrative": "input_text",
    "ProbableCause": "target_text"
})

for col in ["input_text", "target_text"]:
    dataset[col] = (
        dataset[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

dataset = dataset[
    dataset["input_text"].ne("") &
    dataset["target_text"].ne("")
].copy()

dataset = dataset.drop_duplicates(
    subset=["investigation_id"]
).reset_index(drop=True)

print("RECONSTRUCTED EXPERIMENTAL DATASET")
print("=" * 55)
print("Total investigations:", len(dataset))
print("Unique investigation IDs:",
      dataset["investigation_id"].nunique())
print("Missing inputs:",
      dataset["input_text"].eq("").sum())
print("Missing targets:",
      dataset["target_text"].eq("").sum())

print("\nEXAMPLE")
print("Investigation:", dataset.loc[0, "investigation_id"])
print("Analysis:", dataset.loc[0, "input_text"][:350])
print("Probable cause:", dataset.loc[0, "target_text"][:350])


In [ ]:

# Reproducible sample-size-matched reconstruction
# Original manuscript sample size: 20,919

ORIGINAL_N = 20919
SEED = 42

matched_dataset = (
    dataset.sample(
        n=ORIGINAL_N,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("SAMPLE-SIZE-MATCHED DATASET")
print("=" * 55)
print("Full reconstructed dataset:", len(dataset))
print("Matched dataset:", len(matched_dataset))
print("Records excluded from matched sample:",
      len(dataset) - len(matched_dataset))

print("\nQUALITY CHECK")
print("Unique investigation IDs:",
      matched_dataset["investigation_id"].nunique())
print("Missing analysis narratives:",
      matched_dataset["input_text"].eq("").sum())
print("Missing probable causes:",
      matched_dataset["target_text"].eq("").sum())

print("\nSample investigation IDs:")
print(matched_dataset["investigation_id"].head(5).to_list())


In [ ]:

from sklearn.model_selection import train_test_split

SEED = 42

# Original manuscript: 85% train, 15% test
train_all, test_df = train_test_split(
    matched_dataset,
    test_size=0.15,
    random_state=SEED,
    shuffle=True
)

# Original manuscript: 10% of training for validation
train_df, val_df = train_test_split(
    train_all,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

print("ORIGINAL EXPERIMENT SPLIT")
print("=" * 55)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

train_ids = set(train_df["investigation_id"])
val_ids = set(val_df["investigation_id"])
test_ids = set(test_df["investigation_id"])

print("\nLEAKAGE CHECK")
print("Train/validation overlap:", len(train_ids & val_ids))
print("Train/test overlap:", len(train_ids & test_ids))
print("Validation/test overlap:", len(val_ids & test_ids))

assert len(train_df) + len(val_df) + len(test_df) == 20919
assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

print("\nAll checks passed.")


In [ ]:

import sys
import importlib.util

print("PREPROCESSING ENVIRONMENT")
print("=" * 55)
print("Python:", sys.version.split()[0])

packages = [
    "numpy",
    "pandas",
    "nltk",
    "spacy",
    "sklearn",
    "torch",
    "tensorflow",
    "gensim"
]

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package:12s}: {'Available' if available else 'Not installed'}")

print("\nSPACY ENGLISH MODEL")
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("en_core_web_sm: Available")
except Exception:
    print("en_core_web_sm: Not installed")


In [ ]:

import spacy
import numpy as np

print("SPACY MODEL CHECK")
print("=" * 55)

try:
    nlp = spacy.load("en_core_web_sm")
    print("English model: Available")
except OSError:
    print("English model: Not installed")

print("\nCURRENT DATASET WORD LENGTHS")
print("=" * 55)

for column in ["input_text", "target_text"]:
    lengths = matched_dataset[column].str.split().str.len()

    print(f"\n{column}")
    print("Mean:", round(lengths.mean(), 2))
    print("Standard deviation:", round(lengths.std(), 2))
    print("Median:", lengths.median())
    print("Maximum:", lengths.max())

print("\nORIGINAL MANUSCRIPT")
print("Analysis narrative mean: 1116 words")
print("Probable cause mean: 165 words")


In [ ]:

print("CHARACTER LENGTH INVESTIGATION")
print("=" * 60)

for column in ["input_text", "target_text"]:
    char_lengths = matched_dataset[column].str.len()
    word_lengths = matched_dataset[column].str.split().str.len()

    print(f"\n{column.upper()}")
    print("Mean characters:", round(char_lengths.mean(), 2))
    print("SD characters:", round(char_lengths.std(), 2))
    print("Mean words:", round(word_lengths.mean(), 2))
    print("SD words:", round(word_lengths.std(), 2))
    print("Median characters:", round(char_lengths.median(), 2))
    print("Maximum characters:", char_lengths.max())

print("\nORIGINAL REPORTED STATISTICS")
print("Analysis: mean 1116, SD 858.36")
print("Cause: mean 165, SD 93.12")


In [ ]:

print("FULL DATASET CHARACTER STATISTICS")
print("=" * 60)

for column in ["input_text", "target_text"]:
    lengths = dataset[column].str.len()

    print(f"\n{column}")
    print("Records:", len(lengths))
    print("Mean characters:", round(lengths.mean(), 2))
    print("SD characters:", round(lengths.std(), 2))

print("\nORIGINAL MANUSCRIPT")
print("Analysis: 1116 ± 858.36")
print("Probable cause: 165 ± 93.12")


In [ ]:

print("SEQUENCE LENGTH COVERAGE")
print("=" * 60)

for column, limit in [
    ("input_text", 1200),
    ("target_text", 200)
]:
    chars = matched_dataset[column].str.len()
    words = matched_dataset[column].str.split().str.len()

    print(f"\n{column.upper()} — LIMIT {limit}")
    print("Character sequences within limit:",
          int((chars <= limit).sum()))
    print("Character sequences exceeding limit:",
          int((chars > limit).sum()))
    print("Character truncation percentage:",
          round((chars > limit).mean() * 100, 2))

    print("Word sequences within limit:",
          int((words <= limit).sum()))
    print("Word sequences exceeding limit:",
          int((words > limit).sum()))
    print("Word truncation percentage:",
          round((words > limit).mean() * 100, 2))


In [ ]:

import os
import psutil
import torch

memory = psutil.virtual_memory()

print("CPU-ONLY COLAB ENVIRONMENT")
print("=" * 50)
print("Available CPU cores:", os.cpu_count())
print("Total RAM (GB):", round(memory.total / 1024**3, 2))
print("Available RAM (GB):", round(memory.available / 1024**3, 2))
print("PyTorch version:", torch.__version__)
print("Current device: CPU")


In [ ]:

import os

print("SEARCHING FOR EXISTING GLOVE FILES")
print("=" * 55)

locations = [
    "/content",
    "/content/drive/MyDrive"
]

found = []

for location in locations:
    if not os.path.exists(location):
        continue

    for root, dirs, files in os.walk(location):
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        for filename in files:
            name = filename.lower()

            if "glove" in name and name.endswith(
                (".txt", ".zip", ".gz", ".pt", ".npy")
            ):
                path = os.path.join(root, filename)
                found.append(path)

                print(path)

print("\nGloVe files found:", len(found))

if not found:
    print("No existing GloVe files detected.")


In [ ]:

import os
import subprocess
import zipfile

archive = "/content/glove.6B.zip"
destination = "/content/glove.6B.100d.txt"

urls = [
    "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip",
    "https://nlp.stanford.edu/data/glove.6B.zip"
]

if os.path.exists(destination):
    print("GloVe 100d is already available.")
else:
    # Remove any incomplete archive from the failed download
    if os.path.exists(archive):
        os.remove(archive)

    downloaded = False

    for url in urls:
        print("Trying:", url)

        result = subprocess.run(
            [
                "curl", "-L", "--fail",
                "--retry", "2",
                "--connect-timeout", "20",
                "-o", archive, url
            ],
            capture_output=True,
            text=True
        )

        if result.returncode == 0 and zipfile.is_zipfile(archive):
            downloaded = True
            print("Archive downloaded successfully.")
            break

        print("Download attempt failed.")
        if os.path.exists(archive):
            os.remove(archive)

    if downloaded:
        with zipfile.ZipFile(archive) as z:
            z.extract("glove.6B.100d.txt", "/content")
        print("100-dimensional GloVe extracted.")
    else:
        print("Stanford download unavailable.")
        print("We will use an alternative source if needed.")

print("\nGLOVE STATUS")
print("=" * 50)
print("Embedding file available:", os.path.exists(destination))

if os.path.exists(destination):
    print(
        "Size (MB):",
        round(os.path.getsize(destination) / 1024**2, 2)
    )


In [ ]:

import re
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower())

word_counts = Counter()

for column in ["input_text", "target_text"]:
    for text in train_df[column]:
        word_counts.update(tokenize(text))

print("TRAINING VOCABULARY")
print("=" * 55)
print("Unique tokens:", len(word_counts))
print("Total token occurrences:", sum(word_counts.values()))

print("\nMOST COMMON TOKENS")
for word, count in word_counts.most_common(15):
    print(f"{word:20s} {count:,}")

print("\nGLOVE FILE")
print("/content/glove.6B.100d.txt")


In [ ]:

import numpy as np

glove_path = "/content/glove.6B.100d.txt"
embedding_dim = 100

glove_vectors = {}

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]

        if word in word_counts:
            glove_vectors[word] = np.asarray(
                parts[1:], dtype=np.float32
            )

covered = set(glove_vectors)
missing = set(word_counts) - covered

covered_occurrences = sum(
    count for word, count in word_counts.items()
    if word in covered
)

total_occurrences = sum(word_counts.values())

print("GLOVE VOCABULARY COVERAGE")
print("=" * 55)
print("Training vocabulary:", len(word_counts))
print("Words found in GloVe:", len(covered))
print("Words missing from GloVe:", len(missing))

print(
    "Vocabulary coverage:",
    round(100 * len(covered) / len(word_counts), 2), "%"
)

print(
    "Token occurrence coverage:",
    round(100 * covered_occurrences / total_occurrences, 2), "%"
)

print("\nMOST FREQUENT MISSING WORDS")
for word in sorted(
    missing,
    key=lambda w: word_counts[w],
    reverse=True
)[:20]:
    print(word, word_counts[word])

print("\nEXAMPLE GLOVE VECTOR")
print("Word: airplane")
print(glove_vectors.get("airplane", "Not found")[:10])


In [ ]:

def normalize_glove_token(word):
    if word.endswith("'s") and len(word) > 2:
        return word[:-2]
    return word

original_covered = 0
normalized_covered = 0
recovered_words = []

for word, count in word_counts.items():
    if word in glove_vectors:
        original_covered += count
        normalized_covered += count
    else:
        normalized = normalize_glove_token(word)

        if normalized in glove_vectors:
            normalized_covered += count
            recovered_words.append((word, normalized, count))

total = sum(word_counts.values())

print("GLOVE COVERAGE AFTER POSSESSIVE NORMALIZATION")
print("=" * 60)

print("Original token coverage:",
      round(100 * original_covered / total, 2), "%")

print("Normalized token coverage:",
      round(100 * normalized_covered / total, 2), "%")

print("Recovered vocabulary entries:", len(recovered_words))

print("\nMOST FREQUENT RECOVERED WORDS")
for original, normalized, count in sorted(
    recovered_words,
    key=lambda x: x[2],
    reverse=True
)[:15]:
    print(f"{original:20s} -> {normalized:15s} {count:,}")


In [ ]:

import numpy as np

EMBEDDING_DIM = 100
MAX_INPUT_LEN = 1200
MAX_TARGET_LEN = 200

def text_to_glove(text, max_length):
    tokens = tokenize(text)[:max_length]
    matrix = np.zeros(
        (max_length, EMBEDDING_DIM),
        dtype=np.float32
    )

    for i, token in enumerate(tokens):
        normalized = normalize_glove_token(token)
        vector = glove_vectors.get(token)

        if vector is None:
            vector = glove_vectors.get(normalized)

        if vector is not None:
            matrix[i] = vector

    return matrix, len(tokens)

example = train_df.iloc[0]

input_matrix, input_length = text_to_glove(
    example["input_text"], MAX_INPUT_LEN
)

target_matrix, target_length = text_to_glove(
    example["target_text"], MAX_TARGET_LEN
)

print("GLOVE REPRESENTATION CHECK")
print("=" * 55)

print("Investigation:", example["investigation_id"])
print("Input shape:", input_matrix.shape)
print("Target shape:", target_matrix.shape)

print("\nINPUT")
print("Tokens:", input_length)
print("Nonzero embedding rows:",
      np.count_nonzero(np.any(input_matrix != 0, axis=1)))

print("\nTARGET")
print("Tokens:", target_length)
print("Nonzero embedding rows:",
      np.count_nonzero(np.any(target_matrix != 0, axis=1)))

print("\nMemory for this example (MB):",
      round(
          (input_matrix.nbytes + target_matrix.nbytes) / 1024**2,
          3
      ))


In [ ]:

def normalize_glove_token(word):
    if word.endswith("'s") and len(word) > 2:
        return word[:-2]
    return word

original_covered = 0
normalized_covered = 0
recovered_words = []

for word, count in word_counts.items():
    if word in glove_vectors:
        original_covered += count
        normalized_covered += count
    else:
        normalized = normalize_glove_token(word)

        if normalized in glove_vectors:
            normalized_covered += count
            recovered_words.append((word, normalized, count))

total = sum(word_counts.values())

print("GLOVE COVERAGE AFTER POSSESSIVE NORMALIZATION")
print("=" * 60)

print("Original token coverage:",
      round(100 * original_covered / total, 2), "%")

print("Normalized token coverage:",
      round(100 * normalized_covered / total, 2), "%")

print("Recovered vocabulary entries:", len(recovered_words))

print("\nMOST FREQUENT RECOVERED WORDS")
for original, normalized, count in sorted(
    recovered_words,
    key=lambda x: x[2],
    reverse=True
)[:15]:
    print(f"{original:20s} -> {normalized:15s} {count:,}")


In [ ]:

import numpy as np

EMBEDDING_DIM = 100
MAX_INPUT_LEN = 1200
MAX_TARGET_LEN = 200

def text_to_glove(text, max_length):
    tokens = tokenize(text)[:max_length]
    matrix = np.zeros(
        (max_length, EMBEDDING_DIM),
        dtype=np.float32
    )

    for i, token in enumerate(tokens):
        normalized = normalize_glove_token(token)
        vector = glove_vectors.get(token)

        if vector is None:
            vector = glove_vectors.get(normalized)

        if vector is not None:
            matrix[i] = vector

    return matrix, len(tokens)

example = train_df.iloc[0]

input_matrix, input_length = text_to_glove(
    example["input_text"], MAX_INPUT_LEN
)

target_matrix, target_length = text_to_glove(
    example["target_text"], MAX_TARGET_LEN
)

print("GLOVE REPRESENTATION CHECK")
print("=" * 55)

print("Investigation:", example["investigation_id"])
print("Input shape:", input_matrix.shape)
print("Target shape:", target_matrix.shape)

print("\nINPUT")
print("Tokens:", input_length)
print("Nonzero embedding rows:",
      np.count_nonzero(np.any(input_matrix != 0, axis=1)))

print("\nTARGET")
print("Tokens:", target_length)
print("Nonzero embedding rows:",
      np.count_nonzero(np.any(target_matrix != 0, axis=1)))

print("\nMemory for this example (MB):",
      round(
          (input_matrix.nbytes + target_matrix.nbytes) / 1024**2,
          3
      ))


In [ ]:

import torch
import torch.nn as nn

torch.set_num_threads(2)

# Encoder configuration reported in the manuscript
# Kernel sizes are preserved; strides are inferred
# from the reported output shapes where possible.

reported_layers = [
    (256, 36, 134),
    (128, 25, 17),
    (128, 25, 4),
    (64, 16, 1),
    (64, 9, 1),
    (64, 4, 1),
    (64, 2, 1),
    (64, 4, 1),
    (64, 4, 1),
    (64, 2, 1),
    (64, 1, 1),
]

print("REPORTED CONV1D ENCODER SHAPES")
print("=" * 60)

length = 1200
in_channels = 1

for i, (out_channels, kernel, reported_length) in enumerate(
    reported_layers, start=1
):
    possible = []

    for stride in range(1, 101):
        for padding in range(0, kernel + 1):
            calculated = (
                (length + 2 * padding - kernel) // stride
            ) + 1

            if calculated == reported_length:
                possible.append((stride, padding))

    if possible:
        stride, padding = possible[0]
        print(
            f"Layer {i:2d}: "
            f"{length:4d} -> {reported_length:4d} "
            f"| kernel={kernel:2d} "
            f"| possible stride={stride}, padding={padding}"
        )
        length = reported_length
    else:
        print(
            f"Layer {i:2d}: "
            f"No matching stride/padding found "
            f"for {length} -> {reported_length}"
        )
        break

    in_channels = out_channels

print("\nThis checks dimensions only, not model equivalence.")


In [ ]:

# Decoder dimensions reported in Table I
# Input after dense layer and reshape: (1, 64)

decoder_layers = [
    (3, 4),
    (6, 4),
    (19, 4),
    (22, 4),
    (47, 5),
    (47, 4),
    (96, 4),
    (194, 4),
    (197, 3),
    (199, 2),
    (200, 1),
    (200, 1),
]

print("REPORTED CONV1D TRANSPOSE DECODER SHAPES")
print("=" * 65)

length = 1

for i, (target_length, kernel) in enumerate(
    decoder_layers, start=1
):
    possible = []

    for stride in range(1, 11):
        for padding in range(0, kernel + 1):
            for output_padding in range(stride):
                calculated = (
                    (length - 1) * stride
                    - 2 * padding
                    + kernel
                    + output_padding
                )

                if calculated == target_length:
                    possible.append(
                        (stride, padding, output_padding)
                    )

    if possible:
        stride, padding, output_padding = possible[0]

        print(
            f"Layer {i:2d}: "
            f"{length:3d} -> {target_length:3d} "
            f"| kernel={kernel} "
            f"| stride={stride} "
            f"| padding={padding} "
            f"| output_padding={output_padding}"
        )

        length = target_length
    else:
        print(
            f"Layer {i:2d}: "
            f"No matching configuration found "
            f"for {length} -> {target_length}"
        )
        break

print("\nFinal sequence length:", length)
print("Expected sequence length: 200")


In [ ]:

decoder_layers = [
    (3, 4),
    (6, 4),
    (19, 4),
    (22, 4),
    (47, 5),
    (47, 4),
    (96, 4),
    (194, 4),
    (197, 3),
    (199, 2),
    (200, 1),
    (200, 1),
]

print("DECODER FEASIBILITY CHECK")
print("=" * 65)

length = 1

for i, (target, kernel) in enumerate(decoder_layers, 1):
    solutions = []

    for stride in range(1, 21):
        for padding in range(0, 101):
            for output_padding in range(stride):
                result = (
                    (length - 1) * stride
                    - 2 * padding
                    + kernel
                    + output_padding
                )

                if result == target:
                    solutions.append(
                        (stride, padding, output_padding)
                    )

    if solutions:
        best = min(
            solutions,
            key=lambda x: (
                x[0],
                x[1],
                x[2]
            )
        )

        print(
            f"Layer {i:2d}: {length:3d} -> {target:3d} "
            f"| kernel={kernel} "
            f"| stride={best[0]} "
            f"| padding={best[1]} "
            f"| output_padding={best[2]}"
        )
    else:
        print(
            f"Layer {i:2d}: "
            f"NO STANDARD SOLUTION "
            f"for {length} -> {target}"
        )

    length = target

print("\nExpected final output length: 200")


In [ ]:

decoder_layers = [
    (3, 4), (6, 4), (19, 4), (22, 4),
    (47, 5), (47, 4), (96, 4), (194, 4),
    (197, 3), (199, 2), (200, 1), (200, 1)
]

print("DECODER CHECK WITH PRACTICAL SETTINGS")
print("=" * 65)

length = 1
all_feasible = True

for i, (target, kernel) in enumerate(decoder_layers, 1):
    solutions = []

    for stride in [1, 2, 3]:
        for padding in range(kernel + 1):
            for output_padding in range(stride):
                result = (
                    (length - 1) * stride
                    - 2 * padding
                    + kernel
                    + output_padding
                )

                if result == target:
                    solutions.append(
                        (stride, padding, output_padding)
                    )

    if solutions:
        print(
            f"Layer {i:2d}: {length:3d} -> {target:3d} "
            f"| feasible: {solutions[:3]}"
        )
    else:
        print(
            f"Layer {i:2d}: {length:3d} -> {target:3d} "
            f"| NOT feasible under tested settings"
        )
        all_feasible = False

    length = target

print("\nAll transitions feasible:", all_feasible)


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

torch.set_num_threads(2)

class AviationGloveDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        input_matrix, input_length = text_to_glove(
            row["input_text"], MAX_INPUT_LEN
        )

        target_matrix, target_length = text_to_glove(
            row["target_text"], MAX_TARGET_LEN
        )

        return {
            "input": torch.from_numpy(input_matrix),
            "target": torch.from_numpy(target_matrix),
            "input_length": input_length,
            "target_length": target_length
        }

# Small pilot only
pilot_df = train_df.iloc[:32].copy()

pilot_dataset = AviationGloveDataset(pilot_df)

pilot_loader = DataLoader(
    pilot_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

batch = next(iter(pilot_loader))

print("MEMORY-EFFICIENT DATASET CHECK")
print("=" * 55)

print("Pilot investigations:", len(pilot_dataset))
print("Batch input shape:", tuple(batch["input"].shape))
print("Batch target shape:", tuple(batch["target"].shape))

print("Input lengths:", batch["input_length"].tolist())
print("Target lengths:", batch["target_length"].tolist())

batch_memory = (
    batch["input"].numel() * batch["input"].element_size()
    + batch["target"].numel() * batch["target"].element_size()
) / 1024**2

print("Batch tensor memory (MB):", round(batch_memory, 3))

print("CPU device:", torch.device("cpu"))


In [ ]:

import torch
import torch.nn as nn

class ConvVAEEncoder(nn.Module):
    def __init__(self, embedding_dim=100, latent_dim=20):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(embedding_dim, 64, kernel_size=7,
                      stride=2, padding=3),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5,
                      stride=2, padding=2),
            nn.ReLU(),

            nn.Conv1d(128, 128, kernel_size=5,
                      stride=2, padding=2),
            nn.ReLU(),

            nn.Conv1d(128, 64, kernel_size=3,
                      stride=2, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)

    def forward(self, x):
        # Input: (batch, 1200, 100)
        x = x.transpose(1, 2)

        h = self.conv(x).squeeze(-1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std


encoder = ConvVAEEncoder(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

encoder.eval()

with torch.no_grad():
    mu, logvar = encoder(batch["input"])
    z = encoder.reparameterize(mu, logvar)

print("CONV1D VAE ENCODER CHECK")
print("=" * 55)

print("Input:", tuple(batch["input"].shape))
print("Mean:", tuple(mu.shape))
print("Log-variance:", tuple(logvar.shape))
print("Sampled latent vector:", tuple(z.shape))

print("\nAll values finite:",
      bool(torch.isfinite(mu).all()
           and torch.isfinite(logvar).all()
           and torch.isfinite(z).all()))

print("Trainable parameters:",
      sum(p.numel() for p in encoder.parameters()))


In [ ]:

import torch
import torch.nn as nn

class ConvVAEDecoder(nn.Module):
    def __init__(self, latent_dim=20, embedding_dim=100):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 128 * 25)

        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(
                128, 128, kernel_size=4,
                stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose1d(
                128, 64, kernel_size=4,
                stride=2, padding=1
            ),
            nn.ReLU(),

            nn.ConvTranspose1d(
                64, embedding_dim, kernel_size=4,
                stride=2, padding=1
            )
        )

    def forward(self, z):
        h = self.fc(z)
        h = h.reshape(-1, 128, 25)

        output = self.deconv(h)

        # (batch, 100, 200) -> (batch, 200, 100)
        return output.transpose(1, 2)


decoder = ConvVAEDecoder(
    latent_dim=20,
    embedding_dim=100
).to("cpu")

decoder.eval()

with torch.no_grad():
    reconstructed = decoder(z)

print("CONV1D TRANSPOSE DECODER CHECK")
print("=" * 55)

print("Latent input:", tuple(z.shape))
print("Decoder output:", tuple(reconstructed.shape))
print("Expected target:", tuple(batch["target"].shape))

print(
    "Shapes match:",
    reconstructed.shape == batch["target"].shape
)

print(
    "All values finite:",
    bool(torch.isfinite(reconstructed).all())
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in decoder.parameters())
)

print(
    "Output range:",
    round(reconstructed.min().item(), 4),
    "to",
    round(reconstructed.max().item(), 4)
)


In [ ]:

import torch
import torch.nn as nn

class AviationConvVAE(nn.Module):
    def __init__(self, embedding_dim=100, latent_dim=20):
        super().__init__()

        self.encoder = ConvVAEEncoder(
            embedding_dim=embedding_dim,
            latent_dim=latent_dim
        )

        self.decoder = ConvVAEDecoder(
            latent_dim=latent_dim,
            embedding_dim=embedding_dim
        )

    def forward(self, x):
        mu, logvar = self.encoder(x)

        z = self.encoder.reparameterize(
            mu, logvar
        )

        reconstruction = self.decoder(z)

        return reconstruction, mu, logvar


torch.manual_seed(42)

model = AviationConvVAE(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

model.eval()

with torch.no_grad():
    reconstruction, mu, logvar = model(
        batch["input"]
    )

print("COMPLETE CONV1D VAE CHECK")
print("=" * 55)

print("Input:", tuple(batch["input"].shape))
print("Reconstruction:", tuple(reconstruction.shape))
print("Target:", tuple(batch["target"].shape))
print("Latent mean:", tuple(mu.shape))
print("Latent log-variance:", tuple(logvar.shape))

print(
    "\nReconstruction matches target shape:",
    reconstruction.shape == batch["target"].shape
)

print(
    "All outputs finite:",
    bool(
        torch.isfinite(reconstruction).all()
        and torch.isfinite(mu).all()
        and torch.isfinite(logvar).all()
    )
)

print(
    "Total trainable parameters:",
    sum(p.numel() for p in model.parameters())
)

print("\nTraining performed: NO")


In [ ]:

import torch
import torch.nn.functional as F

def vae_loss(reconstruction, target, mu, logvar,
             beta=0.01):

    # Identify actual target embeddings, excluding padding
    mask = (target.abs().sum(dim=-1) > 0)

    # Cosine similarity for each target position
    cosine = F.cosine_similarity(
        reconstruction,
        target,
        dim=-1,
        eps=1e-8
    )

    # Mean reconstruction loss over valid target positions
    reconstruction_loss = (
        (1.0 - cosine)[mask].mean()
    )

    # KL divergence, averaged over investigations
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()

    total_loss = reconstruction_loss + beta * kl_loss

    return total_loss, reconstruction_loss, kl_loss


model.eval()

with torch.no_grad():
    reconstruction, mu, logvar = model(batch["input"])

    total, recon, kl = vae_loss(
        reconstruction,
        batch["target"],
        mu,
        logvar,
        beta=0.01
    )

valid_positions = (
    batch["target"].abs().sum(dim=-1) > 0
).sum().item()

print("VAE LOSS VERIFICATION")
print("=" * 55)

print("Total loss:", round(total.item(), 6))
print("Reconstruction loss:", round(recon.item(), 6))
print("KL divergence:", round(kl.item(), 6))

print("\nValid target positions:", valid_positions)
print("Total target positions:", batch["target"].shape[0] * 200)

print(
    "All losses finite:",
    all(torch.isfinite(x).item()
        for x in [total, recon, kl])
)

print("\nTraining performed: NO")


In [ ]:

import torch

torch.set_num_threads(2)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

model.train()

# Save one parameter tensor before optimization
before = next(model.parameters()).detach().clone()

# One forward pass
reconstruction, mu, logvar = model(batch["input"])

# Calculate loss
total_loss, reconstruction_loss, kl_loss = vae_loss(
    reconstruction,
    batch["target"],
    mu,
    logvar,
    beta=0.01
)

# One backward pass and optimizer update
optimizer.zero_grad()
total_loss.backward()

# Check gradients before updating
gradients_finite = all(
    p.grad is None or torch.isfinite(p.grad).all().item()
    for p in model.parameters()
)

optimizer.step()

# Check whether parameters changed
after = next(model.parameters()).detach().clone()

parameter_change = (
    after - before
).abs().sum().item()

print("SINGLE OPTIMIZATION STEP")
print("=" * 55)

print("Total loss:", round(total_loss.item(), 6))
print("Reconstruction loss:",
      round(reconstruction_loss.item(), 6))
print("KL divergence:", round(kl_loss.item(), 6))

print("\nGradients finite:", gradients_finite)
print("Parameter change:", round(parameter_change, 8))
print("Parameters updated:", parameter_change > 0)

print("\nOptimization steps completed: 1")
print("Device: CPU")


In [ ]:

import time
import torch

torch.set_num_threads(2)

EPOCHS = 3
BETA = 0.01

history = []

print("32-INVESTIGATION CONV1D VAE PILOT")
print("=" * 65)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()

    epoch_total = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0
    samples_seen = 0

    for pilot_batch in pilot_loader:
        inputs = pilot_batch["input"]
        targets = pilot_batch["target"]

        optimizer.zero_grad()

        reconstruction, mu, logvar = model(inputs)

        total_loss, recon_loss, kl_loss = vae_loss(
            reconstruction,
            targets,
            mu,
            logvar,
            beta=BETA
        )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        n = inputs.size(0)
        samples_seen += n

        epoch_total += total_loss.item() * n
        epoch_recon += recon_loss.item() * n
        epoch_kl += kl_loss.item() * n

    results = {
        "epoch": epoch,
        "total_loss": epoch_total / samples_seen,
        "reconstruction_loss": epoch_recon / samples_seen,
        "kl_loss": epoch_kl / samples_seen
    }

    history.append(results)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Total: {results['total_loss']:.6f} | "
        f"Reconstruction: {results['reconstruction_loss']:.6f} | "
        f"KL: {results['kl_loss']:.6f}"
    )

elapsed = time.time() - start_time

print("\nPILOT SUMMARY")
print("=" * 65)
print("Investigations:", len(pilot_dataset))
print("Epochs:", EPOCHS)
print("Total optimization steps:", EPOCHS * len(pilot_loader))
print("Elapsed time (seconds):", round(elapsed, 2))
print("Device: CPU")


In [ ]:

import torch
from torch.utils.data import DataLoader

# Held-out investigations, never used for training
test_pilot_df = test_df.iloc[:32].copy()

test_pilot_dataset = AviationGloveDataset(
    test_pilot_df
)

test_pilot_loader = DataLoader(
    test_pilot_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

model.eval()

total_sum = 0.0
recon_sum = 0.0
kl_sum = 0.0
n_samples = 0

with torch.no_grad():
    for test_batch in test_pilot_loader:
        inputs = test_batch["input"]
        targets = test_batch["target"]

        reconstruction, mu, logvar = model(inputs)

        total, recon, kl = vae_loss(
            reconstruction,
            targets,
            mu,
            logvar,
            beta=0.01
        )

        n = inputs.size(0)
        n_samples += n

        total_sum += total.item() * n
        recon_sum += recon.item() * n
        kl_sum += kl.item() * n

print("HELD-OUT TEST PILOT")
print("=" * 55)
print("Investigations:", n_samples)
print("Mean total loss:", round(total_sum / n_samples, 6))
print("Mean reconstruction loss:",
      round(recon_sum / n_samples, 6))
print("Mean KL divergence:",
      round(kl_sum / n_samples, 6))
print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

correct_losses = []
shuffled_losses = []

with torch.no_grad():
    for test_batch in test_pilot_loader:
        inputs = test_batch["input"]
        targets = test_batch["target"]

        # Deterministic evaluation using latent mean
        mu, logvar = model.encoder(inputs)
        reconstruction = model.decoder(mu)

        # Correct narrative–probable-cause pairing
        _, correct_recon, _ = vae_loss(
            reconstruction, targets, mu, logvar, beta=0.01
        )

        # Circular shift ensures no target remains
        # paired with its original narrative
        shuffled_targets = targets.roll(shifts=1, dims=0)

        _, shuffled_recon, _ = vae_loss(
            reconstruction, shuffled_targets,
            mu, logvar, beta=0.01
        )

        correct_losses.append(correct_recon.item())
        shuffled_losses.append(shuffled_recon.item())

correct_mean = sum(correct_losses) / len(correct_losses)
shuffled_mean = sum(shuffled_losses) / len(shuffled_losses)

print("NARRATIVE–TARGET PAIRING DIAGNOSTIC")
print("=" * 60)
print("Investigations:", len(test_pilot_dataset))
print("Correct pairing loss:", round(correct_mean, 6))
print("Shuffled pairing loss:", round(shuffled_mean, 6))
print("Difference (shuffled - correct):",
      round(shuffled_mean - correct_mean, 6))

print("\nCorrect pairing performs better:",
      correct_mean < shuffled_mean)

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

all_mu = []
all_predictions = []

with torch.no_grad():
    for test_batch in test_pilot_loader:
        inputs = test_batch["input"]

        mu, logvar = model.encoder(inputs)
        predictions = model.decoder(mu)

        all_mu.append(mu)
        all_predictions.append(predictions)

latent_vectors = torch.cat(all_mu, dim=0)
predictions = torch.cat(all_predictions, dim=0)

# Compare each investigation with every other investigation
latent_similarity = F.cosine_similarity(
    latent_vectors.unsqueeze(1),
    latent_vectors.unsqueeze(0),
    dim=-1
)

# Summarize each predicted probable cause
prediction_vectors = predictions.mean(dim=1)

prediction_similarity = F.cosine_similarity(
    prediction_vectors.unsqueeze(1),
    prediction_vectors.unsqueeze(0),
    dim=-1
)

# Exclude self-comparisons
n = latent_vectors.size(0)
off_diagonal = ~torch.eye(n, dtype=torch.bool)

print("LATENT REPRESENTATION DIVERSITY")
print("=" * 60)

print("Investigations:", n)

print(
    "Mean latent cosine similarity:",
    round(latent_similarity[off_diagonal].mean().item(), 6)
)

print(
    "Mean prediction cosine similarity:",
    round(prediction_similarity[off_diagonal].mean().item(), 6)
)

print(
    "Mean latent standard deviation:",
    round(latent_vectors.std(dim=0).mean().item(), 6)
)

print(
    "Mean prediction standard deviation:",
    round(predictions.std(dim=0).mean().item(), 6)
)

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

torch.manual_seed(123)

fresh_encoder = ConvVAEEncoder(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

fresh_encoder.eval()
model.encoder.eval()

fresh_vectors = []
trained_vectors = []

with torch.no_grad():
    for test_batch in test_pilot_loader:
        inputs = test_batch["input"]

        fresh_mu, _ = fresh_encoder(inputs)
        trained_mu, _ = model.encoder(inputs)

        fresh_vectors.append(fresh_mu)
        trained_vectors.append(trained_mu)

fresh_vectors = torch.cat(fresh_vectors, dim=0)
trained_vectors = torch.cat(trained_vectors, dim=0)

def diversity_statistics(vectors):
    similarity = F.cosine_similarity(
        vectors.unsqueeze(1),
        vectors.unsqueeze(0),
        dim=-1
    )

    n = vectors.size(0)
    mask = ~torch.eye(n, dtype=torch.bool)

    return (
        similarity[mask].mean().item(),
        vectors.std(dim=0).mean().item()
    )

fresh_similarity, fresh_std = diversity_statistics(
    fresh_vectors
)

trained_similarity, trained_std = diversity_statistics(
    trained_vectors
)

print("ENCODER INPUT-SENSITIVITY CHECK")
print("=" * 60)

print("Fresh encoder:")
print("  Mean cosine similarity:",
      round(fresh_similarity, 6))
print("  Mean latent standard deviation:",
      round(fresh_std, 6))

print("\nTrained encoder:")
print("  Mean cosine similarity:",
      round(trained_similarity, 6))
print("  Mean latent standard deviation:",
      round(trained_std, 6))

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.encoder.eval()

# Use only examples with 400 or fewer actual tokens
eligible = [
    i for i, length in enumerate(batch["input_length"])
    if int(length) <= 400
]

full_inputs = batch["input"][eligible]
short_inputs = full_inputs[:, :400, :]

with torch.no_grad():
    full_mu, _ = model.encoder(full_inputs)
    short_mu, _ = model.encoder(short_inputs)

similarity = F.cosine_similarity(
    full_mu,
    short_mu,
    dim=-1
)

difference = (
    full_mu - short_mu
).norm(dim=-1)

print("INPUT PADDING SENSITIVITY")
print("=" * 60)

print("Investigations tested:", len(eligible))
print("Original sequence length:", full_inputs.shape[1])
print("Shortened sequence length:", short_inputs.shape[1])

print("\nMean cosine similarity:",
      round(similarity.mean().item(), 6))

print("Mean Euclidean difference:",
      round(difference.mean().item(), 6))

print("\nIndividual cosine similarities:")
for i, value in enumerate(similarity):
    print(f"Investigation {i+1}: {value.item():.6f}")

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.encoder.eval()

standard_vectors = []
length_aware_vectors = []

with torch.no_grad():
    for test_batch in test_pilot_loader:
        inputs = test_batch["input"]
        lengths = test_batch["input_length"]

        # Extract convolutional features before final pooling
        x = inputs.transpose(1, 2)

        features = model.encoder.conv[:-1](x)

        # Standard global average pooling
        standard_pooled = features.mean(dim=-1)

        # Approximate valid output lengths after four stride-2 layers
        valid_lengths = (lengths + 15) // 16
        valid_lengths = valid_lengths.clamp(
            min=1,
            max=features.shape[-1]
        )

        positions = torch.arange(
            features.shape[-1]
        ).unsqueeze(0)

        mask = (
            positions < valid_lengths.unsqueeze(1)
        ).unsqueeze(1)

        # Pool only over estimated valid positions
        masked_features = features * mask

        length_aware_pooled = (
            masked_features.sum(dim=-1)
            / valid_lengths.unsqueeze(1)
        )

        standard_mu = model.encoder.fc_mu(
            standard_pooled
        )

        length_aware_mu = model.encoder.fc_mu(
            length_aware_pooled
        )

        standard_vectors.append(standard_mu)
        length_aware_vectors.append(length_aware_mu)

standard_vectors = torch.cat(
    standard_vectors, dim=0
)

length_aware_vectors = torch.cat(
    length_aware_vectors, dim=0
)

def measure_diversity(vectors):
    normalized = F.normalize(vectors, dim=-1)

    similarity = normalized @ normalized.T

    n = vectors.shape[0]
    mask = ~torch.eye(n, dtype=torch.bool)

    return (
        similarity[mask].mean().item(),
        vectors.std(dim=0).mean().item()
    )

standard_sim, standard_std = measure_diversity(
    standard_vectors
)

aware_sim, aware_std = measure_diversity(
    length_aware_vectors
)

print("PADDING-AWARE POOLING DIAGNOSTIC")
print("=" * 60)

print("Investigations:", len(standard_vectors))

print("\nStandard pooling:")
print("  Mean cosine similarity:",
      round(standard_sim, 6))
print("  Mean latent standard deviation:",
      round(standard_std, 6))

print("\nLength-aware pooling:")
print("  Mean cosine similarity:",
      round(aware_sim, 6))
print("  Mean latent standard deviation:",
      round(aware_std, 6))

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Build baseline from the 32 training investigations
train_targets = torch.stack([
    pilot_dataset[i]["target"]
    for i in range(len(pilot_dataset))
])

# Average target embeddings at each sequence position
# using only nonzero target embeddings
train_mask = (
    train_targets.abs().sum(dim=-1) > 0
)

position_counts = train_mask.sum(dim=0).clamp(min=1)

mean_target = (
    train_targets.sum(dim=0)
    / position_counts.unsqueeze(-1)
)

# Use 32 validation investigations
val_pilot_dataset = AviationGloveDataset(
    val_df.iloc[:32].copy()
)

val_pilot_loader = DataLoader(
    val_pilot_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

model.eval()

vae_losses = []
baseline_losses = []

with torch.no_grad():
    for val_batch in val_pilot_loader:
        inputs = val_batch["input"]
        targets = val_batch["target"]

        # Deterministic VAE prediction
        mu, _ = model.encoder(inputs)
        vae_prediction = model.decoder(mu)

        baseline_prediction = mean_target.unsqueeze(0).expand(
            targets.shape[0], -1, -1
        )

        mask = targets.abs().sum(dim=-1) > 0

        vae_cos = F.cosine_similarity(
            vae_prediction, targets, dim=-1
        )

        baseline_cos = F.cosine_similarity(
            baseline_prediction, targets, dim=-1
        )

        vae_losses.extend(
            (1 - vae_cos)[mask].tolist()
        )

        baseline_losses.extend(
            (1 - baseline_cos)[mask].tolist()
        )

print("VALIDATION BASELINE COMPARISON")
print("=" * 60)

print("Training investigations for baseline: 32")
print("Validation investigations: 32")

print(
    "VAE reconstruction loss:",
    round(sum(vae_losses) / len(vae_losses), 6)
)

print(
    "Constant baseline loss:",
    round(sum(baseline_losses) / len(baseline_losses), 6)
)

print(
    "VAE better than baseline:",
    (sum(vae_losses) / len(vae_losses))
    < (sum(baseline_losses) / len(baseline_losses))
)

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

# Collect all 32 validation investigations
val_inputs = []
val_targets = []

for val_batch in val_pilot_loader:
    val_inputs.append(val_batch["input"])
    val_targets.append(val_batch["target"])

val_inputs = torch.cat(val_inputs, dim=0)
val_targets = torch.cat(val_targets, dim=0)

# Circularly shift the input narratives
# while keeping targets in their original order
shuffled_inputs = val_inputs.roll(shifts=1, dims=0)

def masked_cosine_loss(predictions, targets):
    mask = targets.abs().sum(dim=-1) > 0

    cosine = F.cosine_similarity(
        predictions,
        targets,
        dim=-1,
        eps=1e-8
    )

    return (1.0 - cosine)[mask].mean().item()

with torch.no_grad():
    # Correct narrative–target pairing
    correct_mu, _ = model.encoder(val_inputs)
    correct_predictions = model.decoder(correct_mu)

    # Incorrect narrative–target pairing
    shuffled_mu, _ = model.encoder(shuffled_inputs)
    shuffled_predictions = model.decoder(shuffled_mu)

correct_loss = masked_cosine_loss(
    correct_predictions,
    val_targets
)

shuffled_loss = masked_cosine_loss(
    shuffled_predictions,
    val_targets
)

print("VALIDATION INPUT-SHUFFLING TEST")
print("=" * 60)

print("Validation investigations:", len(val_inputs))
print("Correct input loss:", round(correct_loss, 6))
print("Shuffled input loss:", round(shuffled_loss, 6))
print(
    "Difference (shuffled - correct):",
    round(shuffled_loss - correct_loss, 6)
)

print(
    "\nCorrect inputs perform better:",
    correct_loss < shuffled_loss
)

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
from torch.utils.data import DataLoader

torch.manual_seed(42)
torch.set_num_threads(2)

# Use eight investigations from the training partition only
overfit_df = train_df.iloc[:8].copy()

overfit_dataset = AviationGloveDataset(overfit_df)

overfit_loader = DataLoader(
    overfit_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

overfit_batch = next(iter(overfit_loader))

# Create an independent diagnostic model
overfit_model = AviationConvVAE(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

overfit_optimizer = torch.optim.Adam(
    overfit_model.parameters(),
    lr=0.001
)

print("CONTROLLED OVERFITTING TEST — SETUP")
print("=" * 60)

print("Investigations:", len(overfit_dataset))
print("Input shape:", tuple(overfit_batch["input"].shape))
print("Target shape:", tuple(overfit_batch["target"].shape))

print(
    "Model parameters:",
    sum(p.numel() for p in overfit_model.parameters())
)

print("Separate model created: YES")
print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)

inputs = overfit_batch["input"]
targets = overfit_batch["target"]

mask = targets.abs().sum(dim=-1) > 0

start_time = time.time()

overfit_model.train()

for epoch in range(1, 101):
    overfit_optimizer.zero_grad()

    # Deterministic latent representation
    mu, logvar = overfit_model.encoder(inputs)
    predictions = overfit_model.decoder(mu)

    cosine = F.cosine_similarity(
        predictions,
        targets,
        dim=-1,
        eps=1e-8
    )

    # Reconstruction-only objective for diagnostic
    loss = (1.0 - cosine)[mask].mean()

    loss.backward()
    overfit_optimizer.step()

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Reconstruction loss: {loss.item():.6f}"
        )

elapsed = time.time() - start_time

print("\nCONTROLLED OVERFITTING TEST")
print("=" * 60)
print("Investigations:", len(overfit_dataset))
print("Epochs:", 100)
print("Final reconstruction loss:", round(loss.item(), 6))
print("Elapsed seconds:", round(elapsed, 2))
print("Training performed: YES")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

overfit_model.eval()

inputs = overfit_batch["input"]
targets = overfit_batch["target"]

# Shift narratives, keeping their target causes unchanged
shuffled_inputs = inputs.roll(shifts=1, dims=0)

def reconstruction_loss(predictions, targets):
    mask = targets.abs().sum(dim=-1) > 0
    cosine = F.cosine_similarity(
        predictions, targets, dim=-1, eps=1e-8
    )
    return (1.0 - cosine)[mask].mean().item()

with torch.no_grad():
    correct_mu, _ = overfit_model.encoder(inputs)
    correct_predictions = overfit_model.decoder(correct_mu)

    shuffled_mu, _ = overfit_model.encoder(shuffled_inputs)
    shuffled_predictions = overfit_model.decoder(shuffled_mu)

    correct_loss = reconstruction_loss(
        correct_predictions, targets
    )
    shuffled_loss = reconstruction_loss(
        shuffled_predictions, targets
    )

    # Measure variation across predicted sequences
    prediction_std = correct_predictions.std(
        dim=0
    ).mean().item()

    # Measure latent representation diversity
    latent_similarity = F.cosine_similarity(
        correct_mu.unsqueeze(1),
        correct_mu.unsqueeze(0),
        dim=-1
    )

    n = correct_mu.shape[0]
    off_diagonal = ~torch.eye(n, dtype=torch.bool)

    mean_latent_similarity = latent_similarity[
        off_diagonal
    ].mean().item()

print("EIGHT-INVESTIGATION PAIRING TEST")
print("=" * 60)

print("Investigations:", len(inputs))
print("Correct pairing loss:", round(correct_loss, 6))
print("Shuffled pairing loss:", round(shuffled_loss, 6))
print(
    "Difference (shuffled - correct):",
    round(shuffled_loss - correct_loss, 6)
)
print(
    "Mean latent cosine similarity:",
    round(mean_latent_similarity, 6)
)
print(
    "Mean prediction standard deviation:",
    round(prediction_std, 6)
)

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

# Use validation data only
semantic_df = val_df.iloc[:32].copy()

narrative_vectors = []
cause_vectors = []

def mean_glove_embedding(text):
    matrix, length = text_to_glove(text, MAX_INPUT_LEN)

    vectors = torch.from_numpy(matrix[:length])

    # Exclude tokens without GloVe representations
    valid = vectors.abs().sum(dim=-1) > 0

    if valid.any():
        return vectors[valid].mean(dim=0)

    return torch.zeros(EMBEDDING_DIM)

for _, row in semantic_df.iterrows():
    narrative_vectors.append(
        mean_glove_embedding(row["input_text"])
    )

    cause_vectors.append(
        mean_glove_embedding(row["target_text"])
    )

narrative_vectors = torch.stack(narrative_vectors)
cause_vectors = torch.stack(cause_vectors)

correct_similarity = F.cosine_similarity(
    narrative_vectors,
    cause_vectors,
    dim=-1
)

shuffled_similarity = F.cosine_similarity(
    narrative_vectors,
    cause_vectors.roll(shifts=1, dims=0),
    dim=-1
)

print("SEMANTIC PAIRING DIAGNOSTIC")
print("=" * 60)

print("Validation investigations:", len(semantic_df))

print(
    "Mean correct-pair cosine:",
    round(correct_similarity.mean().item(), 6)
)

print(
    "Mean shuffled-pair cosine:",
    round(shuffled_similarity.mean().item(), 6)
)

print(
    "Difference (correct - shuffled):",
    round(
        (correct_similarity - shuffled_similarity)
        .mean().item(), 6
    )
)

print(
    "Cases where correct pairing scores higher:",
    int((correct_similarity > shuffled_similarity).sum().item()),
    "out of",
    len(semantic_df)
)

print("\nTraining performed: NO")
print("Device: CPU")


In [ ]:

import numpy as np
import torch
import torch.nn.functional as F
import time

start = time.time()

narrative_vectors = []
cause_vectors = []

for _, row in val_df.iterrows():
    narrative_vectors.append(
        mean_glove_embedding(row["input_text"])
    )
    cause_vectors.append(
        mean_glove_embedding(row["target_text"])
    )

narrative_vectors = torch.stack(narrative_vectors)
cause_vectors = torch.stack(cause_vectors)

correct_similarity = F.cosine_similarity(
    narrative_vectors,
    cause_vectors,
    dim=-1
)

# Deterministic mismatched cause pairing
shuffled_similarity = F.cosine_similarity(
    narrative_vectors,
    cause_vectors.roll(shifts=1, dims=0),
    dim=-1
)

difference = correct_similarity - shuffled_similarity

print("FULL VALIDATION SEMANTIC DIAGNOSTIC")
print("=" * 60)

print("Investigations:", len(val_df))

print(
    "Mean correct cosine:",
    round(correct_similarity.mean().item(), 6)
)

print(
    "Mean shuffled cosine:",
    round(shuffled_similarity.mean().item(), 6)
)

print(
    "Mean paired difference:",
    round(difference.mean().item(), 6)
)

print(
    "Correct pairing wins:",
    int((difference > 0).sum().item()),
    "of",
    len(difference)
)

print(
    "Correct pairing win rate:",
    round((difference > 0).float().mean().item() * 100, 2),
    "%"
)

print("Elapsed seconds:", round(time.time() - start, 2))
print("Training performed: NO")
print("Device: CPU")


In [ ]:

import pandas as pd
import numpy as np

diagnostic_results = pd.DataFrame({
    "investigation_id": val_df["investigation_id"].to_numpy(),
    "correct_cosine": correct_similarity.cpu().numpy(),
    "shuffled_cosine": shuffled_similarity.cpu().numpy(),
    "pairing_difference": difference.cpu().numpy(),
    "correct_pair_wins": (
        difference > 0
    ).cpu().numpy()
})

output_path = "/content/aviation_glove_validation_diagnostics.csv"

diagnostic_results.to_csv(
    output_path,
    index=False
)

print("DIAGNOSTIC RESULTS SAVED")
print("=" * 60)
print("File:", output_path)
print("Rows:", len(diagnostic_results))
print("Columns:", list(diagnostic_results.columns))
print("Mean pairing difference:",
      round(diagnostic_results["pairing_difference"].mean(), 6))
print("Training performed: NO")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    batch = next(iter(pilot_loader))
    inputs = batch["input"].to("cpu")

    # Features before the 20-dimensional latent projection
    features = model.encoder.conv(
        inputs.transpose(1, 2)
    ).squeeze(-1)

    # Existing latent mean representations
    mu, logvar = model.encoder(inputs)

    # Compare representations between investigations
    feature_similarity = F.cosine_similarity(
        features.unsqueeze(1),
        features.unsqueeze(0),
        dim=-1
    )

    latent_similarity = F.cosine_similarity(
        mu.unsqueeze(1),
        mu.unsqueeze(0),
        dim=-1
    )

    n = inputs.shape[0]
    off_diagonal = ~torch.eye(n, dtype=torch.bool)

    print("STEP 38: ENCODER DIVERSITY")
    print("=" * 55)
    print("Investigations:", n)
    print("Feature shape:", tuple(features.shape))
    print("Latent shape:", tuple(mu.shape))

    print("Mean feature similarity:",
          round(feature_similarity[off_diagonal].mean().item(), 6))

    print("Mean latent similarity:",
          round(latent_similarity[off_diagonal].mean().item(), 6))

    print("Mean feature standard deviation:",
          round(features.std(dim=0).mean().item(), 6))

    print("Mean latent standard deviation:",
          round(mu.std(dim=0).mean().item(), 6))

    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    batch = next(iter(pilot_loader))
    x = batch["input"].to("cpu").transpose(1, 2)

    # Features before global average pooling
    conv_features = model.encoder.conv[:-1](x)

    # Features after global average pooling
    pooled_features = model.encoder.conv[-1](
        conv_features
    ).squeeze(-1)

    # Flatten features to compare investigations
    flattened = conv_features.flatten(start_dim=1)

    def diversity_report(features, name):
        similarity = F.cosine_similarity(
            features.unsqueeze(1),
            features.unsqueeze(0),
            dim=-1
        )

        n = features.shape[0]
        mask = ~torch.eye(n, dtype=torch.bool)

        print(name)
        print("Shape:", tuple(features.shape))
        print("Mean cross-investigation cosine:",
              round(similarity[mask].mean().item(), 6))
        print("Mean feature standard deviation:",
              round(features.std(dim=0).mean().item(), 6))
        print()

    print("STEP 39: GLOBAL POOLING DIAGNOSTIC")
    print("=" * 55)

    diversity_report(flattened, "BEFORE GLOBAL POOLING")
    diversity_report(pooled_features, "AFTER GLOBAL POOLING")

    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    batch = next(iter(pilot_loader))
    inputs = batch["input"].to("cpu")

    valid_positions = (
        inputs.abs().sum(dim=-1) > 0
    )

    valid_counts = valid_positions.sum(dim=1)
    padding_percent = (
        1 - valid_counts.float() / inputs.shape[1]
    ) * 100

    # Encode actual narratives
    actual_mu, _ = model.encoder(inputs)

    # Encode completely empty narratives
    zero_inputs = torch.zeros_like(inputs)
    zero_mu, _ = model.encoder(zero_inputs)

    zero_similarity = F.cosine_similarity(
        actual_mu,
        zero_mu,
        dim=-1
    )

    print("STEP 40: PADDING DIAGNOSTIC")
    print("=" * 55)

    print("Valid token positions:",
          valid_counts.tolist())

    print("Padding percentages:",
          [round(v, 2) for v in padding_percent.tolist()])

    print("Mean padding percentage:",
          round(padding_percent.mean().item(), 2))

    print("Cosine similarity to zero-input representation:",
          [round(v, 6) for v in zero_similarity.tolist()])

    print("Mean similarity to zero-input representation:",
          round(zero_similarity.mean().item(), 6))

    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    batch = next(iter(pilot_loader))
    inputs = batch["input"].to("cpu")

    lengths = (inputs.abs().sum(dim=-1) > 0).sum(dim=1)

    conv_features = model.encoder.conv[:-1](
        inputs.transpose(1, 2)
    )

    # Four stride-2 convolutional layers
    reduced_lengths = lengths.clone()
    for _ in range(4):
        reduced_lengths = (reduced_lengths + 1) // 2

    sequence_positions = torch.arange(
        conv_features.shape[-1]
    ).unsqueeze(0)

    mask = (
        sequence_positions < reduced_lengths.unsqueeze(1)
    ).unsqueeze(1)

    standard_features = conv_features.mean(dim=-1)

    masked_features = (
        (conv_features * mask).sum(dim=-1)
        / mask.sum(dim=-1).clamp(min=1)
    )

    def mean_pairwise_cosine(features):
        sim = F.cosine_similarity(
            features.unsqueeze(1),
            features.unsqueeze(0),
            dim=-1
        )
        n = features.shape[0]
        off_diag = ~torch.eye(n, dtype=torch.bool)
        return sim[off_diag].mean().item()

    print("STEP 41: PADDING-AWARE POOLING")
    print("=" * 55)
    print("Original lengths:", lengths.tolist())
    print("Reduced lengths:", reduced_lengths.tolist())
    print("Feature-map length:", conv_features.shape[-1])

    print("Standard pooling similarity:",
          round(mean_pairwise_cosine(standard_features), 6))

    print("Padding-aware pooling similarity:",
          round(mean_pairwise_cosine(masked_features), 6))

    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

batch = next(iter(pilot_loader))
inputs = batch["input"].to("cpu")

with torch.no_grad():
    valid = inputs.abs().sum(dim=-1) > 0

    mean_embeddings = (
        (inputs * valid.unsqueeze(-1)).sum(dim=1)
        / valid.sum(dim=1, keepdim=True).clamp(min=1)
    )

    similarity = F.cosine_similarity(
        mean_embeddings.unsqueeze(1),
        mean_embeddings.unsqueeze(0),
        dim=-1
    )

    n = mean_embeddings.shape[0]
    off_diagonal = ~torch.eye(n, dtype=torch.bool)

    print("STEP 42: INPUT EMBEDDING DIVERSITY")
    print("=" * 55)
    print("Investigations:", n)
    print("Mean cross-investigation cosine:",
          round(similarity[off_diagonal].mean().item(), 6))
    print("Minimum cross-investigation cosine:",
          round(similarity[off_diagonal].min().item(), 6))
    print("Maximum cross-investigation cosine:",
          round(similarity[off_diagonal].max().item(), 6))
    print("Mean embedding standard deviation:",
          round(mean_embeddings.std(dim=0).mean().item(), 6))
    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
from torch.utils.data import DataLoader

# Reproducible subset from training data only
controlled_train_df = (
    train_df.sample(n=128, random_state=42)
    .reset_index(drop=True)
)

controlled_train_dataset = AviationGloveDataset(
    controlled_train_df
)

controlled_train_loader = DataLoader(
    controlled_train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

print("STEP 43: CONTROLLED TRAINING SUBSET")
print("=" * 55)
print("Investigations:", len(controlled_train_df))
print("Batches per epoch:", len(controlled_train_loader))
print("Batch size:", 8)

print("Unique investigation IDs:",
      controlled_train_df["investigation_id"].nunique())

print("Overlap with validation:",
      len(set(controlled_train_df["investigation_id"])
          & set(val_df["investigation_id"])))

print("Overlap with test:",
      len(set(controlled_train_df["investigation_id"])
          & set(test_df["investigation_id"])))

batch = next(iter(controlled_train_loader))

print("Input shape:", tuple(batch["input"].shape))
print("Target shape:", tuple(batch["target"].shape))

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn as nn

class DeterministicConvEncoderDecoder(nn.Module):
    def __init__(self, embedding_dim=100, latent_dim=20):
        super().__init__()

        # Same convolutional feature extractor as the VAE
        self.encoder = ConvVAEEncoder(
            embedding_dim=embedding_dim,
            latent_dim=latent_dim
        )

        # Same decoder architecture as the VAE
        self.decoder = ConvVAEDecoder(
            latent_dim=latent_dim,
            embedding_dim=embedding_dim
        )

    def forward(self, x):
        # Deterministic latent representation
        z, _ = self.encoder(x)

        # No stochastic sampling
        reconstruction = self.decoder(z)

        return reconstruction


torch.manual_seed(42)

deterministic_model = DeterministicConvEncoderDecoder(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

test_batch = next(iter(controlled_train_loader))

with torch.no_grad():
    output = deterministic_model(
        test_batch["input"]
    )

print("STEP 44: DETERMINISTIC BASELINE")
print("=" * 55)

print("Input shape:",
      tuple(test_batch["input"].shape))

print("Output shape:",
      tuple(output.shape))

print("Expected target shape:",
      tuple(test_batch["target"].shape))

print("Total parameters:",
      sum(p.numel() for p in deterministic_model.parameters()))

print("VAE parameters:",
      sum(p.numel() for p in model.parameters()))

print("Finite outputs:",
      bool(torch.isfinite(output).all()))

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch

torch.manual_seed(42)

# Fresh VAE
comparison_vae = AviationConvVAE(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

# Fresh deterministic baseline
comparison_deterministic = DeterministicConvEncoderDecoder(
    embedding_dim=100,
    latent_dim=20
).to("cpu")

# Match all shared initial parameters
comparison_deterministic.load_state_dict(
    comparison_vae.state_dict()
)

# Exclude unused log-variance parameters
for parameter in comparison_deterministic.encoder.fc_logvar.parameters():
    parameter.requires_grad = False

def trainable_parameters(model):
    return sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

# Confirm shared weights are identical
shared_weights_match = all(
    torch.equal(
        comparison_vae.state_dict()[name],
        comparison_deterministic.state_dict()[name]
    )
    for name in comparison_vae.state_dict()
)

print("STEP 45: MATCHED MODEL INITIALIZATION")
print("=" * 55)

print("Shared initial weights match:",
      shared_weights_match)

print("VAE trainable parameters:",
      trainable_parameters(comparison_vae))

print("Deterministic trainable parameters:",
      trainable_parameters(comparison_deterministic))

print("VAE latent dimension:", 20)
print("Deterministic latent dimension:", 20)

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F
import time

torch.set_num_threads(2)

det_optimizer = torch.optim.Adam(
    comparison_deterministic.parameters(), lr=0.001
)

vae_optimizer = torch.optim.Adam(
    comparison_vae.parameters(), lr=0.001
)

def masked_cosine_loss(prediction, target):
    mask = target.abs().sum(dim=-1) > 0
    cosine = F.cosine_similarity(
        prediction, target, dim=-1, eps=1e-8
    )
    return (1.0 - cosine)[mask].mean()

torch.manual_seed(42)
start = time.time()

for epoch in range(1, 6):
    comparison_deterministic.train()
    comparison_vae.train()

    det_total = 0.0
    vae_total = 0.0
    vae_kl_total = 0.0
    samples = 0

    for batch in controlled_train_loader:
        x = batch["input"].to("cpu")
        y = batch["target"].to("cpu")
        n = x.shape[0]

        # Deterministic encoder-decoder
        det_optimizer.zero_grad()
        det_prediction = comparison_deterministic(x)
        det_loss = masked_cosine_loss(det_prediction, y)
        det_loss.backward()
        det_optimizer.step()

        # Variational encoder-decoder
        vae_optimizer.zero_grad()
        vae_prediction, mu, logvar = comparison_vae(x)

        vae_reconstruction = masked_cosine_loss(
            vae_prediction, y
        )

        kl = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp(),
            dim=1
        ).mean()

        vae_loss_value = vae_reconstruction + 0.01 * kl

        vae_loss_value.backward()
        vae_optimizer.step()

        det_total += det_loss.item() * n
        vae_total += vae_reconstruction.item() * n
        vae_kl_total += kl.item() * n
        samples += n

    print(
        f"Epoch {epoch:02d} | "
        f"Det reconstruction: {det_total/samples:.6f} | "
        f"VAE reconstruction: {vae_total/samples:.6f} | "
        f"VAE KL: {vae_kl_total/samples:.6f}"
    )

print("=" * 70)
print("Elapsed seconds:", round(time.time()-start, 2))
print("Training investigations:", len(controlled_train_df))
print("Epochs: 5")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

comparison_deterministic.eval()
comparison_vae.eval()

# Validation data only — no test-set access
validation_subset = val_df.iloc[:32].copy()
validation_loader = DataLoader(
    AviationGloveDataset(validation_subset),
    batch_size=32,
    shuffle=False,
    num_workers=0
)

batch = next(iter(validation_loader))
x = batch["input"].to("cpu")
y = batch["target"].to("cpu")

shuffled_x = x.roll(shifts=1, dims=0)

with torch.no_grad():
    # Deterministic model
    det_correct = comparison_deterministic(x)
    det_shuffled = comparison_deterministic(shuffled_x)

    # VAE evaluated using latent mean for reproducibility
    vae_mu_correct, _ = comparison_vae.encoder(x)
    vae_mu_shuffled, _ = comparison_vae.encoder(shuffled_x)

    vae_correct = comparison_vae.decoder(vae_mu_correct)
    vae_shuffled = comparison_vae.decoder(vae_mu_shuffled)

    results = {
        "Deterministic": (
            masked_cosine_loss(det_correct, y).item(),
            masked_cosine_loss(det_shuffled, y).item()
        ),
        "VAE (latent mean)": (
            masked_cosine_loss(vae_correct, y).item(),
            masked_cosine_loss(vae_shuffled, y).item()
        )
    }

print("STEP 47: VALIDATION PAIRING DISCRIMINATION")
print("=" * 65)

for name, (correct, shuffled) in results.items():
    print(name)
    print("  Correct narrative loss:", round(correct, 6))
    print("  Shuffled narrative loss:", round(shuffled, 6))
    print("  Difference (shuffled - correct):",
          round(shuffled - correct, 6))

print("=" * 65)
print("Validation investigations:", len(validation_subset))
print("Training performed: NO")
print("Test partition accessed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

comparison_deterministic.eval()
comparison_vae.eval()

with torch.no_grad():
    det_predictions = comparison_deterministic(x)

    vae_mu, _ = comparison_vae.encoder(x)
    vae_predictions = comparison_vae.decoder(vae_mu)

    def prediction_diversity(predictions, name):
        # Flatten each predicted target sequence
        flattened = predictions.flatten(start_dim=1)

        similarity = F.cosine_similarity(
            flattened.unsqueeze(1),
            flattened.unsqueeze(0),
            dim=-1
        )

        n = flattened.shape[0]
        off_diagonal = ~torch.eye(n, dtype=torch.bool)

        print(name)
        print("Mean cross-investigation cosine:",
              round(similarity[off_diagonal].mean().item(), 6))
        print("Mean output standard deviation:",
              round(predictions.std(dim=0).mean().item(), 6))
        print()

    print("STEP 48: PREDICTION DIVERSITY")
    print("=" * 55)

    prediction_diversity(
        det_predictions,
        "DETERMINISTIC MODEL"
    )

    prediction_diversity(
        vae_predictions,
        "VAE (LATENT MEAN)"
    )

    print("Validation investigations:", len(x))
    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

comparison_deterministic.eval()
comparison_vae.eval()

with torch.no_grad():
    det_out = comparison_deterministic(x)

    vae_mu, vae_logvar = comparison_vae.encoder(x)
    vae_out = comparison_vae.decoder(vae_mu)

    def inspect_outputs(name, outputs):
        # Variation across investigations, for each output feature
        between_case_std = outputs.std(
            dim=0, unbiased=False
        ).mean().item()

        # Mean distance of each output from the batch-average output
        centered = outputs - outputs.mean(dim=0, keepdim=True)
        mean_distance = centered.flatten(1).norm(
            dim=1
        ).mean().item()

        mean_output_norm = outputs.flatten(1).norm(
            dim=1
        ).mean().item()

        relative_variation = (
            mean_distance / max(mean_output_norm, 1e-8)
        )

        print(name)
        print("  Between-case std:",
              f"{between_case_std:.8f}")
        print("  Relative output variation:",
              f"{relative_variation:.8f}")
        print()

    print("STEP 49: BETWEEN-INVESTIGATION VARIATION")
    print("=" * 60)

    inspect_outputs("DETERMINISTIC", det_out)
    inspect_outputs("VAE (LATENT MEAN)", vae_out)

    print("Validation cases:", x.shape[0])
    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

comparison_deterministic.eval()

with torch.no_grad():
    outputs = comparison_deterministic(x)
    flat = outputs.flatten(start_dim=1)

    norms = flat.norm(dim=1)

    normalized = F.normalize(
        flat, p=2, dim=1, eps=1e-8
    )

    # Variation in output direction after removing magnitude
    direction_std = normalized.std(
        dim=0, unbiased=False
    ).mean().item()

    # Variation in output magnitude
    norm_cv = (
        norms.std(unbiased=False) /
        norms.mean().clamp_min(1e-8)
    ).item()

    # Pairwise cosine similarity
    cosine_matrix = normalized @ normalized.T

    mask = ~torch.eye(
        len(flat), dtype=torch.bool
    )

    pairwise_cosine = cosine_matrix[mask]

    print("STEP 50: DETERMINISTIC OUTPUT DIAGNOSTIC")
    print("=" * 60)

    print("Mean output norm:",
          round(norms.mean().item(), 6))

    print("Output norm coefficient of variation:",
          round(norm_cv, 6))

    print("Mean normalized-feature std:",
          round(direction_std, 8))

    print("Mean pairwise cosine:",
          round(pairwise_cosine.mean().item(), 8))

    print("Minimum pairwise cosine:",
          round(pairwise_cosine.min().item(), 8))

    print("Maximum pairwise cosine:",
          round(pairwise_cosine.max().item(), 8))

    print("Training performed: NO")
    print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

print("STEP 51: CONSTANT TARGET BASELINE")
print("=" * 60)

# Construct a position-wise mean target from training data only
train_target_sum = torch.zeros(200, 100)
train_target_count = torch.zeros(200, 1)

reference_loader = DataLoader(
    controlled_train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

for batch in reference_loader:
    targets = batch["target"]
    valid = (targets.abs().sum(dim=-1) > 0).unsqueeze(-1)

    train_target_sum += (targets * valid).sum(dim=0)
    train_target_count += valid.sum(dim=0)

constant_target = train_target_sum / train_target_count.clamp_min(1)

# Reuse the 32 held-out validation cases from Step 47
constant_predictions = constant_target.unsqueeze(0).expand(
    y.shape[0], -1, -1
)

with torch.no_grad():
    constant_loss = masked_cosine_loss(
        constant_predictions, y
    ).item()

    det_loss = masked_cosine_loss(
        comparison_deterministic(x), y
    ).item()

    mu, _ = comparison_vae.encoder(x)
    vae_loss = masked_cosine_loss(
        comparison_vae.decoder(mu), y
    ).item()

print(f"Constant baseline loss: {constant_loss:.6f}")
print(f"Deterministic model loss: {det_loss:.6f}")
print(f"VAE latent-mean loss:    {vae_loss:.6f}")

print()
print("Deterministic improvement over constant:",
      round(constant_loss - det_loss, 6))

print("VAE improvement over constant:",
      round(constant_loss - vae_loss, 6))

print()
print("Training investigations:", len(controlled_train_df))
print("Validation investigations:", len(x))
print("Test partition accessed: NO")
print("Neural training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

print("STEP 52: CONTRASTIVE SIGNAL CHECK")
print("=" * 60)

# Extract mean GloVe embeddings from the controlled subset
input_vectors = []
target_vectors = []

for batch in reference_loader:
    inputs = batch["input"]
    targets = batch["target"]

    input_mask = (inputs.abs().sum(dim=-1) > 0).float()
    target_mask = (targets.abs().sum(dim=-1) > 0).float()

    input_mean = (
        (inputs * input_mask.unsqueeze(-1)).sum(dim=1)
        / input_mask.sum(dim=1, keepdim=True).clamp_min(1)
    )

    target_mean = (
        (targets * target_mask.unsqueeze(-1)).sum(dim=1)
        / target_mask.sum(dim=1, keepdim=True).clamp_min(1)
    )

    input_vectors.append(input_mean)
    target_vectors.append(target_mean)

input_vectors = F.normalize(
    torch.cat(input_vectors), dim=1
)

target_vectors = F.normalize(
    torch.cat(target_vectors), dim=1
)

similarity = input_vectors @ target_vectors.T

correct_scores = similarity.diag()

# Exclude self-pairings from mismatched comparisons
mask = ~torch.eye(
    len(similarity), dtype=torch.bool
)

mismatched_scores = similarity[mask]

# For each narrative, compare its true target
# against the mean score of incorrect targets
mean_negative = (
    similarity.sum(dim=1) - correct_scores
) / (len(similarity) - 1)

pairing_advantage = correct_scores - mean_negative

print("Mean correct-pair cosine:",
      round(correct_scores.mean().item(), 6))

print("Mean mismatched-pair cosine:",
      round(mismatched_scores.mean().item(), 6))

print("Mean pairing advantage:",
      round(pairing_advantage.mean().item(), 6))

print("Cases with positive pairing advantage:",
      int((pairing_advantage > 0).sum().item()),
      "/", len(pairing_advantage))

print("Training investigations:", len(controlled_train_df))
print("Neural training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

def contrastive_pairing_loss(
    narrative_embeddings,
    cause_embeddings,
    temperature=0.1
):
    narrative_embeddings = F.normalize(
        narrative_embeddings, dim=-1
    )
    cause_embeddings = F.normalize(
        cause_embeddings, dim=-1
    )

    logits = (
        narrative_embeddings @ cause_embeddings.T
    ) / temperature

    labels = torch.arange(
        logits.size(0),
        device=logits.device
    )

    # Symmetric narrative-to-cause and cause-to-narrative loss
    loss_forward = F.cross_entropy(logits, labels)
    loss_backward = F.cross_entropy(logits.T, labels)

    return 0.5 * (loss_forward + loss_backward)


# Test on the first eight controlled training pairs
test_narratives = input_vectors[:8]
test_causes = target_vectors[:8]

correct_loss = contrastive_pairing_loss(
    test_narratives,
    test_causes
)

shuffled_loss = contrastive_pairing_loss(
    test_narratives,
    test_causes.roll(shifts=1, dims=0)
)

print("STEP 53: CONTRASTIVE LOSS CHECK")
print("=" * 55)

print("Correct-pair loss:",
      round(correct_loss.item(), 6))

print("Shuffled-pair loss:",
      round(shuffled_loss.item(), 6))

print("Difference (shuffled - correct):",
      round((shuffled_loss - correct_loss).item(), 6))

print("Expected random-pair loss for batch size 8:",
      round(torch.log(torch.tensor(8.0)).item(), 6))

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ContrastiveNarrativeProjection(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        return F.normalize(self.network(x), dim=-1)


contrastive_model = ContrastiveNarrativeProjection(
    input_dim=100,
    hidden_dim=128
).to("cpu")

with torch.no_grad():
    projected = contrastive_model(input_vectors[:8])

print("STEP 54: CONTRASTIVE PROJECTION MODEL")
print("=" * 60)

print("Input shape:", tuple(input_vectors[:8].shape))
print("Output shape:", tuple(projected.shape))

print("Trainable parameters:",
      sum(
          p.numel()
          for p in contrastive_model.parameters()
          if p.requires_grad
      ))

print("Finite outputs:",
      bool(torch.isfinite(projected).all()))

print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import time
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

contrastive_train_dataset = TensorDataset(
    input_vectors.float(),
    target_vectors.float()
)

contrastive_generator = torch.Generator().manual_seed(42)

contrastive_train_loader = DataLoader(
    contrastive_train_dataset,
    batch_size=16,
    shuffle=True,
    generator=contrastive_generator,
    num_workers=0
)

contrastive_optimizer = torch.optim.Adam(
    contrastive_model.parameters(),
    lr=0.001
)

contrastive_model.train()

start_time = time.time()

print("STEP 55: CONTRASTIVE MODEL TRAINING")
print("=" * 65)

for epoch in range(1, 21):
    total_loss = 0.0
    total_samples = 0

    for narrative_batch, cause_batch in contrastive_train_loader:
        contrastive_optimizer.zero_grad()

        predicted_embeddings = contrastive_model(
            narrative_batch
        )

        loss = contrastive_pairing_loss(
            predicted_embeddings,
            cause_batch,
            temperature=0.1
        )

        loss.backward()
        contrastive_optimizer.step()

        batch_size = narrative_batch.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"Training contrastive loss: "
            f"{total_loss / total_samples:.6f}"
        )

print("=" * 65)
print("Elapsed seconds:", round(time.time() - start_time, 2))
print("Training investigations:", len(contrastive_train_dataset))
print("Epochs:", 20)
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

print("STEP 56: HELD-OUT CONTRASTIVE VALIDATION")
print("=" * 65)

contrastive_model.eval()

validation_dataset = AviationGloveDataset(val_df)

validation_eval_loader = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

val_input_vectors = []
val_target_vectors = []

with torch.no_grad():
    for batch in validation_eval_loader:
        inputs = batch["input"]
        targets = batch["target"]

        input_mask = (
            inputs.abs().sum(dim=-1) > 0
        ).float()

        target_mask = (
            targets.abs().sum(dim=-1) > 0
        ).float()

        input_mean = (
            (inputs * input_mask.unsqueeze(-1)).sum(dim=1)
            / input_mask.sum(
                dim=1, keepdim=True
            ).clamp_min(1)
        )

        target_mean = (
            (targets * target_mask.unsqueeze(-1)).sum(dim=1)
            / target_mask.sum(
                dim=1, keepdim=True
            ).clamp_min(1)
        )

        val_input_vectors.append(input_mean)
        val_target_vectors.append(target_mean)

val_input_vectors = F.normalize(
    torch.cat(val_input_vectors), dim=1
)

val_target_vectors = F.normalize(
    torch.cat(val_target_vectors), dim=1
)

with torch.no_grad():
    projected_val_vectors = contrastive_model(
        val_input_vectors
    )

def evaluate_pairing(narrative_vectors, cause_vectors):
    similarity = narrative_vectors @ cause_vectors.T

    correct = similarity.diag()

    mean_negative = (
        similarity.sum(dim=1) - correct
    ) / (similarity.size(1) - 1)

    advantage = correct - mean_negative

    top1_accuracy = (
        similarity.argmax(dim=1)
        == torch.arange(similarity.size(0))
    ).float().mean()

    return {
        "correct_cosine": correct.mean().item(),
        "negative_cosine": mean_negative.mean().item(),
        "pairing_advantage": advantage.mean().item(),
        "positive_advantage_rate":
            (advantage > 0).float().mean().item(),
        "top1_accuracy": top1_accuracy.item()
    }

baseline_results = evaluate_pairing(
    val_input_vectors,
    val_target_vectors
)

projection_results = evaluate_pairing(
    projected_val_vectors,
    val_target_vectors
)

print(f"Validation investigations: {len(val_df)}")
print()

for name, results in [
    ("UNMODIFIED GLOVE", baseline_results),
    ("TRAINED CONTRASTIVE PROJECTION", projection_results)
]:
    print(name)
    for metric, value in results.items():
        print(f"  {metric}: {value:.6f}")
    print()

print("Test partition accessed: NO")
print("Training performed: NO")
print("Device: CPU")


In [ ]:

import torch
import torch.nn.functional as F

def retrieval_metrics(narrative_vectors, cause_vectors):
    similarity = narrative_vectors @ cause_vectors.T
    n = similarity.shape[0]

    # Rank of each investigation's correct probable cause
    correct_scores = similarity.diag().unsqueeze(1)

    ranks = (
        (similarity > correct_scores).sum(dim=1) + 1
    )

    return {
        "Recall@1": (ranks <= 1).float().mean().item(),
        "Recall@5": (ranks <= 5).float().mean().item(),
        "Recall@10": (ranks <= 10).float().mean().item(),
        "MRR": (1.0 / ranks.float()).mean().item(),
        "Median rank": ranks.float().median().item(),
        "Mean rank": ranks.float().mean().item()
    }

print("STEP 57: HELD-OUT RETRIEVAL RANKING")
print("=" * 65)

for name, narrative_vectors in [
    ("UNMODIFIED GLOVE", val_input_vectors),
    ("CONTRASTIVE PROJECTION", projected_val_vectors)
]:
    metrics = retrieval_metrics(
        narrative_vectors,
        val_target_vectors
    )

    print(name)

    for metric, value in metrics.items():
        print(f"  {metric}: {value:.6f}")

    print()

print("Validation investigations:", len(val_df))
print("Training performed: NO")
print("Test partition accessed: NO")
print("Device: CPU")


In [ ]:

import pandas as pd

records = [
    {
        "experiment": "constant_target_baseline",
        "dataset": "32_validation_cases",
        "metric": "masked_cosine_reconstruction_loss",
        "value": 0.342906
    },
    {
        "experiment": "deterministic_conv_encoder_decoder",
        "dataset": "32_validation_cases",
        "metric": "masked_cosine_reconstruction_loss",
        "value": 0.334911
    },
    {
        "experiment": "reconstructed_conv_vae_latent_mean",
        "dataset": "32_validation_cases",
        "metric": "masked_cosine_reconstruction_loss",
        "value": 0.341045
    },
    {
        "experiment": "unmodified_glove",
        "dataset": "1779_validation_cases",
        "metric": "recall_at_1",
        "value": 0.029230
    },
    {
        "experiment": "contrastive_projection",
        "dataset": "1779_validation_cases",
        "metric": "recall_at_1",
        "value": 0.003373
    },
    {
        "experiment": "unmodified_glove",
        "dataset": "1779_validation_cases",
        "metric": "recall_at_10",
        "value": 0.087128
    },
    {
        "experiment": "contrastive_projection",
        "dataset": "1779_validation_cases",
        "metric": "recall_at_10",
        "value": 0.034289
    },
    {
        "experiment": "unmodified_glove",
        "dataset": "1779_validation_cases",
        "metric": "mean_reciprocal_rank",
        "value": 0.051252
    },
    {
        "experiment": "contrastive_projection",
        "dataset": "1779_validation_cases",
        "metric": "mean_reciprocal_rank",
        "value": 0.018334
    },
    {
        "experiment": "unmodified_glove",
        "dataset": "1779_validation_cases",
        "metric": "median_rank",
        "value": 441
    },
    {
        "experiment": "contrastive_projection",
        "dataset": "1779_validation_cases",
        "metric": "median_rank",
        "value": 237
    }
]

results_df = pd.DataFrame(records)

output_path = "/content/aviation_reviewer2_experimental_results.csv"

results_df.to_csv(output_path, index=False)

print("STEP 58: RESULTS SAVED")
print("=" * 60)
print("File:", output_path)
print("Records:", len(results_df))
print("Historical results included: NO")
print("Test-set results included: NO")
print("Training performed: NO")


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

backup_dir = Path(
    "/content/drive/MyDrive/Aviation_Reviewer2_Revision"
)
backup_dir.mkdir(parents=True, exist_ok=True)

files_to_backup = [
    "/content/aviation_reviewer2_experimental_results.csv",
    "/content/aviation_glove_validation_diagnostics.csv"
]

print("STEP 59: EXPERIMENTAL RESULTS BACKUP")
print("=" * 60)

for source in files_to_backup:
    source_path = Path(source)

    if source_path.exists():
        destination = backup_dir / source_path.name
        shutil.copy2(source_path, destination)

        print("Saved:", destination)
        print("Size:", destination.stat().st_size, "bytes")
    else:
        print("NOT FOUND:", source)

print("=" * 60)
print("Backup folder:", backup_dir)
print("Training performed: NO")


In [ ]:

from pathlib import Path
import torch

output_dir = Path(
    "/content/drive/MyDrive/Aviation_Reviewer2_Revision"
)

output_dir.mkdir(parents=True, exist_ok=True)

specification_path = output_dir / "reconstructed_model_specifications.txt"

with open(specification_path, "w", encoding="utf-8") as f:
    f.write("RECONSTRUCTED MODEL SPECIFICATIONS\n")
    f.write("=" * 70 + "\n\n")

    f.write("IMPORTANT: These are new experimental models.\n")
    f.write("They are not verified reproductions of the historical model.\n\n")

    f.write("Dataset:\n")
    f.write("Sample-size-matched cohort: 20,919 investigations\n")
    f.write("Controlled training subset: 128 investigations\n")
    f.write("Validation partition: 1,779 investigations\n")
    f.write("Test partition: 3,138 investigations (not accessed)\n\n")

    f.write("Input representation:\n")
    f.write("Pretrained GloVe 6B, 100-dimensional embeddings\n")
    f.write("Maximum narrative length: 1,200 tokens\n")
    f.write("Maximum target length: 200 tokens\n\n")

    for name, experimental_model in [
        ("Deterministic encoder-decoder", comparison_deterministic),
        ("Reconstructed VAE", comparison_vae),
        ("Contrastive projection", contrastive_model)
    ]:
        f.write("\n" + "=" * 70 + "\n")
        f.write(name + "\n")
        f.write("=" * 70 + "\n")

        total = sum(
            p.numel() for p in experimental_model.parameters()
        )

        trainable = sum(
            p.numel() for p in experimental_model.parameters()
            if p.requires_grad
        )

        f.write(f"Total parameters: {total}\n")
        f.write(f"Trainable parameters: {trainable}\n\n")
        f.write(str(experimental_model) + "\n")

    f.write("\nExperiment notes:\n")
    f.write("Convolutional pilot: 5 epochs, 128 training cases.\n")
    f.write("Contrastive projection: 20 epochs, 128 training cases.\n")
    f.write("All experiments performed on CPU.\n")
    f.write("Historical cosine score not reproduced.\n")

print("STEP 60: MODEL SPECIFICATIONS SAVED")
print("=" * 60)
print("File:", specification_path)
print("Size:", specification_path.stat().st_size, "bytes")
print("Historical architecture claimed as reproduced: NO")
print("Training performed: NO")


In [ ]:

import sys
import importlib.util
from pathlib import Path

print("STEP 63: RECONSTRUCTION ENVIRONMENT CHECK")
print("=" * 65)

required_variables = [
    "dataset",
    "matched_dataset",
    "train_df",
    "val_df",
    "test_df",
    "glove_vectors",
    "text_to_glove",
    "MAX_INPUT_LEN",
    "MAX_TARGET_LEN",
    "EMBEDDING_DIM"
]

for name in required_variables:
    if name in globals():
        obj = globals()[name]

        if hasattr(obj, "shape"):
            detail = f"shape={obj.shape}"
        elif isinstance(obj, dict):
            detail = f"entries={len(obj)}"
        else:
            detail = f"type={type(obj).__name__}"

        print(f"AVAILABLE: {name} | {detail}")
    else:
        print(f"MISSING:   {name}")

print("\nSoftware availability:")
for package in ["tensorflow", "torch", "numpy", "pandas"]:
    spec = importlib.util.find_spec(package)
    print(f"{package}: {'AVAILABLE' if spec else 'MISSING'}")

print("\nSaved evidence:")
backup_dir = Path(
    "/content/drive/MyDrive/Aviation_Reviewer2_Revision"
)

for filename in [
    "aviation_reviewer2_experimental_results.csv",
    "aviation_glove_validation_diagnostics.csv",
    "reconstructed_model_specifications.txt"
]:
    path = backup_dir / filename
    print(f"{filename}: {'SAVED' if path.exists() else 'NOT FOUND'}")

print("\nTraining performed: NO")
print("Historical model modified: NO")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model

print("STEP 64: HISTORICAL CONV1D ENCODER RECONSTRUCTION")
print("=" * 65)

encoder_input = layers.Input(
    shape=(1200, 1),
    name="historical_encoder_input"
)

x = encoder_input

encoder_specs = [
    (256, 36, 9),
    (128, 25, 8),
    (128, 25, 4),
    (64, 16, 4),
    (64, 9, 1),
    (64, 4, 1),
    (64, 2, 1),
    (64, 4, 1),
    (64, 4, 1),
    (64, 2, 1),
    (64, 1, 1)
]

for index, (filters, kernel, stride) in enumerate(
    encoder_specs, start=1
):
    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel,
        strides=stride,
        padding="same",
        name=f"historical_conv_{index}"
    )(x)

    x = layers.LeakyReLU(
        negative_slope=0.2,
        name=f"historical_activation_{index}"
    )(x)

x = layers.Flatten(name="historical_flatten")(x)

latent = layers.Dense(
    20,
    name="historical_latent_projection"
)(x)

historical_encoder = Model(
    encoder_input,
    latent,
    name="historical_encoder_reconstruction"
)

historical_encoder.summary()

print("\nVerification:")
print("Input shape:", historical_encoder.input_shape)
print("Output shape:", historical_encoder.output_shape)
print("Trainable parameters:", historical_encoder.count_params())
print("Reported historical parameters:", 1477908)
print(
    "Parameter count matches:",
    historical_encoder.count_params() == 1477908
)

print("\nTraining performed: NO")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model

print("STEP 65: ENCODER DIMENSION VERIFICATION")
print("=" * 65)

inputs = layers.Input(shape=(1200, 1))

x = inputs

specs = [
    (256, 36, 9, "same"),
    (128, 25, 8, "same"),
    (128, 25, 5, "same"),
    (64, 16, 4, "same"),
    (64, 9, 1, "same"),
    (64, 4, 1, "same"),
    (64, 2, 1, "same"),
    (64, 4, 1, "same"),
    (64, 4, 1, "same"),
    (64, 2, 1, "same"),
    (64, 1, 1, "same")
]

for i, (filters, kernel, stride, padding) in enumerate(specs, 1):
    x = layers.Conv1D(
        filters,
        kernel_size=kernel,
        strides=stride,
        padding=padding,
        name=f"verified_conv_{i}"
    )(x)

    x = layers.LeakyReLU(
        negative_slope=0.2
    )(x)

    print(f"Conv{i:02d} output:", tuple(x.shape[1:]))

x = layers.Flatten()(x)
latent = layers.Dense(20)(x)

verified_encoder = Model(inputs, latent)

print("\nFINAL VERIFICATION")
print("=" * 65)
print("Encoder output:", verified_encoder.output_shape)
print("Encoder parameters:", verified_encoder.count_params())
print("Reported parameters:", 1477908)

print(
    "PARAMETER MATCH:",
    verified_encoder.count_params() == 1477908
)

print("\nTraining performed: NO")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model

print("STEP 66: HISTORICAL DECODER RECONSTRUCTION")
print("=" * 65)

decoder_input = layers.Input(shape=(20,), name="decoder_input")

x = layers.Dense(64, name="decoder_dense")(decoder_input)
x = layers.LeakyReLU(negative_slope=0.2)(x)
x = layers.Reshape((1, 64))(x)

target_lengths = [3, 6, 19, 22, 47, 47, 96, 194, 197, 199, 200, 200]
filters = [64] * 11 + [1]
kernels = [4, 4, 4, 4, 5, 4, 4, 4, 3, 2, 1, 1]

for i, (n_filters, kernel, target_len) in enumerate(
    zip(filters, kernels, target_lengths), start=1
):
    x = layers.Conv1DTranspose(
        filters=n_filters,
        kernel_size=kernel,
        strides=1,
        padding="valid",
        name=f"decoder_deconv_{i}"
    )(x)

    current_len = int(x.shape[1])

    if current_len < target_len:
        x = layers.ZeroPadding1D(
            (0, target_len - current_len),
            name=f"decoder_pad_{i}"
        )(x)
    elif current_len > target_len:
        x = layers.Cropping1D(
            (0, current_len - target_len),
            name=f"decoder_crop_{i}"
        )(x)

    x = layers.LeakyReLU(
        negative_slope=0.2,
        name=f"decoder_activation_{i}"
    )(x)

    print(f"Layer {i:02d}: {tuple(x.shape[1:])}")

historical_decoder = Model(
    decoder_input,
    x,
    name="historical_decoder_reconstruction"
)

print("\nFINAL VERIFICATION")
print("=" * 65)
print("Decoder input:", historical_decoder.input_shape)
print("Decoder output:", historical_decoder.output_shape)
print("Decoder parameters:", historical_decoder.count_params())
print("Reported parameters:", 161857)

print(
    "PARAMETER MATCH:",
    historical_decoder.count_params() == 161857
)

print(
    "COMBINED PARAMETERS:",
    verified_encoder.count_params() +
    historical_decoder.count_params()
)

print("\nTraining performed: NO")


In [ ]:

import tensorflow as tf
from tensorflow.keras import Model

print("STEP 67: COMPLETE ARCHITECTURE VERIFICATION")
print("=" * 65)

# Connect the reconstructed encoder and decoder.
inputs = tf.keras.Input(
    shape=(1200, 1),
    name="narrative_input"
)

latent = verified_encoder(inputs)
outputs = historical_decoder(latent)

reconstructed_model = Model(
    inputs,
    outputs,
    name="reconstructed_historical_encoder_decoder"
)

# Check model dimensions and parameter count.
print("Input shape:", reconstructed_model.input_shape)
print("Latent shape:", verified_encoder.output_shape)
print("Output shape:", reconstructed_model.output_shape)
print("Total parameters:", reconstructed_model.count_params())

# Synthetic forward-pass verification.
tf.random.set_seed(42)

sample_input = tf.random.uniform(
    shape=(2, 1200, 1),
    dtype=tf.float32
)

sample_target = tf.random.uniform(
    shape=(2, 200, 1),
    dtype=tf.float32
)

with tf.GradientTape() as tape:
    predictions = reconstructed_model(
        sample_input,
        training=True
    )

    cosine = tf.keras.losses.cosine_similarity(
        sample_target,
        predictions,
        axis=-1
    )

    loss = tf.reduce_mean(1.0 + cosine)

gradients = tape.gradient(
    loss,
    reconstructed_model.trainable_variables
)

valid_gradients = sum(
    g is not None for g in gradients
)

total_variables = len(
    reconstructed_model.trainable_variables
)

print("\nForward-pass output:", predictions.shape)
print("Synthetic loss:", float(loss.numpy()))
print("Variables with gradients:", valid_gradients)
print("Total trainable variables:", total_variables)

print("\nVerification:")
print(
    "Parameter match:",
    reconstructed_model.count_params() == 1639765
)
print(
    "Output shape match:",
    tuple(predictions.shape) == (2, 200, 1)
)
print(
    "All gradients available:",
    valid_gradients == total_variables
)

print("\nTraining performed: NO")
print("Historical cosine score reproduced: NO")


In [ ]:

print("STEP 68: PREPROCESSING OBJECT INSPECTION")
print("=" * 65)

keywords = [
    "tokenizer",
    "vocab",
    "sequence",
    "scaler",
    "glove",
    "embedding",
    "normalize"
]

matches = []

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(keyword in name.lower() for keyword in keywords):
        matches.append((name, type(value).__name__))

for name, obj_type in sorted(matches):
    print(f"{name}: {obj_type}")

print("\nRepresentation constants:")
for name in [
    "MAX_INPUT_LEN",
    "MAX_TARGET_LEN",
    "EMBEDDING_DIM"
]:
    if name in globals():
        print(f"{name}: {globals()[name]}")

print("\nTraining performed: NO")
print("Historical preprocessing verified: NO")


In [ ]:

import numpy as np

print("STEP 69: GLOVE-TO-SINGLE-CHANNEL DIAGNOSTIC")
print("=" * 65)

sample_texts = train_df["input_text"].iloc[:8].tolist()

representations = []

for text in sample_texts:
    matrix, token_count = text_to_glove(
        text,
        MAX_INPUT_LEN
    )

    # Candidate transformation:
    # Average the 100 embedding dimensions for each token.
    single_channel = matrix.mean(
        axis=1,
        keepdims=True
    )

    representations.append(single_channel)

representations = np.stack(representations)

print("GloVe dimension:", EMBEDDING_DIM)
print("Single-channel tensor shape:", representations.shape)

print("\nNumerical statistics:")
print("Minimum:", float(representations.min()))
print("Maximum:", float(representations.max()))
print("Mean:", float(representations.mean()))
print("Standard deviation:", float(representations.std()))

# Compare representations across investigations.
flat = representations.reshape(8, -1)

norms = np.linalg.norm(flat, axis=1)
safe_norms = np.maximum(norms, 1e-8)

normalized_flat = flat / safe_norms[:, None]
similarity = normalized_flat @ normalized_flat.T

off_diagonal = similarity[
    ~np.eye(8, dtype=bool)
]

print("\nCross-investigation cosine similarity:")
print("Mean:", float(off_diagonal.mean()))
print("Minimum:", float(off_diagonal.min()))
print("Maximum:", float(off_diagonal.max()))

print("\nTraining performed: NO")
print("Historical transformation verified: NO")


In [ ]:

import numpy as np
from sklearn.decomposition import PCA

print("STEP 70: SINGLE-CHANNEL GLOVE PROJECTION COMPARISON")
print("=" * 65)

# Use the available GloVe vocabulary.
vectors = np.asarray(
    list(glove_vectors.values()),
    dtype=np.float32
)

print("Available GloVe vectors:", vectors.shape)

# Fit a one-dimensional PCA projection.
pca = PCA(n_components=1, random_state=42)
pca.fit(vectors)

print("PCA explained variance ratio:",
      float(pca.explained_variance_ratio_[0]))

sample_texts = train_df["input_text"].iloc[:8].tolist()

mean_sequences = []
pca_sequences = []

for text in sample_texts:
    matrix, count = text_to_glove(text, MAX_INPUT_LEN)

    mean_projection = matrix.mean(axis=1)

    # Preserve zero padding explicitly.
    pca_projection = np.zeros(
        MAX_INPUT_LEN,
        dtype=np.float32
    )

    valid_count = min(int(count), MAX_INPUT_LEN)

    if valid_count > 0:
        pca_projection[:valid_count] = (
            pca.transform(matrix[:valid_count])
            .reshape(-1)
        )

    mean_sequences.append(mean_projection)
    pca_sequences.append(pca_projection)

mean_sequences = np.stack(mean_sequences)
pca_sequences = np.stack(pca_sequences)

def off_diagonal_cosine_stats(arr):
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    normalized = arr / np.maximum(norms, 1e-8)
    similarity = normalized @ normalized.T

    values = similarity[~np.eye(len(arr), dtype=bool)]

    return (
        float(values.mean()),
        float(values.min()),
        float(values.max())
    )

print("\nMean-coordinate projection:")
print("Shape:", mean_sequences.shape)
print("Cosine mean/min/max:",
      off_diagonal_cosine_stats(mean_sequences))

print("\nPCA projection:")
print("Shape:", pca_sequences.shape)
print("Cosine mean/min/max:",
      off_diagonal_cosine_stats(pca_sequences))

print("\nTraining performed: NO")
print("Historical projection identified: NO")


In [ ]:

import numpy as np

print("STEP 71: PAIRED REPRESENTATION PREPARATION")
print("=" * 65)

def glove_to_single_channel(text, max_length):
    matrix, token_count = text_to_glove(text, max_length)

    values = matrix.mean(
        axis=1,
        keepdims=True
    )

    return values.astype(np.float32)

def prepare_pairs(frame, n):
    subset = frame.iloc[:n]

    X = np.stack([
        glove_to_single_channel(text, MAX_INPUT_LEN)
        for text in subset["input_text"]
    ])

    Y = np.stack([
        glove_to_single_channel(text, MAX_TARGET_LEN)
        for text in subset["target_text"]
    ])

    return X, Y

X_train_small, Y_train_small = prepare_pairs(train_df, 64)
X_val_small, Y_val_small = prepare_pairs(val_df, 32)

print("Training input:", X_train_small.shape)
print("Training target:", Y_train_small.shape)
print("Validation input:", X_val_small.shape)
print("Validation target:", Y_val_small.shape)

print("\nData checks:")
print("Training finite:", np.isfinite(X_train_small).all()
      and np.isfinite(Y_train_small).all())
print("Validation finite:", np.isfinite(X_val_small).all()
      and np.isfinite(Y_val_small).all())

print("\nTraining performed: NO")


In [ ]:

import numpy as np
import tensorflow as tf

print("STEP 72: RECONSTRUCTED MODEL TRAINING PILOT")
print("=" * 65)

tf.random.set_seed(42)
np.random.seed(42)

# Cosine loss over the complete output sequence.
def sequence_cosine_loss(y_true, y_pred):
    true_flat = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    pred_flat = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])

    true_unit = tf.math.l2_normalize(true_flat, axis=1)
    pred_unit = tf.math.l2_normalize(pred_flat, axis=1)

    cosine = tf.reduce_sum(true_unit * pred_unit, axis=1)
    return tf.reduce_mean(1.0 - cosine)

reconstructed_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=sequence_cosine_loss,
    run_eagerly=False
)

history_72 = reconstructed_model.fit(
    X_train_small,
    Y_train_small,
    validation_data=(X_val_small, Y_val_small),
    epochs=3,
    batch_size=8,
    verbose=2,
    shuffle=True
)

print("\nFINAL RESULTS")
print("=" * 65)

print(
    "Training cosine similarity:",
    1.0 - history_72.history["loss"][-1]
)

print(
    "Validation cosine similarity:",
    1.0 - history_72.history["val_loss"][-1]
)

print("\nTraining completed: YES")
print("Historical model reproduced: NOT YET")


In [ ]:

import numpy as np

print("STEP 73: NARRATIVE-SPECIFICITY EVALUATION")
print("=" * 65)

predictions = reconstructed_model.predict(
    X_val_small,
    batch_size=8,
    verbose=0
)

def cosine_per_case(a, b):
    a = a.reshape(len(a), -1)
    b = b.reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-8)

# Correct targets
correct_scores = cosine_per_case(
    predictions,
    Y_val_small
)

# Deranged permutation: no case keeps its own target.
rng = np.random.default_rng(42)
n = len(Y_val_small)

shift = int(rng.integers(1, n))
shuffled_targets = np.roll(
    Y_val_small,
    shift=shift,
    axis=0
)

shuffled_scores = cosine_per_case(
    predictions,
    shuffled_targets
)

# Constant mean-target baseline.
mean_target = Y_train_small.mean(
    axis=0,
    keepdims=True
)

constant_predictions = np.repeat(
    mean_target,
    n,
    axis=0
)

constant_scores = cosine_per_case(
    constant_predictions,
    Y_val_small
)

print("Correct-target cosine:",
      float(correct_scores.mean()))

print("Shuffled-target cosine:",
      float(shuffled_scores.mean()))

print("Constant-baseline cosine:",
      float(constant_scores.mean()))

print("\nNarrative-specificity gap:",
      float(correct_scores.mean() - shuffled_scores.mean()))

print("\nPrediction diversity:")
flat_predictions = predictions.reshape(n, -1)

print(
    "Mean prediction standard deviation:",
    float(flat_predictions.std(axis=0).mean())
)

print("\nTraining performed: NO")


In [ ]:

import numpy as np

print("STEP 74: TARGET REPRESENTATION DIAGNOSTICS")
print("=" * 65)

targets = Y_train_small.reshape(len(Y_train_small), -1)

norms = np.linalg.norm(targets, axis=1)
nonzero = np.abs(targets) > 1e-8

print("Target sequence length:", targets.shape[1])
print("Mean target norm:", float(norms.mean()))
print("Minimum target norm:", float(norms.min()))
print("Maximum target norm:", float(norms.max()))

print("\nNonzero values:")
print("Mean count:", float(nonzero.sum(axis=1).mean()))
print("Mean proportion:", float(nonzero.mean()))

# Cosine similarity between different training targets.
unit_targets = targets / np.maximum(norms[:, None], 1e-8)
similarity = unit_targets @ unit_targets.T

off_diagonal = similarity[
    ~np.eye(len(targets), dtype=bool)
]

print("\nBetween-target cosine similarity:")
print("Mean:", float(off_diagonal.mean()))
print("Minimum:", float(off_diagonal.min()))
print("Maximum:", float(off_diagonal.max()))

# Measure whether the mean target has a dominant direction.
mean_target_vector = targets.mean(axis=0)

print("\nMean target norm:",
      float(np.linalg.norm(mean_target_vector)))

print(
    "Mean target norm / mean individual norm:",
    float(np.linalg.norm(mean_target_vector) /
          max(norms.mean(), 1e-8))
)

print("\nTraining performed: NO")


In [ ]:

import numpy as np

print("STEP 75: ENCODER LATENT DIVERSITY")
print("=" * 65)

latent_vectors = verified_encoder.predict(
    X_val_small,
    batch_size=8,
    verbose=0
)

print("Latent shape:", latent_vectors.shape)

print("\nLatent statistics:")
print("Mean:", float(latent_vectors.mean()))
print("Standard deviation:", float(latent_vectors.std()))
print(
    "Mean dimension-wise standard deviation:",
    float(latent_vectors.std(axis=0).mean())
)

def pairwise_cosine_stats(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    normalized = values / np.maximum(norms, 1e-8)
    similarity = normalized @ normalized.T

    off_diagonal = similarity[
        ~np.eye(len(values), dtype=bool)
    ]

    return (
        float(off_diagonal.mean()),
        float(off_diagonal.min()),
        float(off_diagonal.max())
    )

print("\nBetween-investigation latent cosine:")
print(
    "Mean / min / max:",
    pairwise_cosine_stats(latent_vectors)
)

print("\nFirst three latent vectors:")
print(np.round(latent_vectors[:3], 5))

print("\nTraining performed: NO")


In [ ]:

import numpy as np
import tensorflow as tf

print("STEP 76: ENCODER COLLAPSE DIAGNOSTIC")
print("=" * 65)

# Clone architecture with fresh random weights.
fresh_encoder = tf.keras.models.clone_model(
    verified_encoder
)

fresh_latents = fresh_encoder.predict(
    X_val_small,
    batch_size=8,
    verbose=0
)

trained_latents = verified_encoder.predict(
    X_val_small,
    batch_size=8,
    verbose=0
)

def diversity_statistics(values):
    dimension_std = values.std(axis=0).mean()

    distances = np.linalg.norm(
        values[:, None, :] - values[None, :, :],
        axis=-1
    )

    upper = distances[
        np.triu_indices(len(values), k=1)
    ]

    return (
        float(dimension_std),
        float(upper.mean()),
        float(upper.max())
    )

fresh_stats = diversity_statistics(fresh_latents)
trained_stats = diversity_statistics(trained_latents)

print("Fresh encoder:")
print("Mean dimension standard deviation:", fresh_stats[0])
print("Mean pairwise distance:", fresh_stats[1])
print("Maximum pairwise distance:", fresh_stats[2])

print("\nTrained encoder:")
print("Mean dimension standard deviation:", trained_stats[0])
print("Mean pairwise distance:", trained_stats[1])
print("Maximum pairwise distance:", trained_stats[2])

print("\nTraining performed: NO")


In [ ]:

import numpy as np

print("STEP 77: ENCODER INPUT-SCALE SENSITIVITY")
print("=" * 65)

sample_inputs = X_val_small[:8]

scales = [1.0, 10.0, 50.0, 100.0]

print("Original input standard deviation:",
      float(sample_inputs.std()))

print("\nScale | Latent std | Mean pairwise distance")
print("-" * 55)

for scale in scales:
    scaled_inputs = sample_inputs * scale

    z = fresh_encoder.predict(
        scaled_inputs,
        batch_size=8,
        verbose=0
    )

    latent_std = float(
        z.std(axis=0).mean()
    )

    distances = np.linalg.norm(
        z[:, None, :] - z[None, :, :],
        axis=-1
    )

    pairwise = distances[
        np.triu_indices(len(z), k=1)
    ]

    print(
        f"{scale:5.1f} | "
        f"{latent_std:.10f} | "
        f"{pairwise.mean():.10f}"
    )

print("\nTraining performed: NO")
print("Historical input scaling identified: NO")


In [ ]:

import numpy as np
import tensorflow as tf

print("STEP 78: LAYER-BY-LAYER ENCODER DIAGNOSTIC")
print("=" * 75)

# Inspect every layer output from the fresh encoder.
inspection_model = tf.keras.Model(
    inputs=fresh_encoder.input,
    outputs=[
        layer.output
        for layer in fresh_encoder.layers
    ]
)

layer_outputs = inspection_model.predict(
    X_val_small[:8],
    batch_size=8,
    verbose=0
)

print(
    f"{'Layer':30s} "
    f"{'Output shape':20s} "
    f"{'Across-case std':>16s} "
    f"{'Output RMS':>14s}"
)
print("-" * 90)

for layer, output in zip(
    fresh_encoder.layers,
    layer_outputs
):
    values = np.asarray(output)

    # Average variation across cases at each output coordinate.
    across_case_std = float(
        values.std(axis=0).mean()
    )

    rms = float(
        np.sqrt(np.mean(values ** 2))
    )

    print(
        f"{layer.name[:30]:30s} "
        f"{str(values.shape):20s} "
        f"{across_case_std:16.10f} "
        f"{rms:14.10f}"
    )

print("\nTraining performed: NO")


In [ ]:

print("STEP 79: RECONSTRUCTED ENCODER ARCHITECTURE AUDIT")
print("=" * 75)

receptive_field = 1
effective_stride = 1

for layer in verified_encoder.layers:
    if isinstance(layer, tf.keras.layers.Conv1D):
        kernel = layer.kernel_size[0]
        stride = layer.strides[0]
        dilation = layer.dilation_rate[0]

        receptive_field += (
            (kernel - 1) *
            dilation *
            effective_stride
        )

        effective_stride *= stride

        print(f"\nLayer: {layer.name}")
        print("  Kernel size:", kernel)
        print("  Stride:", stride)
        print("  Padding:", layer.padding)
        print("  Output shape:", layer.output.shape)
        print("  Theoretical receptive field:", receptive_field)
        print("  Effective stride:", effective_stride)

print("\nOriginal input length:", MAX_INPUT_LEN)
print("Final theoretical receptive field:", receptive_field)
print("Final effective stride:", effective_stride)

print("\nTraining performed: NO")


In [ ]:

import numpy as np
from sklearn.decomposition import PCA

print("STEP 80: GLOVE PROJECTION COMPARISON")
print("=" * 65)

# Fit a one-dimensional PCA projection on the currently loaded
# GloVe vocabulary. This is a diagnostic, not a leakage-controlled
# final preprocessing pipeline.
vocabulary_matrix = np.stack(
    list(glove_vectors.values())
).astype(np.float32)

pca_1d = PCA(n_components=1, random_state=42)
pca_1d.fit(vocabulary_matrix)

def pca_single_channel(text, max_length):
    matrix, _ = text_to_glove(text, max_length)

    # Preserve zero padding as zero, rather than applying
    # PCA centering to padded positions.
    active = np.any(matrix != 0, axis=1)

    output = np.zeros(
        (max_length, 1),
        dtype=np.float32
    )

    if active.any():
        output[active] = pca_1d.transform(
            matrix[active]
        ).astype(np.float32)

    return output

texts = val_df["input_text"].iloc[:8]

X_mean = X_val_small[:8]

X_pca = np.stack([
    pca_single_channel(text, MAX_INPUT_LEN)
    for text in texts
])

def encoder_diversity(X):
    z = fresh_encoder.predict(
        X,
        batch_size=8,
        verbose=0
    )

    distances = np.linalg.norm(
        z[:, None, :] - z[None, :, :],
        axis=-1
    )

    pairwise = distances[
        np.triu_indices(len(z), k=1)
    ]

    return (
        float(z.std(axis=0).mean()),
        float(pairwise.mean())
    )

mean_stats = encoder_diversity(X_mean)
pca_stats = encoder_diversity(X_pca)

print("PCA explained variance ratio:",
      float(pca_1d.explained_variance_ratio_[0]))

print("\nMean-coordinate projection:")
print("Input standard deviation:", float(X_mean.std()))
print("Latent diversity:", mean_stats[0])
print("Mean latent distance:", mean_stats[1])

print("\nPCA projection:")
print("Input standard deviation:", float(X_pca.std()))
print("Latent diversity:", pca_stats[0])
print("Mean latent distance:", pca_stats[1])

print("\nTraining performed: NO")


In [ ]:

import numpy as np

print("STEP 81: SCALE-CONTROLLED GLOVE COMPARISON")
print("=" * 65)

# Match the overall standard deviation of PCA inputs
# to that of the mean-coordinate inputs.
mean_scale = float(X_mean.std())
pca_scale = float(X_pca.std())

X_pca_matched = X_pca * (
    mean_scale / max(pca_scale, 1e-8)
)

# Also compare both representations at unit standard deviation.
X_mean_unit = X_mean / max(mean_scale, 1e-8)
X_pca_unit = X_pca / max(pca_scale, 1e-8)

conditions = {
    "Mean-coordinate (original)": X_mean,
    "PCA (matched scale)": X_pca_matched,
    "Mean-coordinate (unit scale)": X_mean_unit,
    "PCA (unit scale)": X_pca_unit,
}

print(
    f"{'Representation':30s} "
    f"{'Input std':>12s} "
    f"{'Latent distance':>18s}"
)
print("-" * 65)

for name, inputs in conditions.items():
    _, distance = encoder_diversity(inputs)

    print(
        f"{name:30s} "
        f"{inputs.std():12.7f} "
        f"{distance:18.10f}"
    )

print("\nTraining performed: NO")


In [ ]:

import numpy as np
from sklearn.linear_model import Ridge

print("STEP 82: GLOVE RIDGE BASELINE")
print("=" * 65)

def pooled_glove(text):
    matrix, _ = text_to_glove(text, MAX_INPUT_LEN)
    active = np.any(matrix != 0, axis=1)

    if not active.any():
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)

    return matrix[active].mean(axis=0).astype(np.float32)

def pooled_target(text):
    matrix, _ = text_to_glove(text, MAX_TARGET_LEN)
    active = np.any(matrix != 0, axis=1)

    if not active.any():
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)

    return matrix[active].mean(axis=0).astype(np.float32)

# Small, controlled training and validation subsets.
train_subset = train_df.iloc[:512]
val_subset = val_df.iloc[:128]

X_train_82 = np.stack([
    pooled_glove(t) for t in train_subset["input_text"]
])
Y_train_82 = np.stack([
    pooled_target(t) for t in train_subset["target_text"]
])

X_val_82 = np.stack([
    pooled_glove(t) for t in val_subset["input_text"]
])
Y_val_82 = np.stack([
    pooled_target(t) for t in val_subset["target_text"]
])

ridge_82 = Ridge(alpha=10.0)
ridge_82.fit(X_train_82, Y_train_82)

pred_82 = ridge_82.predict(X_val_82)

def cosine_rows(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )
    return numerator / np.maximum(denominator, 1e-8)

correct_82 = cosine_rows(pred_82, Y_val_82)

# Deranged target permutation.
rng = np.random.default_rng(42)
shift = int(rng.integers(1, len(Y_val_82)))
shuffled_82 = cosine_rows(
    pred_82,
    np.roll(Y_val_82, shift, axis=0)
)

mean_target_82 = Y_train_82.mean(axis=0)
constant_82 = cosine_rows(
    np.repeat(
        mean_target_82[None, :],
        len(Y_val_82),
        axis=0
    ),
    Y_val_82
)

print("Training cases:", len(X_train_82))
print("Validation cases:", len(X_val_82))

print("\nCorrect-target cosine:",
      float(correct_82.mean()))
print("Shuffled-target cosine:",
      float(shuffled_82.mean()))
print("Constant-baseline cosine:",
      float(constant_82.mean()))

print("\nCorrect-minus-shuffled gap:",
      float(correct_82.mean() - shuffled_82.mean()))

print("\nTraining performed: Ridge regression only")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 83: RIDGE PROBABLE-CAUSE RETRIEVAL")
print("=" * 65)

def normalize_rows(matrix):
    norms = np.linalg.norm(
        matrix, axis=1, keepdims=True
    )
    return matrix / np.maximum(norms, 1e-8)

pred_normalized = normalize_rows(pred_82)
target_normalized = normalize_rows(Y_val_82)

# Every predicted embedding is compared with all 128 targets.
similarity_matrix = (
    pred_normalized @ target_normalized.T
)

# Correct target for investigation i is target i.
ranks = np.argsort(
    -similarity_matrix,
    axis=1
)

correct_indices = np.arange(len(Y_val_82))

correct_ranks = np.array([
    np.where(ranks[i] == correct_indices[i])[0][0] + 1
    for i in range(len(correct_indices))
])

recall_at_1 = np.mean(correct_ranks <= 1)
recall_at_5 = np.mean(correct_ranks <= 5)
recall_at_10 = np.mean(correct_ranks <= 10)
mrr = np.mean(1.0 / correct_ranks)

print("Validation investigations:", len(correct_ranks))

print("\nRetrieval metrics:")
print("Recall@1:", float(recall_at_1))
print("Recall@5:", float(recall_at_5))
print("Recall@10:", float(recall_at_10))
print("Mean reciprocal rank:", float(mrr))
print("Median correct-target rank:",
      float(np.median(correct_ranks)))

print("\nRandom-ranking reference:")
print("Expected Recall@1:", 1 / len(correct_ranks))
print("Expected Recall@5:", 5 / len(correct_ranks))
print("Expected Recall@10:", 10 / len(correct_ranks))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 84: RIDGE VS CONSTANT RETRIEVAL BASELINE")
print("=" * 65)

# Constant prediction from the training-set mean target.
constant_predictions_84 = np.repeat(
    Y_train_82.mean(axis=0, keepdims=True),
    len(Y_val_82),
    axis=0
)

def evaluate_retrieval(predictions, targets):
    p = normalize_rows(predictions)
    t = normalize_rows(targets)

    scores = p @ t.T

    # Deterministic tie handling.
    order = np.argsort(
        -scores,
        axis=1,
        kind="stable"
    )

    ranks = np.array([
        np.flatnonzero(order[i] == i)[0] + 1
        for i in range(len(targets))
    ])

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "Median rank": float(np.median(ranks))
    }

ridge_results_84 = evaluate_retrieval(
    pred_82,
    Y_val_82
)

constant_results_84 = evaluate_retrieval(
    constant_predictions_84,
    Y_val_82
)

print(f"{'Metric':15s} {'Ridge':>14s} {'Constant':>14s}")
print("-" * 45)

for metric in ridge_results_84:
    print(
        f"{metric:15s} "
        f"{ridge_results_84[metric]:14.6f} "
        f"{constant_results_84[metric]:14.6f}"
    )

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 85: DIRECT GLOVE VS RIDGE VS CONSTANT")
print("=" * 70)

# All three representations have 100 dimensions.
assert X_val_82.shape[1] == Y_val_82.shape[1] == 100

direct_results_85 = evaluate_retrieval(
    X_val_82,
    Y_val_82
)

ridge_results_85 = evaluate_retrieval(
    pred_82,
    Y_val_82
)

constant_results_85 = evaluate_retrieval(
    constant_predictions_84,
    Y_val_82
)

print(
    f"{'Metric':15s} "
    f"{'Direct GloVe':>15s} "
    f"{'Ridge':>15s} "
    f"{'Constant':>15s}"
)
print("-" * 70)

for metric in direct_results_85:
    print(
        f"{metric:15s} "
        f"{direct_results_85[metric]:15.6f} "
        f"{ridge_results_85[metric]:15.6f} "
        f"{constant_results_85[metric]:15.6f}"
    )

# Paired cosine similarity for each approach.
direct_cosine = cosine_rows(
    X_val_82,
    Y_val_82
).mean()

ridge_cosine = cosine_rows(
    pred_82,
    Y_val_82
).mean()

constant_cosine = cosine_rows(
    constant_predictions_84,
    Y_val_82
).mean()

print("\nCorrect-pair cosine similarity:")
print("Direct GloVe:", float(direct_cosine))
print("Ridge:", float(ridge_cosine))
print("Constant:", float(constant_cosine))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import re
import numpy as np

print("STEP 86: INPUT–TARGET TEXT OVERLAP AUDIT")
print("=" * 65)

def audit_tokens(text):
    return re.findall(
        r"[a-z]+",
        str(text).lower()
    )

def overlap_statistics(input_text, target_text):
    input_tokens = audit_tokens(input_text)
    target_tokens = audit_tokens(target_text)

    input_set = set(input_tokens)
    target_set = set(target_tokens)

    if not target_set:
        return 0.0, 0.0, False

    overlap = len(input_set & target_set) / len(target_set)

    # Check whether the complete target occurs inside the input.
    normalized_input = " ".join(input_tokens)
    normalized_target = " ".join(target_tokens)

    exact_containment = (
        bool(normalized_target) and
        normalized_target in normalized_input
    )

    # Jaccard similarity of unique word sets.
    union = input_set | target_set
    jaccard = (
        len(input_set & target_set) / len(union)
        if union else 0.0
    )

    return overlap, jaccard, exact_containment

audit_results = [
    overlap_statistics(x, y)
    for x, y in zip(
        val_subset["input_text"],
        val_subset["target_text"]
    )
]

target_overlap = np.array([
    result[0] for result in audit_results
])

jaccard_scores = np.array([
    result[1] for result in audit_results
])

exact_containment = np.array([
    result[2] for result in audit_results
])

print("Investigations audited:", len(audit_results))

print("\nTarget vocabulary overlap:")
print("Mean:", float(target_overlap.mean()))
print("Median:", float(np.median(target_overlap)))
print("Maximum:", float(target_overlap.max()))

print("\nJaccard similarity:")
print("Mean:", float(jaccard_scores.mean()))
print("Median:", float(np.median(jaccard_scores)))

print("\nComplete target contained in input:")
print("Count:", int(exact_containment.sum()))
print("Percentage:", float(exact_containment.mean() * 100))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
from scipy.stats import spearmanr

print("STEP 87: RETRIEVAL SUCCESS AND TEXT CHARACTERISTICS")
print("=" * 70)

input_lengths = np.array([
    len(audit_tokens(text))
    for text in val_subset["input_text"]
])

target_lengths = np.array([
    len(audit_tokens(text))
    for text in val_subset["target_text"]
])

# Recalculate direct GloVe retrieval ranks.
direct_similarity = (
    normalize_rows(X_val_82) @
    normalize_rows(Y_val_82).T
)

direct_order = np.argsort(
    -direct_similarity,
    axis=1,
    kind="stable"
)

direct_ranks = np.array([
    np.flatnonzero(direct_order[i] == i)[0] + 1
    for i in range(len(val_subset))
])

reciprocal_ranks = 1.0 / direct_ranks

def report_association(name, values):
    rho, p_value = spearmanr(
        values,
        reciprocal_ranks
    )

    print(f"\n{name}")
    print("Spearman rho:", float(rho))
    print("p-value:", float(p_value))

report_association(
    "Investigation narrative length",
    input_lengths
)

report_association(
    "Probable-cause statement length",
    target_lengths
)

report_association(
    "Target-vocabulary overlap",
    target_overlap
)

print("\nRetrieval success by narrative length:")
median_length = np.median(input_lengths)

short = input_lengths <= median_length
long = input_lengths > median_length

print(
    "Shorter narratives Recall@10:",
    float(np.mean(direct_ranks[short] <= 10))
)

print(
    "Longer narratives Recall@10:",
    float(np.mean(direct_ranks[long] <= 10))
)

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 88: TRAINING REPRESENTATION AUDIT")
print("=" * 65)

sample_df = train_df.iloc[:512]

input_lengths_88 = []
target_lengths_88 = []
input_coverage_88 = []
target_coverage_88 = []

for _, row in sample_df.iterrows():
    for text, lengths, coverage in [
        (row["input_text"], input_lengths_88, input_coverage_88),
        (row["target_text"], target_lengths_88, target_coverage_88)
    ]:
        tokens = tokenize(text)
        lengths.append(len(tokens))

        if tokens:
            matched = sum(
                normalize_glove_token(token) in glove_vectors
                for token in tokens
            )
            coverage.append(matched / len(tokens))
        else:
            coverage.append(0.0)

for name, lengths, coverage in [
    ("Investigation narratives", input_lengths_88, input_coverage_88),
    ("Probable-cause statements", target_lengths_88, target_coverage_88)
]:
    lengths = np.asarray(lengths)
    coverage = np.asarray(coverage)

    print(f"\n{name}")
    print("Median tokens:", float(np.median(lengths)))
    print("90th percentile:", float(np.percentile(lengths, 90)))
    print("95th percentile:", float(np.percentile(lengths, 95)))
    print("Maximum tokens:", int(lengths.max()))
    print("Mean GloVe coverage:", float(coverage.mean()))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 89: PREPARE 100D GLOVE NEURAL INPUT")
print("=" * 65)

NEURAL_INPUT_LENGTH = 512
NEURAL_TRAIN_SIZE = 512
NEURAL_VAL_SIZE = 128

def prepare_neural_inputs(frame, n):
    subset = frame.iloc[:n]

    inputs = np.stack([
        text_to_glove(text, NEURAL_INPUT_LENGTH)[0]
        for text in subset["input_text"]
    ]).astype(np.float32)

    targets = np.stack([
        pooled_target(text)
        for text in subset["target_text"]
    ]).astype(np.float32)

    return inputs, targets

X_train_89, Y_train_89 = prepare_neural_inputs(
    train_df, NEURAL_TRAIN_SIZE
)

X_val_89, Y_val_89 = prepare_neural_inputs(
    val_df, NEURAL_VAL_SIZE
)

print("Training inputs:", X_train_89.shape)
print("Training targets:", Y_train_89.shape)
print("Validation inputs:", X_val_89.shape)
print("Validation targets:", Y_val_89.shape)

print("\nInput standard deviation:",
      float(X_train_89.std()))

print("Target standard deviation:",
      float(Y_train_89.std()))

print("All training values finite:",
      bool(
          np.isfinite(X_train_89).all()
          and np.isfinite(Y_train_89).all()
      ))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import tensorflow as tf
import numpy as np

print("STEP 90: BUILD REPRODUCIBLE GLOVE CONVOLUTIONAL MODEL")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

inputs_90 = tf.keras.Input(
    shape=(512, 100),
    name="glove_input"
)

# Identify actual tokens before convolution.
token_mask_90 = tf.keras.layers.Lambda(
    lambda x: tf.cast(
        tf.reduce_any(tf.not_equal(x, 0.0), axis=-1),
        tf.float32
    ),
    name="token_mask"
)(inputs_90)

x_90 = tf.keras.layers.Conv1D(
    64, kernel_size=5, padding="same",
    activation="relu",
    name="conv1"
)(inputs_90)

x_90 = tf.keras.layers.Conv1D(
    64, kernel_size=3, padding="same",
    activation="relu",
    name="conv2"
)(x_90)

# Mask padded positions before averaging.
masked_90 = tf.keras.layers.Multiply(
    name="masked_features"
)([
    x_90,
    tf.keras.layers.Reshape((512, 1))(token_mask_90)
])

sum_90 = tf.keras.layers.Lambda(
    lambda x: tf.reduce_sum(x, axis=1),
    name="feature_sum"
)(masked_90)

count_90 = tf.keras.layers.Lambda(
    lambda x: tf.maximum(
        tf.reduce_sum(x, axis=1, keepdims=True),
        1.0
    ),
    name="token_count"
)(token_mask_90)

pooled_90 = tf.keras.layers.Lambda(
    lambda x: x[0] / x[1],
    name="masked_mean_pool"
)([sum_90, count_90])

latent_90 = tf.keras.layers.Dense(
    20, activation="relu",
    name="latent_20"
)(pooled_90)

outputs_90 = tf.keras.layers.Dense(
    100,
    name="predicted_probable_cause_embedding"
)(latent_90)

neural_model_90 = tf.keras.Model(
    inputs_90,
    outputs_90,
    name="GloveConv20"
)

neural_model_90.summary()

print("\nModel parameters:",
      neural_model_90.count_params())

test_output_90 = neural_model_90(
    X_val_89[:2],
    training=False
)

print("Test output shape:", test_output_90.shape)
print("Finite predictions:",
      bool(np.isfinite(test_output_90.numpy()).all()))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
import tensorflow as tf

print("STEP 91: TRAIN GLOVE CONV20 — CPU PILOT")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

def cosine_distance_loss(y_true, y_pred):
    y_true = tf.math.l2_normalize(y_true, axis=-1)
    y_pred = tf.math.l2_normalize(y_pred, axis=-1)

    cosine = tf.reduce_sum(
        y_true * y_pred,
        axis=-1
    )

    return tf.reduce_mean(1.0 - cosine)

neural_model_90.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=cosine_distance_loss
)

history_91 = neural_model_90.fit(
    X_train_89,
    Y_train_89,
    validation_data=(X_val_89, Y_val_89),
    epochs=10,
    batch_size=16,
    verbose=2,
    shuffle=True
)

pred_91 = neural_model_90.predict(
    X_val_89,
    batch_size=16,
    verbose=0
)

correct_91 = cosine_rows(
    pred_91,
    Y_val_89
)

rng_91 = np.random.default_rng(42)
shift_91 = int(rng_91.integers(1, len(Y_val_89)))

shuffled_91 = cosine_rows(
    pred_91,
    np.roll(Y_val_89, shift_91, axis=0)
)

normalized_pred_91 = normalize_rows(pred_91)

prediction_similarity_91 = (
    normalized_pred_91 @ normalized_pred_91.T
)

off_diagonal_91 = prediction_similarity_91[
    ~np.eye(len(pred_91), dtype=bool)
]

print("\nFINAL DIAGNOSTICS")
print("-" * 50)

print(
    "Correct-target cosine:",
    float(correct_91.mean())
)

print(
    "Shuffled-target cosine:",
    float(shuffled_91.mean())
)

print(
    "Correct-minus-shuffled gap:",
    float(correct_91.mean() - shuffled_91.mean())
)

print(
    "Mean prediction-to-prediction cosine:",
    float(off_diagonal_91.mean())
)

print(
    "Prediction coordinate std:",
    float(pred_91.std(axis=0).mean())
)

print("\nTraining completed.")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 92: SAME-COHORT NEURAL BASELINE COMPARISON")
print("=" * 75)

# Confirm that all predictions refer to the same 128 cases.
assert len(pred_91) == len(pred_82) == len(X_val_82) == len(Y_val_89)
assert np.allclose(Y_val_89, Y_val_82)

methods_92 = {
    "Direct GloVe": X_val_82,
    "Ridge": pred_82,
    "Conv20": pred_91,
    "Constant": constant_predictions_84
}

print(
    f"{'Method':15s} {'Cosine':>10s} "
    f"{'R@1':>9s} {'R@5':>9s} "
    f"{'R@10':>9s} {'MRR':>9s}"
)
print("-" * 75)

results_92 = {}

for name, predictions in methods_92.items():
    cosine = float(
        cosine_rows(predictions, Y_val_89).mean()
    )

    retrieval = evaluate_retrieval(
        predictions,
        Y_val_89
    )

    results_92[name] = {
        "cosine": cosine,
        **retrieval
    }

    print(
        f"{name:15s} "
        f"{cosine:10.6f} "
        f"{retrieval['R@1']:9.4f} "
        f"{retrieval['R@5']:9.4f} "
        f"{retrieval['R@10']:9.4f} "
        f"{retrieval['MRR']:9.4f}"
    )

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import tensorflow as tf
import numpy as np

print("STEP 93: BUILD CONTRASTIVE GLOVE MODEL")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

contrastive_inputs_93 = tf.keras.Input(
    shape=(100,),
    name="investigation_glove"
)

x_93 = tf.keras.layers.Dense(
    128,
    activation="relu",
    name="projection_hidden"
)(contrastive_inputs_93)

x_93 = tf.keras.layers.Dense(
    100,
    name="projection_output"
)(x_93)

normalized_outputs_93 = tf.keras.layers.UnitNormalization(
    axis=-1,
    name="normalized_embedding"
)(x_93)

contrastive_model_93 = tf.keras.Model(
    contrastive_inputs_93,
    normalized_outputs_93,
    name="ContrastiveGloveProjection"
)

contrastive_model_93.summary()

test_predictions_93 = contrastive_model_93(
    X_val_82[:4],
    training=False
).numpy()

print("\nModel parameters:",
      contrastive_model_93.count_params())

print("Test output shape:",
      test_predictions_93.shape)

print("Output norms:",
      np.linalg.norm(test_predictions_93, axis=1))

print("All outputs finite:",
      bool(np.isfinite(test_predictions_93).all()))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import tensorflow as tf
import numpy as np

print("STEP 94: CONTRASTIVE GLOVE TRAINING")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

temperature_94 = 0.07
optimizer_94 = tf.keras.optimizers.Adam(
    learning_rate=0.0005
)

X_train_94 = X_train_82.astype(np.float32)
Y_train_94 = normalize_rows(
    Y_train_82.astype(np.float32)
)

training_dataset_94 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_94, Y_train_94)
    )
    .shuffle(
        len(X_train_94),
        seed=42,
        reshuffle_each_iteration=True
    )
    .batch(32)
)

@tf.function
def contrastive_train_step_94(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predicted = contrastive_model_93(
            x_batch,
            training=True
        )

        similarities = tf.matmul(
            predicted,
            y_batch,
            transpose_b=True
        ) / temperature_94

        labels = tf.range(
            tf.shape(similarities)[0]
        )

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels,
                similarities,
                from_logits=True
            )
        )

    gradients = tape.gradient(
        loss,
        contrastive_model_93.trainable_variables
    )

    optimizer_94.apply_gradients(
        zip(
            gradients,
            contrastive_model_93.trainable_variables
        )
    )

    return loss

loss_history_94 = []

for epoch in range(15):
    epoch_losses = []

    for x_batch, y_batch in training_dataset_94:
        loss = contrastive_train_step_94(
            x_batch,
            y_batch
        )

        epoch_losses.append(float(loss.numpy()))

    mean_loss = float(np.mean(epoch_losses))
    loss_history_94.append(mean_loss)

    print(
        f"Epoch {epoch + 1:02d}/15 "
        f"- contrastive loss: {mean_loss:.6f}"
    )

print("\nTraining completed.")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 95: CONTRASTIVE RETRIEVAL EVALUATION")
print("=" * 75)

# Generate embeddings for the held-out validation inputs.
pred_95 = contrastive_model_93.predict(
    X_val_82.astype(np.float32),
    batch_size=32,
    verbose=0
)

# Compare all methods on identical validation targets.
methods_95 = {
    "Direct GloVe": X_val_82,
    "Ridge": pred_82,
    "Conv20": pred_91,
    "Contrastive": pred_95,
    "Constant": constant_predictions_84
}

print(
    f"{'Method':15s} {'Cosine':>10s} "
    f"{'R@1':>9s} {'R@5':>9s} "
    f"{'R@10':>9s} {'MRR':>9s}"
)
print("-" * 75)

results_95 = {}

for name, predictions in methods_95.items():
    cosine = float(
        cosine_rows(predictions, Y_val_82).mean()
    )

    retrieval = evaluate_retrieval(
        predictions,
        Y_val_82
    )

    results_95[name] = {
        "cosine": cosine,
        **retrieval
    }

    print(
        f"{name:15s} "
        f"{cosine:10.6f} "
        f"{retrieval['R@1']:9.4f} "
        f"{retrieval['R@5']:9.4f} "
        f"{retrieval['R@10']:9.4f} "
        f"{retrieval['MRR']:9.4f}"
    )

# Check whether predictions remain nearly identical.
normalized_pred_95 = normalize_rows(pred_95)
pairwise_95 = normalized_pred_95 @ normalized_pred_95.T

off_diagonal_95 = pairwise_95[
    ~np.eye(len(pred_95), dtype=bool)
]

print("\nContrastive prediction diagnostics:")
print(
    "Mean prediction-to-prediction cosine:",
    float(off_diagonal_95.mean())
)
print(
    "Mean coordinate standard deviation:",
    float(pred_95.std(axis=0).mean())
)

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 96: CORRECT-PAIR DISCRIMINATION AUDIT")
print("=" * 70)

def discrimination_audit(predictions, targets):
    scores = (
        normalize_rows(predictions) @
        normalize_rows(targets).T
    )

    n = len(targets)
    correct = np.diag(scores)

    incorrect_mean = (
        scores.sum(axis=1) - correct
    ) / (n - 1)

    margin = correct - incorrect_mean

    return {
        "correct_cosine": float(correct.mean()),
        "incorrect_cosine": float(incorrect_mean.mean()),
        "mean_margin": float(margin.mean()),
        "positive_margin_pct": float(
            100 * np.mean(margin > 0)
        ),
        "median_margin": float(np.median(margin))
    }

for name, predictions in [
    ("Direct GloVe", X_val_82),
    ("Ridge", pred_82),
    ("Contrastive", pred_95),
    ("Constant", constant_predictions_84)
]:
    result = discrimination_audit(
        predictions,
        Y_val_82
    )

    print(f"\n{name}")
    for metric, value in result.items():
        print(f"  {metric}: {value:.6f}")

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import re
import hashlib

print("STEP 97: TRAIN / VALIDATION / TEST LEAKAGE AUDIT")
print("=" * 70)

splits_97 = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}

def text_fingerprint(text):
    normalized = re.sub(
        r"\s+",
        " ",
        str(text).lower().strip()
    )
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

for field in ["investigation_id", "input_text", "target_text"]:
    print(f"\nFIELD: {field}")

    sets = {}

    for split_name, frame in splits_97.items():
        if field == "investigation_id":
            values = set(frame[field].astype(str))
        else:
            values = set(
                frame[field].map(text_fingerprint)
            )

        sets[split_name] = values

        print(
            f"  {split_name}: "
            f"{len(frame)} rows, "
            f"{len(values)} unique values"
        )

    for a, b in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test")
    ]:
        overlap = len(sets[a] & sets[b])

        print(
            f"  {a} vs {b}: "
            f"{overlap} shared values"
        )

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
import re

print("STEP 98: DUPLICATE-AWARE RETRIEVAL EVALUATION")
print("=" * 75)

def normalize_target_text_98(text):
    return re.sub(
        r"\s+", " ", str(text).lower().strip()
    )

# The original pilot used the first 128 validation rows.
validation_targets_98 = (
    val_df.iloc[:len(Y_val_82)]["target_text"]
    .map(normalize_target_text_98)
    .to_numpy()
)

assert len(validation_targets_98) == len(Y_val_82)

def duplicate_aware_retrieval_98(predictions, targets, texts):
    scores = (
        normalize_rows(predictions) @
        normalize_rows(targets).T
    )

    ranks = []

    for i in range(len(texts)):
        ordering = np.argsort(-scores[i], kind="stable")

        correct_positions = np.flatnonzero(
            texts[ordering] == texts[i]
        )

        ranks.append(int(correct_positions[0]) + 1)

    ranks = np.asarray(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks))
    }

methods_98 = {
    "Direct GloVe": X_val_82,
    "Ridge": pred_82,
    "Conv20": pred_91,
    "Contrastive": pred_95,
    "Constant": constant_predictions_84
}

print(
    f"{'Method':15s} {'R@1':>9s} "
    f"{'R@5':>9s} {'R@10':>9s} {'MRR':>9s}"
)
print("-" * 60)

for name, predictions in methods_98.items():
    result = duplicate_aware_retrieval_98(
        predictions,
        Y_val_82,
        validation_targets_98
    )

    print(
        f"{name:15s} "
        f"{result['R@1']:9.4f} "
        f"{result['R@5']:9.4f} "
        f"{result['R@10']:9.4f} "
        f"{result['MRR']:9.4f}"
    )

print(
    "\nUnique target statements:",
    len(set(validation_targets_98))
)

print("Total validation cases:", len(validation_targets_98))
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
import time

print("STEP 99: FULL TRAINING AND VALIDATION EMBEDDINGS")
print("=" * 70)

start_99 = time.time()

def prepare_pooled_split_99(frame, split_name):
    x_vectors = []
    y_vectors = []

    total = len(frame)

    for i, (_, row) in enumerate(frame.iterrows(), start=1):
        x_vectors.append(
            pooled_target(row["input_text"])
        )
        y_vectors.append(
            pooled_target(row["target_text"])
        )

        if i % 4000 == 0 or i == total:
            print(
                f"{split_name}: {i:,}/{total:,} processed"
            )

    return (
        np.asarray(x_vectors, dtype=np.float32),
        np.asarray(y_vectors, dtype=np.float32)
    )

X_train_99, Y_train_99 = prepare_pooled_split_99(
    train_df, "Train"
)

X_val_99, Y_val_99 = prepare_pooled_split_99(
    val_df, "Validation"
)

print("\nArray shapes:")
print("X_train:", X_train_99.shape)
print("Y_train:", Y_train_99.shape)
print("X_val:", X_val_99.shape)
print("Y_val:", Y_val_99.shape)

print("\nAll arrays finite:", all(
    np.isfinite(a).all()
    for a in [
        X_train_99, Y_train_99,
        X_val_99, Y_val_99
    ]
))

print(
    "Elapsed seconds:",
    round(time.time() - start_99, 2)
)

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
import tensorflow as tf

print("STEP 100: FULL CONTRASTIVE MODEL TRAINING")
print("=" * 75)

tf.keras.utils.set_random_seed(42)

# Fresh model with the same architecture as Step 93.
inputs_100 = tf.keras.Input(shape=(100,))
hidden_100 = tf.keras.layers.Dense(
    128, activation="relu"
)(inputs_100)
projected_100 = tf.keras.layers.Dense(100)(hidden_100)
outputs_100 = tf.keras.layers.UnitNormalization(
    axis=-1
)(projected_100)

model_100 = tf.keras.Model(inputs_100, outputs_100)

optimizer_100 = tf.keras.optimizers.Adam(
    learning_rate=0.0005
)

temperature_100 = 0.07

X_train_tf_100 = X_train_99.astype(np.float32)
Y_train_tf_100 = normalize_rows(
    Y_train_99.astype(np.float32)
)

training_data_100 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_tf_100, Y_train_tf_100)
    )
    .shuffle(
        len(X_train_tf_100),
        seed=42,
        reshuffle_each_iteration=True
    )
    .batch(32)
)

@tf.function
def train_step_100(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predicted = model_100(x_batch, training=True)

        logits = tf.matmul(
            predicted, y_batch, transpose_b=True
        ) / temperature_100

        labels = tf.range(tf.shape(logits)[0])

        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels, logits, from_logits=True
            )
        )

    gradients = tape.gradient(
        loss, model_100.trainable_variables
    )

    optimizer_100.apply_gradients(
        zip(gradients, model_100.trainable_variables)
    )

    return loss

def validation_recall10_100():
    predictions = model_100.predict(
        X_val_99, batch_size=128, verbose=0
    )

    scores = (
        normalize_rows(predictions) @
        normalize_rows(Y_val_99).T
    )

    # Treat identical target statements as equivalent.
    target_texts = (
        val_df["target_text"]
        .map(normalize_target_text_98)
        .to_numpy()
    )

    top10 = np.argpartition(
        -scores, kth=9, axis=1
    )[:, :10]

    correct = np.array([
        np.any(target_texts[top10[i]] == target_texts[i])
        for i in range(len(target_texts))
    ])

    return float(correct.mean())

best_recall_100 = -1.0
best_epoch_100 = None
best_weights_100 = None
history_100 = []

for epoch in range(15):
    losses = []

    for xb, yb in training_data_100:
        loss = train_step_100(xb, yb)
        losses.append(float(loss.numpy()))

    train_loss = float(np.mean(losses))
    val_r10 = validation_recall10_100()

    history_100.append((train_loss, val_r10))

    if val_r10 > best_recall_100:
        best_recall_100 = val_r10
        best_epoch_100 = epoch + 1
        best_weights_100 = [
            w.copy() for w in model_100.get_weights()
        ]

    print(
        f"Epoch {epoch + 1:02d}/15 | "
        f"Loss: {train_loss:.4f} | "
        f"Validation R@10: {val_r10:.4f}"
    )

model_100.set_weights(best_weights_100)

print("\nBEST CHECKPOINT")
print("Epoch:", best_epoch_100)
print("Validation Recall@10:", round(best_recall_100, 6))
print("Parameters:", model_100.count_params())
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 101: FULL VALIDATION RETRIEVAL COMPARISON")
print("=" * 75)

pred_contrastive_101 = model_100.predict(
    X_val_99,
    batch_size=128,
    verbose=0
)

target_texts_101 = (
    val_df["target_text"]
    .map(normalize_target_text_98)
    .to_numpy()
)

def full_retrieval_metrics_101(predictions, targets, texts):
    scores = (
        normalize_rows(predictions) @
        normalize_rows(targets).T
    )

    ranks = np.empty(len(texts), dtype=np.int32)

    for i in range(len(texts)):
        ordering = np.argsort(
            -scores[i],
            kind="stable"
        )

        matching = (
            texts[ordering] == texts[i]
        )

        ranks[i] = np.flatnonzero(matching)[0] + 1

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MedianRank": float(np.median(ranks))
    }

results_101 = {}

for name, predictions in [
    ("Direct GloVe", X_val_99),
    ("Contrastive", pred_contrastive_101)
]:
    metrics = full_retrieval_metrics_101(
        predictions,
        Y_val_99,
        target_texts_101
    )

    results_101[name] = metrics

    print(f"\n{name}")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.6f}")

print("\nValidation investigations:", len(val_df))
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np
import time

print("STEP 102: PREPARE INDEPENDENT TEST EMBEDDINGS")
print("=" * 70)

start_102 = time.time()

X_test_102, Y_test_102 = prepare_pooled_split_99(
    test_df,
    "Test"
)

assert X_test_102.shape == (3138, 100)
assert Y_test_102.shape == (3138, 100)

assert np.isfinite(X_test_102).all()
assert np.isfinite(Y_test_102).all()

test_texts_102 = (
    test_df["target_text"]
    .map(normalize_target_text_98)
    .to_numpy()
)

print("\nTest input shape:", X_test_102.shape)
print("Test target shape:", Y_test_102.shape)
print("Unique target texts:", len(set(test_texts_102)))
print("Total investigations:", len(test_texts_102))

print(
    "Elapsed seconds:",
    round(time.time() - start_102, 2)
)

print("\nModel training performed: NO")
print("Test evaluation performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 103: FINAL INDEPENDENT TEST EVALUATION")
print("=" * 75)

# Use the frozen model selected on validation data.
assert best_epoch_100 == 13

pred_contrastive_103 = model_100.predict(
    X_test_102,
    batch_size=128,
    verbose=0
)

# Evaluate both methods against the same test targets.
results_103 = {}

for name, predictions in [
    ("Direct GloVe", X_test_102),
    ("Contrastive", pred_contrastive_103)
]:
    metrics = full_retrieval_metrics_101(
        predictions,
        Y_test_102,
        test_texts_102
    )

    results_103[name] = metrics

    print(f"\n{name}")
    print("-" * 35)

    for metric, value in metrics.items():
        print(f"{metric}: {value:.6f}")

print("\nFINAL COMPARISON")
print("-" * 75)

print(
    f"{'Metric':15s} "
    f"{'Direct GloVe':>15s} "
    f"{'Contrastive':>15s}"
)

for metric in [
    "R@1", "R@5", "R@10",
    "MRR", "MedianRank"
]:
    print(
        f"{metric:15s} "
        f"{results_103['Direct GloVe'][metric]:15.6f} "
        f"{results_103['Contrastive'][metric]:15.6f}"
    )

print("\nTest investigations:", len(test_df))
print("Model training performed: NO")
print("GPU required: NO")


In [ ]:

import numpy as np

print("STEP 104: PAIRED BOOTSTRAP UNCERTAINTY")
print("=" * 70)

def retrieval_ranks_104(predictions, targets, texts):
    scores = (
        normalize_rows(predictions) @
        normalize_rows(targets).T
    )

    ranks = np.empty(len(texts), dtype=np.int32)

    for i in range(len(texts)):
        ordering = np.argsort(
            -scores[i], kind="stable"
        )

        matches = (
            texts[ordering] == texts[i]
        )

        ranks[i] = np.flatnonzero(matches)[0] + 1

    return ranks

ranks_glove_104 = retrieval_ranks_104(
    X_test_102,
    Y_test_102,
    test_texts_102
)

ranks_contrastive_104 = retrieval_ranks_104(
    pred_contrastive_103,
    Y_test_102,
    test_texts_102
)

rng_104 = np.random.default_rng(42)
n_104 = len(test_texts_102)
bootstrap_indices_104 = rng_104.integers(
    0, n_104, size=(2000, n_104)
)

metrics_104 = {
    "Recall@1": (
        (ranks_glove_104 <= 1).astype(float),
        (ranks_contrastive_104 <= 1).astype(float)
    ),
    "Recall@10": (
        (ranks_glove_104 <= 10).astype(float),
        (ranks_contrastive_104 <= 10).astype(float)
    ),
    "MRR": (
        1.0 / ranks_glove_104,
        1.0 / ranks_contrastive_104
    )
}

for name, (baseline, proposed) in metrics_104.items():
    paired_differences = proposed - baseline

    bootstrap_differences = (
        paired_differences[bootstrap_indices_104].mean(axis=1)
    )

    ci_low, ci_high = np.percentile(
        bootstrap_differences, [2.5, 97.5]
    )

    print(f"\n{name}")
    print(
        "Observed difference:",
        round(float(paired_differences.mean()), 6)
    )
    print(
        "95% bootstrap CI:",
        (round(float(ci_low), 6),
         round(float(ci_high), 6))
    )

print("\nBootstrap resamples: 2000")
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 105: SEARCH FOR ORIGINAL VAE EXPERIMENT ARTIFACTS

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

root = Path("/content/drive/MyDrive")

model_extensions = {
    ".h5", ".hdf5", ".keras",
    ".pt", ".pth", ".ckpt",
    ".pkl", ".pickle", ".joblib"
}

keywords = [
    "vae", "encoder", "decoder",
    "tokenizer", "training_history",
    "checkpoint", "model_weights",
    "glove", "latent"
]

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        path.suffix.lower() in model_extensions
        or any(word in name for word in keywords)
    ):
        matches.append({
            "path": str(path),
            "size_mb": round(
                path.stat().st_size / (1024 ** 2), 3
            )
        })

print("=" * 70)
print("STEP 105: ORIGINAL EXPERIMENT ARTIFACT SEARCH")
print("=" * 70)

print("Total potential artifacts:", len(matches))

for i, item in enumerate(matches[:100], 1):
    print(
        f"{i}. {item['size_mb']} MB | "
        f"{item['path']}"
    )

if not matches:
    print("No matching artifacts found in MyDrive.")

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 106: CHECK EXISTING EXPERIMENT OBJECTS

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 106: EXISTING EXPERIMENT OBJECTS")
print("=" * 70)

objects = [
    "dataset",
    "matched_dataset",
    "train_df",
    "val_df",
    "test_df",
    "glove_vectors",
    "X_train_99",
    "Y_train_99",
    "X_val_99",
    "Y_val_99",
    "X_test_102",
    "Y_test_102",
    "model_100",
    "contrastive_model_93"
]

for name in objects:
    if name in globals():
        obj = globals()[name]

        if hasattr(obj, "shape"):
            detail = f"shape={obj.shape}"
        elif isinstance(obj, dict):
            detail = f"entries={len(obj)}"
        else:
            detail = f"type={type(obj).__name__}"

        print(f"AVAILABLE | {name}: {detail}")
    else:
        print(f"MISSING   | {name}")

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 107: IDENTIFY RECONSTRUCTED ARCHITECTURE OBJECTS

import tensorflow as tf

print("=" * 70)
print("STEP 107: RECONSTRUCTED ARCHITECTURE CHECK")
print("=" * 70)

candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, tf.keras.Model):
        try:
            params = obj.count_params()
            candidates.append(
                (name, params, str(obj.input_shape),
                 str(obj.output_shape))
            )
        except Exception:
            pass

print("Keras models found:", len(candidates))

for name, params, inp, out in candidates:
    print(f"\nModel: {name}")
    print(f"Parameters: {params:,}")
    print(f"Input shape: {inp}")
    print(f"Output shape: {out}")

print("\nHistorical architecture reference:")
print("Encoder parameters: 1,477,908")
print("Decoder parameters: 161,857")
print("Combined parameters: 1,639,765")

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 108: INSPECT RECONSTRUCTED MODEL CONFIGURATION

import tensorflow as tf

print("=" * 70)
print("STEP 108: MODEL CONFIGURATION INSPECTION")
print("=" * 70)

model = reconstructed_model

print("Model name:", model.name)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Total parameters:", f"{model.count_params():,}")

print("\n--- Training configuration ---")

print("Optimizer:",
      type(model.optimizer).__name__
      if getattr(model, "optimizer", None) is not None
      else "NOT COMPILED")

print("Loss:", getattr(model, "loss", None))

print("\n--- Latent representation ---")

print("Encoder output:", verified_encoder.output_shape)
print("Decoder input:", historical_decoder.input_shape)

print("\n--- Potential variational components ---")

for layer in model.layers:
    name = layer.name.lower()
    cls = layer.__class__.__name__.lower()

    if any(term in name or term in cls for term in [
        "mean", "logvar", "variance",
        "sampling", "kl", "latent"
    ]):
        print(layer.name, "|", layer.__class__.__name__)

print("\nModel-level auxiliary losses:",
      len(model.losses))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 109: VERIFY COSINE-SIMILARITY LOSS

import inspect
import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 109: COSINE LOSS VERIFICATION")
print("=" * 70)

print("\n1. LOSS FUNCTION SOURCE")
try:
    print(inspect.getsource(sequence_cosine_loss))
except (OSError, TypeError):
    print("Source unavailable.")
    print("Function:", sequence_cosine_loss)

print("\n2. SYNTHETIC LOSS TEST")

y_true = tf.constant(
    [[[1.0], [2.0], [3.0], [4.0]]],
    dtype=tf.float32
)

y_same = tf.identity(y_true)
y_opposite = -y_true
y_zero = tf.zeros_like(y_true)

for name, prediction in [
    ("Identical", y_same),
    ("Opposite", y_opposite),
    ("Zero", y_zero)
]:
    result = sequence_cosine_loss(y_true, prediction)
    print(
        f"{name:12s} | Loss = "
        f"{float(tf.reduce_mean(result)):.6f}"
    )

print("\n3. MODEL CONFIGURATION")

print("Optimizer:", type(reconstructed_model.optimizer).__name__)
print("Loss:", reconstructed_model.loss)
print("Model-level auxiliary losses:", len(reconstructed_model.losses))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 110: INSPECT EXISTING PREPROCESSING FUNCTIONS

import inspect

print("=" * 70)
print("STEP 110: PREPROCESSING INSPECTION")
print("=" * 70)

names = [
    "tokenize",
    "normalize_glove_token",
    "text_to_glove",
    "pooled_target",
    "sequence_cosine_loss"
]

for name in names:
    print(f"\n{'-' * 60}")
    print("OBJECT:", name)

    obj = globals().get(name)

    if obj is None:
        print("NOT AVAILABLE")
        continue

    try:
        print(inspect.getsource(obj))
    except (OSError, TypeError):
        print("Source unavailable:", repr(obj))

print("\nDATASET SAMPLE")

for column in ["input_text", "target_text"]:
    value = str(train_df.iloc[0][column])
    print(f"\n{column}:")
    print(value[:350])

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 111: SEARCH FOR TOKEN-ID PREPROCESSING OBJECTS

print("=" * 70)
print("STEP 111: TOKEN REPRESENTATION AUDIT")
print("=" * 70)

keywords = [
    "tokenizer", "vocab", "word_index",
    "word_to", "token_to", "sequence",
    "pad_sequences", "embedding_matrix",
    "max_input", "max_target"
]

matches = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(k in name.lower() for k in keywords):
        if callable(obj):
            description = "callable"
        elif hasattr(obj, "shape"):
            description = f"shape={obj.shape}"
        elif isinstance(obj, dict):
            description = f"dict, entries={len(obj)}"
        else:
            description = f"{type(obj).__name__}: {str(obj)[:100]}"

        matches.append((name, description))

for name, description in matches[:100]:
    print(f"{name}: {description}")

print("\nTotal matching objects:", len(matches))
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 112: INSPECT SINGLE-CHANNEL GLOVE REPRESENTATIONS

import numpy as np

print("=" * 70)
print("STEP 112: SINGLE-CHANNEL REPRESENTATION INSPECTION")
print("=" * 70)

for name in ["mean_sequences", "pca_sequences"]:
    arr = np.asarray(globals()[name])

    print(f"\n{name}")
    print("Shape:", arr.shape)
    print("Data type:", arr.dtype)
    print("Finite values:", np.isfinite(arr).all())
    print("Minimum:", float(arr.min()))
    print("Maximum:", float(arr.max()))
    print("Mean:", float(arr.mean()))
    print("Standard deviation:", float(arr.std()))
    print("Nonzero proportion:", float(np.mean(arr != 0)))
    print("First 10 values:", arr[0, :10])

print("\nPOSSIBLE TRANSFORMATION OBJECTS")

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(term in name.lower() for term in [
        "pca", "projection", "mean_sequence",
        "glove_sequence", "single_channel"
    ]):
        print(name, "|", type(obj).__name__)

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 113: VERIFY GLOVE-TO-SINGLE-CHANNEL TRANSFORMATIONS

import inspect
import numpy as np

print("=" * 70)
print("STEP 113: GLOVE PROJECTION FUNCTION AUDIT")
print("=" * 70)

for name in [
    "glove_to_single_channel",
    "pca_single_channel"
]:
    print(f"\n{'-' * 60}")
    print("FUNCTION:", name)

    fn = globals().get(name)

    if fn is None:
        print("NOT FOUND")
        continue

    try:
        print("Signature:", inspect.signature(fn))
        print("\nSource:")
        print(inspect.getsource(fn))
    except (OSError, TypeError, ValueError) as error:
        print("Source unavailable:", error)

print("\nPCA CONFIGURATION")

for name in ["pca", "pca_1d"]:
    obj = globals().get(name)

    if obj is not None:
        print(f"\n{name}:")
        print("Type:", type(obj).__name__)
        print("Components:", getattr(obj, "n_components", None))
        print(
            "Explained variance ratio:",
            getattr(obj, "explained_variance_ratio_", None)
        )

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 114: SINGLE-CHANNEL REPRESENTATION DIAGNOSTIC

import numpy as np

print("=" * 70)
print("STEP 114: MEAN-PROJECTION DIAGNOSTIC")
print("=" * 70)

sample = val_df.iloc[:128]

X_114 = np.stack([
    glove_to_single_channel(text, MAX_INPUT_LEN)
    for text in sample["input_text"]
])

Y_114 = np.stack([
    glove_to_single_channel(text, MAX_TARGET_LEN)
    for text in sample["target_text"]
])

print("Input shape:", X_114.shape)
print("Target shape:", Y_114.shape)

print("Input finite:", np.isfinite(X_114).all())
print("Target finite:", np.isfinite(Y_114).all())

print("Input nonzero proportion:",
      round(float(np.mean(X_114 != 0)), 4))

print("Target nonzero proportion:",
      round(float(np.mean(Y_114 != 0)), 4))

# Constant prediction: mean of training targets
# Use 512 training records for a lightweight diagnostic.
training_targets = np.stack([
    glove_to_single_channel(text, MAX_TARGET_LEN)
    for text in train_df["target_text"].iloc[:512]
])

constant_target = training_targets.mean(axis=0)

def cosine_114(a, b):
    a = a.reshape(len(a), -1)
    b = b.reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-8)

constant_predictions = np.repeat(
    constant_target[None, :, :],
    len(Y_114),
    axis=0
)

scores = cosine_114(Y_114, constant_predictions)

print("\nCONSTANT BASELINE")
print("Mean cosine:", round(float(scores.mean()), 6))
print("SD cosine:", round(float(scores.std()), 6))
print("Minimum:", round(float(scores.min()), 6))
print("Maximum:", round(float(scores.max()), 6))

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 115: MODEL READINESS AUDIT

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 115: MODEL READINESS AUDIT")
print("=" * 70)

model = reconstructed_model

print("Model:", model.name)
print("Parameters:", f"{model.count_params():,}")
print("Input:", model.input_shape)
print("Output:", model.output_shape)

print("\nOPTIMIZER")
optimizer = model.optimizer

print("Type:", type(optimizer).__name__)
print("Learning rate:",
      float(tf.keras.backend.get_value(
          optimizer.learning_rate
      )))

print("Optimizer iterations:",
      int(tf.keras.backend.get_value(
          optimizer.iterations
      )))

print("\nWEIGHT DIAGNOSTICS")

for layer in model.layers:
    if not layer.weights:
        continue

    weights = layer.get_weights()

    if not weights:
        continue

    w = np.asarray(weights[0])

    print(
        f"{layer.name:35s} "
        f"shape={str(w.shape):18s} "
        f"mean={w.mean():.6f} "
        f"std={w.std():.6f}"
    )

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 116: FRESH DETERMINISTIC BASELINE

import tensorflow as tf

print("=" * 70)
print("STEP 116: FRESH DETERMINISTIC BASELINE")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

# Rebuild the complete architecture using its existing
# configuration, without transferring trained weights.
fresh_baseline_116 = tf.keras.Model.from_config(
    reconstructed_model.get_config()
)

fresh_baseline_116.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=sequence_cosine_loss
)

print("Model:", fresh_baseline_116.name)
print("Parameters:", f"{fresh_baseline_116.count_params():,}")
print("Input:", fresh_baseline_116.input_shape)
print("Output:", fresh_baseline_116.output_shape)

print(
    "Optimizer iterations:",
    int(fresh_baseline_116.optimizer.iterations.numpy())
)

print(
    "Same parameter count:",
    fresh_baseline_116.count_params()
    == reconstructed_model.count_params()
)

print("\nTraining performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 117: SMALL TRAINING-SPEED TEST

import time
import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 117: TRAINING-SPEED TEST")
print("=" * 70)

# Use 32 training examples only.
pilot_df_117 = train_df.iloc[:32]

X_117 = np.stack([
    glove_to_single_channel(text, MAX_INPUT_LEN)
    for text in pilot_df_117["input_text"]
]).astype(np.float32)

Y_117 = np.stack([
    glove_to_single_channel(text, MAX_TARGET_LEN)
    for text in pilot_df_117["target_text"]
]).astype(np.float32)

print("Input:", X_117.shape)
print("Target:", Y_117.shape)

print("\nStarting one training batch...")
start = time.perf_counter()

result = fresh_baseline_116.train_on_batch(
    X_117,
    Y_117
)

elapsed = time.perf_counter() - start

print("\nRESULT")
print("Training loss:", float(result))
print("Elapsed seconds:", round(elapsed, 2))
print(
    "Optimizer iterations:",
    int(fresh_baseline_116.optimizer.iterations.numpy())
)

print("\nGPU required: NO")
print("Full-dataset training performed: NO")


In [ ]:

# STEP 118: STEADY-STATE TRAINING SPEED

import time
import numpy as np

print("=" * 70)
print("STEP 118: STEADY-STATE TRAINING SPEED")
print("=" * 70)

batch_times = []
batch_losses = []

for i in range(3):
    start = time.perf_counter()

    loss = fresh_baseline_116.train_on_batch(
        X_117,
        Y_117
    )

    elapsed = time.perf_counter() - start

    batch_times.append(elapsed)
    batch_losses.append(float(loss))

    print(
        f"Batch {i + 1}: "
        f"loss={float(loss):.6f}, "
        f"time={elapsed:.2f}s"
    )

median_time = float(np.median(batch_times))

batches_per_epoch = int(
    np.ceil(len(train_df) / 32)
)

estimated_minutes = (
    median_time * batches_per_epoch / 60
)

print("\nSUMMARY")
print("Median batch time:", round(median_time, 2), "seconds")
print("Batches per full epoch:", batches_per_epoch)
print(
    "Estimated minutes per epoch:",
    round(estimated_minutes, 2)
)
print(
    "Total optimizer iterations:",
    int(fresh_baseline_116.optimizer.iterations.numpy())
)

print("\nFull-dataset training performed: NO")


In [ ]:

# STEP 119: PREPARE FULL RECONSTRUCTION EXPERIMENT

import numpy as np
import tensorflow as tf
import time

print("=" * 70)
print("STEP 119: FULL TRAINING DATA PREPARATION")
print("=" * 70)

def prepare_sequences_119(df):
    X = np.stack([
        glove_to_single_channel(text, MAX_INPUT_LEN)
        for text in df["input_text"]
    ]).astype(np.float32)

    Y = np.stack([
        glove_to_single_channel(text, MAX_TARGET_LEN)
        for text in df["target_text"]
    ]).astype(np.float32)

    return X, Y

start = time.perf_counter()

X_train_119, Y_train_119 = prepare_sequences_119(train_df)
X_val_119, Y_val_119 = prepare_sequences_119(val_df)

elapsed = time.perf_counter() - start

print("Training input:", X_train_119.shape)
print("Training target:", Y_train_119.shape)
print("Validation input:", X_val_119.shape)
print("Validation target:", Y_val_119.shape)

memory_mb = sum(
    array.nbytes
    for array in [
        X_train_119, Y_train_119,
        X_val_119, Y_val_119
    ]
) / (1024 ** 2)

print("Array memory:", round(memory_mb, 2), "MB")
print("Preparation time:", round(elapsed, 2), "seconds")

# Fresh initialization for the actual experiment.
tf.keras.utils.set_random_seed(42)

baseline_model_119 = tf.keras.Model.from_config(
    reconstructed_model.get_config()
)

baseline_model_119.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=sequence_cosine_loss
)

print("\nFRESH MODEL")
print("Parameters:", baseline_model_119.count_params())
print(
    "Optimizer iterations:",
    int(baseline_model_119.optimizer.iterations.numpy())
)

print("\nTest set accessed: NO")
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

# STEP 120: TRAIN DETERMINISTIC BASELINE

import tensorflow as tf
import numpy as np
import time

print("=" * 70)
print("STEP 120: DETERMINISTIC BASELINE TRAINING")
print("=" * 70)

tf.keras.utils.set_random_seed(42)

callbacks_120 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.TerminateOnNaN()
]

start = time.perf_counter()

history_120 = baseline_model_119.fit(
    X_train_119,
    Y_train_119,
    validation_data=(X_val_119, Y_val_119),
    epochs=15,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks_120,
    verbose=2
)

elapsed = time.perf_counter() - start

train_losses = np.asarray(history_120.history["loss"])
val_losses = np.asarray(history_120.history["val_loss"])

best_epoch = int(np.argmin(val_losses)) + 1

print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

print("Epochs completed:", len(train_losses))
print("Best epoch:", best_epoch)
print("Best validation loss:", float(val_losses.min()))
print(
    "Best validation mean cosine:",
    float(1.0 - val_losses.min())
)
print("Final training loss:", float(train_losses[-1]))
print("Elapsed minutes:", round(elapsed / 60, 2))

print(
    "Optimizer iterations:",
    int(baseline_model_119.optimizer.iterations.numpy())
)

print("\nTest set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 121: VALIDATION PREDICTION-COLLAPSE DIAGNOSTIC

import numpy as np

print("=" * 70)
print("STEP 121: VALIDATION PREDICTION-COLLAPSE DIAGNOSTIC")
print("=" * 70)

# Predictions from the best restored model.
pred_121 = baseline_model_119.predict(
    X_val_119,
    batch_size=32,
    verbose=0
)

def cosine_rows_121(a, b):
    a = a.reshape(len(a), -1)
    b = b.reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-8)

# Model validation cosine.
model_scores_121 = cosine_rows_121(
    Y_val_119,
    pred_121
)

# Constant baseline calculated exclusively from training targets.
constant_target_121 = Y_train_119.mean(
    axis=0,
    keepdims=True
)

constant_predictions_121 = np.repeat(
    constant_target_121,
    len(Y_val_119),
    axis=0
)

constant_scores_121 = cosine_rows_121(
    Y_val_119,
    constant_predictions_121
)

# Prediction diversity: compare outputs for different narratives.
rng_121 = np.random.default_rng(42)

indices_a = rng_121.integers(
    0, len(pred_121), size=1000
)

indices_b = rng_121.integers(
    0, len(pred_121), size=1000
)

different = indices_a != indices_b
indices_a = indices_a[different]
indices_b = indices_b[different]

prediction_similarity_121 = cosine_rows_121(
    pred_121[indices_a],
    pred_121[indices_b]
)

print("VALIDATION PERFORMANCE")
print(
    "Model mean cosine:",
    round(float(model_scores_121.mean()), 6)
)
print(
    "Constant mean cosine:",
    round(float(constant_scores_121.mean()), 6)
)
print(
    "Model minus constant:",
    round(float(
        model_scores_121.mean()
        - constant_scores_121.mean()
    ), 6)
)

print("\nPREDICTION DIVERSITY")
print(
    "Mean cosine between different predictions:",
    round(float(prediction_similarity_121.mean()), 6)
)
print(
    "Prediction cosine SD:",
    round(float(prediction_similarity_121.std()), 6)
)
print(
    "Prediction elementwise SD:",
    round(float(pred_121.std(axis=0).mean()), 6)
)

print("\nTest set accessed: NO")
print("Training performed: NO")


In [ ]:

# STEP 122: INPUT REPRESENTATION DIVERSITY

import numpy as np

print("=" * 70)
print("STEP 122: INPUT REPRESENTATION DIVERSITY")
print("=" * 70)

rng_122 = np.random.default_rng(42)

n_122 = min(500, len(X_val_119))

indices_122 = rng_122.choice(
    len(X_val_119),
    size=n_122,
    replace=False
)

# Single-channel mean-projected inputs.
single_122 = X_val_119[indices_122].reshape(n_122, -1)

# Original 100-dimensional pooled GloVe inputs.
pooled_122 = X_val_99[indices_122]

def pairwise_sample_122(arr, pairs=2000):
    a = rng_122.integers(0, len(arr), size=pairs)
    b = rng_122.integers(0, len(arr), size=pairs)

    mask = a != b
    a = a[mask]
    b = b[mask]

    x = arr[a]
    y = arr[b]

    numerator = np.sum(x * y, axis=1)
    denominator = (
        np.linalg.norm(x, axis=1)
        * np.linalg.norm(y, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-8)

single_scores_122 = pairwise_sample_122(single_122)
pooled_scores_122 = pairwise_sample_122(pooled_122)

print("MEAN-PROJECTED SINGLE-CHANNEL INPUT")
print("Mean pairwise cosine:",
      round(float(single_scores_122.mean()), 6))
print("Pairwise cosine SD:",
      round(float(single_scores_122.std()), 6))

print("\nPOOLED 100D GLOVE INPUT")
print("Mean pairwise cosine:",
      round(float(pooled_scores_122.mean()), 6))
print("Pairwise cosine SD:",
      round(float(pooled_scores_122.std()), 6))

print("\nINPUT NORMS")
print(
    "Single-channel mean norm:",
    round(float(
        np.linalg.norm(single_122, axis=1).mean()
    ), 6)
)
print(
    "Pooled GloVe mean norm:",
    round(float(
        np.linalg.norm(pooled_122, axis=1).mean()
    ), 6)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 123: TARGET REPRESENTATION DIAGNOSTIC

import numpy as np

print("=" * 70)
print("STEP 123: TARGET REPRESENTATION DIAGNOSTIC")
print("=" * 70)

rng_123 = np.random.default_rng(42)

Y_123 = Y_val_119.reshape(len(Y_val_119), -1)

# Sample pairs of different validation targets.
a_123 = rng_123.integers(
    0, len(Y_123), size=3000
)
b_123 = rng_123.integers(
    0, len(Y_123), size=3000
)

mask_123 = a_123 != b_123
a_123 = a_123[mask_123]
b_123 = b_123[mask_123]

def cosine_pair_123(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )
    return numerator / np.maximum(denominator, 1e-8)

target_pair_cosine_123 = cosine_pair_123(
    Y_123[a_123],
    Y_123[b_123]
)

# Nonzero sequence length (after projection).
target_lengths_123 = np.count_nonzero(
    Y_123, axis=1
)

# Exact duplicate representations.
unique_targets_123 = np.unique(
    Y_123,
    axis=0
)

print("TARGET PAIRWISE SIMILARITY")
print(
    "Mean cosine:",
    round(float(target_pair_cosine_123.mean()), 6)
)
print(
    "SD cosine:",
    round(float(target_pair_cosine_123.std()), 6)
)
print(
    "Median cosine:",
    round(float(np.median(target_pair_cosine_123)), 6)
)

print("\nTARGET LENGTH")
print(
    "Mean nonzero positions:",
    round(float(target_lengths_123.mean()), 2)
)
print(
    "Median nonzero positions:",
    float(np.median(target_lengths_123))
)
print(
    "Maximum nonzero positions:",
    int(target_lengths_123.max())
)

print("\nEXACT TARGET REPRESENTATION DUPLICATES")
print("Validation records:", len(Y_123))
print("Unique target vectors:", len(unique_targets_123))
print(
    "Duplicate representation count:",
    len(Y_123) - len(unique_targets_123)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 124: DECODER OUTPUT STRUCTURE DIAGNOSTIC

import numpy as np

print("=" * 70)
print("STEP 124: DECODER OUTPUT STRUCTURE")
print("=" * 70)

pred = pred_121.reshape(len(pred_121), 200)
true = Y_val_119.reshape(len(Y_val_119), 200)

# True active positions, based on nonzero target values.
active = true != 0

pred_energy = pred ** 2

total_energy = pred_energy.sum(axis=1)

energy_on_active = (
    (pred_energy * active).sum(axis=1)
    / np.maximum(total_energy, 1e-8)
)

energy_on_padding = 1.0 - energy_on_active

# Output variability across investigations.
position_std = pred.std(axis=0)

# Inspect several prediction vectors.
print("PREDICTION ENERGY")
print(
    "Mean fraction on true active positions:",
    round(float(energy_on_active.mean()), 6)
)
print(
    "Mean fraction on padded positions:",
    round(float(energy_on_padding.mean()), 6)
)

print("\nOUTPUT VARIABILITY")
print(
    "Mean position-wise SD:",
    round(float(position_std.mean()), 6)
)
print(
    "Median position-wise SD:",
    round(float(np.median(position_std)), 6)
)
print(
    "Maximum position-wise SD:",
    round(float(position_std.max()), 6)
)

print("\nFIRST 15 OUTPUT VALUES")

for i in range(3):
    print(
        f"Investigation {i + 1}:",
        np.round(pred[i, :15], 4)
    )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 125: LATENT REPRESENTATION DIAGNOSTIC

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 125: LATENT REPRESENTATION DIAGNOSTIC")
print("=" * 70)

# Extract the encoder from the trained baseline.
encoder_125 = baseline_model_119.layers[1]

print("Encoder:", encoder_125.name)
print("Encoder output shape:", encoder_125.output_shape)

Z_125 = encoder_125.predict(
    X_val_119,
    batch_size=32,
    verbose=0
)

print("Latent matrix shape:", Z_125.shape)
print("All finite:", np.isfinite(Z_125).all())

# Variation across investigations.
latent_std_125 = Z_125.std(axis=0)

print("\nLATENT DIMENSION VARIABILITY")
print("Mean SD:", round(float(latent_std_125.mean()), 6))
print("Median SD:", round(float(np.median(latent_std_125)), 6))
print("Minimum SD:", round(float(latent_std_125.min()), 6))
print("Maximum SD:", round(float(latent_std_125.max()), 6))

# Compare randomly selected latent vectors.
rng_125 = np.random.default_rng(42)

a_125 = rng_125.integers(0, len(Z_125), size=2000)
b_125 = rng_125.integers(0, len(Z_125), size=2000)

mask_125 = a_125 != b_125
a_125 = a_125[mask_125]
b_125 = b_125[mask_125]

za = Z_125[a_125]
zb = Z_125[b_125]

numerator = np.sum(za * zb, axis=1)
denominator = (
    np.linalg.norm(za, axis=1)
    * np.linalg.norm(zb, axis=1)
)

latent_cosine_125 = numerator / np.maximum(
    denominator, 1e-8
)

print("\nLATENT VECTOR SIMILARITY")
print(
    "Mean pairwise cosine:",
    round(float(latent_cosine_125.mean()), 6)
)
print(
    "SD pairwise cosine:",
    round(float(latent_cosine_125.std()), 6)
)

print("\nFIRST THREE LATENT VECTORS")
for i in range(3):
    print(f"Investigation {i + 1}:",
          np.round(Z_125[i], 4))

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 126: EFFECTIVE LATENT DIMENSIONALITY

import numpy as np

print("=" * 70)
print("STEP 126: EFFECTIVE LATENT DIMENSIONALITY")
print("=" * 70)

# Center latent representations.
Z_centered_126 = Z_125 - Z_125.mean(axis=0)

# Singular value decomposition.
_, singular_values_126, _ = np.linalg.svd(
    Z_centered_126,
    full_matrices=False
)

variance_126 = singular_values_126 ** 2

explained_126 = (
    variance_126 / variance_126.sum()
)

cumulative_126 = np.cumsum(explained_126)

# Participation ratio: effective number of dimensions.
effective_dim_126 = (
    variance_126.sum() ** 2
    / np.sum(variance_126 ** 2)
)

print("LATENT VARIANCE STRUCTURE")

for i in range(10):
    print(
        f"Component {i + 1:2d}: "
        f"explained={explained_126[i]:.6f}, "
        f"cumulative={cumulative_126[i]:.6f}"
    )

print("\nSUMMARY")
print(
    "Effective dimensionality:",
    round(float(effective_dim_126), 4)
)

print(
    "Components explaining 90% variance:",
    int(np.searchsorted(cumulative_126, 0.90) + 1)
)

print(
    "Components explaining 95% variance:",
    int(np.searchsorted(cumulative_126, 0.95) + 1)
)

print(
    "Components explaining 99% variance:",
    int(np.searchsorted(cumulative_126, 0.99) + 1)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 127: TRAINED VS UNTRAINED LATENT DIMENSIONALITY

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 127: TRAINED VS UNTRAINED LATENT SPACE")
print("=" * 70)

tf.keras.utils.set_random_seed(127)

untrained_model_127 = tf.keras.Model.from_config(
    reconstructed_model.get_config()
)

untrained_encoder_127 = untrained_model_127.layers[1]

# Use the same validation inputs for both encoders.
Z_untrained_127 = untrained_encoder_127.predict(
    X_val_119,
    batch_size=32,
    verbose=0
)

def latent_stats_127(Z):
    Z = np.asarray(Z, dtype=np.float64)
    centered = Z - Z.mean(axis=0)

    singular_values = np.linalg.svd(
        centered,
        compute_uv=False
    )

    variances = singular_values ** 2
    total = variances.sum()

    if total <= 0:
        return 0.0, 0.0, 0.0

    ratios = variances / total

    effective_dim = (
        total ** 2 / np.sum(variances ** 2)
    )

    return (
        float(ratios[0]),
        float(effective_dim),
        float(np.mean(Z.std(axis=0)))
    )

trained_stats = latent_stats_127(Z_125)
untrained_stats = latent_stats_127(Z_untrained_127)

print("\nTRAINED ENCODER")
print("First component variance:", round(trained_stats[0], 6))
print("Effective dimensionality:", round(trained_stats[1], 4))
print("Mean latent SD:", round(trained_stats[2], 6))

print("\nUNTRAINED ENCODER")
print("First component variance:", round(untrained_stats[0], 6))
print("Effective dimensionality:", round(untrained_stats[1], 4))
print("Mean latent SD:", round(untrained_stats[2], 6))

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 128: CHECK LATENT NUMERICAL SCALE

import numpy as np

print("=" * 70)
print("STEP 128: LATENT NUMERICAL SCALE AUDIT")
print("=" * 70)

for name, Z in [
    ("TRAINED", Z_125),
    ("UNTRAINED", Z_untrained_127)
]:
    Z = np.asarray(Z, dtype=np.float64)
    centered = Z - Z.mean(axis=0)

    singular_values = np.linalg.svd(
        centered,
        compute_uv=False
    )

    total_variance = np.sum(
        centered ** 2
    ) / max(len(Z) - 1, 1)

    print(f"\n{name} ENCODER")
    print("Minimum:", f"{Z.min():.12e}")
    print("Maximum:", f"{Z.max():.12e}")
    print("Mean absolute value:", f"{np.abs(Z).mean():.12e}")
    print("Mean dimension SD:", f"{Z.std(axis=0).mean():.12e}")
    print("Total variance:", f"{total_variance:.12e}")
    print("Largest singular value:", f"{singular_values[0]:.12e}")
    print("Smallest singular value:", f"{singular_values[-1]:.12e}")
    print("Exactly zero entries:", int(np.sum(Z == 0)))
    print("All finite:", bool(np.isfinite(Z).all()))

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 129: TRACE ENCODER SIGNAL ATTENUATION

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 129: UNTRAINED ENCODER SIGNAL TRACE")
print("=" * 70)

encoder_129 = untrained_encoder_127

# Inspect the encoder's internal layer outputs.
trace_layers_129 = [
    layer for layer in encoder_129.layers
    if isinstance(layer, (
        tf.keras.layers.Conv1D,
        tf.keras.layers.Dense,
        tf.keras.layers.Flatten
    ))
]

trace_model_129 = tf.keras.Model(
    inputs=encoder_129.input,
    outputs=[layer.output for layer in trace_layers_129]
)

sample_129 = X_val_119[:16]

outputs_129 = trace_model_129.predict(
    sample_129,
    verbose=0
)

print(
    f"{'Layer':35s} {'Shape':22s} "
    f"{'Mean |value|':>14s} {'Max |value|':>14s}"
)

for layer, output in zip(trace_layers_129, outputs_129):
    arr = np.asarray(output)

    print(
        f"{layer.name[:35]:35s} "
        f"{str(arr.shape):22s} "
        f"{np.mean(np.abs(arr)):14.6e} "
        f"{np.max(np.abs(arr)):14.6e}"
    )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 130: INSPECT ENCODER CONFIGURATION

import tensorflow as tf

print("=" * 85)
print("STEP 130: ENCODER CONFIGURATION AUDIT")
print("=" * 85)

encoder_130 = untrained_encoder_127

for layer in encoder_130.layers:

    if isinstance(layer, tf.keras.layers.Conv1D):

        config = layer.get_config()

        print(f"\n{layer.name}")
        print("  Filters:", config["filters"])
        print("  Kernel size:", config["kernel_size"])
        print("  Strides:", config["strides"])
        print("  Padding:", config["padding"])
        print("  Activation:", config["activation"])
        print("  Use bias:", config["use_bias"])
        print(
            "  Kernel initializer:",
            config["kernel_initializer"]["class_name"]
        )

    elif isinstance(layer, tf.keras.layers.Dense):

        config = layer.get_config()

        print(f"\n{layer.name}")
        print("  Units:", config["units"])
        print("  Activation:", config["activation"])
        print(
            "  Kernel initializer:",
            config["kernel_initializer"]["class_name"]
        )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 131: DECODER ACTIVATION AUDIT

import tensorflow as tf

print("=" * 70)
print("STEP 131: DECODER ACTIVATION AUDIT")
print("=" * 70)

decoder_131 = historical_decoder

for layer in decoder_131.layers:
    if isinstance(
        layer,
        (
            tf.keras.layers.Dense,
            tf.keras.layers.Conv1DTranspose,
            tf.keras.layers.Activation,
            tf.keras.layers.LeakyReLU,
            tf.keras.layers.ReLU
        )
    ):
        config = layer.get_config()

        print(
            f"{layer.name:30s} "
            f"type={layer.__class__.__name__:20s} "
            f"activation={config.get('activation', 'separate/none')}"
        )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 132: PREPARE CONTROLLED VAE ARCHITECTURE

import tensorflow as tf

print("=" * 70)
print("STEP 132: VAE ARCHITECTURE PREPARATION")
print("=" * 70)

encoder_132 = verified_encoder
decoder_132 = historical_decoder

print("ENCODER")
print("Input shape:", encoder_132.input_shape)
print("Output shape:", encoder_132.output_shape)
print("Total parameters:", encoder_132.count_params())

print("\nFINAL ENCODER LAYERS")
for layer in encoder_132.layers[-5:]:
    print(
        layer.name,
        layer.__class__.__name__,
        "output:",
        layer.output.shape
    )

print("\nDECODER")
print("Input shape:", decoder_132.input_shape)
print("Output shape:", decoder_132.output_shape)
print("Total parameters:", decoder_132.count_params())

print("\nREQUIRED VAE COMPONENTS")
print("Latent mean head: Dense(20)")
print("Latent log-variance head: Dense(20)")
print("Sampling: z = mean + exp(0.5 * logvar) * epsilon")
print("Objective: reconstruction loss + beta * KL divergence")

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 133: BUILD GENUINE VAE ARCHITECTURE
# Architecture only — no training

import tensorflow as tf

tf.keras.utils.set_random_seed(133)

LATENT_DIM_133 = 20

# Use the reconstructed encoder architecture as a feature extractor.
# Remove its final Dense(20) layer to expose 64 features.
encoder_features_133 = tf.keras.Model(
    inputs=verified_encoder.input,
    outputs=verified_encoder.get_layer("flatten").output,
    name="vae_encoder_features_133"
)

# Create a new feature extractor with fresh weights.
encoder_features_133 = tf.keras.Model.from_config(
    encoder_features_133.get_config()
)

# Independent latent distribution heads.
z_mean_layer_133 = tf.keras.layers.Dense(
    LATENT_DIM_133,
    name="z_mean_133"
)

z_logvar_layer_133 = tf.keras.layers.Dense(
    LATENT_DIM_133,
    name="z_logvar_133"
)

# Reparameterization trick.
class Sampling133(tf.keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_logvar = inputs

        epsilon = tf.random.normal(
            shape=tf.shape(z_mean)
        )

        return (
            z_mean
            + tf.exp(0.5 * z_logvar) * epsilon
        )

sampling_133 = Sampling133(name="sampling_133")

# Fresh decoder with the reconstructed architecture.
decoder_133 = tf.keras.Model.from_config(
    historical_decoder.get_config()
)

# Build the full VAE computation graph.
vae_inputs_133 = tf.keras.Input(
    shape=(1200, 1),
    name="vae_input_133"
)

features_133 = encoder_features_133(vae_inputs_133)

z_mean_133 = z_mean_layer_133(features_133)
z_logvar_133 = z_logvar_layer_133(features_133)

z_sample_133 = sampling_133([
    z_mean_133,
    z_logvar_133
])

vae_outputs_133 = decoder_133(z_sample_133)

vae_133 = tf.keras.Model(
    inputs=vae_inputs_133,
    outputs=vae_outputs_133,
    name="reconstructed_variational_autoencoder_133"
)

# Also expose posterior statistics for later diagnostics.
posterior_133 = tf.keras.Model(
    inputs=vae_inputs_133,
    outputs=[z_mean_133, z_logvar_133],
    name="vae_posterior_133"
)

print("=" * 70)
print("STEP 133: GENUINE VAE ARCHITECTURE")
print("=" * 70)

print("VAE input:", vae_133.input_shape)
print("VAE output:", vae_133.output_shape)

print("\nEncoder feature dimension:",
      encoder_features_133.output_shape)

print("Latent mean dimension:",
      z_mean_133.shape)

print("Latent log-variance dimension:",
      z_logvar_133.shape)

print("Sampled latent dimension:",
      z_sample_133.shape)

print("\nPARAMETER COUNTS")
print("Feature extractor:",
      encoder_features_133.count_params())
print("Mean head:",
      z_mean_layer_133.count_params())
print("Log-variance head:",
      z_logvar_layer_133.count_params())
print("Decoder:",
      decoder_133.count_params())
print("Total VAE:",
      vae_133.count_params())

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 134: CONFIGURE VAE TRAINING OBJECTIVE
# No training or weight updates in this step

import tensorflow as tf

BETA_134 = 0.001

class VAETrainer134(tf.keras.Model):

    def __init__(
        self,
        feature_extractor,
        mean_head,
        logvar_head,
        decoder,
        beta=0.001
    ):
        super().__init__()

        self.feature_extractor = feature_extractor
        self.mean_head = mean_head
        self.logvar_head = logvar_head
        self.decoder = decoder
        self.beta = beta

        self.total_loss_tracker = tf.keras.metrics.Mean(
            name="total_loss"
        )
        self.reconstruction_tracker = tf.keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_tracker = tf.keras.metrics.Mean(
            name="kl_loss"
        )

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_tracker,
            self.kl_tracker
        ]

    def compute_components(self, x, y, training=False):

        features = self.feature_extractor(
            x, training=training
        )

        z_mean = self.mean_head(features)
        z_logvar = self.logvar_head(features)

        if training:
            epsilon = tf.random.normal(tf.shape(z_mean))
            z = z_mean + tf.exp(0.5 * z_logvar) * epsilon
        else:
            z = z_mean

        prediction = self.decoder(
            z, training=training
        )

        reconstruction = sequence_cosine_loss(
            y, prediction
        )

        kl_per_case = -0.5 * tf.reduce_sum(
            1.0
            + z_logvar
            - tf.square(z_mean)
            - tf.exp(z_logvar),
            axis=1
        )

        kl = tf.reduce_mean(kl_per_case)

        total = reconstruction + self.beta * kl

        return total, reconstruction, kl

    def train_step(self, data):

        x, y = data

        with tf.GradientTape() as tape:
            total, reconstruction, kl = (
                self.compute_components(
                    x, y, training=True
                )
            )

        gradients = tape.gradient(
            total, self.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )

        self.total_loss_tracker.update_state(total)
        self.reconstruction_tracker.update_state(
            reconstruction
        )
        self.kl_tracker.update_state(kl)

        return {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss":
                self.reconstruction_tracker.result(),
            "kl_loss": self.kl_tracker.result()
        }

    def test_step(self, data):

        x, y = data

        total, reconstruction, kl = (
            self.compute_components(
                x, y, training=False
            )
        )

        self.total_loss_tracker.update_state(total)
        self.reconstruction_tracker.update_state(
            reconstruction
        )
        self.kl_tracker.update_state(kl)

        return {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss":
                self.reconstruction_tracker.result(),
            "kl_loss": self.kl_tracker.result()
        }


vae_trainer_134 = VAETrainer134(
    encoder_features_133,
    z_mean_layer_133,
    z_logvar_layer_133,
    decoder_133,
    beta=BETA_134
)

vae_trainer_134.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    )
)

# Evaluate one batch without training.
x_check_134 = tf.convert_to_tensor(
    X_train_119[:32]
)
y_check_134 = tf.convert_to_tensor(
    Y_train_119[:32]
)

total_134, recon_134, kl_134 = (
    vae_trainer_134.compute_components(
        x_check_134,
        y_check_134,
        training=False
    )
)

print("=" * 70)
print("STEP 134: VAE LOSS CONFIGURATION")
print("=" * 70)

print("Beta:", BETA_134)
print("Total loss:", float(total_134))
print("Reconstruction loss:", float(recon_134))
print("KL loss:", float(kl_134))

print(
    "All losses finite:",
    all(
        tf.math.is_finite(v).numpy()
        for v in [total_134, recon_134, kl_134]
    )
)

print(
    "Optimizer iterations:",
    int(vae_trainer_134.optimizer.iterations.numpy())
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 135: VAE TRAINING STABILITY PILOT
# 256 training investigations only

import numpy as np
import time

print("=" * 70)
print("STEP 135: VAE TRAINING STABILITY PILOT")
print("=" * 70)

X_pilot_135 = X_train_119[:256]
Y_pilot_135 = Y_train_119[:256]

start_135 = time.time()

history_135 = vae_trainer_134.fit(
    X_pilot_135,
    Y_pilot_135,
    batch_size=32,
    epochs=2,
    shuffle=True,
    verbose=1
)

elapsed_135 = time.time() - start_135

print("\nPILOT RESULTS")

for epoch in range(len(history_135.history["total_loss"])):
    print(
        f"Epoch {epoch + 1}: "
        f"total={history_135.history['total_loss'][epoch]:.6f}, "
        f"reconstruction="
        f"{history_135.history['reconstruction_loss'][epoch]:.6f}, "
        f"KL={history_135.history['kl_loss'][epoch]:.6f}"
    )

print("\nElapsed seconds:", round(elapsed_135, 2))

print(
    "Optimizer iterations:",
    int(vae_trainer_134.optimizer.iterations.numpy())
)

print("\nTraining performed: YES — pilot only")
print("Training investigations used: 256")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 136: FULL VAE TRAINING
# 16,002 training + 1,779 validation investigations

import tensorflow as tf
import numpy as np
import time

tf.keras.utils.set_random_seed(136)

# Fresh encoder feature extractor
features_136 = tf.keras.Model.from_config(
    encoder_features_133.get_config()
)

# Fresh latent distribution heads
mean_136 = tf.keras.layers.Dense(
    20, name="z_mean_136"
)

logvar_136 = tf.keras.layers.Dense(
    20, name="z_logvar_136"
)

# Fresh decoder
decoder_136 = tf.keras.Model.from_config(
    historical_decoder.get_config()
)

# Fresh VAE trainer
vae_trainer_136 = VAETrainer134(
    features_136,
    mean_136,
    logvar_136,
    decoder_136,
    beta=0.001
)

vae_trainer_136.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    )
)

callbacks_136 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_reconstruction_loss",
        mode="min",
        patience=3,
        restore_best_weights=True
    )
]

print("=" * 70)
print("STEP 136: FULL VAE TRAINING")
print("=" * 70)

print("Training cases:", len(X_train_119))
print("Validation cases:", len(X_val_119))
print("Test cases accessed: 0")
print("Beta:", vae_trainer_136.beta)

start_136 = time.time()

history_136 = vae_trainer_136.fit(
    X_train_119,
    Y_train_119,
    validation_data=(
        X_val_119,
        Y_val_119
    ),
    batch_size=32,
    epochs=15,
    shuffle=True,
    callbacks=callbacks_136,
    verbose=2
)

elapsed_136 = time.time() - start_136

best_epoch_136 = int(
    np.argmin(
        history_136.history[
            "val_reconstruction_loss"
        ]
    ) + 1
)

best_val_loss_136 = float(
    np.min(
        history_136.history[
            "val_reconstruction_loss"
        ]
    )
)

print("\nFINAL TRAINING SUMMARY")
print("Best epoch:", best_epoch_136)
print(
    "Best validation reconstruction loss:",
    round(best_val_loss_136, 6)
)
print(
    "Best validation cosine similarity:",
    round(1.0 - best_val_loss_136, 6)
)
print(
    "Elapsed minutes:",
    round(elapsed_136 / 60, 2)
)
print(
    "Optimizer iterations:",
    int(vae_trainer_136.optimizer.iterations.numpy())
)

print("\nTraining performed: YES")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 137: VAE POSTERIOR COLLAPSE DIAGNOSTIC
# Validation only; no training

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 137: VAE POSTERIOR DIAGNOSTIC")
print("=" * 70)

features_137 = vae_trainer_136.feature_extractor.predict(
    X_val_119,
    batch_size=32,
    verbose=0
)

z_mean_137 = vae_trainer_136.mean_head(
    features_137
).numpy()

z_logvar_137 = vae_trainer_136.logvar_head(
    features_137
).numpy()

kl_per_case_137 = -0.5 * np.sum(
    1.0
    + z_logvar_137
    - z_mean_137**2
    - np.exp(z_logvar_137),
    axis=1
)

mean_std_137 = z_mean_137.std(axis=0)

centered_137 = z_mean_137 - z_mean_137.mean(axis=0)

singular_137 = np.linalg.svd(
    centered_137,
    compute_uv=False
)

variance_137 = singular_137**2

effective_dim_137 = (
    variance_137.sum()**2
    / np.maximum(np.sum(variance_137**2), 1e-30)
)

print("VALIDATION CASES:", len(z_mean_137))

print("\nPOSTERIOR MEAN")
print(
    "Mean absolute value:",
    round(float(np.abs(z_mean_137).mean()), 8)
)
print(
    "Mean dimension SD:",
    round(float(mean_std_137.mean()), 8)
)
print(
    "Effective dimensionality:",
    round(float(effective_dim_137), 4)
)

print("\nPOSTERIOR LOG VARIANCE")
print(
    "Mean log variance:",
    round(float(z_logvar_137.mean()), 8)
)
print(
    "Mean posterior variance:",
    round(float(np.exp(z_logvar_137).mean()), 8)
)

print("\nKL DIVERGENCE")
print(
    "Mean KL per case:",
    round(float(kl_per_case_137.mean()), 8)
)
print(
    "Median KL per case:",
    round(float(np.median(kl_per_case_137)), 8)
)
print(
    "Maximum KL per case:",
    round(float(kl_per_case_137.max()), 8)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 138: BATCH NORMALIZATION AUDIT
# No training or test-set access

import tensorflow as tf

print("=" * 70)
print("STEP 138: BATCH NORMALIZATION AUDIT")
print("=" * 70)

models_138 = {
    "Reconstructed encoder": verified_encoder,
    "Reconstructed decoder": historical_decoder,
    "Trained deterministic model": baseline_model_119,
    "Trained VAE feature extractor":
        vae_trainer_136.feature_extractor,
    "Trained VAE decoder":
        vae_trainer_136.decoder
}

for name, model in models_138.items():

    batchnorm_layers = [
        layer for layer in model._flatten_layers(
            include_self=False,
            recursive=True
        )
        if isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        )
    ]

    print(f"\n{name}")
    print("BatchNormalization layers:", len(batchnorm_layers))

    for layer in batchnorm_layers[:15]:
        print("  ", layer.name)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 139: CURRENT INPUT DISTRIBUTION AUDIT
# Training data only; no training

import numpy as np

print("=" * 70)
print("STEP 139: INPUT DISTRIBUTION AUDIT")
print("=" * 70)

def summarize_139(name, array):
    values = np.asarray(array).reshape(-1)

    nonzero = values[values != 0]

    print(f"\n{name}")
    print("Shape:", array.shape)
    print("Overall minimum:", float(values.min()))
    print("Overall maximum:", float(values.max()))
    print("Overall mean:", float(values.mean()))
    print("Overall SD:", float(values.std()))
    print("Zero fraction:", float(np.mean(values == 0)))

    if len(nonzero):
        print("Nonzero minimum:", float(nonzero.min()))
        print("Nonzero maximum:", float(nonzero.max()))
        print("Nonzero mean:", float(nonzero.mean()))
        print("Nonzero SD:", float(nonzero.std()))
        print(
            "Nonzero percentiles [1, 25, 50, 75, 99]:",
            np.percentile(
                nonzero, [1, 25, 50, 75, 99]
            ).round(6)
        )

summarize_139(
    "TRAINING INPUT NARRATIVES",
    X_train_119
)

summarize_139(
    "TRAINING TARGET PROBABLE CAUSES",
    Y_train_119
)

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 140: SEQUENCE LENGTH AND PADDING DIAGNOSTIC
# Training data only; no model training

import numpy as np

print("=" * 70)
print("STEP 140: SEQUENCE LENGTH AND PADDING DIAGNOSTIC")
print("=" * 70)

def inspect_sequences_140(name, array):

    values = np.asarray(array)[:, :, 0]
    nonzero_mask = values != 0

    nonzero_counts = nonzero_mask.sum(axis=1)

    last_nonzero = np.where(
        nonzero_mask.any(axis=1),
        values.shape[1] - np.argmax(
            nonzero_mask[:, ::-1],
            axis=1
        ),
        0
    )

    internal_zeros = last_nonzero - nonzero_counts

    print(f"\n{name}")
    print("Cases:", len(values))
    print("Maximum sequence length:", values.shape[1])

    print("\nNonzero positions per case:")
    print(
        "Mean:",
        round(float(nonzero_counts.mean()), 2)
    )
    print(
        "Median:",
        float(np.median(nonzero_counts))
    )
    print(
        "Percentiles [5, 25, 75, 95]:",
        np.percentile(
            nonzero_counts, [5, 25, 75, 95]
        ).round(2)
    )

    print("\nLast nonzero position:")
    print(
        "Mean:",
        round(float(last_nonzero.mean()), 2)
    )
    print(
        "Median:",
        float(np.median(last_nonzero))
    )

    print("\nInternal zero positions:")
    print(
        "Mean:",
        round(float(internal_zeros.mean()), 2)
    )
    print(
        "Cases with internal zeros:",
        int(np.sum(internal_zeros > 0))
    )

    print(
        "\nCompletely zero sequences:",
        int(np.sum(nonzero_counts == 0))
    )

inspect_sequences_140(
    "TRAINING INPUT NARRATIVES",
    X_train_119
)

inspect_sequences_140(
    "TRAINING PROBABLE CAUSES",
    Y_train_119
)

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 141: RAW TEXT VS ENCODED LENGTH AUDIT
# Training data only

import numpy as np
import re

print("=" * 70)
print("STEP 141: RAW TEXT VS ENCODED LENGTH AUDIT")
print("=" * 70)

def word_count_141(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

def summarize_lengths_141(name, texts, encoded):
    raw_lengths = np.array([
        word_count_141(text)
        for text in texts
    ])

    encoded_lengths = np.count_nonzero(
        np.asarray(encoded)[:, :, 0],
        axis=1
    )

    print(f"\n{name}")
    print("Number of cases:", len(raw_lengths))

    print("\nRaw whitespace-separated words:")
    print("Mean:", round(raw_lengths.mean(), 2))
    print("Median:", round(np.median(raw_lengths), 2))
    print("SD:", round(raw_lengths.std(), 2))
    print("Maximum:", raw_lengths.max())

    print("\nEncoded nonzero positions:")
    print("Mean:", round(encoded_lengths.mean(), 2))
    print("Median:", round(np.median(encoded_lengths), 2))

    print("\nMean encoded/raw ratio:")
    valid = raw_lengths > 0
    print(
        round(
            float(np.mean(
                encoded_lengths[valid]
                / raw_lengths[valid]
            )),
            4
        )
    )

    print(
        "Cases with raw length above encoding limit:",
        int(np.sum(raw_lengths > encoded.shape[1]))
    )

summarize_lengths_141(
    "ANALYSIS NARRATIVES",
    train_df["input_text"].tolist(),
    X_train_119
)

summarize_lengths_141(
    "PROBABLE CAUSES",
    train_df["target_text"].tolist(),
    Y_train_119
)

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 142: INSPECT CURRENT TEXT FIELDS
# Training data only; no model training

import numpy as np

print("=" * 70)
print("STEP 142: TEXT FIELD INSPECTION")
print("=" * 70)

lengths_142 = train_df["input_text"].fillna("").apply(
    lambda text: len(str(text).split())
).to_numpy()

selected_142 = [
    int(np.argmin(lengths_142)),
    int(np.argsort(lengths_142)[len(lengths_142) // 2]),
    int(np.argmax(lengths_142))
]

labels_142 = [
    "SHORTEST NARRATIVE",
    "MEDIAN-LENGTH NARRATIVE",
    "LONGEST NARRATIVE"
]

for label, index in zip(labels_142, selected_142):
    record = train_df.iloc[index]

    narrative = str(record["input_text"])
    cause = str(record["target_text"])

    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    print("Investigation ID:", record["investigation_id"])
    print("Narrative word count:", len(narrative.split()))
    print("Probable-cause word count:", len(cause.split()))

    print("\nNARRATIVE:")
    print(narrative[:1800])

    print("\nPROBABLE CAUSE:")
    print(cause[:1000])

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 143: RECONSTRUCTED COHORT QUALITY AUDIT
# Metadata and training text only; no model training

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 143: COHORT QUALITY AUDIT")
print("=" * 70)

print("Full reconstructed cohort:", len(matched_dataset))
print("Training cases:", len(train_df))

def audit_text_143(df, label):

    inputs = df["input_text"].fillna("").astype(str).str.strip()
    targets = df["target_text"].fillna("").astype(str).str.strip()

    input_words = inputs.str.split().str.len()
    target_words = targets.str.split().str.len()

    placeholder = inputs.str.contains(
        r"^\s*(analysis pending|pending|not available|"
        r"no analysis|unknown|n/?a)\s*[.!]?\s*$",
        case=False,
        regex=True,
        na=False
    )

    print(f"\n{label}")
    print("Records:", len(df))
    print("Blank narratives:", int((inputs == "").sum()))
    print("Blank probable causes:", int((targets == "").sum()))
    print("Placeholder narratives:", int(placeholder.sum()))
    print("Narratives under 10 words:", int((input_words < 10).sum()))
    print("Narratives under 30 words:", int((input_words < 30).sum()))
    print("Probable causes under 5 words:", int((target_words < 5).sum()))
    print("Mean narrative words:", round(input_words.mean(), 2))
    print("Mean probable-cause words:", round(target_words.mean(), 2))

audit_text_143(
    matched_dataset,
    "FULL RECONSTRUCTED COHORT"
)

audit_text_143(
    train_df,
    "TRAINING PARTITION"
)

print("\nInvestigation ID examples:")
print(
    matched_dataset["investigation_id"]
    .astype(str)
    .head(10)
    .tolist()
)

print("\nTraining performed: NO")
print("Model predictions generated: NO")
print("Test features accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 144: WORD-COUNT VS CHARACTER-COUNT AUDIT
# No training; no model evaluation

import numpy as np

print("=" * 70)
print("STEP 144: TEXT LENGTH MEASUREMENT AUDIT")
print("=" * 70)

def length_audit_144(texts, label):
    texts = texts.fillna("").astype(str)

    words = texts.str.split().str.len().to_numpy()
    characters = texts.str.len().to_numpy()

    print(f"\n{label}")
    print("Mean words:", round(float(words.mean()), 2))
    print("Word SD:", round(float(words.std()), 2))
    print("Mean characters:", round(float(characters.mean()), 2))
    print("Character SD:", round(float(characters.std()), 2))
    print(
        "Mean characters per word:",
        round(float(characters.sum() / max(words.sum(), 1)), 2)
    )

length_audit_144(
    matched_dataset["input_text"],
    "ANALYSIS NARRATIVES"
)

length_audit_144(
    matched_dataset["target_text"],
    "PROBABLE CAUSES"
)

print("\nHISTORICAL MANUSCRIPT VALUES")
print("Narrative: mean 1116, SD 858.36")
print("Probable cause: mean 165, SD 93.12")

print("\nTraining performed: NO")
print("Model predictions generated: NO")
print("GPU required: NO")


In [ ]:

# STEP 145: HISTORICAL LENGTH STATISTICS VERIFICATION

import numpy as np

print("=" * 70)
print("STEP 145: HISTORICAL LENGTH STATISTICS VERIFICATION")
print("=" * 70)

comparisons = {
    "Narrative mean": (1116, 1157.86),
    "Narrative SD": (858.36, 810.41),
    "Probable-cause mean": (165, 162.20),
    "Probable-cause SD": (93.12, 92.73)
}

for name, (historical, reconstructed) in comparisons.items():
    difference = reconstructed - historical
    relative = 100 * difference / historical

    print(f"\n{name}")
    print("Historical:", historical)
    print("Reconstructed:", reconstructed)
    print("Difference:", round(difference, 2))
    print("Relative difference (%):", round(relative, 2))

print("\nTraining performed: NO")
print("Dataset modified: NO")
print("GPU required: NO")


In [ ]:

# STEP 146: COMPLETE PREPROCESSING FUNCTION INVENTORY

import inspect

print("=" * 70)
print("STEP 146: PREPROCESSING FUNCTION INVENTORY")
print("=" * 70)

candidate_names = [
    "tokenize",
    "normalize_glove_token",
    "text_to_glove",
    "glove_to_single_channel",
    "pca_single_channel",
    "pooled_target",
    "pca",
    "pca_1d",
    "mean_projection",
    "pca_projection",
    "sequence_cosine_loss"
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print(f"\n{'=' * 60}")
        print("OBJECT:", name)
        print("TYPE:", type(obj).__name__)

        if callable(obj):
            try:
                print(inspect.getsource(obj))
            except Exception:
                print("Source unavailable in current notebook.")
        else:
            print("Shape:", getattr(obj, "shape", "N/A"))

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 147: GLOVE VOCABULARY COVERAGE AUDIT

import numpy as np
from collections import Counter

print("=" * 70)
print("STEP 147: GLOVE VOCABULARY COVERAGE")
print("=" * 70)

def coverage_147(texts, label):
    total_tokens = 0
    covered_tokens = 0
    missing = Counter()

    for text in texts:
        tokens = tokenize(text)

        for token in tokens:
            total_tokens += 1

            normalized = normalize_glove_token(token)

            if (
                token in glove_vectors
                or normalized in glove_vectors
            ):
                covered_tokens += 1
            else:
                missing[token] += 1

    coverage = (
        covered_tokens / total_tokens
        if total_tokens else 0
    )

    print(f"\n{label}")
    print("Total tokens:", total_tokens)
    print("Covered tokens:", covered_tokens)
    print("Missing tokens:", total_tokens - covered_tokens)
    print("Coverage (%):", round(coverage * 100, 2))
    print("Top 15 missing tokens:", missing.most_common(15))

coverage_147(
    train_df["input_text"],
    "TRAINING NARRATIVES"
)

coverage_147(
    train_df["target_text"],
    "TRAINING PROBABLE CAUSES"
)

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 148: RECONSTRUCT ENCODER WITH BATCH NORMALIZATION
# No training

import tensorflow as tf

tf.keras.utils.set_random_seed(148)

print("=" * 70)
print("STEP 148: BATCH-NORMALIZED ENCODER")
print("=" * 70)

inputs_148 = tf.keras.Input(
    shape=(1200, 1),
    name="encoder_input_148"
)

x_148 = inputs_148

filters_148 = [
    256, 128, 128, 64, 64, 64,
    64, 64, 64, 64, 64
]

kernels_148 = [
    36, 25, 25, 16, 9, 4,
    2, 4, 4, 2, 1
]

strides_148 = [
    9, 8, 5, 4, 1, 1,
    1, 1, 1, 1, 1
]

for i, (filters, kernel, stride) in enumerate(
    zip(filters_148, kernels_148, strides_148),
    start=1
):
    x_148 = tf.keras.layers.Conv1D(
        filters=filters,
        kernel_size=kernel,
        strides=stride,
        padding="same",
        name=f"conv_{i}_148"
    )(x_148)

    x_148 = tf.keras.layers.BatchNormalization(
        name=f"batchnorm_{i}_148"
    )(x_148)

    x_148 = tf.keras.layers.LeakyReLU(
        negative_slope=0.3,
        name=f"activation_{i}_148"
    )(x_148)

x_148 = tf.keras.layers.Flatten(
    name="flatten_148"
)(x_148)

outputs_148 = tf.keras.layers.Dense(
    20,
    name="latent_148"
)(x_148)

encoder_148 = tf.keras.Model(
    inputs_148,
    outputs_148,
    name="batch_normalized_encoder_148"
)

print("Input shape:", encoder_148.input_shape)
print("Output shape:", encoder_148.output_shape)
print("Total parameters:", encoder_148.count_params())

bn_count_148 = sum(
    isinstance(layer, tf.keras.layers.BatchNormalization)
    for layer in encoder_148.layers
)

print("BatchNormalization layers:", bn_count_148)

print("\nLayer output shapes:")
for layer in encoder_148.layers:
    if isinstance(
        layer,
        (
            tf.keras.layers.Conv1D,
            tf.keras.layers.BatchNormalization,
            tf.keras.layers.Dense
        )
    ):
        print(
            layer.name,
            "->",
            layer.output.shape
        )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 149 (CORRECTED): BATCH-NORMALIZED DECODER
# Preserve original decoder layer configurations
# No training

import tensorflow as tf

tf.keras.utils.set_random_seed(149)

print("=" * 70)
print("STEP 149: BATCH-NORMALIZED DECODER")
print("=" * 70)

decoder_input_149 = tf.keras.Input(
    shape=(20,),
    name="decoder_input_149"
)

x_149 = decoder_input_149

conv_count_149 = 0
bn_count_149 = 0

for layer in historical_decoder.layers:

    if isinstance(layer, tf.keras.layers.InputLayer):
        continue

    # Reconstruct the original layer with fresh weights
    new_layer = layer.__class__.from_config(
        layer.get_config()
    )

    x_149 = new_layer(x_149)

    if isinstance(layer, tf.keras.layers.Conv1DTranspose):

        conv_count_149 += 1

        # Batch normalization for hidden transpose convolutions
        # Final output convolution remains unnormalized
        if conv_count_149 < 12:

            x_149 = tf.keras.layers.BatchNormalization(
                name=f"decoder_bn_{conv_count_149}_149"
            )(x_149)

            bn_count_149 += 1

decoder_149 = tf.keras.Model(
    decoder_input_149,
    x_149,
    name="batch_normalized_decoder_149"
)

print("Input shape:", decoder_149.input_shape)
print("Output shape:", decoder_149.output_shape)
print("Conv1DTranspose layers:", conv_count_149)
print("BatchNormalization layers:", bn_count_149)
print("Total parameters:", decoder_149.count_params())

assert decoder_149.output_shape[1:] == (200, 1)
assert conv_count_149 == 12
assert bn_count_149 == 11

print("\nDecoder layer summary:")

for layer in decoder_149.layers:
    if isinstance(
        layer,
        (
            tf.keras.layers.Conv1DTranspose,
            tf.keras.layers.BatchNormalization,
            tf.keras.layers.Cropping1D
        )
    ):
        print(
            layer.name,
            "->",
            layer.output.shape
        )

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 150: MANUSCRIPT RECONSTRUCTION LOSS
# Diagnostic only — no model training

import tensorflow as tf
import numpy as np

print("=" * 70)
print("STEP 150: ORIGINAL RECONSTRUCTION LOSS")
print("=" * 70)

def manuscript_cosine_loss_150(
    y_true,
    y_pred,
    alpha=1.0,
    epsilon=1e-6
):
    true_flat = tf.reshape(
        y_true,
        [tf.shape(y_true)[0], -1]
    )

    pred_flat = tf.reshape(
        y_pred,
        [tf.shape(y_pred)[0], -1]
    )

    true_unit = tf.math.l2_normalize(
        true_flat, axis=1
    )

    pred_unit = tf.math.l2_normalize(
        pred_flat, axis=1
    )

    cosine = tf.reduce_sum(
        true_unit * pred_unit,
        axis=1
    )

    cosine = tf.clip_by_value(
        cosine, -1.0, 1.0
    )

    denominator = tf.maximum(
        1.0 + cosine,
        epsilon
    )

    return tf.reduce_mean(
        alpha / denominator
    )

cosine_values_150 = [
    -0.9, -0.5, 0.0, 0.3, 0.5, 0.7, 0.786, 0.9, 1.0
]

print("\nLoss comparison (alpha = 1)")
print(
    f"{'Cosine':>10} "
    f"{'Original formula':>20} "
    f"{'Current formula':>20}"
)

for cs in cosine_values_150:
    original = 1.0 / (1.0 + cs)
    current = 1.0 - cs

    print(
        f"{cs:>10.3f} "
        f"{original:>20.6f} "
        f"{current:>20.6f}"
    )

# Verify implementation on identical vectors
sample_150 = tf.constant(
    [[[0.2], [0.4], [0.6]]],
    dtype=tf.float32
)

identical_loss_150 = manuscript_cosine_loss_150(
    sample_150,
    sample_150
)

print("\nIdentical-vector loss:", float(identical_loss_150))
print("Expected:", 0.5)

assert np.isclose(
    float(identical_loss_150),
    0.5,
    atol=1e-5
)

print("\nLoss implementation: VERIFIED")
print("Training performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 151: TRAINING-ONLY MINMAX SCALING PROPOSAL
# No training; no validation or test access

import numpy as np

print("=" * 70)
print("STEP 151: MINMAX SCALING DIAGNOSTIC")
print("=" * 70)

# Shared training-only range for input and target values.
# This is a reconstruction assumption, not a recovered
# historical scaler configuration.

x_values_151 = X_train_119[:, :, 0]
y_values_151 = Y_train_119[:, :, 0]

x_active_151 = x_values_151[x_values_151 != 0]
y_active_151 = y_values_151[y_values_151 != 0]

min_151 = min(
    float(x_active_151.min()),
    float(y_active_151.min())
)

max_151 = max(
    float(x_active_151.max()),
    float(y_active_151.max())
)

print("Training-only minimum:", min_151)
print("Training-only maximum:", max_151)

def scale_preserving_padding_151(array):
    array = np.asarray(array, dtype=np.float32)

    scaled = np.zeros_like(array)

    active = array != 0

    scaled[active] = (
        (array[active] - min_151)
        / (max_151 - min_151)
    )

    return scaled

# Small diagnostic sample only
x_sample_151 = scale_preserving_padding_151(
    X_train_119[:32]
)

y_sample_151 = scale_preserving_padding_151(
    Y_train_119[:32]
)

print("\nINPUT SAMPLE")
print("Original min:", float(X_train_119[:32].min()))
print("Original max:", float(X_train_119[:32].max()))
print("Scaled min:", float(x_sample_151.min()))
print("Scaled max:", float(x_sample_151.max()))

print("\nTARGET SAMPLE")
print("Original min:", float(Y_train_119[:32].min()))
print("Original max:", float(Y_train_119[:32].max()))
print("Scaled min:", float(y_sample_151.min()))
print("Scaled max:", float(y_sample_151.max()))

print("\nPADDING CHECK")
print(
    "Input zero-mask preserved:",
    bool(np.array_equal(
        X_train_119[:32] == 0,
        x_sample_151 == 0
    ))
)

print(
    "Target zero-mask preserved:",
    bool(np.array_equal(
        Y_train_119[:32] == 0,
        y_sample_151 == 0
    ))
)

print("\nTraining performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 152: CREATE SCALED TRAINING AND VALIDATION ARRAYS
# No training; test set untouched

import numpy as np

print("=" * 70)
print("STEP 152: SCALED TRAINING AND VALIDATION ARRAYS")
print("=" * 70)

def scale_with_mask_152(array):
    array = np.asarray(array, dtype=np.float32)

    original_padding = (array == 0)

    scaled = (
        (array - min_151)
        / (max_151 - min_151)
    ).astype(np.float32)

    scaled[original_padding] = 0.0

    return scaled, original_padding

X_train_152, X_train_pad_152 = scale_with_mask_152(
    X_train_119
)

Y_train_152, Y_train_pad_152 = scale_with_mask_152(
    Y_train_119
)

X_val_152, X_val_pad_152 = scale_with_mask_152(
    X_val_119
)

Y_val_152, Y_val_pad_152 = scale_with_mask_152(
    Y_val_119
)

arrays_152 = {
    "X_train": X_train_152,
    "Y_train": Y_train_152,
    "X_val": X_val_152,
    "Y_val": Y_val_152
}

for name, array in arrays_152.items():
    print(
        name,
        "shape:", array.shape,
        "min:", round(float(array.min()), 6),
        "max:", round(float(array.max()), 6),
        "finite:", bool(np.isfinite(array).all())
    )

print("\nORIGINAL PADDING PRESERVATION")

for name, scaled, mask in [
    ("X_train", X_train_152, X_train_pad_152),
    ("Y_train", Y_train_152, Y_train_pad_152),
    ("X_val", X_val_152, X_val_pad_152),
    ("Y_val", Y_val_152, Y_val_pad_152)
]:
    print(
        name,
        bool(np.all(scaled[mask] == 0.0))
    )

assert all(
    np.isfinite(array).all()
    for array in arrays_152.values()
)

assert all(
    np.all(scaled[mask] == 0.0)
    for scaled, mask in [
        (X_train_152, X_train_pad_152),
        (Y_train_152, Y_train_pad_152),
        (X_val_152, X_val_pad_152),
        (Y_val_152, Y_val_pad_152)
    ]
)

print("\nScaling completed successfully.")
print("Training performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 153: BUILD BATCH-NORMALIZED VARIATIONAL ENCODER
# No training; no validation or test access

import tensorflow as tf

tf.keras.utils.set_random_seed(153)

print("=" * 70)
print("STEP 153: VARIATIONAL ENCODER")
print("=" * 70)

# Reuse the trained-independent feature architecture
# from Step 148, excluding its final Dense(20) layer.

feature_extractor_153 = tf.keras.Model(
    inputs=encoder_148.input,
    outputs=encoder_148.get_layer("flatten_148").output,
    name="feature_extractor_153"
)

input_153 = tf.keras.Input(
    shape=(1200, 1),
    name="vae_input_153"
)

features_153 = feature_extractor_153(input_153)

z_mean_153 = tf.keras.layers.Dense(
    20,
    name="z_mean_153"
)(features_153)

z_logvar_153 = tf.keras.layers.Dense(
    20,
    name="z_logvar_153"
)(features_153)

variational_encoder_153 = tf.keras.Model(
    inputs=input_153,
    outputs=[z_mean_153, z_logvar_153],
    name="variational_encoder_153"
)

print("Input shape:", variational_encoder_153.input_shape)
print("Mean output shape:", variational_encoder_153.output_shape[0])
print("Log-variance output shape:", variational_encoder_153.output_shape[1])

print("Feature extractor parameters:",
      feature_extractor_153.count_params())

print("Variational encoder parameters:",
      variational_encoder_153.count_params())

bn_count_153 = sum(
    isinstance(layer, tf.keras.layers.BatchNormalization)
    for layer in feature_extractor_153.layers
)

print("BatchNormalization layers:", bn_count_153)

assert variational_encoder_153.output_shape == [
    (None, 20),
    (None, 20)
]

assert bn_count_153 == 11

print("\nArchitecture verified.")
print("Training performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# Only one training batch; no full training

import tensorflow as tf
import numpy as np

def corrected_train_step_154(self, data):

    # Compatible with Keras (x, y, sample_weight) format
    x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)

    with tf.GradientTape() as tape:

        z_mean, z_logvar = self.encoder(x, training=True)

        epsilon = tf.random.normal(shape=tf.shape(z_mean))

        z = z_mean + tf.exp(0.5 * z_logvar) * epsilon

        y_pred = self.decoder(z, training=True)

        reconstruction_loss = manuscript_cosine_loss_150(
            y, y_pred, alpha=1.0
        )

        kl_loss = tf.reduce_mean(
            -0.5 * tf.reduce_sum(
                1.0 + z_logvar
                - tf.square(z_mean)
                - tf.exp(z_logvar),
                axis=1
            )
        )

        total_loss = reconstruction_loss + self.beta * kl_loss

    gradients = tape.gradient(
        total_loss,
        self.trainable_variables
    )

    self.optimizer.apply_gradients(
        zip(gradients, self.trainable_variables)
    )

    self.total_tracker.update_state(total_loss)
    self.recon_tracker.update_state(reconstruction_loss)
    self.kl_tracker.update_state(kl_loss)

    return {
        metric.name: metric.result()
        for metric in self.metrics
    }

# Patch the model class, retaining the existing architecture.
BatchNormalizedVAE154.train_step = corrected_train_step_154

# Recompile so Keras uses the corrected training step.
vae_154.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    )
)

result_154 = vae_154.train_on_batch(
    X_train_152[:16],
    Y_train_152[:16],
    return_dict=True
)

print("=" * 70)
print("STEP 154: CORRECTED VAE PILOT")
print("=" * 70)

for key, value in result_154.items():
    print(f"{key}: {float(value):.6f}")

assert all(
    np.isfinite(float(value))
    for value in result_154.values()
)

print("\nForward/backward pass: SUCCESS")
print("Full training performed: NO")
print("Validation accessed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 155: SHORT VAE TRAINING PILOT
# 512 training examples, 128 validation examples
# CPU-compatible; test set untouched

import numpy as np
import tensorflow as tf

print("=" * 70)
print("STEP 155: SHORT VAE TRAINING PILOT")
print("=" * 70)

# Deterministic subsets of the existing training/validation splits
X_pilot_155 = X_train_152[:512]
Y_pilot_155 = Y_train_152[:512]

X_val_pilot_155 = X_val_152[:128]
Y_val_pilot_155 = Y_val_152[:128]

history_155 = vae_154.fit(
    X_pilot_155,
    Y_pilot_155,
    epochs=5,
    batch_size=16,
    shuffle=True,
    verbose=2
)

# Deterministic validation predictions using latent mean
z_mean_val_155, z_logvar_val_155 = (
    variational_encoder_153.predict(
        X_val_pilot_155,
        batch_size=16,
        verbose=0
    )
)

Y_pred_val_155 = decoder_149.predict(
    z_mean_val_155,
    batch_size=16,
    verbose=0
)

true_flat_155 = Y_val_pilot_155.reshape(128, -1)
pred_flat_155 = Y_pred_val_155.reshape(128, -1)

numerator_155 = np.sum(
    true_flat_155 * pred_flat_155,
    axis=1
)

denominator_155 = (
    np.linalg.norm(true_flat_155, axis=1)
    * np.linalg.norm(pred_flat_155, axis=1)
)

cosines_155 = numerator_155 / np.maximum(
    denominator_155, 1e-8
)

print("\nVALIDATION RESULTS")
print("Mean cosine:", float(cosines_155.mean()))
print("Cosine SD:", float(cosines_155.std()))

print("\nLATENT DIAGNOSTICS")
print(
    "Mean absolute latent mean:",
    float(np.abs(z_mean_val_155).mean())
)
print(
    "Mean posterior variance:",
    float(np.exp(z_logvar_val_155).mean())
)
print(
    "Mean across-case latent SD:",
    float(z_mean_val_155.std(axis=0).mean())
)

print("\nTraining examples:", len(X_pilot_155))
print("Validation examples:", len(X_val_pilot_155))
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 156: CONSTANT-PREDICTION BASELINE AND COLLAPSE CHECK
# Uses existing Step 155 validation predictions
# No training; test set untouched

import numpy as np

print("=" * 70)
print("STEP 156: VALIDATION BASELINE AND COLLAPSE CHECK")
print("=" * 70)

def row_cosine_156(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(len(a), -1)
    b = np.asarray(b, dtype=np.float64).reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-12)

# Baseline derived only from the 512 training targets
mean_target_156 = Y_pilot_155.mean(
    axis=0,
    keepdims=True
)

baseline_predictions_156 = np.repeat(
    mean_target_156,
    len(Y_val_pilot_155),
    axis=0
)

baseline_cosines_156 = row_cosine_156(
    Y_val_pilot_155,
    baseline_predictions_156
)

vae_cosines_156 = row_cosine_156(
    Y_val_pilot_155,
    Y_pred_val_155
)

# Compare predictions across different investigations
prediction_pair_cosines_156 = row_cosine_156(
    Y_pred_val_155[:-1],
    Y_pred_val_155[1:]
)

target_pair_cosines_156 = row_cosine_156(
    Y_val_pilot_155[:-1],
    Y_val_pilot_155[1:]
)

print("\nVALIDATION COSINE SIMILARITY")
print(
    "VAE mean:",
    round(float(vae_cosines_156.mean()), 6)
)
print(
    "Constant baseline mean:",
    round(float(baseline_cosines_156.mean()), 6)
)
print(
    "VAE improvement:",
    round(float(
        vae_cosines_156.mean()
        - baseline_cosines_156.mean()
    ), 6)
)

print("\nCROSS-INVESTIGATION SIMILARITY")
print(
    "Adjacent VAE predictions:",
    round(float(prediction_pair_cosines_156.mean()), 6)
)
print(
    "Adjacent true targets:",
    round(float(target_pair_cosines_156.mean()), 6)
)

print("\nPREDICTION VARIABILITY")
print(
    "Mean position-wise prediction SD:",
    round(float(
        Y_pred_val_155.std(axis=0).mean()
    ), 8)
)

print(
    "Mean position-wise target SD:",
    round(float(
        Y_val_pilot_155.std(axis=0).mean()
    ), 8)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 157: MEASURE COSINE INFLATION FROM MINMAX SCALING
# No training; test set untouched

import numpy as np

print("=" * 70)
print("STEP 157: SCALING EFFECT ON COSINE SIMILARITY")
print("=" * 70)

def cosine_rows_157(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(len(a), -1)
    b = np.asarray(b, dtype=np.float64).reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-12)

# Use exactly the same 512 training and 128 validation cases
# as Step 155, without accessing the test set.

train_raw_157 = Y_train_119[:512]
val_raw_157 = Y_val_119[:128]

train_scaled_157 = Y_train_152[:512]
val_scaled_157 = Y_val_152[:128]

# Constant baseline in original signed space
raw_mean_157 = train_raw_157.mean(axis=0, keepdims=True)
raw_predictions_157 = np.repeat(
    raw_mean_157,
    len(val_raw_157),
    axis=0
)

# Constant baseline in MinMax-scaled space
scaled_mean_157 = train_scaled_157.mean(axis=0, keepdims=True)
scaled_predictions_157 = np.repeat(
    scaled_mean_157,
    len(val_scaled_157),
    axis=0
)

raw_cos_157 = cosine_rows_157(
    val_raw_157,
    raw_predictions_157
)

scaled_cos_157 = cosine_rows_157(
    val_scaled_157,
    scaled_predictions_157
)

print("\nCONSTANT BASELINE")
print(
    "Original signed representation:",
    round(float(raw_cos_157.mean()), 6)
)
print(
    "MinMax-scaled representation:",
    round(float(scaled_cos_157.mean()), 6)
)

print(
    "Difference:",
    round(float(
        scaled_cos_157.mean() - raw_cos_157.mean()
    ), 6)
)

print("\nTARGET DISTRIBUTION")
print(
    "Raw target mean:",
    round(float(train_raw_157.mean()), 6)
)
print(
    "Scaled target mean:",
    round(float(train_scaled_157.mean()), 6)
)

print(
    "Raw target nonzero fraction:",
    round(float(np.mean(train_raw_157 != 0)), 6)
)
print(
    "Scaled target nonzero fraction:",
    round(float(np.mean(train_scaled_157 != 0)), 6)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 158: EVALUATE VAE AFTER REVERSING MINMAX SCALING
# No training; test set untouched

import numpy as np

print("=" * 70)
print("STEP 158: ORIGINAL-SCALE VAE EVALUATION")
print("=" * 70)

# The same 128 validation examples used in Step 155
pred_scaled_158 = np.asarray(
    Y_pred_val_155,
    dtype=np.float64
)

true_raw_158 = np.asarray(
    Y_val_119[:128],
    dtype=np.float64
)

# Recover signed values using the training-fitted range.
# The decoder has no explicit padding-mask output, so this
# inverse is applied to every predicted position.
pred_raw_158 = (
    pred_scaled_158 * (max_151 - min_151)
    + min_151
)

def cosine_rows_158(a, b):
    a = np.asarray(a).reshape(len(a), -1)
    b = np.asarray(b).reshape(len(b), -1)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )

    return numerator / np.maximum(denominator, 1e-12)

# Evaluation A: all 200 output positions
cos_all_158 = cosine_rows_158(
    true_raw_158,
    pred_raw_158
)

# Evaluation B: only positions with nonzero true values.
# This is a diagnostic, not a deployable prediction mask.
true_mask_158 = (true_raw_158 != 0)

pred_masked_158 = np.where(
    true_mask_158,
    pred_raw_158,
    0.0
)

cos_masked_158 = cosine_rows_158(
    true_raw_158,
    pred_masked_158
)

# Constant baseline in original signed space
raw_mean_158 = Y_train_119[:512].mean(
    axis=0,
    keepdims=True
)

raw_baseline_158 = np.repeat(
    raw_mean_158,
    128,
    axis=0
)

baseline_cos_158 = cosine_rows_158(
    true_raw_158,
    raw_baseline_158
)

print("\nVALIDATION COSINE")
print(
    "VAE, inverse-scaled, all positions:",
    round(float(cos_all_158.mean()), 6)
)
print(
    "VAE, inverse-scaled, true-active positions:",
    round(float(cos_masked_158.mean()), 6)
)
print(
    "Constant baseline, original signed space:",
    round(float(baseline_cos_158.mean()), 6)
)

print("\nPREDICTION DISTRIBUTION")
print(
    "Inverse-scaled prediction mean:",
    round(float(pred_raw_158.mean()), 6)
)
print(
    "Inverse-scaled prediction SD:",
    round(float(pred_raw_158.std()), 6)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 159: DECODER OUTPUT RANGE AND PADDING AUDIT
# No training; test set untouched

import numpy as np

print("=" * 70)
print("STEP 159: DECODER OUTPUT DIAGNOSTIC")
print("=" * 70)

pred_159 = np.asarray(Y_pred_val_155, dtype=np.float64)
true_159 = np.asarray(Y_val_pilot_155, dtype=np.float64)

active_159 = Y_val_119[:128] != 0
padding_159 = ~active_159

print("\nDECODER OUTPUT RANGE")
print("Minimum:", round(float(pred_159.min()), 6))
print("Maximum:", round(float(pred_159.max()), 6))
print("Mean:", round(float(pred_159.mean()), 6))
print("SD:", round(float(pred_159.std()), 6))

print("\nVALUES OUTSIDE [0, 1]")
print(
    "Fraction below zero:",
    round(float(np.mean(pred_159 < 0)), 6)
)
print(
    "Fraction above one:",
    round(float(np.mean(pred_159 > 1)), 6)
)

print("\nGROUND-TRUTH PADDING POSITIONS")
print(
    "Fraction of positions padded:",
    round(float(padding_159.mean()), 6)
)
print(
    "Mean absolute prediction at padded positions:",
    round(float(np.abs(pred_159[padding_159]).mean()), 6)
)
print(
    "Mean absolute prediction at active positions:",
    round(float(np.abs(pred_159[active_159]).mean()), 6)
)

print("\nPREDICTION DIVERSITY")
print(
    "Mean position-wise SD across investigations:",
    round(float(pred_159.std(axis=0).mean()), 8)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 160: FINAL EXPERIMENT — FULL VALIDATION BASELINE
# No model training
# No test-set access

import numpy as np

print("=" * 70)
print("STEP 160: FULL VALIDATION CONSTANT BASELINE")
print("=" * 70)

# Original signed GloVe-derived scalar representations
# These arrays have not undergone MinMax scaling.

train_targets_160 = np.asarray(
    Y_train_119,
    dtype=np.float32
)

val_targets_160 = np.asarray(
    Y_val_119,
    dtype=np.float32
)

# Mean target learned from training data only
mean_target_160 = train_targets_160.mean(
    axis=0,
    keepdims=True
)

def cosine_similarity_160(y_true, y_pred):
    y_true = np.asarray(
        y_true,
        dtype=np.float64
    ).reshape(len(y_true), -1)

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64
    ).reshape(len(y_pred), -1)

    numerator = np.sum(
        y_true * y_pred,
        axis=1
    )

    denominator = (
        np.linalg.norm(y_true, axis=1)
        * np.linalg.norm(y_pred, axis=1)
    )

    return numerator / np.maximum(
        denominator,
        1e-12
    )

# Predict the same mean vector for every validation case
baseline_predictions_160 = np.repeat(
    mean_target_160,
    len(val_targets_160),
    axis=0
)

baseline_cosines_160 = cosine_similarity_160(
    val_targets_160,
    baseline_predictions_160
)

print("Training cases:", len(train_targets_160))
print("Validation cases:", len(val_targets_160))

print("\nCONSTANT BASELINE — VALIDATION")
print(
    "Mean cosine:",
    round(float(baseline_cosines_160.mean()), 6)
)
print(
    "Standard deviation:",
    round(float(baseline_cosines_160.std()), 6)
)
print(
    "Median cosine:",
    round(float(np.median(baseline_cosines_160)), 6)
)

print(
    "Minimum cosine:",
    round(float(baseline_cosines_160.min()), 6)
)
print(
    "Maximum cosine:",
    round(float(baseline_cosines_160.max()), 6)
)

print("\nEVALUATION PROTOCOL")
print("Representation: original signed GloVe-derived scalar")
print("Baseline: training-set mean target")
print("Metric: per-investigation cosine similarity")
print("Training performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 161: SUPERVISED GLOVE RIDGE BASELINE
# CPU only; validation evaluation; test untouched

import numpy as np
from sklearn.linear_model import Ridge

print("=" * 70)
print("STEP 161: SUPERVISED GLOVE BASELINE")
print("=" * 70)

def pooled_input_161(text):
    matrix, _ = text_to_glove(
        text,
        MAX_INPUT_LEN
    )

    active = np.any(matrix != 0, axis=1)

    if not np.any(active):
        return np.zeros(
            EMBEDDING_DIM,
            dtype=np.float32
        )

    return matrix[active].mean(
        axis=0
    ).astype(np.float32)

print("Preparing training representations...")

X_ridge_train_161 = np.stack([
    pooled_input_161(text)
    for text in train_df["input_text"]
])

print("Preparing validation representations...")

X_ridge_val_161 = np.stack([
    pooled_input_161(text)
    for text in val_df["input_text"]
])

Y_ridge_train_161 = Y_train_119.reshape(
    len(Y_train_119),
    -1
)

Y_ridge_val_161 = Y_val_119.reshape(
    len(Y_val_119),
    -1
)

ridge_161 = Ridge(
    alpha=100.0,
    solver="lsqr"
)

ridge_161.fit(
    X_ridge_train_161,
    Y_ridge_train_161
)

Y_ridge_pred_161 = ridge_161.predict(
    X_ridge_val_161
)

ridge_cosines_161 = cosine_similarity_160(
    Y_ridge_val_161,
    Y_ridge_pred_161
)

print("\nVALIDATION RESULTS")
print(
    "Ridge mean cosine:",
    round(float(ridge_cosines_161.mean()), 6)
)
print(
    "Ridge cosine SD:",
    round(float(ridge_cosines_161.std()), 6)
)

print(
    "Constant baseline cosine:",
    round(float(baseline_cosines_160.mean()), 6)
)

print(
    "Improvement over constant baseline:",
    round(float(
        ridge_cosines_161.mean()
        - baseline_cosines_160.mean()
    ), 6)
)

print("\nPREDICTION DIVERSITY")
print(
    "Mean position-wise prediction SD:",
    round(float(
        Y_ridge_pred_161.std(axis=0).mean()
    ), 8)
)

print(
    "Mean position-wise target SD:",
    round(float(
        Y_ridge_val_161.std(axis=0).mean()
    ), 8)
)

print("\nTraining cases:", len(X_ridge_train_161))
print("Validation cases:", len(X_ridge_val_161))
print("GPU required: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 162: INPUT-SHUFFLING CONTROL
# Uses the already-trained Ridge model
# No retraining; test set untouched

import numpy as np

print("=" * 70)
print("STEP 162: INPUT-SHUFFLING CONTROL")
print("=" * 70)

rng_162 = np.random.default_rng(42)

# Original validation performance
original_scores_162 = cosine_similarity_160(
    Y_ridge_val_161,
    Y_ridge_pred_161
)

# Repeat input shuffling to estimate control variability
shuffled_means_162 = []

for repeat in range(100):
    permutation = rng_162.permutation(
        len(X_ridge_val_161)
    )

    shuffled_inputs = X_ridge_val_161[
        permutation
    ]

    shuffled_predictions = ridge_161.predict(
        shuffled_inputs
    )

    shuffled_scores = cosine_similarity_160(
        Y_ridge_val_161,
        shuffled_predictions
    )

    shuffled_means_162.append(
        shuffled_scores.mean()
    )

shuffled_means_162 = np.asarray(
    shuffled_means_162
)

print("\nORIGINAL INPUT-TARGET PAIRING")
print(
    "Mean cosine:",
    round(float(original_scores_162.mean()), 6)
)

print("\nSHUFFLED INPUT-TARGET PAIRING")
print(
    "Mean cosine across 100 permutations:",
    round(float(shuffled_means_162.mean()), 6)
)
print(
    "SD across permutations:",
    round(float(shuffled_means_162.std()), 6)
)

print("\nINPUT-DEPENDENT PERFORMANCE")
print(
    "Original minus shuffled mean:",
    round(float(
        original_scores_162.mean()
        - shuffled_means_162.mean()
    ), 6)
)

print(
    "Fraction of shuffled runs matching or exceeding original:",
    round(float(
        np.mean(
            shuffled_means_162 >= original_scores_162.mean()
        )
    ), 4)
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 163: FRESH MATCHED DETERMINISTIC MODEL
# Architecture preparation only
# No training; test set untouched

import tensorflow as tf
import numpy as np

tf.keras.utils.set_random_seed(42)

print("=" * 70)
print("STEP 163: MATCHED DETERMINISTIC ENCODER–DECODER")
print("=" * 70)

# Clone architecture without carrying over trained weights
encoder_163 = tf.keras.models.clone_model(encoder_148)
decoder_163 = tf.keras.models.clone_model(decoder_149)

# Build the complete model
input_163 = tf.keras.Input(
    shape=(1200, 1),
    name="analysis_input"
)

latent_163 = encoder_163(input_163)
output_163 = decoder_163(latent_163)

deterministic_163 = tf.keras.Model(
    inputs=input_163,
    outputs=output_163,
    name="deterministic_encoder_decoder_163"
)

# Explicit cosine reconstruction objective
def signed_cosine_loss_163(y_true, y_pred):
    y_true = tf.reshape(
        y_true,
        [tf.shape(y_true)[0], -1]
    )
    y_pred = tf.reshape(
        y_pred,
        [tf.shape(y_pred)[0], -1]
    )

    cosine = tf.reduce_sum(
        tf.math.l2_normalize(y_true, axis=1)
        * tf.math.l2_normalize(y_pred, axis=1),
        axis=1
    )

    return tf.reduce_mean(1.0 - cosine)

deterministic_163.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=signed_cosine_loss_163
)

print("\nMODEL SUMMARY")
print("Input shape:", deterministic_163.input_shape)
print("Output shape:", deterministic_163.output_shape)
print("Parameters:", deterministic_163.count_params())

print("\nARCHITECTURE VERIFICATION")
print(
    "Encoder Conv1D layers:",
    sum(
        isinstance(layer, tf.keras.layers.Conv1D)
        for layer in encoder_163.layers
    )
)
print(
    "Decoder Conv1DTranspose layers:",
    sum(
        isinstance(
            layer,
            tf.keras.layers.Conv1DTranspose
        )
        for layer in decoder_163.layers
    )
)

print(
    "Encoder BatchNormalization layers:",
    sum(
        isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        )
        for layer in encoder_163.layers
    )
)

print(
    "Decoder BatchNormalization layers:",
    sum(
        isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        )
        for layer in decoder_163.layers
    )
)

print("\nTraining performed: NO")
print("Test set accessed: NO")
print("GPU required: NO")


In [ ]:

# STEP 164: FULL DETERMINISTIC MODEL TRAINING
# 16,002 training cases; 1,779 validation cases
# Test set untouched

import tensorflow as tf
import numpy as np
import time

print("=" * 70)
print("STEP 164: DETERMINISTIC MODEL TRAINING")
print("=" * 70)

X_train_164 = np.asarray(X_train_119, dtype=np.float32)
Y_train_164 = np.asarray(Y_train_119, dtype=np.float32)

X_val_164 = np.asarray(X_val_119, dtype=np.float32)
Y_val_164 = np.asarray(Y_val_119, dtype=np.float32)

print("Training inputs:", X_train_164.shape)
print("Training targets:", Y_train_164.shape)
print("Validation inputs:", X_val_164.shape)
print("Validation targets:", Y_val_164.shape)

callbacks_164 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        min_delta=0.0005
    )
]

start_164 = time.time()

history_164 = deterministic_163.fit(
    X_train_164,
    Y_train_164,
    validation_data=(X_val_164, Y_val_164),
    epochs=12,
    batch_size=16,
    callbacks=callbacks_164,
    shuffle=True,
    verbose=2
)

elapsed_164 = time.time() - start_164

print("\nTRAINING SUMMARY")
print("Epochs completed:", len(history_164.history["loss"]))
print(
    "Best validation loss:",
    round(float(min(history_164.history["val_loss"])), 6)
)
print(
    "Training time (minutes):",
    round(elapsed_164 / 60, 2)
)

print("\nTraining completed: YES")
print("Test set accessed: NO")


In [ ]:

# STEP 164A: SHORT TRAINING BENCHMARK
# CPU-friendly pilot; no test access

import time
import numpy as np

print("=" * 65)
print("STEP 164A: FAST TRAINING BENCHMARK")
print("=" * 65)

# Start from fresh weights, regardless of how far
# the interrupted Step 164 progressed.
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

encoder_164a = tf.keras.models.clone_model(encoder_148)
decoder_164a = tf.keras.models.clone_model(decoder_149)

inputs_164a = tf.keras.Input(shape=(1200, 1))
outputs_164a = decoder_164a(encoder_164a(inputs_164a))

pilot_164a = tf.keras.Model(inputs_164a, outputs_164a)

pilot_164a.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=signed_cosine_loss_163
)

start_164a = time.time()

history_164a = pilot_164a.fit(
    X_train_119[:512],
    Y_train_119[:512],
    validation_data=(
        X_val_119[:128],
        Y_val_119[:128]
    ),
    epochs=1,
    batch_size=16,
    verbose=2
)

elapsed_164a = time.time() - start_164a

print("\nBENCHMARK RESULTS")
print("One-epoch time (seconds):", round(elapsed_164a, 2))
print("Training loss:", round(float(history_164a.history["loss"][-1]), 6))
print("Validation loss:", round(float(history_164a.history["val_loss"][-1]), 6))

print("\nEstimated full epoch (minutes):",
      round(elapsed_164a * (16002 / 512) / 60, 2))

print("Test set accessed: NO")


In [ ]:

# STEP 165: SAVE EXPERIMENT CHECKPOINT
# Run before changing Colab runtime

import os
import json
import numpy as np
from google.colab import drive

drive.mount("/content/drive")

save_dir_165 = (
    "/content/drive/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

os.makedirs(save_dir_165, exist_ok=True)

# Save prepared train/validation arrays
np.savez_compressed(
    os.path.join(
        save_dir_165,
        "train_validation_signed_glove.npz"
    ),
    X_train=X_train_119,
    Y_train=Y_train_119,
    X_val=X_val_119,
    Y_val=Y_val_119
)

# Save split identifiers, not test representations
for name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    frame[["investigation_id"]].to_csv(
        os.path.join(
            save_dir_165,
            f"{name}_investigation_ids.csv"
        ),
        index=False
    )

results_165 = {
    "dataset_total": 20919,
    "training_cases": 16002,
    "validation_cases": 1779,
    "test_cases": 3138,
    "constant_baseline_cosine": 0.289701,
    "ridge_validation_cosine": 0.323661,
    "ridge_shuffled_cosine": 0.269576,
    "deterministic_architecture_parameters": 1646677,
    "evaluation_representation": "signed_scalar_glove",
    "test_evaluated": False
}

with open(
    os.path.join(
        save_dir_165,
        "experiment_manifest.json"
    ),
    "w"
) as file:
    json.dump(
        results_165,
        file,
        indent=2
    )

print("=" * 65)
print("STEP 165: CHECKPOINT SAVED")
print("=" * 65)

print("Folder:", save_dir_165)
print("Files:", sorted(os.listdir(save_dir_165)))
print("Test evaluation performed: NO")


In [ ]:

# STEP 166: VERIFY GPU AND SAVED EXPERIMENT DATA

import os
import json
import numpy as np
import tensorflow as tf
from google.colab import drive

drive.mount("/content/drive")

print("=" * 65)
print("STEP 166: GPU ENVIRONMENT VERIFICATION")
print("=" * 65)

print("\nTensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

save_dir_166 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

with open(
    os.path.join(save_dir_166, "experiment_manifest.json"),
    "r"
) as f:
    manifest_166 = json.load(f)

data_166 = np.load(
    os.path.join(
        save_dir_166,
        "train_validation_signed_glove.npz"
    )
)

X_train_166 = data_166["X_train"]
Y_train_166 = data_166["Y_train"]
X_val_166 = data_166["X_val"]
Y_val_166 = data_166["Y_val"]

print("\nRESTORED DATA")
print("Training inputs:", X_train_166.shape)
print("Training targets:", Y_train_166.shape)
print("Validation inputs:", X_val_166.shape)
print("Validation targets:", Y_val_166.shape)

print("\nMANIFEST")
print("Dataset size:", manifest_166["dataset_total"])
print("Test evaluated:", manifest_166["test_evaluated"])

print("\nGPU available:",
      len(tf.config.list_physical_devices("GPU")) > 0)


In [ ]:
# STEP 167: GPU IDENTIFICATION

import tensorflow as tf

print("=" * 60)
print("STEP 167: GPU IDENTIFICATION")
print("=" * 60)

!nvidia-smi

print("\nTensorFlow GPU devices:")
print(tf.config.list_physical_devices("GPU"))

In [ ]:

# STEP 168: CHECK FOR SAVED MODEL ARCHITECTURE

import os

print("=" * 65)
print("STEP 168: CHECK SAVED ARCHITECTURE FILES")
print("=" * 65)

folder_168 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

for root, dirs, files in os.walk(folder_168):
    for filename in files:
        path = os.path.join(root, filename)
        print(os.path.relpath(path, folder_168))

print("\nGPU available:", len(
    tf.config.list_physical_devices("GPU")
) > 0)


In [ ]:

# STEP 169: REBUILD AND SAVE VERIFIED ARCHITECTURE
# No training; no test-set access.

import os
import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(42)

save_dir_169 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
os.makedirs(save_dir_169, exist_ok=True)

# --------------------------------------------------
# 1. ENCODER: EXACT STEP 148 CONFIGURATION
# --------------------------------------------------

encoder_input_169 = layers.Input(
    shape=(1200, 1), name="encoder_input"
)

x = encoder_input_169

encoder_filters = [
    256, 128, 128, 64, 64, 64,
    64, 64, 64, 64, 64
]
encoder_kernels = [
    36, 25, 25, 16, 9, 4,
    2, 4, 4, 2, 1
]
encoder_strides = [
    9, 8, 5, 4, 1, 1,
    1, 1, 1, 1, 1
]

for i, (nf, k, s) in enumerate(
    zip(encoder_filters, encoder_kernels, encoder_strides), 1
):
    x = layers.Conv1D(
        nf, k, strides=s, padding="same",
        name=f"encoder_conv_{i}"
    )(x)
    x = layers.BatchNormalization(
        name=f"encoder_bn_{i}"
    )(x)
    x = layers.LeakyReLU(
        negative_slope=0.3,
        name=f"encoder_activation_{i}"
    )(x)

x = layers.Flatten(name="encoder_flatten")(x)
z = layers.Dense(20, name="latent_vector")(x)

encoder_169 = Model(
    encoder_input_169, z, name="encoder_169"
)

# --------------------------------------------------
# 2. HISTORICAL DECODER: EXACT STEP 66 CONFIGURATION
# --------------------------------------------------

decoder_input_169 = layers.Input(
    shape=(20,), name="decoder_input"
)

x = layers.Dense(64, name="decoder_dense")(decoder_input_169)
x = layers.LeakyReLU(negative_slope=0.2)(x)
x = layers.Reshape((1, 64))(x)

target_lengths = [
    3, 6, 19, 22, 47, 47,
    96, 194, 197, 199, 200, 200
]
decoder_filters = [64] * 11 + [1]
decoder_kernels = [
    4, 4, 4, 4, 5, 4,
    4, 4, 3, 2, 1, 1
]

for i, (nf, k, target_len) in enumerate(
    zip(decoder_filters, decoder_kernels, target_lengths), 1
):
    x = layers.Conv1DTranspose(
        nf, k, strides=1, padding="valid",
        name=f"decoder_deconv_{i}"
    )(x)

    # Step 149: BN after first 11 transpose convolutions
    if i < 12:
        x = layers.BatchNormalization(
            name=f"decoder_bn_{i}"
        )(x)

    current_len = int(x.shape[1])

    if current_len < target_len:
        x = layers.ZeroPadding1D(
            (0, target_len - current_len),
            name=f"decoder_pad_{i}"
        )(x)
    elif current_len > target_len:
        x = layers.Cropping1D(
            (0, current_len - target_len),
            name=f"decoder_crop_{i}"
        )(x)

    x = layers.LeakyReLU(
        negative_slope=0.2,
        name=f"decoder_activation_{i}"
    )(x)

decoder_169 = Model(
    decoder_input_169, x, name="decoder_169"
)

# --------------------------------------------------
# 3. COMBINED DETERMINISTIC MODEL
# --------------------------------------------------

model_input_169 = layers.Input(
    shape=(1200, 1), name="narrative_input"
)

model_output_169 = decoder_169(
    encoder_169(model_input_169)
)

deterministic_169 = Model(
    model_input_169,
    model_output_169,
    name="deterministic_encoder_decoder_169"
)

# --------------------------------------------------
# 4. ARCHITECTURE VERIFICATION
# --------------------------------------------------

print("=" * 65)
print("STEP 169: ARCHITECTURE VERIFICATION")
print("=" * 65)

print("Encoder parameters:", encoder_169.count_params())
print("Decoder parameters:", decoder_169.count_params())
print("Combined parameters:", deterministic_169.count_params())
print("Input shape:", deterministic_169.input_shape)
print("Output shape:", deterministic_169.output_shape)

assert deterministic_169.input_shape[1:] == (1200, 1)
assert deterministic_169.output_shape[1:] == (200, 1)
assert deterministic_169.count_params() == 1646677, (
    "Parameter count differs from Step 163; inspect before training."
)

# --------------------------------------------------
# 5. SAVE ARCHITECTURES
# --------------------------------------------------

for name, model in [
    ("encoder", encoder_169),
    ("decoder", decoder_169),
    ("deterministic", deterministic_169)
]:
    path = os.path.join(save_dir_169, f"{name}_architecture.json")
    with open(path, "w") as f:
        f.write(model.to_json())
    print("Saved:", path)

print("\nArchitecture verified and saved.")
print("Training performed: NO")
print("Test set accessed: NO")
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


In [ ]:

# STEP 170A: RESTORE TRAINING VARIABLE NAMES
# No training and no test-set access.

import numpy as np
import os

data_path_170a = (
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "train_validation_signed_glove.npz"
)

with np.load(data_path_170a) as data:
    X_train_119 = data["X_train"].astype("float32")
    Y_train_119 = data["Y_train"].astype("float32")
    X_val_119 = data["X_val"].astype("float32")
    Y_val_119 = data["Y_val"].astype("float32")

print("=" * 65)
print("STEP 170A: TRAINING DATA RESTORED")
print("=" * 65)

print("Training inputs:", X_train_119.shape)
print("Training targets:", Y_train_119.shape)
print("Validation inputs:", X_val_119.shape)
print("Validation targets:", Y_val_119.shape)

assert X_train_119.shape == (16002, 1200, 1)
assert Y_train_119.shape == (16002, 200, 1)
assert X_val_119.shape == (1779, 1200, 1)
assert Y_val_119.shape == (1779, 200, 1)

print("\nAll dimensions verified.")
print("Training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 170A: RESTORE TRAINING VARIABLE NAMES
# No training and no test-set access.

import numpy as np
import os

data_path_170a = (
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "train_validation_signed_glove.npz"
)

with np.load(data_path_170a) as data:
    X_train_119 = data["X_train"].astype("float32")
    Y_train_119 = data["Y_train"].astype("float32")
    X_val_119 = data["X_val"].astype("float32")
    Y_val_119 = data["Y_val"].astype("float32")

print("=" * 65)
print("STEP 170A: TRAINING DATA RESTORED")
print("=" * 65)

print("Training inputs:", X_train_119.shape)
print("Training targets:", Y_train_119.shape)
print("Validation inputs:", X_val_119.shape)
print("Validation targets:", Y_val_119.shape)

assert X_train_119.shape == (16002, 1200, 1)
assert Y_train_119.shape == (16002, 200, 1)
assert X_val_119.shape == (1779, 1200, 1)
assert Y_val_119.shape == (1779, 200, 1)

print("\nAll dimensions verified.")
print("Training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 170: FULL GPU TRAINING — DETERMINISTIC MODEL

import os
import time
import numpy as np
import tensorflow as tf

save_dir_170 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
os.makedirs(save_dir_170, exist_ok=True)

tf.keras.utils.set_random_seed(42)

# Reuse verified Step 169 architecture
model_170 = deterministic_169

# Row-wise cosine loss across the full 200-element output
def signed_cosine_loss_170(y_true, y_pred):
    yt = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    yp = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])

    yt = tf.math.l2_normalize(yt, axis=1)
    yp = tf.math.l2_normalize(yp, axis=1)

    cosine = tf.reduce_sum(yt * yp, axis=1)
    return tf.reduce_mean(1.0 - cosine)

model_170.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=signed_cosine_loss_170
)

checkpoint_170 = os.path.join(
    save_dir_170, "deterministic_best_170.keras"
)

callbacks_170 = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_170,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        min_delta=0.0005,
        restore_best_weights=True
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(save_dir_170, "deterministic_log_170.csv")
    )
]

print("=" * 65)
print("STEP 170: DETERMINISTIC GPU TRAINING")
print("=" * 65)

print("Training investigations:", len(X_train_119))
print("Validation investigations:", len(X_val_119))
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", checkpoint_170)
print("Test set accessed: NO")

start_170 = time.time()

history_170 = model_170.fit(
    X_train_119,
    Y_train_119,
    validation_data=(X_val_119, Y_val_119),
    epochs=12,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks_170,
    verbose=2
)

elapsed_170 = time.time() - start_170

best_epoch_170 = int(
    np.argmin(history_170.history["val_loss"]) + 1
)
best_loss_170 = float(
    np.min(history_170.history["val_loss"])
)

print("\n" + "=" * 65)
print("TRAINING SUMMARY")
print("=" * 65)
print("Best epoch:", best_epoch_170)
print("Best validation cosine:", round(1.0 - best_loss_170, 6))
print("Training minutes:", round(elapsed_170 / 60, 2))
print("Best model saved:", checkpoint_170)
print("Test set accessed: NO")


In [ ]:

# STEP 171: VALIDATION EVALUATION AND PREDICTION DIVERSITY
# No training. Held-out test set remains untouched.

import os
import numpy as np
import tensorflow as tf

checkpoint_171 = (
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "deterministic_best_170.keras"
)

model_171 = tf.keras.models.load_model(
    checkpoint_171,
    compile=False
)

pred_171 = model_171.predict(
    X_val_119,
    batch_size=64,
    verbose=0
)

true_171 = Y_val_119.reshape(len(Y_val_119), -1)
pred_flat_171 = pred_171.reshape(len(pred_171), -1)

def row_cosine_171(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )
    return numerator / np.maximum(denominator, 1e-12)

scores_171 = row_cosine_171(true_171, pred_flat_171)

# Measure variation across predicted investigation outputs
prediction_std_171 = np.mean(
    np.std(pred_flat_171, axis=0)
)

target_std_171 = np.mean(
    np.std(true_171, axis=0)
)

# Compare each prediction with the next prediction
adjacent_cosine_171 = row_cosine_171(
    pred_flat_171[:-1],
    pred_flat_171[1:]
)

print("=" * 65)
print("STEP 171: INDEPENDENT VALIDATION EVALUATION")
print("=" * 65)

print("Validation investigations:", len(scores_171))
print("Mean cosine:", round(float(scores_171.mean()), 6))
print("Cosine SD:", round(float(scores_171.std()), 6))
print("Median cosine:", round(float(np.median(scores_171)), 6))

print("\nPREDICTION DIVERSITY")
print("Mean prediction-position SD:",
      round(float(prediction_std_171), 8))
print("Mean target-position SD:",
      round(float(target_std_171), 8))
print("Adjacent prediction cosine:",
      round(float(adjacent_cosine_171.mean()), 6))

print("\nREFERENCE BASELINES")
print("Constant baseline:", 0.289701)
print("Ridge baseline:", 0.323661)

print("\nTraining performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 172: INPUT-SHUFFLING CONTROL
# No training and no test-set access.

import numpy as np

rng_172 = np.random.default_rng(42)

original_mean_172 = float(scores_171.mean())

shuffle_means_172 = []

for repetition in range(100):
    indices = rng_172.permutation(len(pred_flat_171))

    shuffled_predictions = pred_flat_171[indices]

    shuffled_scores = row_cosine_171(
        true_171,
        shuffled_predictions
    )

    shuffle_means_172.append(
        float(shuffled_scores.mean())
    )

shuffle_means_172 = np.array(shuffle_means_172)

print("=" * 65)
print("STEP 172: DETERMINISTIC MODEL SHUFFLING CONTROL")
print("=" * 65)

print("Original validation cosine:",
      round(original_mean_172, 6))

print("Mean shuffled cosine:",
      round(float(shuffle_means_172.mean()), 6))

print("Shuffled cosine SD:",
      round(float(shuffle_means_172.std()), 6))

print("Original minus shuffled:",
      round(
          original_mean_172 -
          float(shuffle_means_172.mean()),
          6
      ))

print("Shuffles matching/exceeding original:",
      int(np.sum(shuffle_means_172 >= original_mean_172)),
      "out of 100")

print("\nTraining performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 173: SAVE VERIFIED DETERMINISTIC RESULTS

import os
import json
import numpy as np

folder_173 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
os.makedirs(folder_173, exist_ok=True)

results_173 = {
    "model": "deterministic_batch_normalized_encoder_decoder",
    "representation": "signed_scalar_glove",
    "training_cases": 16002,
    "validation_cases": 1779,
    "best_epoch": 2,
    "validation_cosine_mean": float(scores_171.mean()),
    "validation_cosine_sd": float(scores_171.std()),
    "prediction_position_sd": float(prediction_std_171),
    "target_position_sd": float(target_std_171),
    "adjacent_prediction_cosine": float(
        adjacent_cosine_171.mean()
    ),
    "shuffled_cosine_mean": float(
        shuffle_means_172.mean()
    ),
    "shuffled_cosine_sd": float(
        shuffle_means_172.std()
    ),
    "shuffling_repetitions": 100,
    "constant_baseline_cosine": 0.289701,
    "ridge_baseline_cosine": 0.323661,
    "test_evaluated": False
}

path_173 = os.path.join(
    folder_173,
    "deterministic_validation_results.json"
)

with open(path_173, "w") as f:
    json.dump(results_173, f, indent=2)

print("Saved:", path_173)
print("Validation cosine:", results_173["validation_cosine_mean"])
print("Test set accessed: NO")


In [ ]:

# STEP 174: CONSTRUCT GENUINE VARIATIONAL ENCODER
# No training. No test-set access.

import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(42)

# Reuse the verified Step 169 encoder's feature layers,
# excluding its final deterministic latent Dense layer.
feature_extractor_174 = Model(
    inputs=encoder_169.input,
    outputs=encoder_169.get_layer("encoder_flatten").output,
    name="vae_feature_extractor_174"
)

vae_input_174 = layers.Input(
    shape=(1200, 1),
    name="vae_input_174"
)

features_174 = feature_extractor_174(vae_input_174)

z_mean_174 = layers.Dense(
    20, name="z_mean_174"
)(features_174)

z_logvar_174 = layers.Dense(
    20, name="z_logvar_174"
)(features_174)

variational_encoder_174 = Model(
    vae_input_174,
    [z_mean_174, z_logvar_174],
    name="variational_encoder_174"
)

# Reparameterization:
# z = mean + exp(0.5 * logvar) * epsilon
class Sampling174(layers.Layer):
    def call(self, inputs):
        mean, logvar = inputs
        epsilon = tf.random.normal(tf.shape(mean))
        return mean + tf.exp(0.5 * logvar) * epsilon

sampling_174 = Sampling174(name="sampling_174")

# Use a fresh decoder with the verified architecture.
decoder_174 = tf.keras.models.clone_model(decoder_169)

print("=" * 65)
print("STEP 174: GENUINE VAE ARCHITECTURE")
print("=" * 65)

print("Feature extractor output:",
      feature_extractor_174.output_shape)
print("Latent mean shape:",
      variational_encoder_174.output_shape[0])
print("Latent log-variance shape:",
      variational_encoder_174.output_shape[1])
print("Decoder output:", decoder_174.output_shape)

sample_mean_174, sample_logvar_174 = variational_encoder_174(
    X_val_119[:2], training=False
)

sample_z_174 = sampling_174(
    [sample_mean_174, sample_logvar_174]
)

sample_output_174 = decoder_174(
    sample_z_174, training=False
)

assert sample_z_174.shape == (2, 20)
assert sample_output_174.shape == (2, 200, 1)

print("Sampled latent shape:", sample_z_174.shape)
print("Reconstruction shape:", sample_output_174.shape)
print("VAE components verified: YES")
print("Training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 175: DEFINE AND VERIFY VAE TRAINER
# No dataset training; no test-set access.

import tensorflow as tf
from tensorflow.keras import Model

tf.keras.utils.set_random_seed(42)

class VAETrainer175(Model):

    def __init__(self, encoder, decoder, sampling, beta=0.001):
        super().__init__(name="genuine_vae_175")
        self.encoder = encoder
        self.decoder = decoder
        self.sampling = sampling
        self.beta = beta

        self.total_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_tracker = tf.keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_tracker,
            self.recon_tracker,
            self.kl_tracker
        ]

    def compute_losses(self, x, y, training):
        mean, logvar = self.encoder(x, training=training)
        z = self.sampling([mean, logvar])
        prediction = self.decoder(z, training=training)

        yt = tf.reshape(y, [tf.shape(y)[0], -1])
        yp = tf.reshape(prediction, [tf.shape(prediction)[0], -1])

        yt = tf.math.l2_normalize(yt, axis=1)
        yp = tf.math.l2_normalize(yp, axis=1)

        cosine = tf.reduce_sum(yt * yp, axis=1)
        reconstruction = tf.reduce_mean(1.0 - cosine)

        # Clip only for numerical stability of exp(logvar).
        safe_logvar = tf.clip_by_value(logvar, -20.0, 20.0)

        kl_per_case = -0.5 * tf.reduce_sum(
            1.0 + safe_logvar
            - tf.square(mean)
            - tf.exp(safe_logvar),
            axis=1
        )
        kl = tf.reduce_mean(kl_per_case)

        total = reconstruction + self.beta * kl
        return total, reconstruction, kl

    def train_step(self, data):
        x, y = data

        with tf.GradientTape() as tape:
            total, reconstruction, kl = self.compute_losses(
                x, y, training=True
            )

        gradients = tape.gradient(
            total, self.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )

        self.total_tracker.update_state(total)
        self.recon_tracker.update_state(reconstruction)
        self.kl_tracker.update_state(kl)

        return {
            "loss": self.total_tracker.result(),
            "reconstruction_loss": self.recon_tracker.result(),
            "kl_loss": self.kl_tracker.result()
        }

    def test_step(self, data):
        x, y = data

        total, reconstruction, kl = self.compute_losses(
            x, y, training=False
        )

        self.total_tracker.update_state(total)
        self.recon_tracker.update_state(reconstruction)
        self.kl_tracker.update_state(kl)

        return {
            "loss": self.total_tracker.result(),
            "reconstruction_loss": self.recon_tracker.result(),
            "kl_loss": self.kl_tracker.result()
        }

vae_175 = VAETrainer175(
    encoder=variational_encoder_174,
    decoder=decoder_174,
    sampling=sampling_174,
    beta=0.001
)

vae_175.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    )
)

# Numerical verification on two validation examples.
total_175, recon_175, kl_175 = vae_175.compute_losses(
    tf.convert_to_tensor(X_val_119[:2]),
    tf.convert_to_tensor(Y_val_119[:2]),
    training=False
)

print("=" * 65)
print("STEP 175: VAE OBJECTIVE VERIFICATION")
print("=" * 65)

print("Total loss:", float(total_175.numpy()))
print("Reconstruction loss:", float(recon_175.numpy()))
print("KL divergence:", float(kl_175.numpy()))
print("Beta:", vae_175.beta)

assert tf.math.is_finite(total_175)
assert tf.math.is_finite(recon_175)
assert tf.math.is_finite(kl_175)
assert float(kl_175.numpy()) >= 0

print("\nVAE objective verified: YES")
print("Dataset training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 176: FULL VAE GPU TRAINING
# No held-out test-set access.

import os
import time
import numpy as np
import tensorflow as tf

folder_176 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
os.makedirs(folder_176, exist_ok=True)

checkpoint_176 = os.path.join(
    folder_176, "vae_best_176.weights.h5"
)

log_176 = os.path.join(
    folder_176, "vae_training_log_176.csv"
)

callbacks_176 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_176,
        monitor="val_reconstruction_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_reconstruction_loss",
        mode="min",
        patience=3,
        min_delta=0.0005,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        log_176,
        append=False
    )
]

print("=" * 65)
print("STEP 176: FULL VAE GPU TRAINING")
print("=" * 65)

print("Training investigations:", len(X_train_119))
print("Validation investigations:", len(X_val_119))
print("GPU:", tf.config.list_physical_devices("GPU"))
print("KL weight:", vae_175.beta)
print("Best checkpoint:", checkpoint_176)
print("Test set accessed: NO")

start_176 = time.time()

history_176 = vae_175.fit(
    X_train_119,
    Y_train_119,
    validation_data=(X_val_119, Y_val_119),
    epochs=12,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks_176,
    verbose=2
)

elapsed_176 = time.time() - start_176

best_epoch_176 = int(
    np.argmin(
        history_176.history["val_reconstruction_loss"]
    ) + 1
)

best_recon_176 = float(
    np.min(
        history_176.history["val_reconstruction_loss"]
    )
)

print("\n" + "=" * 65)
print("STEP 176: TRAINING SUMMARY")
print("=" * 65)

print("Best epoch:", best_epoch_176)
print("Best validation reconstruction loss:",
      round(best_recon_176, 6))
print("Approximate validation cosine:",
      round(1.0 - best_recon_176, 6))
print("Training minutes:",
      round(elapsed_176 / 60, 2))
print("Best checkpoint:", checkpoint_176)
print("Test set accessed: NO")


In [ ]:

# STEP 176A: REPAIR VAE CHECKPOINTING
# Preserve weights learned during Epoch 1.
# No additional training or test-set access.

import os
import tensorflow as tf

folder_176a = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
os.makedirs(folder_176a, exist_ok=True)

# Define the forward pass so Keras can recognize the
# outer VAE as a built model.
def vae_forward_176a(self, inputs, training=False):
    mean, logvar = self.encoder(inputs, training=training)

    # Deterministic forward pass for inspection/saving.
    # Stochastic sampling remains in compute_losses().
    return self.decoder(mean, training=training)

VAETrainer175.call = vae_forward_176a

# Invoke the model to establish its built state.
sample_176a = vae_175(
    tf.convert_to_tensor(X_val_119[:2]),
    training=False
)

assert sample_176a.shape == (2, 200, 1)
assert vae_175.built

# Preserve weights already learned during Epoch 1.
checkpoint_176a = os.path.join(
    folder_176a,
    "vae_epoch1_recovered.weights.h5"
)

vae_175.save_weights(checkpoint_176a)

print("=" * 65)
print("STEP 176A: VAE CHECKPOINT REPAIR")
print("=" * 65)

print("VAE built:", vae_175.built)
print("Forward output shape:", sample_176a.shape)
print("Recovered weights saved:", checkpoint_176a)
print("Checkpoint exists:", os.path.isfile(checkpoint_176a))
print("Optimizer iterations:",
      int(vae_175.optimizer.iterations.numpy()))

print("\nAdditional training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 176B: RESUME VAE GPU TRAINING
# Continue from the completed first epoch.
# No test-set access.

import os
import time
import numpy as np
import tensorflow as tf

folder_176b = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

checkpoint_176b = os.path.join(
    folder_176b, "vae_best_176b.weights.h5"
)

log_176b = os.path.join(
    folder_176b, "vae_training_log_176b.csv"
)

# Confirm the model is built and retains Epoch 1 training.
assert vae_175.built
assert int(vae_175.optimizer.iterations.numpy()) == 501, (
    "Unexpected optimizer state. Do not restart training."
)

callbacks_176b = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_176b,
        monitor="val_reconstruction_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_reconstruction_loss",
        mode="min",
        patience=3,
        min_delta=0.0005,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        log_176b,
        append=False
    )
]

print("=" * 65)
print("STEP 176B: CONTINUED VAE GPU TRAINING")
print("=" * 65)

print("Completed optimizer updates:",
      int(vae_175.optimizer.iterations.numpy()))
print("Training investigations:", len(X_train_119))
print("Validation investigations:", len(X_val_119))
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", checkpoint_176b)
print("Test set accessed: NO")

start_176b = time.time()

history_176b = vae_175.fit(
    X_train_119,
    Y_train_119,
    validation_data=(X_val_119, Y_val_119),
    initial_epoch=1,
    epochs=12,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks_176b,
    verbose=2
)

elapsed_176b = time.time() - start_176b

best_epoch_176b = int(
    np.argmin(
        history_176b.history["val_reconstruction_loss"]
    ) + 2
)

best_recon_176b = float(
    np.min(
        history_176b.history["val_reconstruction_loss"]
    )
)

print("\n" + "=" * 65)
print("STEP 176B: TRAINING SUMMARY")
print("=" * 65)

print("Best epoch:", best_epoch_176b)
print("Best validation reconstruction loss:",
      round(best_recon_176b, 6))
print("Approximate validation cosine:",
      round(1.0 - best_recon_176b, 6))
print("Additional training minutes:",
      round(elapsed_176b / 60, 2))
print("Best checkpoint:", checkpoint_176b)
print("Test set accessed: NO")


In [ ]:

# STEP 177: INDEPENDENT VAE VALIDATION
# Compare Epoch 1 and Epoch 2 checkpoints.
# No training. No test-set access.

import os
import numpy as np
import tensorflow as tf

folder_177 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

checkpoints_177 = {
    "Epoch 1": os.path.join(
        folder_177, "vae_epoch1_recovered.weights.h5"
    ),
    "Epoch 2": os.path.join(
        folder_177, "vae_best_176b.weights.h5"
    )
}

def cosine_rows_177(a, b):
    a = np.asarray(a).reshape(len(a), -1)
    b = np.asarray(b).reshape(len(b), -1)

    return np.sum(a * b, axis=1) / np.maximum(
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1),
        1e-12
    )

results_177 = {}

for label, path in checkpoints_177.items():

    assert os.path.exists(path), f"Missing checkpoint: {path}"

    vae_175.load_weights(path)

    # Deterministic inference using posterior mean.
    z_mean, z_logvar = vae_175.encoder.predict(
        X_val_119, batch_size=64, verbose=0
    )

    predictions = vae_175.decoder.predict(
        z_mean, batch_size=64, verbose=0
    )

    scores = cosine_rows_177(Y_val_119, predictions)

    flat = predictions.reshape(len(predictions), -1)

    rng = np.random.default_rng(42)
    shuffled_means = []

    for _ in range(100):
        indices = rng.permutation(len(flat))
        shuffled_means.append(
            cosine_rows_177(
                Y_val_119,
                flat[indices]
            ).mean()
        )

    results_177[label] = {
        "mean_cosine": float(scores.mean()),
        "cosine_sd": float(scores.std()),
        "prediction_position_sd": float(
            np.mean(np.std(flat, axis=0))
        ),
        "adjacent_prediction_cosine": float(
            cosine_rows_177(flat[:-1], flat[1:]).mean()
        ),
        "shuffled_mean_cosine": float(
            np.mean(shuffled_means)
        ),
        "shuffle_gap": float(
            scores.mean() - np.mean(shuffled_means)
        )
    }

print("=" * 65)
print("STEP 177: INDEPENDENT VAE VALIDATION")
print("=" * 65)

for label, metrics in results_177.items():
    print(f"\n{label}")
    for key, value in metrics.items():
        print(f"  {key}: {value:.6f}")

best_label_177 = max(
    results_177,
    key=lambda k: results_177[k]["mean_cosine"]
)

print("\nBEST CHECKPOINT:", best_label_177)
print("Constant baseline: 0.289701")
print("Deterministic baseline: 0.296295")
print("Ridge baseline: 0.323661")

print("\nTraining performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 178: SAVE VERIFIED VAE RESULTS
# No training or test-set access.

import os
import json
import shutil

folder_178 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

results_path_178 = os.path.join(
    folder_178, "vae_validation_results.json"
)

selected_checkpoint_178 = os.path.join(
    folder_178, "vae_selected_validation.weights.h5"
)

source_checkpoint_178 = os.path.join(
    folder_178, "vae_best_176b.weights.h5"
)

assert best_label_177 == "Epoch 2"
assert os.path.isfile(source_checkpoint_178)

with open(results_path_178, "w") as f:
    json.dump(
        {
            "model": "conditional_variational_encoder_decoder",
            "representation": "signed_scalar_glove",
            "training_cases": 16002,
            "validation_cases": 1779,
            "latent_dimensions": 20,
            "beta": 0.001,
            "inference": "posterior_mean",
            "selected_checkpoint": best_label_177,
            "checkpoint_results": results_177,
            "constant_baseline_cosine": 0.289701,
            "deterministic_baseline_cosine": 0.296295,
            "ridge_baseline_cosine": 0.323661,
            "test_evaluated": False
        },
        f,
        indent=2
    )

shutil.copy2(
    source_checkpoint_178,
    selected_checkpoint_178
)

print("=" * 65)
print("STEP 178: VAE RESULTS SAVED")
print("=" * 65)
print("Results:", results_path_178)
print("Selected checkpoint:", selected_checkpoint_178)
print("Selected validation cosine:",
      results_177["Epoch 2"]["mean_cosine"])
print("Test set accessed: NO")


In [ ]:

# STEP 179: PAIRED VALIDATION COMPARISON
# No training. No test-set access.

import numpy as np
import tensorflow as tf

def cosine_rows_179(a, b):
    a = np.asarray(a).reshape(len(a), -1)
    b = np.asarray(b).reshape(len(b), -1)
    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )
    return numerator / np.maximum(denominator, 1e-12)

# Reload selected VAE weights.
vae_175.load_weights(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "vae_selected_validation.weights.h5"
)

# Posterior-mean VAE predictions.
z_mean_179, _ = vae_175.encoder.predict(
    X_val_119, batch_size=64, verbose=0
)

vae_pred_179 = vae_175.decoder.predict(
    z_mean_179, batch_size=64, verbose=0
)

# Reload deterministic model.
det_179 = tf.keras.models.load_model(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "deterministic_best_170.keras",
    compile=False
)

det_pred_179 = det_179.predict(
    X_val_119, batch_size=64, verbose=0
)

# Paired per-investigation scores.
vae_scores_179 = cosine_rows_179(
    Y_val_119, vae_pred_179
)

det_scores_179 = cosine_rows_179(
    Y_val_119, det_pred_179
)

difference_179 = vae_scores_179 - det_scores_179

# Bootstrap confidence interval for mean paired difference.
rng_179 = np.random.default_rng(179)
n_179 = len(difference_179)

bootstrap_179 = np.array([
    difference_179[
        rng_179.integers(0, n_179, n_179)
    ].mean()
    for _ in range(2000)
])

ci_179 = np.quantile(
    bootstrap_179, [0.025, 0.975]
)

# Paired sign-flip permutation test.
observed_179 = abs(difference_179.mean())

permuted_179 = np.array([
    abs(
        np.mean(
            difference_179 *
            rng_179.choice([-1, 1], size=n_179)
        )
    )
    for _ in range(2000)
])

p_179 = (
    1 + np.sum(permuted_179 >= observed_179)
) / (len(permuted_179) + 1)

print("=" * 65)
print("STEP 179: PAIRED VALIDATION COMPARISON")
print("=" * 65)

print("Validation investigations:", n_179)
print("VAE mean cosine:",
      round(float(vae_scores_179.mean()), 6))
print("Deterministic mean cosine:",
      round(float(det_scores_179.mean()), 6))
print("Mean paired difference (VAE - deterministic):",
      round(float(difference_179.mean()), 6))
print("95% bootstrap CI:",
      np.round(ci_179, 6))
print("Paired permutation p-value:",
      round(float(p_179), 6))
print("VAE higher on individual investigations:",
      int(np.sum(difference_179 > 0)))
print("Deterministic higher:",
      int(np.sum(difference_179 < 0)))

print("\nTraining performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 180: SAVE PAIRED VALIDATION STATISTICS
# No training. No test-set access.

import os
import json
import numpy as np

folder_180 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

results_180 = {
    "comparison": "VAE versus deterministic encoder-decoder",
    "dataset": "validation",
    "validation_cases": int(len(difference_179)),
    "vae_mean_cosine": float(vae_scores_179.mean()),
    "deterministic_mean_cosine": float(det_scores_179.mean()),
    "mean_paired_difference": float(difference_179.mean()),
    "bootstrap_95_ci": [
        float(ci_179[0]),
        float(ci_179[1])
    ],
    "paired_permutation_p_value": float(p_179),
    "vae_wins": int(np.sum(difference_179 > 0)),
    "deterministic_wins": int(np.sum(difference_179 < 0)),
    "bootstrap_resamples": 2000,
    "permutation_resamples": 2000,
    "validation_used_for_checkpoint_selection": True,
    "test_set_accessed": False
}

path_180 = os.path.join(
    folder_180,
    "vae_deterministic_paired_validation.json"
)

with open(path_180, "w") as f:
    json.dump(results_180, f, indent=2)

print("=" * 65)
print("STEP 180: STATISTICAL RESULTS SAVED")
print("=" * 65)
print("Saved file:", path_180)
print("Mean difference:",
      round(results_180["mean_paired_difference"], 6))
print("95% CI:", results_180["bootstrap_95_ci"])
print("Permutation p-value:",
      round(results_180["paired_permutation_p_value"], 6))
print("Test set accessed: NO")


In [ ]:

# STEP 181: VAE ERROR AND DIVERSITY ANALYSIS
# Validation only. No training. No test-set access.

import numpy as np
import json
import os

folder_181 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

# Uses predictions and per-case scores from Step 179.
assert len(vae_scores_179) == 1779
assert len(det_scores_179) == 1779

vae_flat_181 = vae_pred_179.reshape(1779, -1)
det_flat_181 = det_pred_179.reshape(1779, -1)

# Validation score distribution.
quantiles_181 = [0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0]

vae_quantiles_181 = np.quantile(
    vae_scores_179, quantiles_181
)

det_quantiles_181 = np.quantile(
    det_scores_179, quantiles_181
)

# Prediction diversity.
vae_position_sd_181 = np.std(
    vae_flat_181, axis=0
)

det_position_sd_181 = np.std(
    det_flat_181, axis=0
)

# Relative prediction magnitude.
vae_norms_181 = np.linalg.norm(
    vae_flat_181, axis=1
)

det_norms_181 = np.linalg.norm(
    det_flat_181, axis=1
)

# Posterior distribution diagnostics.
z_mean_181, z_logvar_181 = vae_175.encoder.predict(
    X_val_119, batch_size=64, verbose=0
)

safe_logvar_181 = np.clip(z_logvar_181, -20, 20)

kl_cases_181 = -0.5 * np.sum(
    1 + safe_logvar_181
    - z_mean_181**2
    - np.exp(safe_logvar_181),
    axis=1
)

# Paired performance differences.
paired_181 = vae_scores_179 - det_scores_179

summary_181 = {
    "validation_cases": 1779,
    "vae_score_quantiles": vae_quantiles_181.tolist(),
    "deterministic_score_quantiles": det_quantiles_181.tolist(),
    "vae_mean_position_sd": float(
        vae_position_sd_181.mean()
    ),
    "deterministic_mean_position_sd": float(
        det_position_sd_181.mean()
    ),
    "vae_prediction_norm_mean": float(
        vae_norms_181.mean()
    ),
    "deterministic_prediction_norm_mean": float(
        det_norms_181.mean()
    ),
    "vae_posterior_kl_mean": float(
        kl_cases_181.mean()
    ),
    "vae_posterior_kl_median": float(
        np.median(kl_cases_181)
    ),
    "vae_better_by_more_than_0_05": int(
        np.sum(paired_181 > 0.05)
    ),
    "deterministic_better_by_more_than_0_05": int(
        np.sum(paired_181 < -0.05)
    ),
    "vae_negative_cosine_cases": int(
        np.sum(vae_scores_179 < 0)
    ),
    "vae_cosine_above_0_5_cases": int(
        np.sum(vae_scores_179 > 0.5)
    ),
    "test_set_accessed": False
}

path_181 = os.path.join(
    folder_181,
    "vae_error_diversity_validation.json"
)

with open(path_181, "w") as f:
    json.dump(summary_181, f, indent=2)

print("=" * 65)
print("STEP 181: VAE ERROR AND DIVERSITY ANALYSIS")
print("=" * 65)

for key, value in summary_181.items():
    print(f"{key}: {value}")

print("\nSaved:", path_181)
print("Training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 182: VAE INPUT-DEPENDENCE CHECKS
# Validation only. No training or test-set access.

import numpy as np

assert len(vae_scores_179) == len(Y_val_119) == 1779

# Constant predictor: training-set mean target vector.
# Use training targets only to construct the baseline.
constant_vector_182 = np.mean(
    Y_train_119.reshape(len(Y_train_119), -1),
    axis=0
)

constant_pred_182 = np.broadcast_to(
    constant_vector_182,
    (len(Y_val_119), len(constant_vector_182))
)

constant_scores_182 = cosine_rows_179(
    Y_val_119, constant_pred_182
)

# Shuffle already-computed VAE predictions relative to targets.
# This preserves the prediction distribution while breaking
# investigation-to-target alignment.
rng_182 = np.random.default_rng(182)
shuffle_means_182 = []

for _ in range(1000):
    indices = rng_182.permutation(len(vae_pred_179))
    shuffled_scores = cosine_rows_179(
        Y_val_119,
        vae_pred_179[indices]
    )
    shuffle_means_182.append(shuffled_scores.mean())

shuffle_means_182 = np.asarray(shuffle_means_182)

observed_182 = float(vae_scores_179.mean())
constant_mean_182 = float(constant_scores_182.mean())
shuffle_mean_182 = float(shuffle_means_182.mean())

# Empirical randomization tail probability.
shuffle_p_182 = (
    1 + np.sum(shuffle_means_182 >= observed_182)
) / (len(shuffle_means_182) + 1)

print("=" * 65)
print("STEP 182: VAE INPUT-DEPENDENCE CHECKS")
print("=" * 65)

print("Validation investigations:", len(Y_val_119))
print("VAE cosine:", round(observed_182, 6))
print("Constant baseline cosine:",
      round(constant_mean_182, 6))
print("Shuffled prediction cosine:",
      round(shuffle_mean_182, 6))
print("VAE minus constant:",
      round(observed_182 - constant_mean_182, 6))
print("VAE minus shuffled:",
      round(observed_182 - shuffle_mean_182, 6))
print("Shuffle randomization p-value:",
      round(float(shuffle_p_182), 6))

print("\nTraining performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 183: SAVE FIGURE DATA BEFORE CLOSING COLAB
# No training. No test-set access.

import os
import numpy as np

folder_183 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

path_183 = os.path.join(
    folder_183,
    "validation_figure_data.npz"
)

np.savez_compressed(
    path_183,
    vae_cosine=vae_scores_179.astype(np.float32),
    deterministic_cosine=det_scores_179.astype(np.float32),
    paired_difference=difference_179.astype(np.float32),
    vae_predictions=vae_pred_179.astype(np.float32),
    deterministic_predictions=det_pred_179.astype(np.float32)
)

with np.load(path_183) as saved:
    print("Saved arrays:", saved.files)
    print("Validation cases:", len(saved["vae_cosine"]))

print("File:", path_183)
print("File exists:", os.path.isfile(path_183))
print("Training performed: NO")
print("Test set accessed: NO")


In [ ]:

# STEP 184: CHECK TEST DATA AVAILABILITY
# No training. No model evaluation.

import os
import json
import numpy as np

folder_184 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

print("=" * 65)
print("STEP 184: TEST DATA READINESS")
print("=" * 65)

print("\nSaved experiment files:")
for filename in sorted(os.listdir(folder_184)):
    if filename.lower().endswith(
        (".npz", ".csv", ".json", ".pkl", ".pickle", ".parquet")
    ):
        print(" ", filename)

print("\nNPZ array contents:")
for filename in sorted(os.listdir(folder_184)):
    if filename.endswith(".npz"):
        path = os.path.join(folder_184, filename)
        with np.load(path, allow_pickle=False) as data:
            print("\n", filename)
            for key in data.files:
                print("  ", key, data[key].shape)

manifest_path_184 = os.path.join(
    folder_184, "experiment_manifest.json"
)

if os.path.isfile(manifest_path_184):
    with open(manifest_path_184) as f:
        manifest_184 = json.load(f)

    print("\nExperiment manifest:")
    print(json.dumps(manifest_184, indent=2)[:6000])

print("\nTraining performed: NO")
print("Test predictions generated: NO")


In [ ]:

# STEP 185: FIND DATASET AND PREPROCESSING VARIABLES
# Inspection only. No training or testing.

import pandas as pd
import numpy as np

print("=" * 65)
print("STEP 185: DATASET AND PREPROCESSING CHECK")
print("=" * 65)

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    try:
        if isinstance(value, pd.DataFrame):
            print(
                f"DATAFRAME: {name}",
                "shape:", value.shape,
                "columns:", list(value.columns[:12])
            )

        elif isinstance(value, dict):
            if any(
                term in name.lower()
                for term in [
                    "glove", "vocab", "embedding",
                    "token", "split", "lookup"
                ]
            ):
                print(
                    f"DICTIONARY: {name}",
                    "entries:", len(value)
                )

        elif isinstance(value, np.ndarray):
            if any(
                term in name.lower()
                for term in [
                    "test", "glove", "embedding"
                ]
            ):
                print(
                    f"ARRAY: {name}",
                    "shape:", value.shape
                )

    except Exception:
        pass

print("\nInspection completed.")
print("Training performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 186: LOCATE SOURCE DATA AND PREPROCESSING FILES
# No training. No test evaluation.

import os

roots_186 = [
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility",
    "/content/drive/MyDrive"
]

extensions_186 = (
    ".csv", ".parquet", ".pkl", ".pickle",
    ".jsonl", ".json", ".ipynb",
    ".npy", ".npz", ".zip"
)

keywords_186 = (
    "ntsb", "glove", "narrative",
    "probable", "cause", "investigation",
    "vae", "preprocess", "dataset"
)

matches_186 = []
scanned_186 = 0

# Search MyDrive once, with a cap to keep it quick.
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
    ]

    for filename in files:
        scanned_186 += 1

        if (
            filename.lower().endswith(extensions_186)
            and any(
                word in filename.lower()
                or word in root.lower()
                for word in keywords_186
            )
        ):
            path = os.path.join(root, filename)

            if path not in matches_186:
                matches_186.append(path)

    if scanned_186 >= 20000:
        break

print("=" * 65)
print("STEP 186: SOURCE FILE SEARCH")
print("=" * 65)
print("Files scanned:", scanned_186)
print("Potential matches:", len(matches_186))

for path in matches_186[:60]:
    print(path)

if len(matches_186) > 60:
    print("Additional matches not displayed:",
          len(matches_186) - 60)

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 187: FIND FILES RELEVANT TO THE VAE EXPERIMENT
# No training. No test evaluation.

import os

print("=" * 65)
print("STEP 187: VAE SOURCE FILE IDENTIFICATION")
print("=" * 65)

priority_terms_187 = (
    "20919", "23732", "glove", "vae",
    "probable_cause", "probablecause",
    "investigation", "preprocess",
    "embedding", "train_test"
)

relevant_187 = [
    path for path in matches_186
    if any(
        term in os.path.basename(path).lower()
        for term in priority_terms_187
    )
]

print("Relevant filename matches:", len(relevant_187))

for path in relevant_187:
    print(path)

print("\nOTHER MATCHES NOT PREVIOUSLY DISPLAYED:")
for path in matches_186[60:]:
    print(path)

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 188: INSPECT REVIEWER 2 SOURCE FOLDER
# No training. No test evaluation.

import os
import pandas as pd

folder_188 = "/content/drive/MyDrive/Aviation_Reviewer2_Revision"

print("=" * 65)
print("STEP 188: REVIEWER 2 SOURCE FILES")
print("=" * 65)

if os.path.isdir(folder_188):
    for root, dirs, files in os.walk(folder_188):
        for filename in sorted(files):
            path = os.path.join(root, filename)
            size_mb = os.path.getsize(path) / (1024 ** 2)
            print(f"{size_mb:9.2f} MB  {path}")
else:
    print("Folder not found.")

print("\nTEST ID FILE:")
test_ids_path_188 = (
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility/"
    "test_investigation_ids.csv"
)

test_ids_188 = pd.read_csv(test_ids_path_188)

print("Shape:", test_ids_188.shape)
print("Columns:", list(test_ids_188.columns))
print(test_ids_188.head(3).to_string(index=False))

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 189: LOCATE LARGE SOURCE DATA FILES
# No training. No test evaluation.

import os

root_189 = "/content/drive/MyDrive"

extensions_189 = (
    ".csv", ".parquet", ".pkl", ".pickle",
    ".feather", ".jsonl", ".xlsx",
    ".accdb", ".mdb", ".zip"
)

candidates_189 = []

for root, dirs, files in os.walk(root_189):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
    ]

    for filename in files:
        if filename.lower().endswith(extensions_189):
            path = os.path.join(root, filename)

            try:
                size_mb = os.path.getsize(path) / (1024 ** 2)

                if size_mb >= 5:
                    candidates_189.append(
                        (size_mb, path)
                    )
            except OSError:
                pass

candidates_189.sort(reverse=True)

print("=" * 65)
print("STEP 189: LARGE DATASET FILE SEARCH")
print("=" * 65)

print("Files >= 5 MB:", len(candidates_189))

for size_mb, path in candidates_189[:80]:
    print(f"{size_mb:9.2f} MB  {path}")

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 190: VERIFY ORIGINAL NTSB SOURCE DATA
# No training. No model testing.

import os
import pandas as pd

candidates_190 = [
    "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv",
    "/content/drive/MyDrive/ntsb_analytical_corpus_7207.csv",
    "/content/drive/MyDrive/AeroSafe_Research/NTSB_narratives.csv"
]

test_ids_set_190 = set(
    test_ids_188["investigation_id"]
    .astype(str)
    .str.strip()
)

print("=" * 65)
print("STEP 190: SOURCE DATA VERIFICATION")
print("=" * 65)

for path in candidates_190:

    print("\nFILE:", path)

    if not os.path.isfile(path):
        print("Not found")
        continue

    try:
        df_190 = pd.read_csv(path, low_memory=False)

        print("Rows:", len(df_190))
        print("Columns:", list(df_190.columns))

        id_columns = [
            col for col in df_190.columns
            if any(
                term in col.lower()
                for term in ["investigation", "event_id", "ntsb"]
            )
        ]

        for col in id_columns[:5]:
            source_ids = set(
                df_190[col]
                .dropna()
                .astype(str)
                .str.strip()
            )

            matched = len(
                test_ids_set_190.intersection(source_ids)
            )

            print(
                f"ID column '{col}':",
                f"{matched}/3138 test IDs matched"
            )

    except Exception as error:
        print("Read error:", str(error)[:300])

print("\nTraining performed: NO")
print("Test predictions generated: NO")


In [ ]:

# STEP 192: FIND ORIGINAL DATASET IN COLAB
# No training. No test evaluation.

import os

extensions_192 = (
    ".csv", ".parquet", ".pkl", ".pickle",
    ".jsonl", ".json", ".npz", ".npy",
    ".feather", ".zip"
)

matches_192 = []

for root, dirs, files in os.walk("/content"):
    # Exclude Google Drive because we already searched it.
    dirs[:] = [
        d for d in dirs
        if d not in ["drive", ".config", "sample_data"]
        and not d.startswith(".")
    ]

    for filename in files:
        if filename.lower().endswith(extensions_192):
            path = os.path.join(root, filename)

            try:
                size_mb = os.path.getsize(path) / (1024 ** 2)
                matches_192.append((size_mb, path))
            except OSError:
                pass

matches_192.sort(reverse=True)

print("=" * 65)
print("STEP 192: COLAB LOCAL DATASET SEARCH")
print("=" * 65)

print("Files found:", len(matches_192))

for size_mb, path in matches_192[:80]:
    print(f"{size_mb:9.2f} MB  {path}")

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 191: INSPECT SOURCE FILE FORMATS
# No training. No test evaluation.

import os
import csv

paths_191 = [
    "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv",
    "/content/drive/MyDrive/AeroSafe_Research/NTSB_narratives.csv"
]

print("=" * 65)
print("STEP 191: SOURCE FILE FORMAT INSPECTION")
print("=" * 65)

for path in paths_191:
    print("\nFILE:", path)

    if not os.path.exists(path):
        print("File missing")
        continue

    with open(path, "rb") as f:
        sample = f.read(4096)

    print("First 100 raw bytes:", repr(sample[:100]))

    for encoding in ["utf-8-sig", "cp1252", "latin-1"]:
        try:
            decoded = sample.decode(encoding)
            print("Encoding works:", encoding)

            print("First 350 characters:")
            print(repr(decoded[:350]))
            break

        except UnicodeDecodeError:
            continue

    print("-" * 50)

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:
# STEP 193: VERIFY SAVED RESULTS AND RELEASE GPU
# No training or testing.

import os
import gc
import tensorflow as tf

folder = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

required_files = [
    "train_validation_signed_glove.npz",
    "train_investigation_ids.csv",
    "validation_investigation_ids.csv",
    "test_investigation_ids.csv",
    "experiment_manifest.json",
    "deterministic_best_170.keras",
    "vae_selected_validation.weights.h5",
    "vae_validation_results.json",
    "vae_deterministic_paired_validation.json",
    "vae_error_diversity_validation.json",
    "validation_figure_data.npz"
]

print("=" * 60)
print("FINAL BACKUP VERIFICATION")
print("=" * 60)

missing = []

for filename in required_files:
    path = os.path.join(folder, filename)
    exists = os.path.isfile(path)
    print(("SAVED" if exists else "MISSING"), filename)

    if not exists:
        missing.append(filename)

if missing:
    print("\nWARNING: Some files are missing.")
    print("Do not disconnect until these are checked.")
else:
    print("\nALL ESSENTIAL FILES VERIFIED.")

    # Release TensorFlow resources.
    tf.keras.backend.clear_session()
    gc.collect()

    print("TensorFlow session cleared.")
    print("Safe to disconnect Colab runtime.")

In [ ]:

# STEP 194A: DIAGNOSE SOURCE FILE STRUCTURE
# Read-only inspection. No training or testing.

import os
import csv
from collections import Counter

source_194a = "/content/drive/MyDrive/ntsb_dataset_2000_2020.csv"

print("=" * 65)
print("STEP 194A: SOURCE FILE DIAGNOSTICS")
print("=" * 65)

print("File exists:", os.path.isfile(source_194a))
print("File size (MB):",
      round(os.path.getsize(source_194a) / 1024**2, 2))

with open(source_194a, "r",
          encoding="utf-8-sig", errors="replace",
          newline="") as f:

    for i in range(5):
        line = f.readline()
        print(
            f"Physical line {i+1}:",
            "characters =", len(line),
            "tabs =", line.count("\t"),
            "preview =", repr(line[:160])
        )

print("\nCSV reader inspection:")

counts_194a = Counter()
records_194a = 0
examples_194a = []

try:
    with open(source_194a, "r",
              encoding="utf-8-sig",
              errors="replace",
              newline="") as f:

        reader = csv.reader(
            f,
            delimiter="\t",
            quotechar='"'
        )

        header = next(reader)
        print("Header:", header)
        expected = len(header)

        for row in reader:
            records_194a += 1
            counts_194a[len(row)] += 1

            if len(row) != expected and len(examples_194a) < 5:
                examples_194a.append({
                    "record": records_194a,
                    "fields": len(row),
                    "preview": repr(row[:2])[:180]
                })

    print("Parsed records:", records_194a)
    print("Field-count distribution:", counts_194a)
    print("Malformed examples:", examples_194a)

except Exception as e:
    print("CSV reader error:", type(e).__name__, str(e)[:300])

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 195: INSPECT SAVED EXPERIMENT MANIFEST
# Read-only. No training or testing.

import json
import os

folder_195 = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
manifest_path_195 = os.path.join(
    folder_195, "experiment_manifest.json"
)

print("=" * 65)
print("STEP 195: EXPERIMENT MANIFEST")
print("=" * 65)

with open(manifest_path_195, "r", encoding="utf-8") as f:
    manifest_195 = json.load(f)

print(json.dumps(manifest_195, indent=2, ensure_ascii=False))

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 196: FIND POSSIBLE ORIGINAL DATASET FILES
# CPU only. No training or testing.

import os

root_196 = "/content/drive/MyDrive"

extensions_196 = (
    ".csv", ".tsv", ".parquet", ".pkl", ".pickle",
    ".json", ".jsonl", ".xlsx", ".feather",
    ".db", ".sqlite", ".zip", ".gz"
)

matches_196 = []

for root, dirs, files in os.walk(root_196):
    for filename in files:
        if filename.lower().endswith(extensions_196):
            path = os.path.join(root, filename)

            try:
                size_mb = os.path.getsize(path) / (1024 ** 2)
                matches_196.append((size_mb, path))
            except OSError:
                pass

matches_196.sort(reverse=True)

print("=" * 65)
print("STEP 196: POSSIBLE SOURCE DATASETS")
print("=" * 65)

print("Total candidate files:", len(matches_196))

for size_mb, path in matches_196[:100]:
    print(f"{size_mb:9.2f} MB | {path}")

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

# STEP 198: UPLOAD ORIGINAL CAROL JSON FILES
# CPU only. No training or testing.

from google.colab import files
import os
import json

print("Select all FIVE carol_investigations_data JSON files.")

uploaded_198 = files.upload()

print("\n" + "=" * 65)
print("STEP 198: ORIGINAL CAROL DATA INSPECTION")
print("=" * 65)

for filename in uploaded_198:
    path = os.path.join("/content", filename)

    print("\nFILE:", filename)
    print("Size (MB):", round(os.path.getsize(path) / 1024**2, 2))

    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        print("Top-level type:", type(data).__name__)

        if isinstance(data, list):
            print("Records:", len(data))
            if data and isinstance(data[0], dict):
                print("First record keys:", list(data[0].keys())[:40])

        elif isinstance(data, dict):
            print("Top-level keys:", list(data.keys())[:30])

            for key, value in data.items():
                if isinstance(value, list):
                    print("List field:", key, "records:", len(value))
                    if value and isinstance(value[0], dict):
                        print(
                            "First record keys:",
                            list(value[0].keys())[:40]
                        )

    except Exception as e:
        print("Inspection error:", type(e).__name__, str(e)[:200])

print("\nTraining performed: NO")
print("Test evaluation performed: NO")


In [ ]:

from pathlib import Path
import shutil

source = Path("/content")
destination = Path(
    "/content/drive/MyDrive/"
    "NTSB_VAE_Reproducibility/CAROL_Original_JSON"
)

destination.mkdir(parents=True, exist_ok=True)

files = sorted(source.glob("carol_investigations_data*.json"))

print("Files found:", len(files))

for file in files:
    target = destination / file.name

    if target.exists() and target.stat().st_size == file.stat().st_size:
        print("Already saved:", file.name)
    else:
        shutil.copy2(file, target)
        print("Saved:", file.name)

print("\nVerification:")
for file in sorted(destination.glob("carol_investigations_data*.json")):
    print(file.name, f"{file.stat().st_size / 1024**2:.2f} MB")

print("\nTotal saved files:",
      len(list(destination.glob("carol_investigations_data*.json"))))


In [ ]:

import json
import pandas as pd
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")
data_dir = root / "CAROL_Original_JSON"

# Inspect saved partition files
for split in ["train", "validation", "test"]:
    path = root / f"{split}_investigation_ids.csv"

    if not path.exists():
        print(f"\n{split.upper()}: FILE NOT FOUND")
        continue

    df = pd.read_csv(path, dtype=str)

    print(f"\n{split.upper()} PARTITION")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("First 3 rows:")
    print(df.head(3).to_string(index=False))

# Inspect a small sample from the original CAROL data
files = sorted(data_dir.glob("carol_investigations_data*.json"))

print("\nCAROL SAMPLE RECORDS")

for path in files:
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print(f"\n{path.name}: {len(records)} records")

    for record in records[:2]:
        print({
            "MKey": record.get("MKey"),
            "NTSBNumber": record.get("NTSBNumber"),
            "Mode": record.get("Mode"),
            "EventDateTimeUTC": record.get("EventDateTimeUTC"),
            "AnalysisNarrative_present":
                bool(record.get("AnalysisNarrative")),
            "ProbableCause_present":
                bool(record.get("ProbableCause"))
        })

    del records

print("\nStep 200 complete.")


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive")

keywords = [
    "investigation_ids",
    "train_validation",
    "manifest",
    "split"
]

print("=" * 65)
print("STEP 201: FIND SAVED EXPERIMENT FILES")
print("=" * 65)

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if any(keyword in name for keyword in keywords):
        matches.append(path)

for path in sorted(matches):
    print(path)

print("\nTotal matching files:", len(matches))
print("\nStep 201 complete.")


In [ ]:

from pathlib import Path

print("=" * 65)
print("STEP 202: EXPERIMENT FILE RECOVERY")
print("=" * 65)

locations = [
    Path("/content"),
    Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility"),
    Path("/content/drive/MyDrive")
]

extensions = {".keras", ".h5", ".npz", ".csv", ".ipynb"}

for location in locations:
    print(f"\nLOCATION: {location}")

    if not location.exists():
        print("Not found")
        continue

    matches = [
        p for p in location.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    ]

    print("Matching files:", len(matches))

    for p in sorted(matches)[:60]:
        print(" ", p)

    if len(matches) > 60:
        print(f" ... and {len(matches) - 60} more")

print("\nStep 202 complete.")


In [ ]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
project_root = drive_root / "NTSB_VAE_Reproducibility"

print("=" * 60)
print("STEP 203: GOOGLE DRIVE VERIFICATION")
print("=" * 60)

print("\nGoogle Drive mounted:", drive_root.exists())

print("\nProject folder exists:", project_root.exists())

if drive_root.exists():
    print("\nTop-level Google Drive items:")
    for p in sorted(drive_root.iterdir()):
        print(" ", p.name, "[DIR]" if p.is_dir() else "[FILE]")

if project_root.exists():
    print("\nProject folder contents:")
    for p in sorted(project_root.iterdir()):
        print(" ", p.name, "[DIR]" if p.is_dir() else "[FILE]")

print("\nStep 203 complete.")


In [ ]:

import os

print("=" * 65)
print("STEP 205: ORIGINAL RUNTIME RECOVERY")
print("=" * 65)

names = [
    "vae_175",
    "X_train_119",
    "Y_train_119",
    "X_val_119",
    "Y_val_119",
    "results_177",
    "best_label_177",
    "history_176b"
]

for name in names:
    if name in globals():
        obj = globals()[name]
        shape = getattr(obj, "shape", None)

        print(
            f"{name}: AVAILABLE",
            f"| shape: {shape}" if shape is not None else ""
        )
    else:
        print(f"{name}: NOT AVAILABLE")

folder = "/content/drive/MyDrive/NTSB_VAE_Reproducibility"

print("\nCheckpoint verification:")

for filename in [
    "vae_best_176b.weights.h5",
    "vae_selected_validation.weights.h5",
    "vae_validation_results.json",
    "train_validation_signed_glove.npz"
]:
    path = os.path.join(folder, filename)
    print(filename, "EXISTS" if os.path.isfile(path) else "MISSING")

print("\nStep 205 complete.")


In [ ]:

from google.colab import drive
from pathlib import Path

# Use a new, empty mount location
mount_point = "/content/drive_recovered"

drive.mount(mount_point)

project = Path(mount_point) / "MyDrive" / "NTSB_VAE_Reproducibility"

print("\n" + "=" * 65)
print("STEP 208: RECOVER ORIGINAL EXPERIMENT FILES")
print("=" * 65)

print("Project folder exists:", project.exists())

if project.exists():
    for item in sorted(project.iterdir()):
        if item.is_file():
            print(
                item.name,
                f"| {item.stat().st_size / 1024**2:.2f} MB"
            )
        else:
            print(item.name, "| FOLDER")

print("\nStep 208 complete.")


In [ ]:

from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/"
    "NTSB_VAE_Reproducibility/CAROL_Original_JSON"
)

destination = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility/CAROL_Original_JSON"
)

print("=" * 65)
print("STEP 209: SAVE CAROL DATA TO RECOVERED GOOGLE DRIVE")
print("=" * 65)

files = sorted(source.glob("carol_investigations_data*.json"))

print("Source files found:", len(files))

assert len(files) == 5, "Expected five original CAROL JSON files."

destination.mkdir(parents=True, exist_ok=True)

for file in files:
    target = destination / file.name

    if target.exists() and target.stat().st_size == file.stat().st_size:
        print("Already saved:", file.name)
    else:
        shutil.copy2(file, target)
        print("Copied:", file.name)

print("\nVerification:")

saved = sorted(destination.glob("carol_investigations_data*.json"))

for file in saved:
    print(file.name, f"{file.stat().st_size / 1024**2:.2f} MB")

assert len(saved) == 5, "Not all five CAROL files were saved."

for original in files:
    copied = destination / original.name
    assert copied.stat().st_size == original.stat().st_size

print("\nSUCCESS: All five CAROL files saved to recovered Drive.")
print("Step 209 complete.")


In [ ]:

import json
import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 210: VERIFY ORIGINAL DATASET PARTITIONS")
print("=" * 65)

# Read original saved partition files
splits = {}

for name in ["train", "validation", "test"]:
    path = root / f"{name}_investigation_ids.csv"

    assert path.exists(), f"Missing partition file: {path}"

    df = pd.read_csv(path, dtype=str)

    splits[name] = df

    print(f"\n{name.upper()}")
    print("Records:", len(df))
    print("Columns:", df.columns.tolist())
    print("Sample:")
    print(df.head(3).to_string(index=False))

# Read experiment manifest
manifest_path = root / "experiment_manifest.json"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("\nEXPERIMENT MANIFEST")
print(json.dumps(manifest, indent=2)[:4000])

# Inspect original CAROL record identifiers
data_dir = root / "CAROL_Original_JSON"

carol_ids = set()
total_records = 0

for path in sorted(data_dir.glob("carol_investigations_data*.json")):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    total_records += len(records)

    for record in records:
        if record.get("NTSBNumber") is not None:
            carol_ids.add(str(record["NTSBNumber"]).strip())

    del records

print("\nCAROL SOURCE")
print("Raw records:", total_records)
print("Unique NTSB numbers:", len(carol_ids))

print("\nStep 210 complete.")


In [ ]:

import json
import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 211: RECORD MATCHING AND PARTITION INTEGRITY")
print("=" * 65)

# Load saved investigation IDs
split_ids = {}

for split in ["train", "validation", "test"]:
    df = pd.read_csv(
        root / f"{split}_investigation_ids.csv",
        dtype=str
    )

    ids = df["investigation_id"].str.strip()

    split_ids[split] = set(ids)

    print(
        f"{split.upper()}: "
        f"{len(ids)} rows, "
        f"{len(split_ids[split])} unique IDs"
    )

# Check overlap between partitions
print("\nPARTITION OVERLAP")

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = split_ids[a] & split_ids[b]
    print(f"{a} / {b}: {len(overlap)}")

# Read CAROL records and index by investigation ID
carol = {}

for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        investigation_id = str(
            record.get("NTSBNumber", "")
        ).strip()

        if investigation_id:
            carol[investigation_id] = record

    del records

print("\nMATCHING RESULTS")

for split, ids in split_ids.items():
    matched = ids & carol.keys()
    missing = ids - carol.keys()

    complete = sum(
        bool(carol[i].get("AnalysisNarrative"))
        and bool(carol[i].get("ProbableCause"))
        for i in matched
    )

    print(f"\n{split.upper()}")
    print("Saved IDs:", len(ids))
    print("Matched CAROL records:", len(matched))
    print("Missing IDs:", len(missing))
    print("Records with both text fields:", complete)

    if missing:
        print("First missing IDs:", sorted(missing)[:10])

print("\nStep 211 complete.")


In [ ]:

import numpy as np
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

npz_path = root / "train_validation_signed_glove.npz"

print("=" * 65)
print("STEP 212: SAVED NUMERICAL REPRESENTATION")
print("=" * 65)

with np.load(npz_path, allow_pickle=False) as data:
    print("\nAvailable arrays:", data.files)

    for name in data.files:
        arr = data[name]

        print(f"\n{name}")
        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Minimum:", float(np.min(arr)))
        print("Maximum:", float(np.max(arr)))
        print("Mean:", float(np.mean(arr)))
        print("Nonzero proportion:",
              float(np.count_nonzero(arr) / arr.size))

        print("First record, first 15 values:",
              arr[0].reshape(-1)[:15])

print("\nStep 212 complete.")


In [ ]:

from pathlib import Path

print("=" * 65)
print("STEP 217: LOCATE GLOVE EMBEDDINGS")
print("=" * 65)

locations = [
    Path("/content"),
    Path("/content/drive_recovered/MyDrive"),
    Path("/content/drive/MyDrive"),
]

found = []

for location in locations:
    if not location.exists():
        continue

    print("\nSearching:", location)

    for path in location.rglob("glove.6B.100d.txt"):
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"FOUND: {path}")
        print(f"Size: {size_mb:.2f} MB")
        found.append(str(path))

print("\nRESULT")
if found:
    print("GloVe files found:", len(found))
else:
    print("GloVe file not found in searched locations.")

print("\nStep 217 complete.")


In [ ]:

import subprocess
import zipfile
import shutil
from pathlib import Path

print("=" * 65)
print("STEP 218: RESTORE GLOVE 6B 100D")
print("=" * 65)

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

destination = root / "glove.6B.100d.txt"
archive = Path("/content/glove.6B.zip")
extracted = Path("/content/glove.6B.100d.txt")

if destination.exists() and destination.stat().st_size > 100_000_000:
    print("GloVe already available in recovered Drive.")

else:
    urls = [
        "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip",
        "https://nlp.stanford.edu/data/glove.6B.zip"
    ]

    downloaded = False

    for url in urls:
        print("\nTrying:", url)

        result = subprocess.run(
            [
                "curl", "-L", "--fail",
                "--retry", "2",
                "--connect-timeout", "30",
                "-o", str(archive), url
            ],
            capture_output=True,
            text=True
        )

        if result.returncode == 0 and zipfile.is_zipfile(archive):
            downloaded = True
            print("Download successful.")
            break

        print("Download failed:", result.stderr[-300:])

    if not downloaded:
        print("\nDownload unavailable. No experiment files changed.")

    else:
        with zipfile.ZipFile(archive) as z:
            z.extract("glove.6B.100d.txt", "/content")

        shutil.copy2(extracted, destination)

        print("\nGloVe saved to recovered Google Drive.")

        archive.unlink(missing_ok=True)

print("\nVERIFICATION")

if destination.exists():
    print("Path:", destination)
    print(
        "Size:",
        round(destination.stat().st_size / 1024**2, 2),
        "MB"
    )
else:
    print("GloVe restoration incomplete.")

print("\nStep 218 complete.")


In [ ]:

import numpy as np
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 219: VERIFY GLOVE SCALAR VALUES")
print("=" * 65)

glove_path = root / "glove.6B.100d.txt"

assert glove_path.exists(), "GloVe file missing."

words = {
    "aircraft", "pilot", "engine",
    "weather", "runway", "failure",
    "maintenance", "collision"
}

mean_scalars = {}

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        word, *values = line.split()

        if word in words:
            vector = np.asarray(values, dtype=np.float32)
            mean_scalars[word] = float(np.mean(vector))

print("\nGLOVE MEAN PROJECTIONS")

for word, value in sorted(mean_scalars.items()):
    print(f"{word:15s} {value:+.8f}")

with np.load(
    root / "train_validation_signed_glove.npz"
) as data:
    x_train = data["X_train"][0].reshape(-1)
    x_val = data["X_val"][0].reshape(-1)

print("\nMATCHES IN FIRST TRAINING RECORD")

for word, value in sorted(mean_scalars.items()):
    matches = np.where(
        np.isclose(x_train, value, atol=1e-7, rtol=0)
    )[0]
    print(f"{word:15s} {len(matches)}")

print("\nMATCHES IN FIRST VALIDATION RECORD")

for word, value in sorted(mean_scalars.items()):
    matches = np.where(
        np.isclose(x_val, value, atol=1e-7, rtol=0)
    )[0]
    print(f"{word:15s} {len(matches)}")

print("\nStep 219 complete.")


In [ ]:

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 220: INSPECT ORIGINAL TEXT AND SAVED SEQUENCE")
print("=" * 65)

train_ids = pd.read_csv(
    root / "train_investigation_ids.csv",
    dtype=str
)

first_id = train_ids.iloc[0]["investigation_id"]

print("\nFirst training investigation:", first_id)

record = None

for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for item in records:
        if str(item.get("NTSBNumber", "")).strip() == first_id:
            record = item
            break

    del records

    if record is not None:
        break

assert record is not None, "Investigation not found."

narrative = str(record.get("AnalysisNarrative", ""))
cause = str(record.get("ProbableCause", ""))

print("\nORIGINAL ANALYSIS NARRATIVE")
print(narrative[:800])

print("\nORIGINAL PROBABLE CAUSE")
print(cause[:400])

tokens = re.findall(
    r"[a-z]+(?:'[a-z]+)?",
    narrative.lower()
)

print("\nFIRST 25 REGEX TOKENS")
print(tokens[:25])

with np.load(
    root / "train_validation_signed_glove.npz"
) as data:
    x = data["X_train"][0].reshape(-1)
    y = data["Y_train"][0].reshape(-1)

print("\nFIRST 25 SAVED INPUT VALUES")
print(x[:25])

print("\nFIRST 25 SAVED TARGET VALUES")
print(y[:25])

print("\nStep 220 complete.")


In [ ]:

import re
import numpy as np
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 221: REPRODUCE SAVED INPUT AND TARGET")
print("=" * 65)

def tokenize(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

input_tokens = tokenize(narrative)
target_tokens = tokenize(cause)

required_words = set(input_tokens + target_tokens)

scalar_lookup = {}

with open(root / "glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        word, *values = line.split()

        if word in required_words:
            vector = np.asarray(values, dtype=np.float32)
            scalar_lookup[word] = np.mean(vector)

def encode(tokens, max_length):
    result = np.zeros(max_length, dtype=np.float32)

    for i, word in enumerate(tokens[:max_length]):
        result[i] = scalar_lookup.get(word, 0.0)

    return result

x_rebuilt = encode(input_tokens, 1200)
y_rebuilt = encode(target_tokens, 200)

with np.load(
    root / "train_validation_signed_glove.npz"
) as data:
    x_saved = data["X_train"][0].reshape(-1)
    y_saved = data["Y_train"][0].reshape(-1)

print("\nINPUT COMPARISON")
print("Tokens:", len(input_tokens))
print("Exact match:", np.array_equal(x_rebuilt, x_saved))
print("Maximum absolute difference:",
      np.max(np.abs(x_rebuilt - x_saved)))
print("Differing positions:",
      np.count_nonzero(
          ~np.isclose(x_rebuilt, x_saved, atol=1e-7, rtol=0)
      ))

print("\nTARGET COMPARISON")
print("Tokens:", len(target_tokens))
print("Exact match:", np.array_equal(y_rebuilt, y_saved))
print("Maximum absolute difference:",
      np.max(np.abs(y_rebuilt - y_saved)))
print("Differing positions:",
      np.count_nonzero(
          ~np.isclose(y_rebuilt, y_saved, atol=1e-7, rtol=0)
      ))

print("\nStep 221 complete.")


In [ ]:

import numpy as np

print("=" * 65)
print("STEP 222: INSPECT PREPROCESSING DIFFERENCES")
print("=" * 65)

def inspect_differences(name, tokens, rebuilt, saved):
    differences = np.where(
        ~np.isclose(
            rebuilt,
            saved,
            atol=1e-7,
            rtol=0
        )
    )[0]

    print(f"\n{name}")
    print("Differing positions:", len(differences))

    for position in differences:
        token = (
            tokens[position]
            if position < len(tokens)
            else "<PADDING>"
        )

        print("\nPosition:", int(position))
        print("Token:", repr(token))
        print("In GloVe lookup:", token in scalar_lookup)
        print("Regenerated value:", float(rebuilt[position]))
        print("Saved value:", float(saved[position]))

inspect_differences(
    "INPUT",
    input_tokens,
    x_rebuilt,
    x_saved
)

inspect_differences(
    "TARGET",
    target_tokens,
    y_rebuilt,
    y_saved
)

print("\nStep 222 complete.")


In [ ]:

import numpy as np

print("=" * 65)
print("STEP 223: VERIFY POSSESSIVE-WORD NORMALIZATION")
print("=" * 65)

base_words = {
    "airplane",
    "manufacturer",
    "company",
    "pilot"
}

base_scalars = {}

with open(root / "glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        word, *values = line.split()

        if word in base_words:
            base_scalars[word] = np.mean(
                np.asarray(values, dtype=np.float32)
            )

print("\nBASE-WORD GLOVE SCALARS")

for word, value in sorted(base_scalars.items()):
    print(f"{word:15s} {float(value):+.8f}")

def encode_with_possessives(tokens, max_length):
    result = np.zeros(max_length, dtype=np.float32)

    for i, word in enumerate(tokens[:max_length]):
        value = scalar_lookup.get(word)

        if value is None and word.endswith("'s"):
            value = base_scalars.get(word[:-2])

        if value is not None:
            result[i] = value

    return result

x_corrected = encode_with_possessives(input_tokens, 1200)
y_corrected = encode_with_possessives(target_tokens, 200)

print("\nINPUT VERIFICATION")
print("Exact match:", np.array_equal(x_corrected, x_saved))
print(
    "Positions differing beyond 1e-7:",
    np.count_nonzero(
        ~np.isclose(x_corrected, x_saved, atol=1e-7, rtol=0)
    )
)

print("\nTARGET VERIFICATION")
print("Exact match:", np.array_equal(y_corrected, y_saved))
print(
    "Positions differing beyond 1e-7:",
    np.count_nonzero(
        ~np.isclose(y_corrected, y_saved, atol=1e-7, rtol=0)
    )
)

print("\nStep 223 complete.")


In [ ]:

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 224: MULTI-RECORD PREPROCESSING VERIFICATION")
print("=" * 65)

# Select investigation IDs in their original saved order
selected = {}

for split in ["train", "validation"]:
    df = pd.read_csv(
        root / f"{split}_investigation_ids.csv",
        dtype=str
    )
    selected[split] = df["investigation_id"].iloc[:20].tolist()

needed_ids = set(selected["train"] + selected["validation"])

# Recover only the selected CAROL records
records_by_id = {}

for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for item in records:
        investigation_id = str(
            item.get("NTSBNumber", "")
        ).strip()

        if investigation_id in needed_ids:
            records_by_id[investigation_id] = item

    del records

assert len(records_by_id) == len(needed_ids)

def tokenize(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

# Gather the vocabulary required for this verification
required_words = set()

for item in records_by_id.values():
    for field in ["AnalysisNarrative", "ProbableCause"]:
        for token in tokenize(item.get(field, "")):
            required_words.add(token)
            if token.endswith("'s"):
                required_words.add(token[:-2])

# Load mean-projected GloVe scalars
lookup = {}

with open(root / "glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        word, *values = line.split()

        if word in required_words:
            lookup[word] = np.mean(
                np.asarray(values, dtype=np.float32)
            )

def encode(text, length):
    result = np.zeros(length, dtype=np.float32)

    for i, word in enumerate(tokenize(text)[:length]):
        value = lookup.get(word)

        if value is None and word.endswith("'s"):
            value = lookup.get(word[:-2])

        if value is not None:
            result[i] = value

    return result

with np.load(
    root / "train_validation_signed_glove.npz"
) as data:

    for split in ["train", "validation"]:
        x_key = "X_train" if split == "train" else "X_val"
        y_key = "Y_train" if split == "train" else "Y_val"

        x_saved = data[x_key]
        y_saved = data[y_key]

        input_matches = 0
        target_matches = 0
        first_failure = None

        for i, investigation_id in enumerate(selected[split]):
            record = records_by_id[investigation_id]

            x = encode(record["AnalysisNarrative"], 1200)
            y = encode(record["ProbableCause"], 200)

            x_ok = np.array_equal(
                x, x_saved[i].reshape(-1)
            )
            y_ok = np.array_equal(
                y, y_saved[i].reshape(-1)
            )

            input_matches += int(x_ok)
            target_matches += int(y_ok)

            if first_failure is None and not (x_ok and y_ok):
                first_failure = {
                    "investigation_id": investigation_id,
                    "input_match": x_ok,
                    "target_match": y_ok
                }

        print(f"\n{split.upper()}")
        print("Input matches:", input_matches, "/ 20")
        print("Target matches:", target_matches, "/ 20")
        print("First failure:", first_failure)

print("\nStep 224 complete.")


In [ ]:

import numpy as np

print("=" * 65)
print("STEP 225: DIAGNOSE VALIDATION INPUT MISMATCHES")
print("=" * 65)

with np.load(
    root / "train_validation_signed_glove.npz"
) as data:
    x_val_saved = data["X_val"]

mismatched_records = 0

for i, investigation_id in enumerate(selected["validation"]):
    record = records_by_id[investigation_id]

    tokens = tokenize(record["AnalysisNarrative"])
    rebuilt = encode(record["AnalysisNarrative"], 1200)
    saved = x_val_saved[i].reshape(-1)

    differences = np.where(
        ~np.isclose(rebuilt, saved, atol=1e-7, rtol=0)
    )[0]

    if len(differences) == 0:
        continue

    mismatched_records += 1

    print("\nInvestigation:", investigation_id)
    print("Differing positions:", len(differences))

    for position in differences[:8]:
        word = (
            tokens[position]
            if position < len(tokens)
            else "<PADDING>"
        )

        print(
            f"  Position {position}: "
            f"token={word!r}, "
            f"rebuilt={rebuilt[position]:+.8f}, "
            f"saved={saved[position]:+.8f}"
        )

print("\nTotal mismatched records:", mismatched_records)
print("\nStep 225 complete.")


In [ ]:

import json
import re
import pandas as pd
from collections import Counter
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 226: VERIFY TRAINING VOCABULARY RESTRICTION")
print("=" * 65)

suspect_words = [
    "cambridge", "watered", "warmest",
    "immobilized", "schempp", "alexander",
    "reciprocals", "allegations", "conceal",
    "expediency", "obviating", "foregoing"
]

train_ids = set(
    pd.read_csv(
        root / "train_investigation_ids.csv",
        dtype=str
    )["investigation_id"].str.strip()
)

counts = Counter()
matched_records = 0

for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        investigation_id = str(
            record.get("NTSBNumber", "")
        ).strip()

        if investigation_id not in train_ids:
            continue

        matched_records += 1

        for field in ["AnalysisNarrative", "ProbableCause"]:
            text = str(record.get(field, "")).lower()

            tokens = re.findall(
                r"[a-z]+(?:'[a-z]+)?",
                text
            )

            counts.update(tokens)

    del records

print("\nTraining records checked:", matched_records)

print("\nWORD FREQUENCY IN TRAINING DATA")
print(f"{'Word':20s} {'Occurrences':>12s}")

for word in suspect_words:
    print(f"{word:20s} {counts[word]:12d}")

print("\nWords absent from training:")
print([w for w in suspect_words if counts[w] == 0])

print("\nStep 226 complete.")


In [ ]:

import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 227: VERIFY TRAINING-VOCABULARY RESTRICTION")
print("=" * 65)

# Build vocabulary from training investigations only
train_ids = set(
    pd.read_csv(
        root / "train_investigation_ids.csv",
        dtype=str
    )["investigation_id"].str.strip()
)

training_vocabulary = set()

for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        investigation_id = str(
            record.get("NTSBNumber", "")
        ).strip()

        if investigation_id not in train_ids:
            continue

        for field in ["AnalysisNarrative", "ProbableCause"]:
            training_vocabulary.update(
                tokenize(record.get(field, ""))
            )

    del records

print("Training vocabulary size:", len(training_vocabulary))

# Include base forms for possessive handling
required_words = set(training_vocabulary)

for word in training_vocabulary:
    if word.endswith("'s"):
        required_words.add(word[:-2])

# Load GloVe scalars for training vocabulary
lookup = {}

with open(root / "glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        word, *values = line.split()

        if word in required_words:
            lookup[word] = np.mean(
                np.asarray(values, dtype=np.float32)
            )

def encode_training_vocab(text, length):
    result = np.zeros(length, dtype=np.float32)

    for i, word in enumerate(tokenize(text)[:length]):

        if word not in training_vocabulary:
            continue

        value = lookup.get(word)

        if value is None and word.endswith("'s"):
            value = lookup.get(word[:-2])

        if value is not None:
            result[i] = value

    return result

# Compare first 20 records from each saved partition
with np.load(
    root / "train_validation_signed_glove.npz"
) as data:

    for split in ["train", "validation"]:
        x_key = "X_train" if split == "train" else "X_val"
        y_key = "Y_train" if split == "train" else "Y_val"

        x_saved = data[x_key]
        y_saved = data[y_key]

        input_matches = 0
        target_matches = 0

        for i, investigation_id in enumerate(selected[split]):
            record = records_by_id[investigation_id]

            x = encode_training_vocab(
                record["AnalysisNarrative"], 1200
            )
            y = encode_training_vocab(
                record["ProbableCause"], 200
            )

            input_matches += int(
                np.array_equal(x, x_saved[i].reshape(-1))
            )
            target_matches += int(
                np.array_equal(y, y_saved[i].reshape(-1))
            )

        print(f"\n{split.upper()}")
        print("Input exact matches:", input_matches, "/ 20")
        print("Target exact matches:", target_matches, "/ 20")

print("\nStep 227 complete.")


In [ ]:

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STEP 228: GENERATE INDEPENDENT TEST REPRESENTATIONS")
print("=" * 65)

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

# Reuse verified preprocessing from Step 227
assert "training_vocabulary" in globals()
assert "lookup" in globals()
assert "encode_training_vocab" in globals()

# Preserve original test ordering
test_ids = pd.read_csv(
    root / "test_investigation_ids.csv",
    dtype=str
)["investigation_id"].str.strip().tolist()

test_id_set = set(test_ids)
test_records = {}

# Retrieve test investigations from original CAROL files
for path in sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        investigation_id = str(
            record.get("NTSBNumber", "")
        ).strip()

        if investigation_id in test_id_set:
            test_records[investigation_id] = record

    del records

assert len(test_records) == len(test_ids), (
    "Missing test investigations."
)

# Construct numerical representations
X_test = np.zeros(
    (len(test_ids), 1200, 1),
    dtype=np.float32
)

Y_test = np.zeros(
    (len(test_ids), 200, 1),
    dtype=np.float32
)

for i, investigation_id in enumerate(test_ids):
    record = test_records[investigation_id]

    X_test[i, :, 0] = encode_training_vocab(
        record["AnalysisNarrative"], 1200
    )

    Y_test[i, :, 0] = encode_training_vocab(
        record["ProbableCause"], 200
    )

# Save independent test tensors
output_path = root / "independent_test_signed_glove.npz"

np.savez_compressed(
    output_path,
    X_test=X_test,
    Y_test=Y_test
)

print("\nTEST REPRESENTATIONS")
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

print("\nNONZERO PROPORTIONS")
print("X_test:", np.count_nonzero(X_test) / X_test.size)
print("Y_test:", np.count_nonzero(Y_test) / Y_test.size)

print("\nSAVED FILE")
print(output_path)

# Verify saved file
with np.load(output_path) as data:
    assert data["X_test"].shape == (3138, 1200, 1)
    assert data["Y_test"].shape == (3138, 200, 1)

print("\nSUCCESS: Independent test representations saved.")
print("Step 228 complete.")


In [ ]:

import json
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 229: INSPECT SAVED MODEL ARCHITECTURES")
print("=" * 65)

files = [
    "encoder_architecture.json",
    "decoder_architecture.json",
    "deterministic_architecture.json"
]

for filename in files:
    path = root / filename

    print(f"\n{'=' * 55}")
    print(filename)
    print("=" * 55)

    with open(path, "r") as f:
        architecture = json.load(f)

    print("Top-level keys:", list(architecture.keys()))

    config = architecture.get("config", {})

    print("Model name:", config.get("name"))
    print("Model class:", architecture.get("class_name"))

    layers = config.get("layers", [])

    print("\nLayers:")
    for layer in layers:
        layer_config = layer.get("config", {})

        print(
            layer.get("class_name"),
            "|",
            layer_config.get("name"),
            "|",
            "units:", layer_config.get("units"),
            "|",
            "filters:", layer_config.get("filters")
        )

print("\nSAVED CHECKPOINTS")
for filename in [
    "deterministic_best_170.keras",
    "vae_best_176b.weights.h5"
]:
    path = root / filename
    print(filename, "| Exists:", path.exists())

print("\nStep 229 complete.")


In [ ]:

import numpy as np
import tensorflow as tf
from pathlib import Path

print("=" * 65)
print("STEP 230: INDEPENDENT TEST — DETERMINISTIC BASELINE")
print("=" * 65)

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

# Load saved independent test representations
with np.load(root / "independent_test_signed_glove.npz") as data:
    X_test = data["X_test"]
    Y_test = data["Y_test"]

# Restore trained deterministic model
model_path = root / "deterministic_best_170.keras"

deterministic_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("Model loaded:", deterministic_model.name)
print("Parameters:", deterministic_model.count_params())

# Predict without retraining
Y_pred_det = deterministic_model.predict(
    X_test,
    batch_size=32,
    verbose=1
)

assert Y_pred_det.shape == Y_test.shape, (
    f"Prediction shape mismatch: {Y_pred_det.shape}"
)

# Per-investigation cosine similarity
true_flat = Y_test.reshape(len(Y_test), -1)
pred_flat = Y_pred_det.reshape(len(Y_pred_det), -1)

numerator = np.sum(true_flat * pred_flat, axis=1)

denominator = (
    np.linalg.norm(true_flat, axis=1)
    * np.linalg.norm(pred_flat, axis=1)
)

cosine_scores = np.divide(
    numerator,
    denominator,
    out=np.zeros_like(numerator),
    where=denominator > 0
)

print("\nINDEPENDENT TEST RESULTS")
print("Investigations:", len(cosine_scores))
print("Mean cosine similarity:", float(np.mean(cosine_scores)))
print("Median cosine similarity:", float(np.median(cosine_scores)))
print("Standard deviation:", float(np.std(cosine_scores)))
print("Zero-norm pairs:", int(np.sum(denominator == 0)))

# Save predictions and individual scores
output = root / "deterministic_independent_test_results.npz"

np.savez_compressed(
    output,
    Y_pred=Y_pred_det,
    cosine_scores=cosine_scores
)

print("\nSaved:", output)
print("Step 230 complete.")


In [ ]:

import h5py
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

checkpoint = root / "vae_best_176b.weights.h5"

print("=" * 65)
print("STEP 231: INSPECT SAVED VAE CHECKPOINT")
print("=" * 65)

with h5py.File(checkpoint, "r") as f:
    print("\nTop-level groups:", list(f.keys()))

    if "layers" in f:
        print("\nSaved layers:")
        for name, group in f["layers"].items():
            print(name, "| subgroups:", list(group.keys())[:8])

    if "vars" in f:
        print("\nTop-level variables:", list(f["vars"].keys()))

print("\nStep 231 complete.")


In [ ]:

import h5py
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

checkpoint = root / "vae_best_176b.weights.h5"

print("=" * 65)
print("STEP 232: INSPECT VAE ENCODER AND DECODER WEIGHTS")
print("=" * 65)

with h5py.File(checkpoint, "r") as f:
    for component in ["encoder", "decoder"]:
        print(f"\n{component.upper()}")

        group = f[component]
        print("Groups:", list(group.keys()))

        if "layers" in group:
            for name, layer in group["layers"].items():
                shapes = []

                def collect(path, obj):
                    if isinstance(obj, h5py.Dataset):
                        shapes.append((path, obj.shape))

                layer.visititems(collect)

                if shapes:
                    print(name, shapes[:5])

print("\nStep 232 complete.")


In [ ]:

import json
import numpy as np
import tensorflow as tf
from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

print("=" * 65)
print("STEP 233: RESTORE VARIATIONAL MODEL")
print("=" * 65)

# Restore original convolutional encoder backbone
with open(root / "encoder_architecture.json") as f:
    encoder_json = json.load(f)

encoder_backbone = tf.keras.models.model_from_json(
    json.dumps(encoder_json)
)

# Restore original decoder
with open(root / "decoder_architecture.json") as f:
    decoder_json = json.load(f)

decoder = tf.keras.models.model_from_json(
    json.dumps(decoder_json)
)

# Extract encoder features before the deterministic latent layer
feature_extractor = tf.keras.Model(
    encoder_backbone.input,
    encoder_backbone.get_layer("encoder_flatten").output
)

# Separate variational projections
features = feature_extractor.output

z_mean = tf.keras.layers.Dense(
    20, name="z_mean"
)(features)

z_log_var = tf.keras.layers.Dense(
    20, name="z_log_var"
)(features)

vae_encoder = tf.keras.Model(
    feature_extractor.input,
    [z_mean, z_log_var],
    name="vae_encoder"
)

print("Encoder and decoder architectures reconstructed.")
print("VAE encoder outputs:", vae_encoder.output_shape)
print("Decoder output:", decoder.output_shape)

print("\nStep 233 architecture preparation complete.")


In [ ]:

import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from pathlib import Path

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")

print("=" * 65)
print("STEP 236: RESTORE TRAINED VAE CHECKPOINT")
print("=" * 65)

# Original encoder backbone
with open(root / "encoder_architecture.json") as f:
    encoder_base = tf.keras.models.model_from_json(f.read())

feature_extractor = Model(
    encoder_base.input,
    encoder_base.get_layer("encoder_flatten").output,
    name="vae_feature_extractor_174"
)

vae_input = layers.Input(shape=(1200, 1), name="vae_input_174")
features = feature_extractor(vae_input)

z_mean = layers.Dense(20, name="z_mean_174")(features)
z_logvar = layers.Dense(20, name="z_logvar_174")(features)

variational_encoder = Model(
    vae_input,
    [z_mean, z_logvar],
    name="variational_encoder_174"
)

# Original sampling layer
class Sampling174(layers.Layer):
    def call(self, inputs):
        mean, logvar = inputs
        epsilon = tf.random.normal(tf.shape(mean))
        return mean + tf.exp(0.5 * logvar) * epsilon

sampling = Sampling174(name="sampling_174")

# Original decoder
with open(root / "decoder_architecture.json") as f:
    decoder = tf.keras.models.model_from_json(f.read())

# Original trainer structure
class VAETrainer175(Model):
    def __init__(self, encoder, decoder, sampling, beta=0.001):
        super().__init__(name="genuine_vae_175")
        self.encoder = encoder
        self.decoder = decoder
        self.sampling = sampling
        self.beta = beta

    def call(self, x, training=False):
        mean, logvar = self.encoder(x, training=training)
        z = self.sampling([mean, logvar])
        return self.decoder(z, training=training)

vae = VAETrainer175(
    variational_encoder,
    decoder,
    sampling,
    beta=0.001
)

# Build variables before loading weights
dummy = tf.zeros((1, 1200, 1), dtype=tf.float32)
_ = vae(dummy, training=False)

# Restore saved trained parameters
checkpoint = root / "vae_best_176b.weights.h5"
vae.load_weights(str(checkpoint))

print("Checkpoint loaded successfully.")
print("Model:", vae.name)
print("Parameters:", vae.count_params())
print("Encoder outputs:", vae.encoder.output_shape)
print("Decoder output:", vae.decoder.output_shape)
print("Training performed: NO")
print("Step 236 complete.")


In [ ]:

import numpy as np
from pathlib import Path

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")

print("=" * 65)
print("STEP 237: INDEPENDENT TEST — TRAINED VAE")
print("=" * 65)

# Load independent test representations
with np.load(root / "independent_test_signed_glove.npz") as data:
    X_test = data["X_test"]
    Y_test = data["Y_test"]

# Obtain latent means without stochastic sampling
z_mean, z_logvar = vae.encoder.predict(
    X_test,
    batch_size=32,
    verbose=1
)

# Decode latent means
Y_pred_vae = vae.decoder.predict(
    z_mean,
    batch_size=32,
    verbose=1
)

assert Y_pred_vae.shape == Y_test.shape

# Per-investigation cosine similarity
true_flat = Y_test.reshape(len(Y_test), -1)
pred_flat = Y_pred_vae.reshape(len(Y_pred_vae), -1)

numerator = np.sum(true_flat * pred_flat, axis=1)

denominator = (
    np.linalg.norm(true_flat, axis=1)
    * np.linalg.norm(pred_flat, axis=1)
)

cosine_scores_vae = np.divide(
    numerator,
    denominator,
    out=np.zeros_like(numerator),
    where=denominator > 0
)

print("\nINDEPENDENT TEST RESULTS")
print("Investigations:", len(cosine_scores_vae))
print("Mean cosine:", float(np.mean(cosine_scores_vae)))
print("Median cosine:", float(np.median(cosine_scores_vae)))
print("Standard deviation:", float(np.std(cosine_scores_vae)))
print("Zero-norm pairs:", int(np.sum(denominator == 0)))

# Compare with saved deterministic baseline
with np.load(root / "deterministic_independent_test_results.npz") as data:
    cosine_scores_det = data["cosine_scores"]

print("\nCOMPARISON WITH DETERMINISTIC BASELINE")
print("VAE:", float(np.mean(cosine_scores_vae)))
print("Deterministic:", float(np.mean(cosine_scores_det)))
print(
    "VAE minus deterministic:",
    float(np.mean(cosine_scores_vae - cosine_scores_det))
)

# Save predictions and scores
output = root / "vae_independent_test_results.npz"

np.savez_compressed(
    output,
    Y_pred=Y_pred_vae,
    cosine_scores=cosine_scores_vae,
    z_mean=z_mean,
    z_logvar=z_logvar
)

print("\nSaved:", output)
print("Step 237 complete.")


In [ ]:

import numpy as np
import json
from pathlib import Path
from scipy import stats

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")

print("=" * 65)
print("STEP 238: PAIRED INDEPENDENT TEST COMPARISON")
print("=" * 65)

with np.load(root / "vae_independent_test_results.npz") as f:
    vae_scores = f["cosine_scores"].astype(np.float64)

with np.load(root / "deterministic_independent_test_results.npz") as f:
    det_scores = f["cosine_scores"].astype(np.float64)

assert len(vae_scores) == len(det_scores) == 3138
assert np.all(np.isfinite(vae_scores))
assert np.all(np.isfinite(det_scores))

differences = vae_scores - det_scores
observed = float(np.mean(differences))

# Paired bootstrap confidence interval
rng = np.random.default_rng(42)
n_bootstrap = 10000
bootstrap_means = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    indices = rng.integers(0, len(differences), len(differences))
    bootstrap_means[i] = differences[indices].mean()

ci_low, ci_high = np.percentile(
    bootstrap_means, [2.5, 97.5]
)

# Two-sided paired t-test
t_stat, p_value = stats.ttest_rel(vae_scores, det_scores)

results = {
    "test_cases": len(differences),
    "vae_mean_cosine": float(vae_scores.mean()),
    "deterministic_mean_cosine": float(det_scores.mean()),
    "mean_paired_difference": observed,
    "bootstrap_95_ci": [float(ci_low), float(ci_high)],
    "paired_t_statistic": float(t_stat),
    "paired_t_p_value": float(p_value),
    "bootstrap_iterations": n_bootstrap,
    "random_seed": 42
}

print("\nRESULTS")
for key, value in results.items():
    print(f"{key}: {value}")

output = root / "vae_deterministic_independent_test_statistics.json"

with open(output, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", output)
print("Step 238 complete.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")

print("=" * 65)
print("STEP 239: MODEL-SPECIFIC ERROR ANALYSIS")
print("=" * 65)

with np.load(root / "vae_independent_test_results.npz") as f:
    vae_scores = f["cosine_scores"]

with np.load(root / "deterministic_independent_test_results.npz") as f:
    det_scores = f["cosine_scores"]

ids = pd.read_csv(root / "test_investigation_ids.csv")

assert len(ids) == len(vae_scores) == len(det_scores)

results = pd.DataFrame({
    "investigation_id": ids["investigation_id"].astype(str),
    "vae_cosine": vae_scores,
    "deterministic_cosine": det_scores
})

results["vae_advantage"] = (
    results["vae_cosine"] - results["deterministic_cosine"]
)

print("\nMODEL COMPARISON")
print("VAE better:", int((results.vae_advantage > 0).sum()))
print("Deterministic better:", int((results.vae_advantage < 0).sum()))
print("Ties:", int((results.vae_advantage == 0).sum()))

print("\nFIVE LOWEST VAE COSINE SCORES")
print(
    results.nsmallest(5, "vae_cosine").to_string(index=False)
)

print("\nFIVE HIGHEST VAE COSINE SCORES")
print(
    results.nlargest(5, "vae_cosine").to_string(index=False)
)

print("\nFIVE LARGEST VAE ADVANTAGES")
print(
    results.nlargest(5, "vae_advantage").to_string(index=False)
)

print("\nFIVE LARGEST DETERMINISTIC ADVANTAGES")
print(
    results.nsmallest(5, "vae_advantage").to_string(index=False)
)

output = root / "vae_model_specific_test_errors.csv"
results.to_csv(output, index=False)

print("\nSaved:", output)
print("Step 239 complete.")


In [ ]:

import json
import pandas as pd
from pathlib import Path

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")
carol_dir = root / "CAROL_Original_JSON"

print("=" * 65)
print("STEP 240: INSPECT REPRESENTATIVE TEST NARRATIVES")
print("=" * 65)

scores = pd.read_csv(root / "vae_model_specific_test_errors.csv")

# Representative extremes from the independent test
selected_ids = [
    "NYC08IA206",  # Lowest VAE cosine
    "ATL07LA120",  # Highest VAE cosine
    "MIA08LA156",  # Largest VAE advantage
    "GAA16CA065",  # Second-largest VAE advantage
    "CEN20LA145",  # Largest deterministic advantage
    "ANC12CA115"   # Second-largest deterministic advantage
]

records = {}

for path in sorted(carol_dir.glob("*.json")):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        data = data.get("data", data.get("results", []))

    for record in data:
        if not isinstance(record, dict):
            continue

        investigation_id = str(record.get("NTSBNumber", "")).strip()

        if investigation_id in selected_ids:
            records[investigation_id] = record

for investigation_id in selected_ids:
    print("\n" + "=" * 65)
    print("INVESTIGATION:", investigation_id)
    print("=" * 65)

    row = scores.loc[
        scores["investigation_id"] == investigation_id
    ].iloc[0]

    print("VAE cosine:", round(float(row["vae_cosine"]), 4))
    print("Deterministic cosine:",
          round(float(row["deterministic_cosine"]), 4))
    print("VAE advantage:",
          round(float(row["vae_advantage"]), 4))

    record = records.get(investigation_id)

    if record is None:
        print("Original CAROL record not found.")
        continue

    narrative = str(record.get("AnalysisNarrative") or "")
    probable_cause = str(record.get("ProbableCause") or "")

    print("\nNARRATIVE WORD COUNT:", len(narrative.split()))
    print("PROBABLE CAUSE WORD COUNT:", len(probable_cause.split()))

    print("\nANALYSIS NARRATIVE:")
    print(narrative[:1800])

    print("\nPROBABLE CAUSE:")
    print(probable_cause[:1000])

print("\nStep 240 complete.")


In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from pathlib import Path

root = Path("/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility")

print("=" * 65)
print("STEP 241: LENGTH AND REPRESENTATION COVERAGE ANALYSIS")
print("=" * 65)

scores = pd.read_csv(root / "vae_model_specific_test_errors.csv")

with np.load(root / "independent_test_signed_glove.npz") as f:
    X = f["X_test"]
    Y = f["Y_test"]

assert len(scores) == len(X) == len(Y)

# Nonzero encoded positions (representation coverage proxies)
input_nonzero = np.count_nonzero(X.reshape(len(X), -1), axis=1)
target_nonzero = np.count_nonzero(Y.reshape(len(Y), -1), axis=1)

scores["input_nonzero_positions"] = input_nonzero
scores["target_nonzero_positions"] = target_nonzero

print("\nSPEARMAN CORRELATIONS")

for variable in [
    "input_nonzero_positions",
    "target_nonzero_positions"
]:
    for metric in ["vae_cosine", "vae_advantage"]:
        rho, p = spearmanr(scores[variable], scores[metric])
        print(
            f"{variable} vs {metric}: "
            f"rho={rho:.4f}, p={p:.6g}"
        )

# Group investigations into quartiles of encoded input coverage
scores["coverage_quartile"] = pd.qcut(
    scores["input_nonzero_positions"],
    q=4,
    labels=False,
    duplicates="drop"
) + 1

summary = scores.groupby("coverage_quartile").agg(
    investigations=("vae_cosine", "size"),
    mean_vae_cosine=("vae_cosine", "mean"),
    mean_deterministic_cosine=("deterministic_cosine", "mean"),
    mean_vae_advantage=("vae_advantage", "mean")
)

print("\nINPUT REPRESENTATION COVERAGE QUARTILES")
print(summary.to_string())

output = root / "vae_lexical_coverage_test_analysis.csv"
scores.to_csv(output, index=False)

print("\nSaved:", output)
print("Step 241 complete.")


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# STEP 242: PUBLICATION-QUALITY REVIEWER 2 FIGURES
# CPU ONLY — NO TRAINING OR MODEL INFERENCE
# ============================================================

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

fig_dir = root / "Reviewer2_Publication_Figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Load verified independent test results
with np.load(root / "vae_independent_test_results.npz") as f:
    vae = f["cosine_scores"].astype(np.float64)

with np.load(root / "deterministic_independent_test_results.npz") as f:
    det = f["cosine_scores"].astype(np.float64)

analysis = pd.read_csv(
    root / "vae_lexical_coverage_test_analysis.csv"
)

assert len(vae) == len(det) == len(analysis) == 3138

difference = vae - det
mean_diff = difference.mean()

# Reproduce paired bootstrap interval
rng = np.random.default_rng(42)
boot = np.empty(10000)

for i in range(10000):
    idx = rng.integers(0, len(difference), len(difference))
    boot[i] = difference[idx].mean()

ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

# Journal-friendly formatting
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300
})

navy = "#17324D"
gold = "#B78A38"
gray = "#687782"

def save_figure(name):
    plt.savefig(
        fig_dir / f"{name}.pdf",
        bbox_inches="tight"
    )
    plt.savefig(
        fig_dir / f"{name}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ============================================================
# FIGURE 1: PAIRED VAE IMPROVEMENT
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 3.6))

ax.errorbar(
    mean_diff,
    0,
    xerr=np.array([
        [mean_diff - ci_low],
        [ci_high - mean_diff]
    ]),
    fmt="o",
    color=navy,
    ecolor=navy,
    markersize=9,
    capsize=6,
    linewidth=2
)

ax.axvline(
    0,
    color=gray,
    linestyle="--",
    linewidth=1.2,
    label="No difference"
)

ax.set_yticks([0])
ax.set_yticklabels(["VAE − deterministic"])
ax.set_xlabel("Paired difference in cosine similarity")

ax.set_title(
    "Independent test comparison (n = 3,138)"
)

ax.set_xlim(
    min(-0.001, ci_low - 0.001),
    ci_high + 0.0015
)

ax.text(
    mean_diff,
    0.17,
    f"Mean = {mean_diff:.4f}\n"
    f"95% CI [{ci_low:.4f}, {ci_high:.4f}]",
    ha="center",
    va="bottom",
    fontsize=9
)

ax.set_ylim(-0.4, 0.48)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
save_figure("Figure1_Paired_VAE_Improvement")

# ============================================================
# FIGURE 2: REPRESENTATION DENSITY QUARTILES
# ============================================================

summary = analysis.groupby(
    "coverage_quartile"
).agg(
    vae=("vae_cosine", "mean"),
    deterministic=("deterministic_cosine", "mean"),
    count=("vae_cosine", "size")
).reset_index()

x = np.arange(len(summary))

fig, ax = plt.subplots(figsize=(6.2, 3.8))

ax.plot(
    x,
    summary["vae"],
    marker="o",
    linewidth=2,
    markersize=7,
    color=navy,
    label="VAE"
)

ax.plot(
    x,
    summary["deterministic"],
    marker="s",
    linewidth=2,
    markersize=6,
    linestyle="--",
    color=gold,
    label="Deterministic baseline"
)

ax.set_xticks(x)
ax.set_xticklabels([
    f"Q{int(q)}\n(n={int(n)})"
    for q, n in zip(
        summary["coverage_quartile"],
        summary["count"]
    )
])

ax.set_xlabel(
    "Input representation-density quartile"
)
ax.set_ylabel("Mean cosine similarity")
ax.set_title(
    "Reconstruction similarity by representation density"
)

ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
save_figure("Figure2_Representation_Density")

# ============================================================
# FIGURE 3: PAIRED DIFFERENCE DISTRIBUTION
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 3.8))

ax.hist(
    difference,
    bins=55,
    color=navy,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.35
)

ax.axvline(
    0,
    color=gray,
    linestyle="--",
    linewidth=1.5,
    label="Equal performance"
)

ax.axvline(
    mean_diff,
    color=gold,
    linewidth=2,
    label=f"Mean difference = {mean_diff:.4f}"
)

ax.set_xlabel(
    "Per-investigation cosine difference (VAE − deterministic)"
)
ax.set_ylabel("Number of investigations")
ax.set_title(
    "Distribution of paired test performance differences"
)

ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)

fig.tight_layout()
save_figure("Figure3_Paired_Difference_Distribution")

print("=" * 65)
print("STEP 242 COMPLETE")
print("=" * 65)

print("\nSaved figures:")
for path in sorted(fig_dir.iterdir()):
    print(path.name)

print("\nLocation:", fig_dir)
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt

fig_dir = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility/"
    "Reviewer2_Publication_Figures"
)

# Regenerate the figures by rerunning Step 242,
# but first remove these three title-setting statements:
#
# Figure 1:
# ax.set_title("Independent test comparison (n = 3,138)")
#
# Figure 2:
# ax.set_title(
#     "Reconstruction similarity by representation density"
# )
#
# Figure 3:
# ax.set_title(
#     "Distribution of paired test performance differences"
# )
#
# Keep all other code unchanged.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# STEP 242 — FINAL PUBLICATION FIGURES FOR REVIEWER 2
# NO PLOT TITLES | NO TRAINING | CPU ONLY
# ============================================================

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

fig_dir = root / "Reviewer2_Publication_Figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD VERIFIED INDEPENDENT TEST RESULTS
# ------------------------------------------------------------

with np.load(root / "vae_independent_test_results.npz") as f:
    vae = f["cosine_scores"].astype(np.float64)

with np.load(
    root / "deterministic_independent_test_results.npz"
) as f:
    det = f["cosine_scores"].astype(np.float64)

analysis = pd.read_csv(
    root / "vae_lexical_coverage_test_analysis.csv"
)

assert len(vae) == len(det) == len(analysis) == 3138
assert np.all(np.isfinite(vae))
assert np.all(np.isfinite(det))

difference = vae - det
mean_diff = difference.mean()

# ------------------------------------------------------------
# PAIRED BOOTSTRAP: 95% CONFIDENCE INTERVAL
# ------------------------------------------------------------

rng = np.random.default_rng(42)
boot = np.empty(10000)

for i in range(10000):
    idx = rng.integers(
        0, len(difference), len(difference)
    )
    boot[i] = difference[idx].mean()

ci_low, ci_high = np.percentile(
    boot, [2.5, 97.5]
)

# ------------------------------------------------------------
# PUBLICATION FORMATTING
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300
})

navy = "#17324D"
gold = "#B78A38"
gray = "#687782"

def save_figure(name):
    plt.savefig(
        fig_dir / f"{name}.pdf",
        bbox_inches="tight"
    )
    plt.savefig(
        fig_dir / f"{name}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

# ============================================================
# FIGURE 1 — PAIRED VAE IMPROVEMENT WITH 95% CI
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 3.2))

ax.errorbar(
    mean_diff,
    0,
    xerr=np.array([
        [mean_diff - ci_low],
        [ci_high - mean_diff]
    ]),
    fmt="o",
    color=navy,
    ecolor=navy,
    markersize=9,
    capsize=6,
    linewidth=2
)

ax.axvline(
    0,
    color=gray,
    linestyle="--",
    linewidth=1.2
)

ax.set_yticks([0])
ax.set_yticklabels(["VAE − deterministic"])

ax.set_xlabel(
    "Paired difference in cosine similarity"
)

ax.set_xlim(
    min(-0.001, ci_low - 0.001),
    ci_high + 0.0015
)

ax.text(
    mean_diff,
    0.16,
    f"Mean = {mean_diff:.4f}\n"
    f"95% CI [{ci_low:.4f}, {ci_high:.4f}]",
    ha="center",
    va="bottom",
    fontsize=9
)

ax.set_ylim(-0.35, 0.43)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
save_figure("Figure1_Paired_VAE_Improvement")

# ============================================================
# FIGURE 2 — REPRESENTATION-DENSITY QUARTILES
# ============================================================

summary = analysis.groupby(
    "coverage_quartile"
).agg(
    vae=("vae_cosine", "mean"),
    deterministic=("deterministic_cosine", "mean"),
    count=("vae_cosine", "size")
).reset_index()

x = np.arange(len(summary))

fig, ax = plt.subplots(figsize=(6.2, 3.6))

ax.plot(
    x,
    summary["vae"],
    marker="o",
    linewidth=2,
    markersize=7,
    color=navy,
    label="VAE"
)

ax.plot(
    x,
    summary["deterministic"],
    marker="s",
    linewidth=2,
    markersize=6,
    linestyle="--",
    color=gold,
    label="Deterministic baseline"
)

ax.set_xticks(x)

ax.set_xticklabels([
    f"Q{int(q)}\n(n={int(n)})"
    for q, n in zip(
        summary["coverage_quartile"],
        summary["count"]
    )
])

ax.set_xlabel(
    "Input representation-density quartile"
)

ax.set_ylabel("Mean cosine similarity")

ax.legend(
    frameon=False,
    loc="upper right"
)

ax.grid(axis="y", alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
save_figure("Figure2_Representation_Density")

# ============================================================
# FIGURE 3 — DISTRIBUTION OF PAIRED DIFFERENCES
# ============================================================

fig, ax = plt.subplots(figsize=(6.2, 3.6))

ax.hist(
    difference,
    bins=55,
    color=navy,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.35
)

ax.axvline(
    0,
    color=gray,
    linestyle="--",
    linewidth=1.5,
    label="Equal performance"
)

ax.axvline(
    mean_diff,
    color=gold,
    linewidth=2,
    label=f"Mean difference = {mean_diff:.4f}"
)

ax.set_xlabel(
    "Per-investigation cosine difference (VAE − deterministic)"
)

ax.set_ylabel("Number of investigations")

ax.legend(
    frameon=False,
    loc="upper right"
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)

fig.tight_layout()
save_figure("Figure3_Paired_Difference_Distribution")

# ============================================================
# COMPLETION
# ============================================================

print("=" * 65)
print("FINAL PUBLICATION FIGURES GENERATED")
print("=" * 65)

print("\nSaved files:")

for path in sorted(fig_dir.iterdir()):
    if path.suffix.lower() in [".pdf", ".png"]:
        print(path.name)

print("\nLocation:", fig_dir)
print("Plot headings: REMOVED")
print("Training performed: NO")
print("GPU required: NO")
print("Step 242 complete.")


In [ ]:

from google.colab import drive
from pathlib import Path

# ============================================================
# STEP 243A — RECONNECT DRIVE AND LOCATE ARCHITECTURE FILES
# CPU ONLY | NO TRAINING
# ============================================================

# Reuse the previous mount if it is available.
mount_path = Path("/content/drive_recovered")

if not (mount_path / "MyDrive").exists():
    mount_path.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mount_path))

# Locate the project folder.
root = mount_path / "MyDrive" / "NTSB_VAE_Reproducibility"

print("\nProject folder:", root)
print("Project folder exists:", root.exists())

files = [
    "encoder_architecture.json",
    "decoder_architecture.json",
    "deterministic_architecture.json"
]

if root.exists():
    print("\nArchitecture files:")

    for filename in files:
        path = root / filename
        print(
            "FOUND" if path.is_file() else "NOT FOUND",
            "-", filename
        )

    print("\nOther model-related files:")

    for path in sorted(root.iterdir()):
        if (
            path.suffix.lower() in [".json", ".keras", ".h5"]
            or "architecture" in path.name.lower()
        ):
            print(path.name)
else:
    print("\nProject folder not found at expected location.")
    print("Available folders in MyDrive:")

    mydrive = mount_path / "MyDrive"

    if mydrive.exists():
        for path in sorted(mydrive.iterdir()):
            if path.is_dir():
                print(path.name)

print("\nSTEP 243A COMPLETE")


In [ ]:

import json
import pandas as pd
from pathlib import Path

# ============================================================
# STEP 243B — EXTRACT SAVED MODEL ARCHITECTURES
# CPU ONLY | NO TRAINING | NO GPU
# ============================================================

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

architecture_files = {
    "VAE Encoder": "encoder_architecture.json",
    "VAE Decoder": "decoder_architecture.json",
    "Deterministic Model": "deterministic_architecture.json"
}

all_rows = []

for model_name, filename in architecture_files.items():

    path = root / filename

    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    layers = config.get("config", {}).get("layers", [])

    print("\n" + "=" * 90)
    print(model_name.upper())
    print("File:", filename)
    print("Number of layers:", len(layers))
    print("=" * 90)

    rows = []

    for i, layer in enumerate(layers, start=1):

        cfg = layer.get("config", {})

        row = {
            "Model": model_name,
            "Index": i,
            "Layer": cfg.get("name", ""),
            "Type": layer.get("class_name", ""),
            "Filters": cfg.get("filters", ""),
            "Units": cfg.get("units", ""),
            "Kernel": str(cfg.get("kernel_size", "")),
            "Strides": str(cfg.get("strides", "")),
            "Padding": cfg.get("padding", ""),
            "Activation": cfg.get("activation", ""),
            "Output Padding": str(
                cfg.get("output_padding", "")
            )
        }

        rows.append(row)
        all_rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        print(df.to_string(index=False))
    else:
        print("No layers found in standard Keras configuration.")
        print("Top-level keys:", list(config.keys()))

# Save all extracted layers
output = root / "verified_architecture_layers.csv"

pd.DataFrame(all_rows).to_csv(
    output,
    index=False
)

print("\n" + "=" * 90)
print("STEP 243B COMPLETE")
print("=" * 90)

print("Saved:", output)
print("Total extracted layers:", len(all_rows))
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

import json
import h5py
from pathlib import Path

# ============================================================
# STEP 243C — VERIFY VAE LATENT IMPLEMENTATION
# CPU ONLY | NO TRAINING | NO MODEL MODIFICATION
# ============================================================

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

print("=" * 75)
print("1. ENCODER LATENT LAYERS")
print("=" * 75)

with open(root / "encoder_architecture.json") as f:
    encoder = json.load(f)

for layer in encoder.get("config", {}).get("layers", []):
    name = layer.get("config", {}).get("name", "")
    layer_type = layer.get("class_name", "")

    if (
        "latent" in name.lower()
        or "mean" in name.lower()
        or "variance" in name.lower()
        or "logvar" in name.lower()
        or layer_type == "Dense"
    ):
        print(
            name,
            "| Type:", layer_type,
            "| Units:", layer.get("config", {}).get("units")
        )

print("\n" + "=" * 75)
print("2. VAE WEIGHT FILE STRUCTURE")
print("=" * 75)

weights_path = root / "vae_best_176b.weights.h5"

with h5py.File(weights_path, "r") as f:

    def inspect(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "| Shape:", obj.shape)

    f.visititems(inspect)

print("\n" + "=" * 75)
print("3. ARCHITECTURE METADATA")
print("=" * 75)

manifest_path = root / "experiment_manifest.json"

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)

    print(json.dumps(manifest, indent=2)[:12000])

print("\nSTEP 243C COMPLETE")
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

import json
import pandas as pd
import tensorflow as tf
from pathlib import Path

# ============================================================
# STEP 243D — VERIFIED LAYER-BY-LAYER ARCHITECTURE
# CPU ONLY | NO TRAINING
# ============================================================

root = Path(
    "/content/drive_recovered/MyDrive/NTSB_VAE_Reproducibility"
)

models = {
    "Encoder feature extractor": "encoder_architecture.json",
    "Decoder": "decoder_architecture.json"
}

all_rows = []

for model_name, filename in models.items():

    with open(root / filename, "r") as f:
        model_json = f.read()

    model = tf.keras.models.model_from_json(model_json)

    print("\n" + "=" * 90)
    print(model_name.upper())
    print("=" * 90)

    rows = []

    for layer in model.layers:

        if isinstance(layer, tf.keras.layers.InputLayer):
            continue

        # Keras symbolic output shape
        try:
            output_shape = str(tuple(layer.output.shape))
        except Exception:
            output_shape = "N/A"

        cfg = layer.get_config()

        row = {
            "Model": model_name,
            "Layer": layer.name,
            "Type": layer.__class__.__name__,
            "Filters/Units": cfg.get(
                "filters", cfg.get("units", "")
            ),
            "Output shape": output_shape,
            "Kernel": str(cfg.get("kernel_size", "")),
            "Stride": str(cfg.get("strides", "")),
            "Parameters": layer.count_params()
        }

        rows.append(row)
        all_rows.append(row)

    df = pd.DataFrame(rows)

    print(df.to_string(index=False))
    print("\nModel parameters:", model.count_params())

output = root / "verified_architecture_detailed.csv"

pd.DataFrame(all_rows).to_csv(
    output,
    index=False
)

print("\n" + "=" * 90)
print("STEP 243D COMPLETE")
print("=" * 90)
print("Saved:", output)
print("Training performed: NO")
print("GPU required: NO")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

root = Path(
    "/content/drive_recovered/MyDrive/"
    "NTSB_VAE_Reproducibility"
)

if not root.exists():
    root = Path(
        "/content/drive/MyDrive/"
        "NTSB_VAE_Reproducibility"
    )

print("Project folder:", root)
print("Exists:", root.exists())

extensions = {".csv", ".npz", ".npy", ".parquet"}

files = sorted(
    p for p in root.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

for i, p in enumerate(files):
    print(f"{i:03d} | {p.relative_to(root)}")

print("\nTotal candidate files:", len(files))


In [ ]:

from pathlib import Path

# Use the currently mounted Google Drive
root = Path(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
)

print("Project folder:", root)
print("Exists:", root.is_dir())

if root.is_dir():
    extensions = {".csv", ".npz", ".npy", ".parquet"}

    files = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in extensions
    )

    for i, p in enumerate(files):
        print(f"{i:03d} | {p.relative_to(root)}")

    print("\nTotal candidate files:", len(files))
else:
    print("\nProject folder not found.")
    print("Available folders in MyDrive:")

    mydrive = Path("/content/drive/MyDrive")

    for p in mydrive.iterdir():
        if p.is_dir():
            print(" -", p.name)


In [ ]:

import numpy as np
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
)

vae_path = root / "vae_independent_test_results.npz"
det_path = root / "deterministic_independent_test_results.npz"

for name, path in [
    ("VAE", vae_path),
    ("Deterministic", det_path)
]:
    print(f"\n{name} results")
    print("-" * 45)

    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(
                f"{key}: "
                f"shape={arr.shape}, "
                f"dtype={arr.dtype}"
            )


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project directories
root = Path(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
)

output_dir = root / "Reviewer2_Publication_Figures"
output_dir.mkdir(parents=True, exist_ok=True)

# Load reproduced independent-test scores
with np.load(
    root / "vae_independent_test_results.npz",
    allow_pickle=False
) as data:
    scores = data["cosine_scores"].astype(np.float64)

# Validate the scores
assert scores.shape == (3138,), (
    f"Unexpected score shape: {scores.shape}"
)
assert np.all(np.isfinite(scores)), (
    "Non-finite cosine scores detected."
)

mean_score = float(np.mean(scores))
median_score = float(np.median(scores))
sd_score = float(np.std(scores, ddof=1))

print("Independent test investigations:", len(scores))
print(f"Mean cosine similarity: {mean_score:.9f}")
print(f"Median cosine similarity: {median_score:.9f}")
print(f"Standard deviation: {sd_score:.9f}")
print(f"Minimum: {scores.min():.6f}")
print(f"Maximum: {scores.max():.6f}")

# Confirm agreement with reproduced result
assert np.isclose(
    mean_score, 0.29958532296482326, atol=1e-5
), "Mean differs from the verified reproduced result."

# Publication styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(7.2, 4.6))

ax.hist(
    scores,
    bins=35,
    color="#2878B5",  # Blue histogram bars,
    edgecolor="white",
    linewidth=0.5,
    alpha=0.9
)

ax.axvline(
    mean_score,
    color="#C28B2C",
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_score:.4f}"
)

ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Number of investigations")
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

# Save publication versions
pdf_path = output_dir / (
    "Figure4_Reproduced_VAE_Test_Distribution.pdf"
)
png_path = output_dir / (
    "Figure4_Reproduced_VAE_Test_Distribution.png"
)

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")

plt.show()
plt.close(fig)

print("\nSaved:")
print(pdf_path)
print(png_path)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/NTSB_VAE_Reproducibility"
)

output_dir = root / "Reviewer2_Publication_Figures"
output_dir.mkdir(parents=True, exist_ok=True)

with np.load(
    root / "vae_independent_test_results.npz",
    allow_pickle=False
) as data:
    scores = data["cosine_scores"].astype(np.float64)

assert scores.shape == (3138,)
assert np.all(np.isfinite(scores))
assert np.isclose(scores.mean(), 0.299585323, atol=1e-5)

indices = np.arange(1, len(scores) + 1)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, ax = plt.subplots(figsize=(8.2, 4.6))


ax.scatter(
    indices,
    scores,
    s=20,               # Larger dots (previously 10)
    color="#2878B5",    # Publication blue
    alpha=0.75,         # Darker and more visible
    edgecolors="none"
)


# ax.scatter(
#     indices,
#     scores,
#     s=10,
#     color="#2878B5",
#     alpha=0.55,
#     edgecolors="none"
# )

ax.axhline(
    scores.mean(),
    color="#C28B2C",
    linestyle="--",
    linewidth=1.8,
    label=f"Mean = {scores.mean():.4f}"
)

ax.set_xlabel("Test investigation index")
ax.set_ylabel("Cosine similarity")
ax.set_xlim(0, len(scores) + 1)
ax.set_ylim(-1, 1)
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

pdf_path = (
    output_dir /
    "Figure5_Reproduced_Individual_Test_Scores.pdf"
)
png_path = (
    output_dir /
    "Figure5_Reproduced_Individual_Test_Scores.png"
)

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")

plt.show()
plt.close(fig)

print("Saved:", pdf_path)
print("Saved:", png_path)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import json
import re
from pathlib import Path

# Search the code cells in the current Colab notebook.
# Save the notebook as .ipynb first, then upload it
# to the Colab session if necessary.

keywords = [
    "train_test_split",
    "train_idx",
    "val_idx",
    "test_idx",
    "random_state",
    "shuffle",
    "16002",
    "1779",
    "3138",
    "18450",
    "vocab",
    "tokenizer",
    "fit_on_texts",
    "model.fit",
    "validation_data",
    "EarlyStopping",
]

notebooks = list(Path("/content").glob("*.ipynb"))

if not notebooks:
    print("No .ipynb notebook found in /content.")
    print("Upload the reconstructed experiment notebook first.")
else:
    for notebook_path in notebooks:
        print(f"\nNOTEBOOK: {notebook_path.name}")

        with open(notebook_path, encoding="utf-8") as f:
            notebook = json.load(f)

        for i, cell in enumerate(notebook.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            matches = [
                k for k in keywords
                if re.search(re.escape(k), source, re.I)
            ]

            if matches:
                print(f"\n--- Cell {i} ---")
                print("Matched:", ", ".join(matches))
                print(source[:3500])


In [ ]:

# Inspect variables that may contain data partitions
# or their original investigation identifiers.

patterns = [
    "train", "val", "test", "split",
    "idx", "indices", "vocab", "token"
]

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(p in name.lower() for p in patterns):
        try:
            print(
                f"{name:35s}",
                f"type={type(value).__name__:20s}",
                f"length={len(value)}"
            )
        except (TypeError, ValueError):
            print(
                f"{name:35s}",
                f"type={type(value).__name__}"
            )


In [ ]:

# Replace these names with the actual partition variables.

# Example:
# print("Training:", len(X_train))
# print("Validation:", len(X_val))
# print("Test:", len(X_test))

expected = {
    "Training": 16002,
    "Validation": 1779,
    "Test": 3138
}

print("Expected partition sizes:")
for name, count in expected.items():
    print(f"{name}: {count:,}")

print("Total:", sum(expected.values()))


In [ ]:

# Find actual training, validation, and test variables
# currently available in the notebook.

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    if any(word in name.lower() for word in
           ["train", "val", "test", "split", "idx"]):
        try:
            print(f"{name}: {len(value)} records")
        except (TypeError, ValueError):
            pass


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

folder = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

files = {
    "Train": "train_investigation_ids.csv",
    "Validation": "validation_investigation_ids.csv",
    "Test": "test_investigation_ids.csv"
}

expected = {
    "Train": 16002,
    "Validation": 1779,
    "Test": 3138
}

ids = {}

print("=== PARTITION VERIFICATION ===")

for name, filename in files.items():
    df = pd.read_csv(folder / filename, dtype=str)

    print(f"\n{name}")
    print("Columns:", list(df.columns))
    print("Rows:", len(df))
    print("Expected:", expected[name])
    print("Count matches:", len(df) == expected[name])

    if df.shape[1] != 1:
        print("CHECK REQUIRED: More than one column.")
        print("Select the actual investigation-ID column.")
        continue

    series = df.iloc[:, 0].str.strip()
    ids[name] = set(series.dropna())

    print("Unique IDs:", len(ids[name]))
    print("Missing IDs:", series.isna().sum())
    print("Duplicate IDs:", series.duplicated().sum())

if len(ids) == 3:
    print("\n=== OVERLAP CHECK ===")

    pairs = [
        ("Train", "Validation"),
        ("Train", "Test"),
        ("Validation", "Test")
    ]

    for a, b in pairs:
        overlap = ids[a] & ids[b]
        print(f"{a} vs {b}: {len(overlap)} overlapping IDs")

    print("\nTotal unique investigation IDs:",
          len(ids["Train"] | ids["Validation"] | ids["Test"]))

print("\n=== SAVED NUMERICAL ARRAYS ===")

for filename in [
    "train_validation_signed_glove.npz",
    "independent_test_signed_glove.npz"
]:
    path = folder / filename

    with np.load(path, allow_pickle=False) as data:
        print(f"\n{filename}")
        for key in data.files:
            array = data[key]
            print(f"{key}: shape={array.shape}, dtype={array.dtype}")


In [ ]:

import numpy as np
from pathlib import Path

folder = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

for filename in [
    "vae_independent_test_results.npz",
    "deterministic_independent_test_results.npz",
    "validation_figure_data.npz"
]:
    print(f"\n=== {filename} ===")

    with np.load(folder / filename, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")


In [ ]:

import numpy as np
from pathlib import Path

folder = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

with np.load(folder / "independent_test_signed_glove.npz") as f:
    Y_true = f["Y_test"]

with np.load(folder / "vae_independent_test_results.npz") as f:
    vae_pred = f["Y_pred"]
    vae_saved = f["cosine_scores"]

with np.load(folder / "deterministic_independent_test_results.npz") as f:
    det_pred = f["Y_pred"]
    det_saved = f["cosine_scores"]

def calculate_cosine(y_true, y_pred):
    a = y_true.reshape(len(y_true), -1).astype(np.float64)
    b = y_pred.reshape(len(y_pred), -1).astype(np.float64)

    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1) *
        np.linalg.norm(b, axis=1)
    )

    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0
    )

vae_recomputed = calculate_cosine(Y_true, vae_pred)
det_recomputed = calculate_cosine(Y_true, det_pred)

print("=== INDEPENDENT TEST REPRODUCTION ===")

for name, recomputed, saved in [
    ("VAE", vae_recomputed, vae_saved),
    ("Deterministic", det_recomputed, det_saved)
]:
    print(f"\n{name}")
    print(f"Recomputed mean: {recomputed.mean():.6f}")
    print(f"Saved mean:      {saved.mean():.6f}")
    print(f"Recomputed SD:   {recomputed.std():.6f}")
    print(f"Maximum score discrepancy: "
          f"{np.max(np.abs(recomputed - saved)):.8f}")

difference = vae_recomputed - det_recomputed

print("\n=== PAIRED COMPARISON ===")
print(f"VAE mean:           {vae_recomputed.mean():.6f}")
print(f"Deterministic mean: {det_recomputed.mean():.6f}")
print(f"Mean difference:    {difference.mean():.6f}")
print(f"Number of pairs:    {len(difference)}")
print(f"VAE higher:         {(difference > 0).sum()}")
print(f"Deterministic higher: {(difference < 0).sum()}")


In [ ]:

import numpy as np

# Uses 'difference' from the previous verification cell.

rng = np.random.default_rng(42)
n = len(difference)
B = 10000

bootstrap_means = np.empty(B)

for i in range(B):
    sample = rng.choice(difference, size=n, replace=True)
    bootstrap_means[i] = sample.mean()

ci = np.percentile(bootstrap_means, [2.5, 97.5])

print("=== PAIRED BOOTSTRAP ===")
print(f"Mean difference: {difference.mean():.6f}")
print(f"95% CI: [{ci[0]:.6f}, {ci[1]:.6f}]")
print(f"Sample SD of VAE scores: "
      f"{vae_recomputed.std(ddof=1):.6f}")


In [ ]:

import json
from pathlib import Path

keywords = [
    "vocab", "word_to", "word2idx", "Counter",
    "train_test_split", "random_state",
    "X_test", "Y_test", "test_ds",
    "EarlyStopping", "ModelCheckpoint",
    "validation_data", "model.fit"
]

notebooks = list(Path("/content").rglob("*.ipynb"))

print(f"Found {len(notebooks)} notebooks")

for path in notebooks:
    try:
        with open(path, encoding="utf-8") as f:
            nb = json.load(f)
    except Exception:
        continue

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matches = [
            word for word in keywords
            if word.lower() in source.lower()
        ]

        if matches:
            print(f"\nNotebook: {path.name}")
            print(f"Cell: {i}")
            print("Matches:", ", ".join(matches))
            print(source[:1500])


In [ ]:

import json
from pathlib import Path

matches = list(Path("/content").rglob("AI_PAPER.ipynb"))

if not matches:
    print("AI_PAPER.ipynb not found.")
else:
    path = matches[0]
    print("Notebook:", path)

    with open(path, encoding="utf-8") as f:
        notebook = json.load(f)

    for i in [379, 380]:
        print(f"\n{'=' * 70}")
        print(f"CELL {i}")
        print("=" * 70)
        print("".join(notebook["cells"][i].get("source", [])))


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

print("Project folder exists:", root.exists())
print("Training IDs exist:",
      (root / "train_investigation_ids.csv").exists())

json_dir = root / "CAROL_Original_JSON"
json_files = sorted(json_dir.glob("carol_investigations_data*.json"))

print("CAROL JSON files found:", len(json_files))
print("JSON folder exists:", json_dir.exists())


In [ ]:

import re
import json
import pandas as pd
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

# Load training investigation IDs only
train_ids = set(
    pd.read_csv(
        root / "train_investigation_ids.csv",
        dtype=str
    )["investigation_id"].str.strip()
)

def tokenize(text):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        str(text).lower()
    )

training_vocabulary = set()
matched_ids = set()

json_files = sorted(
    (root / "CAROL_Original_JSON").glob(
        "carol_investigations_data*.json"
    )
)

for path in json_files:
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        investigation_id = str(
            record.get("NTSBNumber", "")
        ).strip()

        if investigation_id not in train_ids:
            continue

        matched_ids.add(investigation_id)

        for field in ["AnalysisNarrative", "ProbableCause"]:
            training_vocabulary.update(
                tokenize(record.get(field, ""))
            )

    del records

print("=== TRAINING VOCABULARY VERIFICATION ===")
print("Training IDs:", len(train_ids))
print("Matched training IDs:", len(matched_ids))
print("Original JSON files:", len(json_files))
print("Training vocabulary size:", len(training_vocabulary))

print("\n=== CHECKS ===")
print("All training records located:",
      matched_ids == train_ids)

print("Vocabulary matches reported 18,450:",
      len(training_vocabulary) == 18450)

print("\nNo models trained. No files modified.")


In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI_PAPER.ipynb"
)

with open(path, encoding="utf-8") as f:
    notebook = json.load(f)

terms = [
    "vae.fit(",
    "deterministic.fit(",
    "model.fit(",
    "validation_data=",
    "EarlyStopping(",
    "ModelCheckpoint(",
    "vae.predict(",
    "deterministic.predict("
]

print("=== TRAINING AND MODEL-SELECTION AUDIT ===")

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matched = [
        term for term in terms
        if term.lower() in source.lower()
    ]

    if matched:
        print(f"\nCELL {i}")
        print("Matches:", ", ".join(matched))
        print(source[:1000])


In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI_PAPER.ipynb"
)

with open(path, encoding="utf-8") as f:
    notebook = json.load(f)

for i in [328, 334, 336]:
    print("\n" + "=" * 70)
    print(f"CELL {i}")
    print("=" * 70)
    print("".join(notebook["cells"][i].get("source", [])))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

with np.load(root / "vae_independent_test_results.npz") as d:
    vae = d["cosine_scores"].astype(np.float64)

with np.load(root / "deterministic_independent_test_results.npz") as d:
    deterministic = d["cosine_scores"].astype(np.float64)

assert len(vae) == len(deterministic) == 3138

difference = vae - deterministic
mean_diff = difference.mean()

rng = np.random.default_rng(42)
boot = np.empty(10000)

for i in range(len(boot)):
    indices = rng.integers(0, len(difference), len(difference))
    boot[i] = difference[indices].mean()

ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4.2),
    gridspec_kw={"width_ratios": [1.1, 1]}
)

# Panel A: Mean independent-test performance
means = [deterministic.mean(), vae.mean()]
colors = ["#66788A", "#174A72"]

axes[0].bar(
    ["Deterministic", "VAE"],
    means,
    color=colors,
    width=0.55
)

for i, value in enumerate(means):
    axes[0].text(
        i, value + 0.006, f"{value:.6f}",
        ha="center", fontsize=10
    )

axes[0].set_ylim(0, 0.36)
axes[0].set_ylabel("Mean cosine similarity")
axes[0].set_title("(a) Independent-test performance")
axes[0].spines[["top", "right"]].set_visible(False)

# Panel B: Paired mean difference and bootstrap CI
axes[1].errorbar(
    mean_diff, 0,
    xerr=[[mean_diff - ci_low], [ci_high - mean_diff]],
    fmt="o", markersize=8,
    color="#174A72", capsize=6, linewidth=2
)

axes[1].axvline(
    0, color="#888888", linestyle="--", linewidth=1
)

axes[1].set_yticks([])
axes[1].set_xlabel("VAE − deterministic cosine similarity")
axes[1].set_title("(b) Paired improvement (95% CI)")
axes[1].set_xlim(-0.001, 0.006)
axes[1].spines[["top", "right", "left"]].set_visible(False)

axes[1].text(
    mean_diff, 0.14,
    f"{mean_diff:.6f}\n[{ci_low:.6f}, {ci_high:.6f}]",
    ha="center", fontsize=9
)

fig.suptitle(
    "Independent-test comparison (n = 3,138)",
    fontsize=12, fontweight="bold"
)

plt.tight_layout()

pdf_path = root / "independent_test_comparison_verified.pdf"
png_path = root / "independent_test_comparison_verified.png"

fig.savefig(pdf_path, bbox_inches="tight")
fig.savefig(png_path, dpi=300, bbox_inches="tight")

plt.show()

print("VAE mean:", round(vae.mean(), 6))
print("Deterministic mean:", round(deterministic.mean(), 6))
print("Paired difference:", round(mean_diff, 6))
print("95% CI:", np.round([ci_low, ci_high], 6))
print("PDF:", pdf_path)
print("PNG:", png_path)


In [ ]:
import json
import os
import re

notebook_path = "/content/drive/MyDrive/Colab Notebooks/AI_PAPER.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "index_word",
    "word_index",
    "sequences_to_texts",
    "decode",
    "denormaliz",
    "inverse_transform",
    "wordcloud",
    "word_cloud",
    "predicted_words",
    "predicted_text",
    "tokenizer",
    "argmax",
    "vocab",
]

matches = []

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    found = [k for k in keywords if k.lower() in source.lower()]

    if found:
        matches.append((i, found, source))

print(f"Found {len(matches)} potentially relevant cells.\n")

for i, found, source in matches:
    print("=" * 75)
    print(f"CELL {i} | Keywords: {', '.join(found)}")
    print(source[:1800])
    print()

In [ ]:

from pathlib import Path
import numpy as np

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

print("REQUIRED FILES")
print("=" * 65)

for filename in [
    "glove.6B.100d.txt",
    "train_investigation_ids.csv",
    "independent_test_signed_glove.npz",
    "vae_independent_test_results.npz",
]:
    matches = list(root.rglob(filename))
    print(f"{filename}:")
    for path in matches[:3]:
        print("  FOUND:", path)
    if not matches:
        print("  NOT FOUND")

print("\nPREDICTION ARRAYS")
print("=" * 65)

matches = list(root.rglob("vae_independent_test_results.npz"))

if matches:
    with np.load(matches[0], allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(key, arr.shape, arr.dtype)
else:
    print("Prediction file not located.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

# 1. Inspect the independent-test representation file
path = root / "independent_test_signed_glove.npz"

print("INDEPENDENT TEST ARRAYS")
print("=" * 65)

with np.load(path, allow_pickle=False) as data:
    for key in data.files:
        arr = data[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")
        if np.issubdtype(arr.dtype, np.number):
            print(
                "  range:",
                float(np.nanmin(arr)),
                "to",
                float(np.nanmax(arr))
            )

# 2. Inspect the training investigation IDs
print("\nTRAINING IDENTIFIERS")
print("=" * 65)

ids = pd.read_csv(root / "train_investigation_ids.csv")
print("Shape:", ids.shape)
print("Columns:", ids.columns.tolist())
print(ids.head(3).to_string(index=False))

# 3. Inspect saved predictions
print("\nVAE PREDICTIONS")
print("=" * 65)

with np.load(root / "vae_independent_test_results.npz",
             allow_pickle=False) as data:
    pred = data["Y_pred"]

print("Shape:", pred.shape)
print("Minimum:", float(pred.min()))
print("Maximum:", float(pred.max()))
print("Mean:", float(pred.mean()))
print("Standard deviation:", float(pred.std()))

print("\nFirst 15 predicted values:")
print(np.round(pred[0, :15, 0], 6))


In [ ]:

import numpy as np
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

with np.load(root / "vae_independent_test_results.npz") as f:
    pred = f["Y_pred"].reshape(3138, 200)

with np.load(root / "independent_test_signed_glove.npz") as f:
    true = f["Y_test"].reshape(3138, 200)

pred_norm = np.linalg.norm(pred, axis=1)
true_norm = np.linalg.norm(true, axis=1)

print("NORM COMPARISON")
print("=" * 60)

for name, arr in [
    ("Predicted norms", pred_norm),
    ("Reference norms", true_norm),
]:
    print(name)
    print("  Median:", np.median(arr))
    print("  Mean:", np.mean(arr))
    print("  Minimum:", np.min(arr))
    print("  Maximum:", np.max(arr))

# Diagnostic rescaling using reference norms.
# This is NOT a deployable inference method because
# it uses the expert reference target.
scale = true_norm / np.maximum(pred_norm, 1e-12)
pred_rescaled = pred * scale[:, None]

def row_cosine(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = (
        np.linalg.norm(a, axis=1)
        * np.linalg.norm(b, axis=1)
    )
    return numerator / np.maximum(denominator, 1e-12)

raw_cosine = row_cosine(pred, true)
rescaled_cosine = row_cosine(pred_rescaled, true)

print("\nCOSINE SIMILARITY")
print("=" * 60)
print("Original mean:", raw_cosine.mean())
print("Rescaled mean:", rescaled_cosine.mean())
print("Maximum absolute difference:",
      np.max(np.abs(raw_cosine - rescaled_cosine)))

print("\nRESCALED VALUE RANGE")
print("=" * 60)
print("Minimum:", pred_rescaled.min())
print("Maximum:", pred_rescaled.max())

print("\nFIRST RECORD")
print("Original first 10:",
      np.round(pred[0, :10], 5))
print("Rescaled first 10:",
      np.round(pred_rescaled[0, :10], 5))
print("Reference first 10:",
      np.round(true[0, :10], 5))


In [ ]:

from pathlib import Path
import numpy as np

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

print("SAVED NUMERICAL DATASETS")
print("=" * 70)

for path in sorted(root.rglob("*.npz")):
    print("\nFILE:", path.relative_to(root))
    try:
        with np.load(path, allow_pickle=False) as data:
            for key in data.files:
                arr = data[key]
                print(" ", key, arr.shape, arr.dtype)
    except Exception as e:
        print(" Could not inspect:", type(e).__name__)

print("\nOTHER POSSIBLE PREPROCESSING FILES")
print("=" * 70)

for pattern in ["*.pkl", "*.joblib", "*vocab*.json",
                "*token*.json", "*train*.npy"]:
    for path in sorted(root.rglob(pattern))[:15]:
        print(path.relative_to(root))


In [ ]:

import numpy as np
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")

with np.load(root / "train_validation_signed_glove.npz") as data:
    Y_train = data["Y_train"].reshape(-1, 200)

with np.load(root / "independent_test_signed_glove.npz") as data:
    Y_test = data["Y_test"].reshape(-1, 200)

with np.load(root / "vae_independent_test_results.npz") as data:
    Y_vae = data["Y_pred"].reshape(-1, 200)

with np.load(root / "deterministic_independent_test_results.npz") as data:
    Y_det = data["Y_pred"].reshape(-1, 200)

def cosine_per_record(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return numerator / np.maximum(denominator, 1e-12)

# Training-only mean target
mean_target = Y_train.mean(axis=0)

# Identical prediction for every test investigation
Y_mean = np.broadcast_to(mean_target, Y_test.shape)

scores = {
    "VAE": cosine_per_record(Y_vae, Y_test),
    "Deterministic": cosine_per_record(Y_det, Y_test),
    "Mean-target baseline": cosine_per_record(Y_mean, Y_test)
}

print("INDEPENDENT TEST PERFORMANCE")
print("=" * 70)

for name, values in scores.items():
    print(
        f"{name:22s} "
        f"Mean={values.mean():.6f} "
        f"SD={values.std(ddof=1):.6f} "
        f"Median={np.median(values):.6f}"
    )

difference = scores["VAE"] - scores["Mean-target baseline"]

print("\nVAE VERSUS MEAN-TARGET BASELINE")
print("=" * 70)
print("Mean paired difference:", round(difference.mean(), 6))
print("VAE wins:", int(np.sum(difference > 0)))
print("Baseline wins:", int(np.sum(difference < 0)))
print("Ties:", int(np.sum(difference == 0)))
print("VAE win percentage:", round(100 * np.mean(difference > 0), 2))

print("\nPREDICTION VARIABILITY")
print("=" * 70)
print("Mean across-position VAE prediction SD:",
      np.std(Y_vae, axis=0).mean())
print("Mean across-position deterministic prediction SD:",
      np.std(Y_det, axis=0).mean())
print("Mean across-position reference target SD:",
      np.std(Y_test, axis=0).mean())


In [ ]:

import numpy as np
from pathlib import Path

root = Path("/content/drive/MyDrive/NTSB_VAE_Reproducibility")
glove_path = root / "glove.6B.100d.txt"

# Read the original 100-dimensional GloVe embeddings.
# Keep only alphabetic vocabulary for this diagnostic.
words = []
scalars = []

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")

        if len(parts) != 101:
            continue

        word = parts[0]

        if not word.isalpha():
            continue

        vector = np.asarray(parts[1:], dtype=np.float32)
        words.append(word)
        scalars.append(float(vector.mean()))

scalars = np.asarray(scalars, dtype=np.float32)

print("GLOVE SCALAR REPRESENTATION AUDIT")
print("=" * 65)
print("Vocabulary examined:", len(words))
print("Scalar minimum:", scalars.min())
print("Scalar maximum:", scalars.max())
print("Scalar SD:", scalars.std())

order = np.argsort(scalars)
sorted_values = scalars[order]
sorted_words = np.asarray(words)[order]

differences = np.diff(sorted_values)

print("\nADJACENT SCALAR DISTANCES")
print("=" * 65)
print("Median:", np.median(differences))
print("5th percentile:", np.percentile(differences, 5))
print("95th percentile:", np.percentile(differences, 95))

print("\nEXAMPLE: WORDS CLOSEST TO 'aircraft'")
print("=" * 65)

if "aircraft" in words:
    target = scalars[words.index("aircraft")]
    nearest = np.argsort(np.abs(scalars - target))[:15]

    for idx in nearest:
        print(
            f"{words[idx]:20s} "
            f"scalar={scalars[idx]: .8f} "
            f"distance={abs(scalars[idx] - target):.8f}"
        )


In [ ]:

import json
from pathlib import Path

notebook_path = Path("/content/drive/MyDrive/AI_PAPER.ipynb")

if not notebook_path.exists():
    print("Notebook not found at:", notebook_path)
    print("Please tell me its actual location in Drive.")
else:
    with open(notebook_path, encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))
        if any(term in source for term in [
            "vocab =",
            "vocabulary =",
            "train_vocab",
            "word_to_scalar",
            "def tokenize",
            "def text_to",
        ]):
            print(f"\n{'='*65}\nCELL {i}\n{'='*65}")
            print(source[:2500])
